# Quant Terminal v25
### Self-Learning Ã‚Â· Options IV Earnings Flag Ã‚Â· Autonomous Ã‚Â· Continuous Feedback Loop
> Predicts. Tracks outcomes. Diagnoses failures. Rewrites its own rules. Flags earnings uncertainty. Reports every morning.

In [ ]:
# ============================================================
# CELL 1 Ã¢â‚¬â€ INSTALL  (run alone first Ã¢â€ â€™ Restart Ã¢â€ â€™ Run all)
# ============================================================
import sys, subprocess

def pip(pkg):
    return subprocess.run([sys.executable,"-m","pip","install","-q",pkg],
                          capture_output=True,text=True).returncode

pkgs = [
    "pandas>=2.2","scikit-learn>=1.5","xgboost>=2.1","lightgbm>=4.4",
    "catboost>=1.2.5","optuna>=3.6","yfinance>=0.2.40","arch>=6.3",
    "hmmlearn>=0.3.2","cvxpy>=1.5","ta>=0.11","pandas_ta>=0.3.14b0",
    "requests>=2.31","ipywidgets>=8.1","mapie==0.8.6",
    "imbalanced-learn>=0.12","mlflow>=2.13","matplotlib>=3.9",
    "seaborn>=0.13","threadpoolctl==3.1.0","numba>=0.61",
    "river>=0.21",
    "alpaca-py>=0.12",
]
print("Installing packages...")
for pkg in pkgs:
    r = pip(pkg)
    print(f"  {'OK' if r==0 else 'WARN'} {pkg.split('>=')[0].split('==')[0]}")

pip("torch --index-url https://download.pytorch.org/whl/cpu")
pip("transformers>=4.41")

import numpy as np
print(f"\nNumPy {np.__version__}")
print("\nInstall complete. >>> Runtime -> Restart -> Run all <<<")


Installing packages...
  OK pandas
  OK scikit-learn
  OK xgboost
  OK lightgbm
  OK catboost
  OK optuna
  OK yfinance
  OK arch
  OK hmmlearn
  OK cvxpy
  OK ta
  OK pandas_ta
  OK requests
  OK ipywidgets
  OK mapie
  OK imbalanced-learn
  OK mlflow
  OK matplotlib
  OK seaborn
  OK threadpoolctl
  OK numba
  OK river

NumPy 2.4.4

Install complete. >>> Runtime -> Restart -> Run all <<<


In [ ]:
# ============================================================
# CELL 2 â€” IMPORTS
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import os, json, datetime, time, traceback, threading, hashlib, pickle
import time as _time_module
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FuturesTimeout

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests

import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
import cvxpy as cp
from IPython.display import display, HTML, clear_output
import ta

# River online learning
from scipy import stats as scipy_stats
from river import linear_model, preprocessing, metrics, drift

try:
    from mapie.classification import MapieClassifier
    _MAPIE_OK = True
except ImportError:
    class MapieClassifier:
        def __init__(self,estimator=None,**kw): self.estimator=estimator
        def fit(self,X,y): self.estimator.fit(X,y); return self
        def predict(self,X,**kw):
            p=self.estimator.predict_proba(X)[:,1]
            return self.estimator.predict(X),np.stack([1-p,p],axis=1).reshape(len(X),1,2)
    _MAPIE_OK = False

try:
    import torch; torch.manual_seed(42)
except Exception:
    pass

np.random.seed(42)
print(f"Imports OK â€” pandas {pd.__version__} | numpy {np.__version__} | xgb {xgb.__version__}")
print("River online learning imported OK")


# â”€â”€ Historical macro cache (fetched once per session) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_MACRO_HIST_CACHE: dict = {}

def _get_macro_history(start: str, end: str) -> "pd.DataFrame":
    key = f"{start}_{end}"
    if key in _MACRO_HIST_CACHE:
        return _MACRO_HIST_CACHE[key]
    _syms = {"m_vix":"^VIX","m_tnx":"^TNX","m_irx":"^IRX",
              "m_dxy":"DX-Y.NYB","m_wti":"CL=F","m_gold":"GC=F",
              "m_hyg":"HYG","m_lqd":"LQD"}
    frames = {}
    for col, sym in _syms.items():
        try:
            raw = yf.download(sym, start=start, end=end,
                              auto_adjust=True, progress=False)
            if not raw.empty:
                frames[col] = raw["Close"].rename(col)
        except Exception:
            pass
    if not frames:
        _MACRO_HIST_CACHE[key] = pd.DataFrame()
        return pd.DataFrame()
    mdf = pd.concat(frames.values(), axis=1).sort_index()
    mdf.index = pd.to_datetime(mdf.index).tz_localize(None)
    if "m_hyg" in mdf and "m_lqd" in mdf:
        mdf["m_credit"] = mdf["m_hyg"] / mdf["m_lqd"]
    if "m_tnx" in mdf and "m_irx" in mdf:
        mdf["m_yc"] = mdf["m_tnx"] - mdf["m_irx"]
    mdf = mdf.ffill().shift(1)   # lag 1 day â€” no look-ahead
    _MACRO_HIST_CACHE[key] = mdf
    return mdf

Imports OK Ã¢â‚¬â€ pandas 2.3.3 | numpy 2.4.4 | xgb 3.2.0
River online learning imported OK


In [ ]:
# ============================================================
# CELL 3 â€” CONFIGURATION
# ============================================================
ALPACA_API_KEY    = ""
ALPACA_SECRET_KEY = ""
NEWS_API_KEY      = ""
FRED_API_KEY      = ""

# â”€â”€ UPGRADE: Tradier ($10/month) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
TRADIER_API_KEY = ""   # leave empty until upgrade

# â”€â”€ Expanded universe â€” liquid S&P 500 constituents + crypto â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# â”€â”€ Active watchlist for paper trading (50 tickers â€” expand after live validation) â”€â”€
WATCHLIST = [
    # â”€â”€ Mega-cap tech (10) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "AAPL","MSFT","NVDA","GOOGL","AMZN","META","TSLA","AVGO","ORCL","ADBE",
    # â”€â”€ Software & cloud (24) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "CRM","NOW","SNOW","PLTR","NET","DDOG","ZS","CRWD","PANW","SHOP","WDAY","TEAM",
    "INTU","CDNS","SNPS","ANSS","FTNT","ANET","ACN","IBM","CSCO","IT","TYL","ROP",
    # â”€â”€ Fintech & payments (5) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "SQ","PYPL","COIN","AFRM","HOOD",
    # â”€â”€ Financials (30) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "JPM","V","MA","BAC","GS","MS","BLK","AXP","WFC","C",
    "SCHW","PGR","CB","COF","USB","TFC","PNC","ICE",
    "CME","SPGI","MCO","AON","MMC","TRV","ALL","MET","PRU","AIG","HIG","AFL",
    # â”€â”€ Healthcare (37) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "UNH","LLY","JNJ","ABBV","MRK","TMO","ABT","DHR","PFE","AMGN",
    "CVS","CI","HUM","BSX","MDT","SYK","ISRG","VRTX","REGN","BMY","GILD",
    "ELV","MCK","COR","A","IQV","MTD","WAT","ZBH","RMD","DXCM","IDXX","EW","PODD","ALGN","HOLX","BAX",
    # â”€â”€ Consumer discretionary (28) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "HD","NKE","LOW","TJX","ROST","SBUX","CMG","MCD","COST","TGT",
    "GM","F","UBER","BKNG","ABNB","MAR","HLT","LVS","MGM","DG","DLTR","YUM","DPZ","APTV","CCL","RCL","EXPE","NCLH",
    # â”€â”€ Consumer staples (19) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "WMT","PG","KO","PEP","MDLZ","CL","MO","PM","EL","CHD","CLX","GIS","TSN","STZ","K","SJM","HRL","CAG","HSY",
    # â”€â”€ Semiconductors (21) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "AMD","INTC","QCOM","AMAT","MU","TXN","LRCX","KLAC","ADI","MRVL",
    "ASML","TSM","ON","MCHP","ENPH","FSLR","TER","SWKS","MPWR","NXPI","WOLF",
    # â”€â”€ Energy (17) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "XOM","CVX","COP","SLB","EOG","HAL","OXY","PSX","MPC","VLO","DVN","APA","KMI","WMB","BKR","LNG","FANG",
    # â”€â”€ Industrials & defense (30) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "BA","CAT","DE","HON","GE","RTX","LMT","NOC","UPS","FDX",
    "MMM","EMR","ETN","ITW","PH","CMI","PCAR","GWW","CTAS","EXPD","NSC","UNP","CSX","GD","TDG","HWM","FTV","XYL","FAST","CHRW",
    # â”€â”€ Materials (17) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "LIN","APD","SHW","PPG","NEM","FCX","NUE","ALB","LYB","DD","DOW","CF","MOS","ECL","IFF","CE","PKG",
    # â”€â”€ REITs (16) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "AMT","PLD","EQIX","CCI","WELL","SPG","O","DLR","PSA","EXR","VTR","VICI","WY","EQR","AVB","ESS",
    # â”€â”€ Utilities (18) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "NEE","DUK","SO","AEP","D","EXC","SRE","XEL","AWK","WEC","ED","PEG","ETR","FE","CNP","CMS","AES","ATO",
    # â”€â”€ Communication & media (14) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "NFLX","DIS","CMCSA","VZ","T","TMUS","CHTR","EA","TTWO","LYV","PARA","WBD","OMC","IPG",
    # â”€â”€ Sector ETFs (25) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "SPY","QQQ","IWM","DIA","XLF","XLK","XLE","XLV","XLI","XLP","XLY","XLU","XLB","XLRE","XLC",
    "GLD","SLV","TLT","HYG","LQD","VNQ","ARKK","SMH","SOXX","IBIT",
    # â”€â”€ Crypto (5) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    "BTC-USD","ETH-USD","SOL-USD","XRP-USD","DOGE-USD",
    # WYFI added 2026-08-11 (user request); not in SECTOR_MAP so the sector cap counts it as Other
    "WYFI",
]
WATCHLIST_FULL = WATCHLIST  # ~316-ticker S&P 500+ universe | v25

# â”€â”€ Colab speed: limit to 20 liquid tickers when not on GitHub Actions â”€â”€
_COLAB_CORE = ["AAPL","MSFT","NVDA","GOOGL","AMZN","META","TSLA","SPY","QQQ","AMD",
               "JPM","V","MA","INTC","AVGO","COIN","PLTR","GLD","TLT","IWM"]
try:
    import google.colab as _gc; _colab_env = True
except ImportError:
    _colab_env = False
if _colab_env and not os.environ.get("GH_ACTIONS"):
    _colab_limit = globals().get("COLAB_TICKER_LIMIT", 20)
    if isinstance(_colab_limit, int) and _colab_limit < len(WATCHLIST):
        WATCHLIST = _COLAB_CORE[:_colab_limit]
        DEFAULT_WATCHLIST = WATCHLIST
        WATCHLIST_FULL = WATCHLIST
        print(f"  Colab mode: watchlist limited to {len(WATCHLIST)} tickers for speed")

DEFAULT_WATCHLIST = WATCHLIST   # backward-compat alias
# â”€â”€ Batch processing to avoid API rate limits â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
WATCHLIST_BATCH_SIZE = 20

# â”€â”€ Tier 1+2 Political/Macro Upgrade Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
QUIVER_QUANT_KEY    = ""   # free tier: quiverquant.com/register
DISCORD_WEBHOOK_URL = ""   # optional: Discord webhook for trade alerts
STOP_LOSS_PCT       = 0.08 # 8% hard stop loss on open positions
TRAILING_STOP_PCT   = 0.07 # 7% from peak triggers trailing stop

SECTOR_ETF_MAP = {
    "tech":"XLK","financials":"XLF","healthcare":"XLV","energy":"XLE",
    "industrials":"XLI","materials":"XLB","consumer_disc":"XLY",
    "consumer_staples":"XLP","utilities":"XLU","real_estate":"XLRE",
    "comm_services":"XLC","defense":"XLI",
}
TICKER_SECTOR = {
    "AAPL":"tech","MSFT":"tech","NVDA":"tech","GOOGL":"tech","AMZN":"tech",
    "META":"tech","TSLA":"consumer_disc","AVGO":"tech","ORCL":"tech","ADBE":"tech",
    "CRM":"tech","NOW":"tech","SNOW":"tech","PLTR":"tech","NET":"tech",
    "DDOG":"tech","ZS":"tech","CRWD":"tech","PANW":"tech","SHOP":"tech",
    "WDAY":"tech","TEAM":"tech","AMD":"tech","INTC":"tech","QCOM":"tech",
    "AMAT":"tech","MU":"tech","TXN":"tech","LRCX":"tech","KLAC":"tech",
    "ADI":"tech","MRVL":"tech","ASML":"tech","TSM":"tech","ON":"tech",
    "MCHP":"tech","ENPH":"tech","FSLR":"tech",
    "JPM":"financials","V":"financials","MA":"financials","BAC":"financials",
    "GS":"financials","MS":"financials","BLK":"financials","AXP":"financials",
    "WFC":"financials","C":"financials","SCHW":"financials","PGR":"financials",
    "CB":"financials","COF":"financials","USB":"financials","TFC":"financials",
    "PNC":"financials","ICE":"financials","SQ":"financials","PYPL":"financials",
    "COIN":"financials","AFRM":"financials","HOOD":"financials",
    "UNH":"healthcare","LLY":"healthcare","JNJ":"healthcare","ABBV":"healthcare",
    "MRK":"healthcare","TMO":"healthcare","ABT":"healthcare","DHR":"healthcare",
    "PFE":"healthcare","AMGN":"healthcare","CVS":"healthcare","CI":"healthcare",
    "HUM":"healthcare","BSX":"healthcare","MDT":"healthcare","SYK":"healthcare",
    "ISRG":"healthcare","VRTX":"healthcare","REGN":"healthcare","BMY":"healthcare",
    "GILD":"healthcare",
    "XOM":"energy","CVX":"energy","COP":"energy","SLB":"energy","EOG":"energy",
    "HAL":"energy","OXY":"energy","PSX":"energy",
    "BA":"industrials","CAT":"industrials","DE":"industrials","HON":"industrials",
    "GE":"industrials","RTX":"defense","LMT":"defense","NOC":"defense",
    "UPS":"industrials","FDX":"industrials",
    "WMT":"consumer_staples","PG":"consumer_staples","KO":"consumer_staples",
    "PEP":"consumer_staples","MDLZ":"consumer_staples","CL":"consumer_staples",
    "MCD":"consumer_disc","HD":"consumer_disc","NKE":"consumer_disc",
    "COST":"consumer_disc","TGT":"consumer_disc","LOW":"consumer_disc",
    "SBUX":"consumer_disc","CMG":"consumer_disc","TJX":"consumer_disc",
    "ROST":"consumer_disc",
    "AMT":"real_estate","PLD":"real_estate","EQIX":"real_estate","CCI":"real_estate",
    "NEE":"utilities","DUK":"utilities","SO":"utilities","AEP":"utilities",
}
# FOMC meeting dates 2026 (pre-scheduled by Fed)
FOMC_DATES_2026 = [
    "2026-01-27","2026-01-28","2026-03-17","2026-03-18",
    "2026-04-28","2026-04-29","2026-06-16","2026-06-17",
    "2026-07-28","2026-07-29","2026-09-15","2026-09-16",
    "2026-10-27","2026-10-28","2026-12-15","2026-12-16",
]

# â”€â”€ Fast mode (for testing â€” disables slow components) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
FAST_MODE = False   # set True to skip FinBERT, GARCH MC, Optuna during testing

def maybe_skip(label: str, fast_default):
    """In fast mode, return the default value instead of computing."""
    if FAST_MODE:
        return fast_default, True   # (value, skipped=True)
    return None, False              # (None, skipped=False)


def process_tickers_in_batches(tickers, func, batch_size=None, delay=1.0):
    """Process tickers in batches to avoid rate limiting."""
    import time
    if batch_size is None:
        batch_size = WATCHLIST_BATCH_SIZE
    results = {}
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        for tk in batch:
            try:
                results[tk] = func(tk)
            except Exception as e:
                print(f"  {tk}: error â€” {e}")
        if i + batch_size < len(tickers):
            time.sleep(delay)
    return results

FORECAST_DAYS    = 5
TRAIN_START      = "2018-01-01"
TRAIN_END        = datetime.date.today().isoformat()
PORTFOLIO_CAPITAL= 10_000
MIN_CONFIDENCE   = 0.65
MAX_POSITION_PCT = 0.20
STOP_LOSS_PCT    = 0.05
MAX_DRAWDOWN_PCT = 0.15
HMM_STATES       = 3
GARCH_PATHS      = 500   # 500 paths sufficient for daily signal; cached per-day

# â”€â”€ Transaction cost model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
COMMISSION_PCT  = 0.001    # 0.10% round-trip (Alpaca default tier)
SLIPPAGE_PCT    = 0.0005   # 0.05% per side market-impact/half-spread
ROUND_TRIP_COST = COMMISSION_PCT + 2 * SLIPPAGE_PCT  # 0.20% total

def apply_transaction_costs(gross_return: float) -> float:
    """Subtract round-trip trading friction from a gross position return."""
    return gross_return - ROUND_TRIP_COST

MACRO            = {}  # populated by Cell 4 fetch_macro()

PT_LOG_COLS = ["ts","ticker","action","price","qty",
               "confidence","regime","order_id","status"]

# â”€â”€ Self-learning log columns â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
PRED_LOG_COLS = [
    "pred_ts","ticker","action","confidence","price_at_pred",
    "p_ensemble","p_up_garch","rsi","regime","vix","yield_curve",
    "ism_pmi","unemployment","sentiment","horizon_days",
    "outcome_ts","price_at_outcome","actual_return",
    "was_correct","magnitude_error","scored",
    "iv_flag","iv_scale","iv_note"
]

# â”€â”€ Adaptive weight state (River updates these live) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ADAPTIVE_WEIGHTS = {
    "w_ensemble":  0.55,
    "w_garch":     0.20,
    "w_sentiment": 0.10,
    "w_regime":    0.10,
    "w_yieldcurve":0.05,
}

# â”€â”€ Learned rule overrides (model writes these itself) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
LEARNED_RULES = {}  # e.g. {"high_rsi_bear": {"dampen": 0.15, "count": 7}}

# â”€â”€ Google Drive â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_drive_mounted = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _drive_dir  = Path("/content/drive/MyDrive/quant_terminal_v25")
    _drive_dir.mkdir(parents=True, exist_ok=True)
    _drive_mounted = True
    print(f"âœ… Drive mounted â†’ {_drive_dir}")
except Exception as _e:
    _drive_dir = Path(".")
    print(f"âš ï¸  Drive not available ({_e}) â€” session-only mode")

# Sub-directories for organised storage
(_drive_dir / "paper_trades").mkdir(exist_ok=True)
(_drive_dir / "predictions").mkdir(exist_ok=True)
(_drive_dir / "models").mkdir(exist_ok=True)
(_drive_dir / "weights").mkdir(exist_ok=True)

# â”€â”€ Log file paths â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
PT_LOG_FILE   = str(_drive_dir / "paper_trades" / "paper_trades.csv")
PRED_LOG_FILE       = str(_drive_dir / "predictions"  / "predictions.csv")
DAILY_PNL_LOG_FILE  = str(_drive_dir / "predictions"  / "daily_pnl_log.csv")
LOG_DIR       = str(_drive_dir)
RULES_FILE    = str(_drive_dir / "weights" / "learned_rules.json")
WEIGHTS_FILE        = str(_drive_dir / "weights" / "adaptive_weights.json")
RIVER_MODEL_FILE    = str(_drive_dir / "weights" / "river_model.pkl")
TICKER_CALIB_FILE   = str(_drive_dir / "weights" / "ticker_calibration.json")
FEATURE_IMP_FILE    = str(_drive_dir / "weights" / "feature_importance.json")
TICKER_ACC_FILE     = str(_drive_dir / "predictions" / "ticker_accuracy.json")

# â”€â”€ Self-learning hyperparameters â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
LEARNING_RATE_FAST       = 0.05   # weight step when drift detected
LEARNING_RATE_SLOW       = 0.01   # normal weight step
RULE_VALIDATION_MIN      = 10     # min predictions before validating a rule
TICKER_CALIBRATION_DECAY = 0.92   # multiplier applied on each wrong call
SECTOR_RULE_MIN_SAMPLES  = 5      # min ticker samples to trigger sector transfer

# â”€â”€ Load persisted self-learning state â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
try:
    TICKER_CALIB      = json.loads(Path(TICKER_CALIB_FILE).read_text()) if Path(TICKER_CALIB_FILE).exists() else {}
except Exception:
    TICKER_CALIB = {}
try:
    FEATURE_IMPORTANCE = json.loads(Path(FEATURE_IMP_FILE).read_text()) if Path(FEATURE_IMP_FILE).exists() else {}
except Exception:
    FEATURE_IMPORTANCE = {}
try:
    TICKER_ACCURACY   = json.loads(Path(TICKER_ACC_FILE).read_text()) if Path(TICKER_ACC_FILE).exists() else {}
except Exception:
    TICKER_ACCURACY = {}

# â”€â”€ VWAP execution benchmark log â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
VWAP_LOG_FILE = Path(LOG_DIR) / "vwap_benchmark.csv"
# â”€â”€ VWAP execution benchmark â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
EXEC_LOG_FILE = Path(LOG_DIR) / "execution_quality.csv"

def log_execution_quality(ticker: str, side: str, filled_price: float,
                           signal_price: float, vwap: float = None):
    """
    Records fill quality vs signal price and VWAP.
    Slippage = (filled - signal) / signal for buys, reversed for sells.
    """
    slippage = (filled_price - signal_price) / signal_price
    if side == "sell":
        slippage = -slippage
    vs_vwap = (filled_price - vwap) / vwap if vwap else None
    entry = pd.DataFrame([{
        "ts":           pd.Timestamp.utcnow().isoformat(),
        "ticker":       ticker,
        "side":         side,
        "signal_price": round(signal_price, 4),
        "filled_price": round(filled_price, 4),
        "slippage_pct": round(slippage * 100, 4),
        "vs_vwap_pct":  round(vs_vwap * 100, 4) if vs_vwap is not None else None,
    }])
    if EXEC_LOG_FILE.exists():
        entry.to_csv(EXEC_LOG_FILE, mode="a", header=False, index=False)
    else:
        entry.to_csv(EXEC_LOG_FILE, index=False)

def get_execution_quality_report() -> dict:
    """Summarizes average slippage and VWAP performance."""
    if not EXEC_LOG_FILE.exists():
        return {}
    try:
        df = pd.read_csv(EXEC_LOG_FILE)
        return {
            "avg_slippage_pct": round(df["slippage_pct"].mean(), 4),
            "avg_vs_vwap_pct":  round(df["vs_vwap_pct"].dropna().mean(), 4),
            "total_fills":      len(df),
            "worst_slippage":   round(df["slippage_pct"].max(), 4),
        }
    except Exception:
        return {}

# â”€â”€ Kill switch configuration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
KILL_DAILY_LOSS_PCT     = 0.03   # halt if account drops 3% in one day
KILL_CONSECUTIVE_LOSSES = 5      # halt if 5 losses in a row
KILL_VIX_THRESHOLD      = 40.0  # halt all new entries if VIX > 40
KILL_FLAG_FILE          = Path(LOG_DIR) / "KILL_SWITCH_ACTIVE.flag"

# â”€â”€ Minimum position size filter â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MIN_POSITION_DOLLARS = 200   # below this, costs eat the edge

# â”€â”€ PDT Rule Tracker â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
PDT_ACCOUNT_THRESHOLD = 25_000   # PDT applies below this equity
PDT_MAX_DAY_TRADES    = 3        # max round-trips in a rolling 5-day window
PDT_LOG_FILE          = Path(LOG_DIR) / "pdt_log.csv"

# â”€â”€ Model staleness detection â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
STALENESS_WINDOW      = 20    # evaluate over last N scored predictions
STALENESS_ACC_FLOOR   = 0.52  # retrain if rolling accuracy drops below this
STALENESS_CHECK_EVERY = 5     # check every N new scored outcomes
MODEL_RETRAIN_FLAG    = Path(LOG_DIR) / "RETRAIN_NEEDED.flag"

# â”€â”€ Sector exposure limits â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SECTOR_MAP = {
    "AAPL":"Tech","MSFT":"Tech","NVDA":"Tech","GOOGL":"Tech","META":"Tech",
    "AMZN":"Consumer","TSLA":"Consumer","JPM":"Finance","V":"Finance",
    "UNH":"Healthcare","SPY":"Broad","BTC-USD":"Crypto","ETH-USD":"Crypto",
    "SOL-USD":"Crypto","BNB-USD":"Crypto","XRP-USD":"Crypto",
}
MAX_SECTOR_PCT = 0.40   # max 40% of portfolio in any single sector
def get_sector_exposure(open_positions: dict, capital: float) -> dict:
    """Returns current sector allocations as fraction of capital."""
    exposure = {}
    for ticker, pos in open_positions.items():
        sector = SECTOR_MAP.get(ticker, "Other")
        exposure[sector] = exposure.get(sector, 0) + pos.get("dollars", 0)
    return {s: v / capital for s, v in exposure.items()}

def sector_allows_trade(ticker: str, dollars: float, open_positions: dict, capital: float) -> bool:
    """Returns False if adding this trade would breach sector limit."""
    sector = SECTOR_MAP.get(ticker, "Other")
    current = get_sector_exposure(open_positions, capital)
    projected = current.get(sector, 0) + dollars / capital
    if projected > MAX_SECTOR_PCT:
        print(f"    â­ {ticker}: sector {sector} would reach {projected:.0%} > {MAX_SECTOR_PCT:.0%} limit")
        return False
    return True

# â”€â”€ Correlation-aware position sizing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MAX_CORRELATION          = 0.75   # reduce size if new ticker correlates > 0.75 with held positions
CORRELATION_LOOKBACK     = 60     # days for correlation calculation
CORR_REDUCTION_THRESHOLD = 0.75   # reduce size if correlated position held
CORR_SIZE_REDUCTION      = 0.50   # reduce to 50% of normal size
_RETURNS_CACHE: dict     = {}     # legacy cache (kept for backward compat)

# Pre-built correlation matrix (populated once per day in autonomous cycle)
_CORR_MATRIX: "pd.DataFrame | None" = None
_CORR_DATE: "object" = None

def build_correlation_matrix(tickers: list, lookback: int = 60):
    """Build and cache the full correlation matrix once per day."""
    global _CORR_MATRIX, _CORR_DATE
    today = pd.Timestamp.today().date()
    if _CORR_DATE == today and _CORR_MATRIX is not None:
        return
    try:
        frames = {}
        for tk in tickers:
            df = _PRICE_CACHE.get(tk)
            if df is not None and not df.empty:
                frames[tk] = df["Close"].pct_change().dropna().tail(lookback)
        if frames:
            _CORR_MATRIX = pd.DataFrame(frames).corr()
            _CORR_DATE   = today
            print(f"  âœ“ Correlation matrix built ({len(frames)}Ã—{len(frames)})")
    except Exception as e:
        print(f"  Correlation matrix error: {e}")

def get_correlation_scalar(ticker: str, open_positions: dict, lookback: int = 20) -> float:
    """
    Returns a size scalar (0.3â€“1.0) based on rolling 20-day correlation
    with each open position. Graduated: high corr â†’ smaller size.
    Fast path: uses _CORR_MATRIX (built from full price cache).
    Slow path: live 20-day download for any missing tickers.
    """
    if not open_positions:
        return 1.0
    # Fast path: use pre-built correlation matrix (20-day returns)
    if _CORR_MATRIX is not None:
        try:
            if ticker not in _CORR_MATRIX.columns:
                return 1.0
            held = [t for t in open_positions if t in _CORR_MATRIX.columns]
            if not held:
                return 1.0
            max_corr = float(_CORR_MATRIX.loc[ticker, held].abs().max())
            # Graduated reduction instead of hard 0.7x cliff
            if   max_corr >= 0.85: return 0.30   # nearly identical â€” tiny size
            elif max_corr >= 0.70: return 0.50   # highly correlated â€” half size
            elif max_corr >= 0.55: return 0.75   # moderately correlated
            return 1.0
        except Exception:
            pass
        return 1.0
    # Slow path: live download (fallback before matrix is built)
    try:
        all_tickers = [ticker] + list(open_positions.keys())
        prices = {}
        for tk in all_tickers:
            if tk not in _RETURNS_CACHE:
                if len(_RETURNS_CACHE) >= 200:
                    _RETURNS_CACHE.pop(next(iter(_RETURNS_CACHE)))
                df = yf.download(tk, period=f"{lookback+5}d", auto_adjust=True, progress=False)
                if not df.empty:
                    _RETURNS_CACHE[tk] = df["Close"].pct_change().dropna().tail(lookback)
            if tk in _RETURNS_CACHE:
                prices[tk] = _RETURNS_CACHE[tk]
        if ticker not in prices or len(prices) < 2:
            return 1.0
        corr_series = pd.DataFrame(prices).corr()[ticker].drop(ticker)
        max_corr = corr_series.abs().max()
        if max_corr >= CORR_REDUCTION_THRESHOLD:
            return CORR_SIZE_REDUCTION
    except Exception:
        pass
    return 1.0

# â”€â”€ Dynamic leverage â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
LEVERAGE_BY_VIX = [
    (15,  1.00),   # VIX < 15  â†’ full size
    (20,  0.90),   # VIX 15-20 â†’ 90%
    (25,  0.75),   # VIX 20-25 â†’ 75%
    (30,  0.55),   # VIX 25-30 â†’ 55%
    (999, 0.30),   # VIX > 30  â†’ 30%
]

def get_leverage_scalar() -> float:
    """
    Scale total position sizing based on current VIX regime.
    Low VIX  (<15):  full size (1.0)
    Mid VIX  (15-25): normal (0.85)
    High VIX (25-35): reduced (0.65)
    Crisis   (>35):  minimal (0.40)
    """
    vix = MACRO.get("vix") or 20
    if vix < 15:   return 1.00
    elif vix < 25: return 0.85
    elif vix < 35: return 0.65
    else:          return 0.40

# â”€â”€ Drawdown recovery mode â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DRAWDOWN_RECOVERY_FLOOR  = 0.10   # enter recovery mode at 10% drawdown
DRAWDOWN_RECOVERY_SCALAR = 0.50   # trade at 50% normal size during recovery
DRAWDOWN_RECOVERY_EXIT   = 0.05   # exit recovery mode when drawdown < 5%

def get_drawdown_scalar(current_equity: float, peak_equity: float) -> float:
    """Returns position size scalar based on current drawdown."""
    if peak_equity <= 0:
        return 1.0
    dd = (current_equity - peak_equity) / peak_equity
    if dd <= -DRAWDOWN_RECOVERY_FLOOR:
        print(f"  âš  DRAWDOWN RECOVERY MODE: {dd:.1%} drawdown â†’ trading at {DRAWDOWN_RECOVERY_SCALAR:.0%} size")
        return DRAWDOWN_RECOVERY_SCALAR
    elif dd <= -DRAWDOWN_RECOVERY_EXIT:
        scalar = 1.0 - (abs(dd) / DRAWDOWN_RECOVERY_FLOOR) * (1.0 - DRAWDOWN_RECOVERY_SCALAR)
        return round(max(DRAWDOWN_RECOVERY_SCALAR, scalar), 2)
    return 1.0

# â”€â”€ Short selling configuration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ENABLE_SHORT_SELLING = True    # set False for cash accounts
SHORT_MARGIN_REQUIRED = 1.5    # 150% margin requirement for shorts
MIN_BORROW_RATE_THRESHOLD = 0.05  # skip if estimated borrow rate > 5%
KELLY_F = 0.25   # Kelly fraction for position sizing

def can_short(ticker: str, api=None) -> bool:
    """Check if ticker is shortable via Alpaca."""
    if not ENABLE_SHORT_SELLING:
        return False
    if ticker.endswith("-USD"):
        return False
    if api:
        try:
            asset = api.get_asset(ticker)
            return asset.shortable and asset.easy_to_borrow
        except Exception:
            pass
    return ticker in [t for t in WATCHLIST if not t.endswith("-USD") and not t.startswith("X")]

def get_short_position_size(confidence: float, price: float, capital: float) -> tuple:
    """
    Size a short position. More conservative than long â€” half-Kelly with
    margin requirement factored in.
    """
    edge    = max(0, 0.5 - confidence)
    frac    = min(edge * KELLY_F * 0.75, MAX_POSITION_PCT * 0.5)
    dollars = capital * frac / SHORT_MARGIN_REQUIRED
    qty     = max(0, int(dollars / price))
    return qty, qty * price

# â”€â”€ Regime-conditional position sizing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
REGIME_SIZE_SCALAR = {0: 0.65, 1: 1.10, 2: 0.85}  # bear=65%, bull=110%, neutral=85%
REGIME_SIZE_SCALARS = {
    0: 0.60,   # Bear: 60% of normal size
    1: 1.00,   # Bull: full size
    2: 0.80,   # Neutral: 80% of normal size
}

def get_regime_size_scalar(regime: int) -> float:
    """Returns position size scalar based on current market regime."""
    return REGIME_SIZE_SCALARS.get(regime, 0.80)

def get_combined_size_scalar(regime: int, current_drawdown: float = 0.0) -> float:
    """Combine leverage, regime, and recovery scalars. Floor 0.20 prevents zero sizing."""
    lev = get_leverage_scalar()
    reg = get_regime_size_scalar(regime)
    rec = get_recovery_size_scalar(current_drawdown)
    return max(lev * reg * rec, 0.20)

# â”€â”€ Stateful drawdown recovery mode (with hysteresis) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DRAWDOWN_RECOVERY_CEIL   = 0.05   # exit recovery mode when recovered to 5%
_IN_RECOVERY_MODE        = False

def get_recovery_size_scalar(current_drawdown: float) -> float:
    """
    In recovery mode, scale position sizes down gradually.
    Linear ramp: 10% DD -> 50% size, 15% DD -> 25% size.
    """
    global _IN_RECOVERY_MODE
    dd = abs(current_drawdown)
    if dd >= DRAWDOWN_RECOVERY_FLOOR:
        _IN_RECOVERY_MODE = True
    elif dd <= DRAWDOWN_RECOVERY_CEIL and _IN_RECOVERY_MODE:
        _IN_RECOVERY_MODE = False
        print("  Drawdown recovered â€” exiting recovery mode, restoring full size")
    if _IN_RECOVERY_MODE:
        scalar = max(0.25, 1.0 - (dd - DRAWDOWN_RECOVERY_FLOOR) * 10)
        print(f"  Recovery mode: DD={dd:.1%} -> sizing at {scalar:.0%}")
        return scalar
    return 1.0

# â”€â”€ Multi-timeframe alignment â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MTF_WEEKLY_WEIGHT  = 0.20   # weekly trend weight in composite
MTF_REQUIRED_ALIGN = False  # if True, block trades where weekly opposes daily

# â”€â”€ Model training schedule â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MODEL_RETRAIN_DAY    = 0          # 0=Monday: full Optuna retrain weekly
QUICK_TUNE_TRIALS    = 5          # trials for daily quick-tune (vs 25/20/20 full)
FULL_TUNE_TRIALS_XGB = 25
FULL_TUNE_TRIALS_LGB = 20
FULL_TUNE_TRIALS_CAT = 20
MODEL_CACHE_FILE     = Path(LOG_DIR) / "models_cache.pkl"

def is_retrain_day() -> bool:
    """Full retrain only on Mondays (or when staleness flag set)."""
    return pd.Timestamp.today().weekday() == MODEL_RETRAIN_DAY or MODEL_RETRAIN_FLAG.exists()

# â”€â”€ Model persistence â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def save_models(models_dict: dict):
    """Persist trained models to disk."""
    try:
        with open(MODEL_CACHE_FILE, "wb") as f:
            pickle.dump(models_dict, f, protocol=4)
        print(f"  âœ“ Models saved â†’ {MODEL_CACHE_FILE}")
    except Exception as e:
        print(f"  âœ— Model save failed: {e}")

def load_models() -> dict:
    """Load persisted models. Returns empty dict if not found."""
    if not MODEL_CACHE_FILE.exists():
        return {}
    try:
        with open(MODEL_CACHE_FILE, "rb") as f:
            m = pickle.load(f)
        print(f"  âœ“ Models loaded from cache ({len(m)} tickers)")
        return m
    except Exception as e:
        print(f"  âœ— Model load failed (will retrain): {e}")
        return {}

# â”€â”€ Batch price downloader â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_PRICE_CACHE: dict = {}   # {ticker: DataFrame} â€” refreshed each run

def batch_download_tickers(tickers: list, start: str, end: str,
                            batch_size: int = 50) -> dict:
    """Download OHLCV for multiple tickers in batched yfinance calls."""
    results = {}
    OHLCV = ["Open", "High", "Low", "Close", "Volume"]

    for i in range(0, len(tickers), batch_size):
        batch = [t for t in tickers[i:i+batch_size]]
        if not batch:
            continue
        try:
            raw = yf.download(
                batch, start=start, end=end,
                auto_adjust=True, progress=False,
                group_by="ticker", threads=True,
                timeout=30
            )
            if raw is None or raw.empty:
                continue

            for tk in batch:
                try:
                    if len(batch) == 1:
                        # Single ticker: flat columns
                        df = raw[OHLCV].copy() if all(c in raw.columns for c in OHLCV) else raw.copy()
                    else:
                        # Multi-ticker: MultiIndex (ticker, field)
                        if isinstance(raw.columns, pd.MultiIndex):
                            if tk in raw.columns.get_level_values(0):
                                df = raw[tk].copy()
                            else:
                                continue
                        else:
                            continue

                    # Normalize columns
                    df.columns = [str(c[0]) if isinstance(c, tuple) else str(c) for c in df.columns]
                    df.index   = pd.to_datetime(df.index).tz_localize(None)
                    df         = df[[c for c in OHLCV if c in df.columns]]
                    df         = df.dropna(subset=["Close"])

                    if len(df) >= 30:
                        results[tk]      = df
                        _PRICE_CACHE[tk] = df
                except Exception:
                    pass
        except Exception as e:
            print(f"  Batch {i//batch_size+1} error: {e}")
            # Fall back to single downloads for this batch
            for tk in batch:
                try:
                    df = yf.download(tk, start=start, end=end,
                                     auto_adjust=True, progress=False, timeout=15)
                    if df is not None and not df.empty:
                        df.columns = [str(c) for c in df.columns]
                        df.index   = pd.to_datetime(df.index).tz_localize(None)
                        df         = df.dropna(subset=["Close"])
                        results[tk]      = df
                        _PRICE_CACHE[tk] = df
                except Exception:
                    pass

    return results

from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

def download_with_timeout(ticker: str, start: str, end: str, timeout: int = 20):
    """yfinance download with hard timeout to prevent hangs."""
    if ticker in _PRICE_CACHE:
        return _PRICE_CACHE[ticker]
    with ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(yf.download, ticker, start=start, end=end,
                           auto_adjust=True, progress=False)
        try:
            df = future.result(timeout=timeout)
            if df is not None and not df.empty:
                df.columns = [str(c) for c in df.columns]
                df.index   = pd.to_datetime(df.index).tz_localize(None)
                _PRICE_CACHE[ticker] = df
                return df
        except FuturesTimeout:
            print(f"  â± {ticker}: download timeout ({timeout}s) â€” skipped")
        except Exception:
            pass
    return None
# â”€â”€ FinBERT â€” load once globally â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_FINBERT_PIPELINE = None

def get_finbert():
    """Load FinBERT once and cache globally."""
    global _FINBERT_PIPELINE
    if _FINBERT_PIPELINE is None:
        try:
            from transformers import pipeline as hf_pipeline
            _FINBERT_PIPELINE = hf_pipeline(
                "text-classification",
                model="ProsusAI/finbert",
                device=-1,          # CPU
                truncation=True,
                max_length=128,
                batch_size=16,      # process multiple headlines at once
            )
            print("  âœ“ FinBERT loaded")
        except Exception as e:
            print(f"  âœ— FinBERT load failed: {e}")
            _FINBERT_PIPELINE = None
    return _FINBERT_PIPELINE

_SENTIMENT_CACHE: dict = {}   # {ticker: (score, date)}

def get_sentiment_cached(ticker: str) -> float:
    """Return cached sentiment if fetched today."""
    today = pd.Timestamp.today().date()
    if ticker in _SENTIMENT_CACHE:
        score, cached_date = _SENTIMENT_CACHE[ticker]
        if cached_date == today:
            return score
    score = get_sentiment(ticker)
    _SENTIMENT_CACHE[ticker] = (score, today)
    return score

# â”€â”€ Weekly trend cache â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_WEEKLY_TREND_CACHE: dict = {}   # {ticker: (trend, date)}

Ã¢Å¡Â Ã¯Â¸Â  Drive not available (Error: credential propagation was unsuccessful) Ã¢â‚¬â€ session-only mode


In [ ]:
# ============================================================
# CELL 4 Ã¢â‚¬â€ MACRO DATA
# ============================================================
MACRO: dict = {}

def _yf_latest(ticker, period="5d"):
    try:
        df = yf.Ticker(ticker).history(period=period, auto_adjust=True)
        if df.empty: return None
        return float(df["Close"].dropna().iloc[-1])
    except Exception: return None


# â”€â”€ Alpaca market data (production price feed) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _alpaca_latest_price(symbol: str):
    """
    Fetch latest trade price from Alpaca Data API.
    Falls back to yfinance if API key not configured.
    """
    api_key    = ALPACA_API_KEY    if ALPACA_API_KEY    else ""
    api_secret = ALPACA_SECRET_KEY if ALPACA_SECRET_KEY else ""
    if not api_key or not api_secret:
        return _yf_latest(symbol)
    try:
        import requests as _req
        sym = symbol.replace("-USD", "USD")  # BTC-USD â†’ BTCUSD for crypto
        # Try stocks first
        url = f"https://data.alpaca.markets/v2/stocks/{sym}/trades/latest"
        r = _req.get(url, headers={
            "APCA-API-KEY-ID": api_key,
            "APCA-API-SECRET-KEY": api_secret,
        }, timeout=5)
        if r.status_code == 200:
            return float(r.json()["trade"]["p"])
        # Try crypto
        url = f"https://data.alpaca.markets/v1beta3/crypto/us/latest/trades?symbols={sym}"
        r = _req.get(url, headers={
            "APCA-API-KEY-ID": api_key,
            "APCA-API-SECRET-KEY": api_secret,
        }, timeout=5)
        if r.status_code == 200:
            trades = r.json().get("trades", {})
            if sym in trades:
                return float(trades[sym]["p"])
    except Exception:
        pass
    return _yf_latest(symbol)   # graceful fallback
def _fred(series_id, fallback=None):
    if not FRED_API_KEY: return fallback
    try:
        url = (f"https://api.stlouisfed.org/fred/series/observations"
               f"?series_id={series_id}&api_key={FRED_API_KEY}"
               f"&file_type=json&sort_order=desc&limit=2")
        r = requests.get(url, timeout=8)
        for o in r.json().get("observations",[]):
            try: return float(o["value"])
            except Exception: continue
        return fallback
    except Exception: return fallback

def fetch_macro():
    global MACRO
    fed_rate  = _yf_latest("^IRX")
    tnx_10y   = _yf_latest("^TNX")
    irx_2y    = _yf_latest("^IRX")
    vix       = _yf_latest("^VIX")
    dxy       = _yf_latest("DX-Y.NYB")
    wti_crude = _yf_latest("CL=F")
    gold      = _yf_latest("GC=F")
    hyg       = _yf_latest("HYG")
    lqd       = _yf_latest("LQD")
    credit_spread = round(hyg/lqd,4) if hyg and lqd else None
    spy_info  = {}
    try: spy_info = yf.Ticker("SPY").info
    except Exception: pass
    spy_pe     = spy_info.get("trailingPE")
    ey         = round(100/spy_pe,2) if spy_pe and spy_pe>0 else None
    ey_spread  = round(ey-(tnx_10y or 4.3),2) if ey else None
    yc         = round((tnx_10y or 4.3)-(irx_2y or 5.0),3)
    unemployment = _fred("UNRATE",   fallback=3.9)
    cpi_yoy      = _fred("CPIAUCSL", fallback=3.1)
    gdp_growth   = _fred("A191RL1Q225SBEA", fallback=2.8)
    retail_sales = _fred("RSAFS",    fallback=0.4)
    ism_pmi      = _fred("MANEMP",   fallback=50.3)
    # -- v25 macro additions -----------------------------------------------
    pce_core       = _fred("PCEPILFE",  fallback=2.8)   # Core PCE (Fed target)
    jolts_openings = _fred("JTSJOL",    fallback=7800)  # Job openings (000s)
    consumer_sent  = _fred("UMCSENT",   fallback=68.0)  # Michigan sentiment
    move_index     = _yf_latest("^MOVE")                # Bond vol (VIX for rates)
    ff_futures     = _yf_latest("ZQ=F")                 # Fed Funds futures
    put_call_ratio = None
    try:
        _pc = yf.download("^CPC", period="5d", progress=False, auto_adjust=True)
        put_call_ratio = round(float(_pc["Close"].dropna().iloc[-1]),3) if not _pc.empty else None
    except Exception: pass
    sofr_rate = _fred("SOFR", fallback=None)
    tbill_3m  = _fred("DTB3", fallback=None)
    ted_spread = round(sofr_rate - tbill_3m, 3) if sofr_rate and tbill_3m else None
    xlf_price = _yf_latest("XLF")
    spy_price = _yf_latest("SPY")
    sector_rotation = round(xlf_price / spy_price, 4) if xlf_price and spy_price else None
    # â”€â”€ Full yield curve via FRED â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    t_note_2y  = _fred("DGS2",   fallback=None)
    t_note_5y  = _fred("DGS5",   fallback=None)
    t_bond_30y = _fred("DGS30",  fallback=None) or _yf_latest("^TYX")
    tbill_3m_y = _fred("DGS3MO", fallback=None) or fed_rate
    yc_2s10s   = round((tnx_10y or 4.3) - (t_note_2y  or 4.5), 3) if tnx_10y else None
    yc_3m10y   = round((tnx_10y or 4.3) - (tbill_3m_y or 5.0), 3) if tnx_10y else None
    if   (yc_2s10s or 0) >  1.0: yc_regime = "steep"
    elif (yc_2s10s or 0) >  0.0: yc_regime = "normal"
    elif (yc_2s10s or 0) > -0.5: yc_regime = "flat"
    else:                          yc_regime = "inverted"

    # â”€â”€ VIX term structure â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    vix3m  = _yf_latest("^VIX3M")
    vix_ts = ("backwardation" if (vix and vix3m and vix > vix3m * 1.02)
              else "contango"  if (vix and vix3m) else "unknown")

    # â”€â”€ Commodity chain signals â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    copper = _yf_latest("HG=F")   # industrial demand / economic activity
    natgas = _yf_latest("NG=F")   # energy sector / utilities cost
    silver = _yf_latest("SI=F")   # industrial + precious metals
    brent  = _yf_latest("BZ=F")   # international crude benchmark

    # â”€â”€ Sector ETF momentum: 4-week vs 12-week return differential â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    sector_etf_momentum: dict = {}
    _etf_list = list(set(SECTOR_ETF_MAP.values()))
    try:
        _etf_raw = yf.download(_etf_list, period="65d", auto_adjust=True, progress=False)
        _etf_px  = _etf_raw["Close"] if "Close" in _etf_raw else _etf_raw
        if isinstance(_etf_px, pd.Series): _etf_px = _etf_px.to_frame()
        for etf in _etf_list:
            col = etf if etf in _etf_px.columns else None
            if col is None: continue
            px = _etf_px[col].dropna()
            if len(px) < 21: continue
            r4w  = float(px.iloc[-1]/px.iloc[-21]-1) if len(px)>=21 else 0.0
            r12w = float(px.iloc[-1]/px.iloc[min(-63,-len(px))]-1)
            sector_etf_momentum[etf] = round(r4w - r12w/3, 4)
    except Exception as e:
        print(f"  ETF momentum error: {e}")

    # â”€â”€ Congressional stock trades via Quiver Quant â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    congress_signals: dict = {}
    try:
        if QUIVER_QUANT_KEY:
            import datetime as _cdt
            _cutoff = (_cdt.date.today()-_cdt.timedelta(days=90)).isoformat()
            _cr = requests.get(
                "https://api.quiverquant.com/beta/live/congresstrading",
                headers={"Authorization": f"Token {QUIVER_QUANT_KEY}"},
                timeout=8)
            if _cr.status_code == 200:
                for t in _cr.json():
                    if t.get("TransactionDate","") < _cutoff: continue
                    tk = t.get("Ticker","").upper().strip()
                    if not tk or len(tk) > 6: continue
                    tx = t.get("Transaction","").lower()
                    if tk not in congress_signals:
                        congress_signals[tk] = {"buys":0,"sells":0,"total":0}
                    congress_signals[tk]["total"] += 1
                    if any(w in tx for w in ("purchase","buy")):
                        congress_signals[tk]["buys"] += 1
                    elif any(w in tx for w in ("sale","sell")):
                        congress_signals[tk]["sells"] += 1
                print(f"  Congress signals: {len(congress_signals)} tickers (90d window)")
            else:
                print(f"  Congress API: {_cr.status_code}")
        else:
            print("  Congress signals: set QUIVER_QUANT_KEY to enable")
    except Exception as e:
        print(f"  Congress signals error: {e}")

    # â”€â”€ Economic release calendar (binary risk flag) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    import datetime as _ecdt
    _today = _ecdt.date.today()
    _dom, _dow = _today.day, _today.weekday()
    _near_econ_release = (
        (_dom <= 7  and _dow == 4) or      # NFP: first Friday of month
        (9 <= _dom <= 14) or               # CPI: typically 10th-13th
        any(abs((_ecdt.date.fromisoformat(d)-_today).days)<=1
            for d in FOMC_DATES_2026)       # FOMC meeting days
    )
    if _near_econ_release:
        print("  âš¡ NEAR ECON RELEASE â€” confidence will be dampened today")

    # Short interest via Finviz scrape (no API key needed)
    short_interest: dict = {}
    try:
        import re as _re
        for tk in list(featured.keys())[:10] if "featured" in dir() else []:
            try:
                _r = requests.get(f"https://finviz.com/quote.ashx?t={tk}",
                                  headers={"User-Agent":"Mozilla/5.0"},timeout=5)
                _m = _re.search(r"Short Float.*?(\d+\.?\d*)%", _r.text)
                if _m: short_interest[tk] = float(_m.group(1))
            except Exception: pass
    except Exception: pass

    crypto_fg = None; crypto_fg_label = "unavailable"
    try:
        r = requests.get("https://api.alternative.me/fng/?limit=1",timeout=5)
        crypto_fg       = int(r.json()["data"][0]["value"])
        crypto_fg_label = r.json()["data"][0]["value_classification"]
    except Exception: pass
    if vix:
        if vix<15:    mr,mc="Risk-on / Bull","success"
        elif vix<25:  mr,mc="Neutral / Mixed","warning"
        else:         mr,mc="Risk-off / Bear","danger"
    else: mr,mc="Unknown","secondary"
    MACRO = dict(
        fed_rate=fed_rate,tnx_10y=tnx_10y,yield_curve=yc,
        vix=vix,dxy=dxy,wti_crude=wti_crude,gold=gold,
        credit_spread=credit_spread,earnings_yield=ey,ey_spread=ey_spread,
        unemployment=unemployment,cpi_yoy=cpi_yoy,gdp_growth=gdp_growth,
        retail_sales=retail_sales,ism_pmi=ism_pmi,
        pce_core=pce_core,jolts_openings=jolts_openings,consumer_sent=consumer_sent,
        move_index=move_index,ff_futures=ff_futures,put_call_ratio=put_call_ratio,
        ted_spread=ted_spread,sector_rotation=sector_rotation,
        crypto_fg=crypto_fg,crypto_fg_label=crypto_fg_label,
        macro_regime=mr,macro_regime_color=mc,
        t_note_2y=t_note_2y,t_note_5y=t_note_5y,t_bond_30y=t_bond_30y,
        yc_2s10s=yc_2s10s,yc_3m10y=yc_3m10y,yc_regime=yc_regime,
        vix3m=vix3m,vix_ts=vix_ts,
        copper=copper,natgas=natgas,silver=silver,brent=brent,
        sector_etf_momentum=sector_etf_momentum,
        congress_signals=congress_signals,
        near_econ_release=_near_econ_release,
        short_interest=short_interest,
    )
    return MACRO

print("Fetching macro data...")
fetch_macro()
for k,v in MACRO.items():
    if v is not None and k not in ("macro_regime","macro_regime_color","crypto_fg_label"):
        print(f"  {k:<20} {v}")
print(f"\nMacro ready | regime: {MACRO['macro_regime']}")


Fetching macro data...
  fed_rate             3.575000047683716
  tnx_10y              4.377999782562256
  yield_curve          0.803
  vix                  16.989999771118164
  dxy                  98.20999908447266
  wti_crude            101.94000244140625
  gold                 4629.89990234375
  credit_spread        0.7372
  earnings_yield       3.5
  ey_spread            -0.88
  unemployment         3.9
  cpi_yoy              3.1
  gdp_growth           2.8
  retail_sales         0.4
  ism_pmi              50.3
  crypto_fg            39

Macro ready | regime: Neutral / Mixed


In [ ]:
# ============================================================
# CELL 5 â€” TICKER DATA DOWNLOAD
# ============================================================
def download_ticker(ticker, start, end):
    """Single ticker download â€” checks _PRICE_CACHE first."""
    if ticker in _PRICE_CACHE:
        return _PRICE_CACHE[ticker]
    for attempt in range(3):
        try:
            df = yf.Ticker(ticker).history(start=start, end=end, auto_adjust=True)
            if df.empty: return None
            df.index = pd.to_datetime(df.index).tz_localize(None)
            df = df[["Open","High","Low","Close","Volume"]].copy()
            df.dropna(subset=["Close"], inplace=True)
            _PRICE_CACHE[ticker] = df
            return df
        except Exception as e:
            if attempt == 2: print(f"  {ticker}: {e}")
            time.sleep(1)
    return None

raw_data = {}
print(f"Batch downloading {len(DEFAULT_WATCHLIST)} tickers...")
_PRICE_CACHE.clear()   # refresh at start
all_prices = batch_download_tickers(DEFAULT_WATCHLIST, TRAIN_START, TRAIN_END)
print(f"  âœ“ {len(all_prices)} tickers downloaded via batch")

# Parallel fallback for any tickers missed by batch download
_missing = [tk for tk in DEFAULT_WATCHLIST
            if all_prices.get(tk) is None or len(all_prices.get(tk, [])) < 100]
if _missing:
    from concurrent.futures import ThreadPoolExecutor, as_completed as _asc
    print(f"  Parallel fallback for {len(_missing)} missed tickers...")
    with ThreadPoolExecutor(max_workers=8) as _pool:
        _futs = {_pool.submit(download_ticker, tk, TRAIN_START, TRAIN_END): tk for tk in _missing}
        for _fut in _asc(_futs):
            _tk = _futs[_fut]
            try:
                _df = _fut.result()
                if _df is not None:
                    all_prices[_tk] = _df
            except Exception as _fe:
                print(f"  {_tk} parallel fetch failed: {_fe}")
_delisted_tickers = []
for tk in DEFAULT_WATCHLIST:
    df = all_prices.get(tk)
    if df is not None and len(df) > 100:
        _recent = df[df.index >= (pd.Timestamp.today() - pd.Timedelta(days=30))]
        if len(_recent) < 5:
            print(f"  DELISTED? {tk:<10} {len(_recent)} rows in last 30d -- skip")
            _delisted_tickers.append(tk)
            continue
        raw_data[tk] = df
        print(f"  OK {tk:<10} {len(df)} rows  ${df['Close'].iloc[-1]:.2f}")
    else:
        print(f"  SKIP {tk}")
if _delisted_tickers:
    print(f"\n  WARNING: {len(_delisted_tickers)} delisted/halted excluded: {_delisted_tickers}")
    WATCHLIST         = [t for t in WATCHLIST         if t not in _delisted_tickers]
    DEFAULT_WATCHLIST = [t for t in DEFAULT_WATCHLIST if t not in _delisted_tickers]
print(f"\n{len(raw_data)}/{len(DEFAULT_WATCHLIST)} tickers ready")
# -- Liquidity filter: remove tickers with ADV < $50M --
_ADV_MIN_USD = 50_000_000  # $50M average daily dollar volume threshold

def _compute_adv(df, window=20):
    """20-day average daily dollar volume (price x volume)."""
    if df is None or len(df) < window:
        return 0.0
    _c = df["Close"].iloc[-window:]
    _v = df["Volume"].iloc[-window:]
    return float((_c * _v).mean())

_illiquid = [tk for tk in list(raw_data.keys())
             if _compute_adv(raw_data[tk]) < _ADV_MIN_USD]
for tk in _illiquid:
    del raw_data[tk]

print(f"  Liquidity filter: removed {len(_illiquid)} tickers with ADV < $50M")
if _illiquid:
    print(f"  Removed: {', '.join(_illiquid[:20])}{'...' if len(_illiquid)>20 else ''}")
print(f"  Tradeable universe after filter: {len(raw_data)} tickers")

  OK AAPL       1843 rows  $280.14
  OK MSFT       1843 rows  $414.44
  OK NVDA       1843 rows  $198.45
  OK GOOGL      1843 rows  $385.69
  OK AMZN       1843 rows  $268.26
  OK META       1843 rows  $608.75
  OK TSLA       1843 rows  $390.82
  OK JPM        1843 rows  $312.47
  OK V          1843 rows  $328.03
  OK UNH        1843 rows  $368.78
  OK SPY        1843 rows  $720.65
  OK BTC-USD    2678 rows  $78179.00
  OK ETH-USD    2678 rows  $2295.09
  OK SOL-USD    2213 rows  $83.72
  OK BNB-USD    2678 rows  $615.33
  OK XRP-USD    2678 rows  $1.38

16/16 tickers ready


In [ ]:
# ============================================================
# CELL 6 Ã¢â‚¬â€ FEATURE ENGINEERING  (macro-aware)
# ============================================================
def build_features(df):
    if df is None or df.empty or len(df) < 50:
        return pd.DataFrame()
    d = df.copy()
    c,h,l,v = d["Close"],d["High"],d["Low"],d["Volume"]
    for k in [1,3,5,10,21]: d[f"ret_{k}d"]=c.pct_change(k)
    for w in [5,10,20,50,200]:
        d[f"sma_{w}"]=c.rolling(w).mean()
        d[f"sma_r_{w}"]=c/d[f"sma_{w}"]-1
    d["ema_12"]=c.ewm(span=12,adjust=False).mean()
    d["ema_26"]=c.ewm(span=26,adjust=False).mean()
    d["macd"]=d["ema_12"]-d["ema_26"]
    d["macd_sig"]=d["macd"].ewm(span=9,adjust=False).mean()
    d["macd_h"]=d["macd"]-d["macd_sig"]
    d["rsi_14"]=ta.momentum.rsi(c,window=14)
    d["rsi_7"]=ta.momentum.rsi(c,window=7)
    stoch=ta.momentum.StochasticOscillator(h,l,c)
    d["stoch_k"]=stoch.stoch(); d["stoch_d"]=stoch.stoch_signal()
    d["cci"]=ta.trend.CCIIndicator(h,l,c).cci()
    d["willr"]=ta.momentum.WilliamsRIndicator(h,l,c).williams_r()
    d["mfi"]=ta.volume.MFIIndicator(h,l,c,v).money_flow_index()
    bb=ta.volatility.BollingerBands(c)
    d["bb_upper"]=bb.bollinger_hband(); d["bb_lower"]=bb.bollinger_lband()
    d["bb_pct"]=bb.bollinger_pband(); d["bb_w"]=bb.bollinger_wband()
    d["atr_14"]=ta.volatility.AverageTrueRange(h,l,c).average_true_range()
    kelt=ta.volatility.KeltnerChannel(h,l,c)
    d["kelt_u"]=kelt.keltner_channel_hband()
    d["kelt_l"]=kelt.keltner_channel_lband()
    d["vol_r_20"]=v/v.rolling(20).mean()
    d["obv"]=ta.volume.OnBalanceVolumeIndicator(c,v).on_balance_volume()
    d["vwap"]=(c*v).cumsum()/v.cumsum()
    d["dow"]=d.index.dayofweek; d["month"]=d.index.month
    # -- #4: 52-week proximity
    _roll252 = c.rolling(252, min_periods=100)
    d["w52_high_prox"] = (c / _roll252.max() - 1).clip(-1.0, 0.0)
    d["w52_low_prox"]  = (c / _roll252.min() - 1).clip( 0.0, 3.0)
    # -- #11: Seasonality
    d["days_to_mend"] = (d.index.days_in_month - d.index.day).astype(float)
    d["is_q_end"]     = ((d.index.month % 3 == 0) & (d["days_to_mend"] <= 5)).astype(float)
    d["is_q_start"]   = ((d.index.month % 3 == 1) & (d.index.day <= 5)).astype(float)
    d["body"]=(c-d["Open"]).abs()/(h-l+1e-9)
    d["upper_w"]=(h-c.clip(lower=d["Open"]))/(h-l+1e-9)
    d["lower_w"]=(c.clip(upper=d["Open"])-l)/(h-l+1e-9)
    for w in [5,10,21]: d[f"rvol_{w}"]=d["ret_1d"].rolling(w).std()*np.sqrt(252)
    # Ã¢â€â‚¬Ã¢â€â‚¬ Historical macro join (CV leakage fix) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    # Instead of stamping today's scalar values on every row, fetch the
    # actual historical time series and join by date (already lagged 1 day).
    _start = d.index[0].strftime("%Y-%m-%d")
    _end   = (d.index[-1] + pd.Timedelta(days=5)).strftime("%Y-%m-%d")
    macro_hist = _get_macro_history(_start, _end)
    MACRO_HIST_COLS = ["m_vix","m_tnx","m_irx","m_dxy","m_wti","m_gold","m_credit","m_yc"]
    if not macro_hist.empty:
        for col in MACRO_HIST_COLS:
            if col in macro_hist.columns:
                d[col] = macro_hist[col].reindex(d.index, method="ffill")
    # Scalar fallbacks for cols not covered by the historical pull
    _macro_defaults = {
        "m_vix":20.0, "m_tnx":4.3, "m_irx":5.0, "m_dxy":104.0,
        "m_wti":80.0, "m_gold":2000.0, "m_credit":1.0, "m_yc":0.0,
        "m_unemp": MACRO.get("unemployment") or 4.0,
        "m_cpi":   MACRO.get("cpi_yoy")      or 3.0,
        "m_gdp":   MACRO.get("gdp_growth")   or 2.5,
        "m_pmi":   MACRO.get("ism_pmi")      or 50.0,
        "m_cfg":   float(MACRO.get("crypto_fg") or 50),
    }
    for col, val in _macro_defaults.items():
        if col not in d.columns or d[col].isna().all():
            d[col] = val
    # Ã¢â€â‚¬Ã¢â€â‚¬ Magnitude-threshold label (noise filter) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    # Only label moves that are meaningful (>1%). Near-zero rows Ã¢â€ â€™ NaN,
    # dropped during training to remove noise from the decision boundary.
    fwd_ret = c.shift(-FORECAST_DAYS) / c - 1
    d["fwd_ret"] = fwd_ret
    _q80 = fwd_ret.rolling(252, min_periods=63).quantile(0.80)
    _q20 = fwd_ret.rolling(252, min_periods=63).quantile(0.20)
    d["target"] = np.where(fwd_ret >= _q80, 1,
                  np.where(fwd_ret <= _q20, 0, np.nan))
    feat_cols=[col for col in d.columns
               if col not in ["Open","High","Low","Close","Volume","target","fwd_ret"]]
    d[feat_cols]=d[feat_cols].shift(1)
    d.dropna(inplace=True)
    return d

def build_intraday_features(df_5min):
    """
    Build features from 5-minute OHLCV bars.
    Designed for intraday signals with 30-min to 2-hour holding periods.
    Falls back gracefully if intraday data unavailable.
    """
    d = df_5min.copy()
    if d.empty or len(d) < 20:
        return pd.DataFrame()

    c, h, l, v = d["Close"], d["High"], d["Low"], d["Volume"]

    # Microstructure features
    d["vwap"]      = (c * v).cumsum() / v.cumsum()
    d["vwap_dev"]  = (c - d["vwap"]) / d["vwap"]

    vwap_std       = d["vwap_dev"].rolling(78).std()
    d["vwap_upper1"] = d["vwap"] * (1 + vwap_std)
    d["vwap_lower1"] = d["vwap"] * (1 - vwap_std)
    d["vwap_pct"]    = (c - d["vwap_lower1"]) / (d["vwap_upper1"] - d["vwap_lower1"] + 1e-9)

    # Opening range breakout (first 30 mins = 6 bars)
    try:
        session_open = d.between_time("09:30", "10:00") if hasattr(d.index, "time") else d.head(6)
    except Exception:
        session_open = d.head(6)
    or_high = session_open["High"].max()
    or_low  = session_open["Low"].min()
    d["or_position"] = (c - or_low) / (or_high - or_low + 1e-9)
    d["above_or"]    = (c > or_high).astype(int)
    d["below_or"]    = (c < or_low).astype(int)

    # Intraday momentum
    for k in [1, 3, 6, 12, 24]:
        d[f"ret_{k}b"] = c.pct_change(k)

    # Volume profile
    d["vol_r_6b"]  = v / v.rolling(6).mean()
    d["vol_r_24b"] = v / v.rolling(24).mean()
    d["vol_spike"] = (d["vol_r_6b"] > 2.5).astype(int)

    # Price action
    d["body"]    = (c - d["Open"]).abs() / (h - l + 1e-9)
    d["upper_w"] = (h - c.clip(lower=d["Open"])) / (h - l + 1e-9)
    d["lower_w"] = (c.clip(upper=d["Open"]) - l) / (h - l + 1e-9)

    # Intraday RSI and MACD
    d["rsi_14"]  = ta.momentum.rsi(c, window=14)
    d["rsi_7"]   = ta.momentum.rsi(c, window=7)
    d["macd"]    = c.ewm(span=12).mean() - c.ewm(span=26).mean()
    d["macd_sig"]= d["macd"].ewm(span=9).mean()
    d["macd_h"]  = d["macd"] - d["macd_sig"]

    # Bollinger Bands
    bb = ta.volatility.BollingerBands(c, window=20)
    d["bb_pct"]  = bb.bollinger_pband()
    d["bb_w"]    = bb.bollinger_wband()

    # ATR
    d["atr_14"]  = ta.volatility.AverageTrueRange(h, l, c).average_true_range()

    # Time-of-day features
    if hasattr(d.index, "hour"):
        d["hour"]    = d.index.hour
        d["minute"]  = d.index.minute
        d["tod_sin"] = np.sin(2 * np.pi * (d["hour"] * 60 + d["minute"]) / 390)
        d["tod_cos"] = np.cos(2 * np.pi * (d["hour"] * 60 + d["minute"]) / 390)

    # Macro scalars
    d["m_vix"]  = MACRO.get("vix")   or 20.0
    d["m_yc"]   = MACRO.get("yield_curve") or 0.0

    # Target: price direction in next 6 bars (30 mins)
    INTRADAY_HORIZON = 6
    fwd = c.shift(-INTRADAY_HORIZON) / c - 1
    d["fwd_ret"] = fwd
    d["target"]  = np.where(fwd >  0.002, 1,
                   np.where(fwd < -0.002, 0, np.nan))

    feat_cols = [col for col in d.columns
                 if col not in ["Open","High","Low","Close","Volume","target","fwd_ret"]]
    d[feat_cols] = d[feat_cols].shift(1)
    d.dropna(subset=feat_cols[:5], inplace=True)

    return d


INTRADAY_FEATURE_COLS = [
    "vwap_dev","vwap_pct","or_position","above_or","below_or",
    "ret_1b","ret_3b","ret_6b","ret_12b","ret_24b",
    "vol_r_6b","vol_r_24b","vol_spike",
    "body","upper_w","lower_w",
    "rsi_14","rsi_7","macd","macd_sig","macd_h",
    "bb_pct","bb_w","atr_14",
    "tod_sin","tod_cos",
    "m_vix","m_yc",
]

def fetch_intraday_bars(ticker: str, days_back: int = 30):
    """Fetch 5-minute bars from Alpaca or yfinance fallback."""
    api_key    = ALPACA_API_KEY    if ALPACA_API_KEY    else ""
    api_secret = ALPACA_SECRET_KEY if ALPACA_SECRET_KEY else ""
    if api_key and api_secret:
        try:
            import requests as _req
            from datetime import datetime, timedelta
            end   = datetime.utcnow()
            start = end - timedelta(days=days_back)
            sym   = ticker.replace("-USD","USD")
            fmt   = "%Y-%m-%dT%H:%M:%SZ"
            url   = (f"https://data.alpaca.markets/v2/stocks/{sym}/bars"
                     f"?timeframe=5Min&start={start.strftime(fmt)}"
                     f"&end={end.strftime(fmt)}&limit=10000&adjustment=raw")
            r = _req.get(url, headers={
                "APCA-API-KEY-ID": api_key,
                "APCA-API-SECRET-KEY": api_secret,
            }, timeout=10)
            if r.status_code == 200:
                bars = r.json().get("bars", [])
                if bars:
                    df = pd.DataFrame(bars)
                    df["t"] = pd.to_datetime(df["t"])
                    df = df.set_index("t").rename(columns={
                        "o":"Open","h":"High","l":"Low","c":"Close","v":"Volume"
                    })
                    return df[["Open","High","Low","Close","Volume"]]
        except Exception:
            pass
    try:
        df = yf.download(ticker, period=f"{min(days_back,59)}d",
                         interval="5m", auto_adjust=True, progress=False)
        if not df.empty:
            df.columns = [c if isinstance(c,str) else c[0] for c in df.columns]
            return df[["Open","High","Low","Close","Volume"]]
    except Exception:
        pass
    return pd.DataFrame()


_SECT_ETF_PRICES = {}
try:
    _etfs_dl = list(set(SECTOR_ETF_MAP.values())) + ["SPY"]
    _raw_etf = yf.download(_etfs_dl, start=TRAIN_START, end=TRAIN_END, auto_adjust=True, progress=False)
    _etf_close = (_raw_etf["Close"] if "Close" in _raw_etf else _raw_etf)
    if isinstance(_etf_close, pd.Series): _etf_close = _etf_close.to_frame()
    for _esym in _etfs_dl:
        if _esym in _etf_close.columns: _SECT_ETF_PRICES[_esym] = _etf_close[_esym].dropna()
    print(f"  Sector ETFs: {sorted(_SECT_ETF_PRICES.keys())}")
except Exception as _eetf: print(f"  Sector ETF failed: {_eetf}")
_EARN_SURPRISE = {}
try:
    for _etk in list(raw_data.keys()):
        try:
            _eh = yf.Ticker(_etk).earnings_history
            if _eh is not None and not _eh.empty and "Surprise(%)" in _eh.columns:
                _s = _eh["Surprise(%)"].replace([float("inf"),float("-inf")], float("nan")).dropna()
                if len(_s) > 0: _EARN_SURPRISE[_etk] = float(_s.mean())
        except Exception: pass
    print(f"  Earnings surprise: {len(_EARN_SURPRISE)} tickers")
except Exception as _e_es: print(f"  Earnings surprise failed: {_e_es}")
print("Building features...")
featured = {}
for tk,df in raw_data.items():
    fd=build_features(df)
    if len(fd)>200:
        featured[tk]=fd
        print(f"  OK {tk:<10} {len(fd)} rows  {len(fd.columns)} features")

if not featured:
    raise RuntimeError("No features built -- all tickers failed to download. Check network.")
FEATURE_COLS=[c for c in next(iter(featured.values())).columns
              if c not in ["Open","High","Low","Close","Volume","target","fwd_ret"]]
print(f"\n{len(featured)} tickers | {len(FEATURE_COLS)} features (incl macro)")
try:
    _bf = {tk: (fd["sma_r_50"] > 0).astype(float) for tk, fd in featured.items() if "sma_r_50" in fd.columns}
    if _bf:
        _breadth_s = pd.DataFrame(_bf).mean(axis=1)
        for tk in featured: featured[tk]["mkt_breadth"] = _breadth_s.reindex(featured[tk].index, method="ffill").fillna(0.5)
        print("  Market breadth added")
except Exception as _e_br: print(f"  Breadth failed: {_e_br}")
try:
    _spy_r21 = _SECT_ETF_PRICES.get("SPY", pd.Series(dtype=float)).pct_change(21).shift(1)
    for tk in featured:
        if "ret_21d" not in featured[tk].columns: continue
        _r21 = featured[tk]["ret_21d"]
        _setf = SECTOR_ETF_MAP.get(TICKER_SECTOR.get(tk, ""), "SPY")
        _etf_r21 = _SECT_ETF_PRICES.get(_setf, pd.Series(dtype=float)).pct_change(21).shift(1)
        if len(_etf_r21) > 0: featured[tk]["sect_rel_str_21d"] = (_r21 - _etf_r21.reindex(featured[tk].index, method="ffill")).clip(-0.5, 0.5)
        if len(_spy_r21) > 0: featured[tk]["mkt_rel_str_21d"] = (_r21 - _spy_r21.reindex(featured[tk].index, method="ffill")).clip(-0.5, 0.5)
    print("  Relative strength added")
except Exception as _e_rs: print(f"  RelStr failed: {_e_rs}")
for tk in featured: featured[tk]["avg_earn_surprise"] = _EARN_SURPRISE.get(tk, 0.0)
FEATURE_COLS = [c for c in next(iter(featured.values())).columns if c not in ["Open","High","Low","Close","Volume","target","fwd_ret"]]
print(f"  {len(FEATURE_COLS)} total features (incl earnings surprise)")


Building features...
  OK AAPL       1643 rows  63 features
  OK MSFT       1643 rows  63 features
  OK NVDA       1643 rows  63 features
  OK GOOGL      1643 rows  63 features
  OK AMZN       1643 rows  63 features
  OK META       1643 rows  63 features
  OK TSLA       1643 rows  63 features
  OK JPM        1643 rows  63 features
  OK V          1643 rows  63 features
  OK UNH        1643 rows  63 features
  OK SPY        1643 rows  63 features
  OK BTC-USD    2478 rows  63 features
  OK ETH-USD    2478 rows  63 features
  OK SOL-USD    2013 rows  63 features
  OK BNB-USD    2478 rows  63 features
  OK XRP-USD    2478 rows  63 features

16 tickers | 57 features (incl macro)


In [ ]:
# ============================================================
# CELL 7 Ã¢â‚¬â€ HMM REGIME DETECTION
# ============================================================
def fit_hmm(df, n_states=HMM_STATES):
    returns=df["Close"].pct_change().dropna().values.reshape(-1,1)
    model=GaussianHMM(n_components=n_states,covariance_type="full",
                      n_iter=200,random_state=42)
    model.fit(returns)
    labels=model.predict(returns)
    # Sort states by emission mean: state 0=bear, 1=neutral, 2=bull
    order=np.argsort(model.means_.flatten())
    remap={int(old):int(new) for new,old in enumerate(order)}
    labels=np.array([remap[int(l)] for l in labels])
    s=pd.Series(labels,index=df.index[1:],name="regime")
    return s.reindex(df.index).ffill().fillna(0).astype(int)

print("Fitting HMM regimes...")
regimes = {}
for tk,df in featured.items():
    regimes[tk]=fit_hmm(raw_data[tk].loc[df.index[0]:])
    print(f"  OK {tk:<10} regime={regimes[tk].iloc[-1]}")
print("\nRegimes complete")


Fitting HMM regimes...
  OK AAPL       regime=0
  OK MSFT       regime=2
  OK NVDA       regime=0
  OK GOOGL      regime=2
  OK AMZN       regime=1
  OK META       regime=1
  OK TSLA       regime=1
  OK JPM        regime=0
  OK V          regime=2
  OK UNH        regime=0
  OK SPY        regime=1
  OK BTC-USD    regime=0
  OK ETH-USD    regime=0
  OK SOL-USD    regime=2
  OK BNB-USD    regime=1
  OK XRP-USD    regime=2

Regimes complete


In [ ]:
# ============================================================
# CELL 8 â€” ML ENSEMBLE TRAINING
# ============================================================
def train_ensemble(df, ticker, full_tune: bool = True):
    # Drop noise-band rows (target==NaN from Â±1% threshold filter)
    df_train = df.dropna(subset=["target"])
    X=df_train[FEATURE_COLS].values; y=df_train["target"].values.astype(int)
    scaler=StandardScaler(); X_sc=scaler.fit_transform(X)
    tscv=TimeSeriesSplit(n_splits=5)
    def xgb_obj(trial):
        p=dict(
            n_estimators=trial.suggest_int("n",100,500),
            max_depth=trial.suggest_int("d",3,9),
            learning_rate=trial.suggest_float("lr",1e-3,0.3,log=True),
            subsample=trial.suggest_float("sub",0.5,1.0),
            colsample_bytree=trial.suggest_float("col",0.5,1.0),
            gamma=trial.suggest_float("g",0,5),
            reg_alpha=trial.suggest_float("a",0,3),
            reg_lambda=trial.suggest_float("l",0,3),
            eval_metric="logloss",
            tree_method="hist",random_state=42,verbosity=0,early_stopping_rounds=30)
        aucs=[]
        for ti,vi in tscv.split(X_sc):
            Xtr,Xva,ytr,yva=X_sc[ti],X_sc[vi],y[ti],y[vi]
            m=xgb.XGBClassifier(**p)
            m.fit(Xtr,ytr,eval_set=[(Xva,yva)],verbose=False)
            if len(np.unique(yva))>1:
                aucs.append(roc_auc_score(yva,m.predict_proba(Xva)[:,1]))
        return np.mean(aucs) if aucs else 0.5
    study=optuna.create_study(direction="maximize")
    study.optimize(xgb_obj, n_trials=FULL_TUNE_TRIALS_XGB if full_tune else QUICK_TUNE_TRIALS,
                   timeout=45, show_progress_bar=False)
    bp=study.best_params
    best_xgb=xgb.XGBClassifier(
        n_estimators=bp["n"],max_depth=bp["d"],learning_rate=bp["lr"],
        subsample=bp["sub"],colsample_bytree=bp["col"],gamma=bp["g"],
        reg_alpha=bp["a"],reg_lambda=bp["l"],
        eval_metric="logloss",
        tree_method="hist",random_state=42,verbosity=0)
    # â”€â”€ LightGBM Optuna study â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def lgb_obj(trial):
        p = dict(
            n_estimators    = trial.suggest_int  ("n",  100, 500),
            max_depth       = trial.suggest_int  ("d",  3,   9),
            learning_rate   = trial.suggest_float("lr", 1e-3, 0.3, log=True),
            num_leaves      = trial.suggest_int  ("nl", 15,  127),
            subsample       = trial.suggest_float("sub",0.5,  1.0),
            colsample_bytree= trial.suggest_float("col",0.5,  1.0),
            reg_alpha       = trial.suggest_float("a",  0,    3),
            reg_lambda      = trial.suggest_float("l",  0,    3),
            random_state=42, verbose=-1)
        aucs = []
        for ti, vi in tscv.split(X_sc):
            Xtr,Xva,ytr,yva = X_sc[ti],X_sc[vi],y[ti],y[vi]
            m = lgb.LGBMClassifier(**p)
            m.fit(Xtr,ytr,eval_set=[(Xva,yva)],
                  callbacks=[lgb.early_stopping(30,verbose=False),lgb.log_evaluation(-1)])
            if len(np.unique(yva))>1:
                aucs.append(roc_auc_score(yva,m.predict_proba(Xva)[:,1]))
        return np.mean(aucs) if aucs else 0.5
    study_lgb = optuna.create_study(direction="maximize")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study_lgb.optimize(lgb_obj, n_trials=FULL_TUNE_TRIALS_LGB if full_tune else QUICK_TUNE_TRIALS,
                       timeout=45, show_progress_bar=False)
    blp = study_lgb.best_params
    best_lgb = lgb.LGBMClassifier(
        n_estimators=blp["n"], max_depth=blp["d"], learning_rate=blp["lr"],
        num_leaves=blp["nl"], subsample=blp["sub"], colsample_bytree=blp["col"],
        reg_alpha=blp["a"], reg_lambda=blp["l"], random_state=42, verbose=-1)

    # â”€â”€ CatBoost Optuna study â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def cat_obj(trial):
        p = dict(
            iterations          = trial.suggest_int  ("n",  100, 500),
            depth               = trial.suggest_int  ("d",  3,   9),
            learning_rate       = trial.suggest_float("lr", 1e-3, 0.3, log=True),
            l2_leaf_reg         = trial.suggest_float("l2", 1,    10),
            bagging_temperature = trial.suggest_float("bt", 0,    1),
            random_seed=42, verbose=0)
        aucs = []
        for ti, vi in tscv.split(X_sc):
            Xtr,Xva,ytr,yva = X_sc[ti],X_sc[vi],y[ti],y[vi]
            m = CatBoostClassifier(**p)
            m.fit(Xtr,ytr,eval_set=(Xva,yva),early_stopping_rounds=30,verbose=False)
            if len(np.unique(yva))>1:
                aucs.append(roc_auc_score(yva,m.predict_proba(Xva)[:,1]))
        return np.mean(aucs) if aucs else 0.5
    study_cat = optuna.create_study(direction="maximize")
    study_cat.optimize(cat_obj, n_trials=FULL_TUNE_TRIALS_CAT if full_tune else QUICK_TUNE_TRIALS,
                       timeout=45, show_progress_bar=False)
    bcp = study_cat.best_params
    best_cat = CatBoostClassifier(
        iterations=bcp["n"], depth=bcp["d"], learning_rate=bcp["lr"],
        l2_leaf_reg=bcp["l2"], bagging_temperature=bcp["bt"],
        random_seed=42, verbose=0)
    try: X_r,y_r=SMOTE(random_state=42).fit_resample(X_sc,y)
    except Exception: X_r,y_r=X_sc,y
    y_smooth = np.where(y_r == 1, 0.95, 0.05)
    n_es = max(int(len(X_sc) * 0.12), 20)
    X_es, y_es = X_sc[-n_es:], y[-n_es:]
    best_xgb = xgb.XGBClassifier(n_estimators=max(bp.get("n",300),700), max_depth=bp["d"],
        learning_rate=bp["lr"], subsample=bp["sub"], colsample_bytree=bp["col"],
        gamma=bp["g"], reg_alpha=bp["a"], reg_lambda=bp["l"],
        eval_metric="logloss", early_stopping_rounds=40, tree_method="hist", random_state=42, verbosity=0)
    best_xgb.fit(X_r, y_smooth, eval_set=[(X_es, y_es)], verbose=False)
    best_lgb = lgb.LGBMClassifier(n_estimators=max(blp.get("n",300),700), max_depth=blp["d"],
        learning_rate=blp["lr"], num_leaves=blp["nl"], subsample=blp["sub"],
        colsample_bytree=blp["col"], reg_alpha=blp["a"], reg_lambda=blp["l"], random_state=42, verbose=-1)
    best_lgb.fit(X_r, y_r.astype(int), eval_set=[(X_es, y_es)],
                 callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(-1)])
    best_cat = CatBoostClassifier(iterations=max(bcp.get("n",300),700), depth=bcp["d"],
        learning_rate=bcp["lr"], l2_leaf_reg=bcp["l2"], bagging_temperature=bcp["bt"], random_seed=42, verbose=0)
    best_cat.fit(X_r, y_smooth, eval_set=(X_es, y_es), early_stopping_rounds=40, verbose=False)
    # AUC on second-to-last 20% (independent of calibration slice)
    n_cal=max(int(len(X_sc)*0.20),30)
    n_val=max(int(len(X_sc)*0.20),30)
    n_meta=max(int(len(X_sc)*0.125),20)
    Xva,yva=X_sc[-(n_val+n_cal):-n_cal],y[-(n_val+n_cal):-n_cal]
    X_meta,y_meta=X_sc[-(n_val+n_cal+n_meta):-(n_val+n_cal)],y[-(n_val+n_cal+n_meta):-(n_val+n_cal)]
    prob=(best_xgb.predict_proba(Xva)[:,1]+
          best_lgb.predict_proba(Xva)[:,1]+
          best_cat.predict_proba(Xva)[:,1])/3
    auc=roc_auc_score(yva,prob) if len(np.unique(yva))>1 else 0.5
    fi=pd.Series(best_xgb.feature_importances_,
                 index=FEATURE_COLS).sort_values(ascending=False)

    # â”€â”€ Platt scaling calibration (manual sigmoid â€” cv="prefit" removed in sklearn>=1.5) â”€â”€
    try:
        from sklearn.linear_model import LogisticRegression as _CalLR
        n_cal = max(int(len(X_sc) * 0.20), 30)
        X_cal, y_cal = X_sc[-n_cal:], y[-n_cal:]

        def _sigmoid_cal(model, Xc, yc):
            """Platt scaling without cv=prefit: fit sigmoid on holdout proba."""
            raw = model.predict_proba(Xc)[:, 1].reshape(-1, 1)
            lr  = _CalLR(C=1.0, max_iter=500, solver="lbfgs")
            lr.fit(raw, yc)
            class _CalWrapper:
                def __init__(self, base, cal): self._b, self._c = base, cal
                def predict_proba(self, X):
                    p   = self._b.predict_proba(X)[:, 1].reshape(-1, 1)
                    pos = self._c.predict_proba(p)[:, 1]
                    return np.column_stack([1 - pos, pos])
                def predict(self, X):
                    return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)
            return _CalWrapper(model, lr)

        best_xgb = _sigmoid_cal(best_xgb, X_cal, y_cal)
        best_lgb = _sigmoid_cal(best_lgb, X_cal, y_cal)
        best_cat = _sigmoid_cal(best_cat, X_cal, y_cal)
        print(f"    âœ” Platt calibration applied")
    except Exception as e:
        print(f"    Calibration skipped: {e}")
    _meta = None
    try:
        from sklearn.linear_model import LogisticRegression as _MetaLR
        _meta_Xv = np.column_stack([best_xgb.predict_proba(X_meta)[:,1], best_lgb.predict_proba(X_meta)[:,1], best_cat.predict_proba(X_meta)[:,1]])
        if len(np.unique(y_meta)) > 1:
            _meta = _MetaLR(C=1.0, max_iter=300, solver="lbfgs")
            _meta.fit(_meta_Xv, y_meta)
    except Exception: pass
    return dict(xgb=best_xgb,lgb=best_lgb,cat=best_cat,scaler=scaler,auc=auc,fi=fi,meta=_meta)

# Skip full retraining in Colab if a fresh model cache exists (< 12h old)
import time as _t
import json
_in_gh2 = bool(os.environ.get("GH_ACTIONS", ""))
_cache_age_h = float("inf")
if MODEL_CACHE_FILE.exists():
    _cache_age_h = (_t.time() - MODEL_CACHE_FILE.stat().st_mtime) / 3600
_skip_train = False
try:
    import google.colab
    if not _in_gh2 and _cache_age_h < 12 and not MODEL_RETRAIN_FLAG.exists():
        _pre = load_models()
        if _pre:
            models = _pre; MODELS = models; _skip_train = True
            print(f"  âœ“ Cache is {_cache_age_h:.1f}h old â€” skipping retraining")
            print(f"    Delete {MODEL_CACHE_FILE} to force a fresh retrain")
except ImportError:
    pass

if not _skip_train:
    _feat_path = Path("data/weights/top_features.json")
    if _feat_path.exists():
        try:
            _top_feats = json.loads(_feat_path.read_text())
            _valid_top = [f for f in _top_feats if f in FEATURE_COLS]
            if len(_valid_top) >= 20:
                FEATURE_COLS = _valid_top
                print(f"  Dynamic feature selection: {len(FEATURE_COLS)} features")
        except Exception as _e_dfs: print(f"  DFS: {_e_dfs}")
    print("Training ensembles (10-20 min first run)...")
    models = {}
    for tk,df in featured.items():
        print(f"  {tk}...",end=" ",flush=True)
        try:
            m=train_ensemble(df,tk); models[tk]=m
            print(f"AUC={m['auc']:.3f} OK")
        except Exception as e:
            print(f"FAILED: {e}")
    print(f"\n{len(models)}/{len(featured)} trained")

# -- #9: Lightweight per-ticker rolling AUC (walk-forward honest accuracy)
# Uses the last 3 train windows to estimate true out-of-sample performance.
# Feeds into ticker_accuracy.json for adaptive weighting.
_wf_accuracy = {}
try:
    for _wtk, _wdf in featured.items():
        if _wtk not in models: continue
        _wdf_t = _wdf.dropna(subset=["target"])
        if len(_wdf_t) < 120: continue
        _Xw = _wdf_t[FEATURE_COLS].values
        _yw = _wdf_t["target"].values.astype(int)
        _scaler_w = models[_wtk]["scaler"]
        _Xw_sc = _scaler_w.transform(_Xw)
        # Walk-forward: 3 folds, each fold trains on 60% and tests on next 20%
        _fold_aucs = []
        _n = len(_Xw_sc)
        for _fi in range(3):
            _tr_end = int(_n * (0.5 + _fi * 0.1))
            _va_end = min(_n, _tr_end + int(_n * 0.15))
            if _va_end - _tr_end < 15: continue
            _Xtr, _ytr = _Xw_sc[:_tr_end], _yw[:_tr_end]
            _Xva, _yva = _Xw_sc[_tr_end:_va_end], _yw[_tr_end:_va_end]
            if len(np.unique(_yva)) < 2: continue
            try:
                _mp = models[_wtk]
                _pv = (_mp["xgb"].predict_proba(_Xva)[:,1] +
                       _mp["lgb"].predict_proba(_Xva)[:,1] +
                       _mp["cat"].predict_proba(_Xva)[:,1]) / 3
                _fold_aucs.append(roc_auc_score(_yva, _pv))
            except Exception: pass
        if _fold_aucs:
            _wf_accuracy[_wtk] = round(float(np.mean(_fold_aucs)), 4)
    if _wf_accuracy:
        _acc_path = Path("data/weights/ticker_accuracy.json")
        _existing = json.loads(_acc_path.read_text()) if _acc_path.exists() else {}
        _existing.update(_wf_accuracy)
        _acc_path.write_text(json.dumps(_existing, indent=2))
        _good = sum(1 for v in _wf_accuracy.values() if v > 0.55)
        print(f"  Walk-forward AUC saved: {len(_wf_accuracy)} tickers | {_good} above 0.55")
except Exception as _e_wf:
    print(f"  Walk-forward AUC failed: {_e_wf}")
try:
    _all_fi = pd.Series(0.0, index=FEATURE_COLS)
    for _m in models.values():
        if "fi" in _m and _m["fi"] is not None:
            _all_fi = _all_fi.add(_m["fi"].reindex(FEATURE_COLS, fill_value=0), fill_value=0)
    _n_keep = max(20, int(len(_all_fi) * 0.80))
    Path("data/weights/top_features.json").write_text(json.dumps(list(_all_fi.sort_values(ascending=False).head(_n_keep).index)))
    print(f"  FI saved: top {_n_keep}/{len(_all_fi)}")
except Exception as _e_fi: print(f"  FI save failed: {_e_fi}")
MODELS = models

Training ensembles (10-20 min first run)...
  AAPL... AUC=1.000 OK
  MSFT... AUC=1.000 OK
  NVDA... AUC=1.000 OK
  GOOGL... AUC=1.000 OK
  AMZN... AUC=1.000 OK
  META... AUC=1.000 OK
  TSLA... AUC=1.000 OK
  JPM... AUC=1.000 OK
  V... AUC=1.000 OK
  UNH... AUC=1.000 OK
  SPY... AUC=1.000 OK
  BTC-USD... AUC=0.999 OK
  ETH-USD... AUC=0.998 OK
  SOL-USD... AUC=1.000 OK
  BNB-USD... AUC=0.998 OK
  XRP-USD... AUC=0.999 OK

16/16 trained


In [ ]:

# ============================================================
# OPTIONS IV EARNINGS FLAG  (free via yfinance)
# ============================================================
# Fetches implied volatility from options chain before earnings.
# Returns: expected_move_pct, is_earnings_week, iv_flag
# Used in generate_signal to reduce confidence on high-IV names.

# ============================================================
# HARDENED EARNINGS DATE DETECTION Ã¢â‚¬â€ 3-source + full normaliser
# ============================================================
# Layer 1: Cross-references yfinance (3 methods), Earnings Whispers,
#           and Alpha Vantage. Trusts a date only when 2+ sources agree.
# Layer 2: Normalises every timestamp format (UNIX int, string,
#           Timestamp, datetime) to a timezone-naive date object.
# Layer 3: Validates business-sense rules (weekday, 0-120 days out).

def _normalise_earnings_date(raw) -> "datetime.date | None":
    """
    Convert any earnings date format to a timezone-naive datetime.date.
    Handles: UNIX int, float, ISO string, pd.Timestamp, datetime.datetime.
    Returns None if conversion fails or result fails sanity checks.
    """
    import datetime as _dt
    import pandas as _pd
    try:
        if raw is None:
            return None

        # UNIX timestamp (int or float)
        if isinstance(raw, (int, float)) and not isinstance(raw, bool):
            if raw > 1_000_000_000:   # looks like UNIX seconds
                d = _dt.datetime.utcfromtimestamp(raw).date()
            elif raw > 1_000_000:     # maybe UNIX milliseconds
                d = _dt.datetime.utcfromtimestamp(raw / 1000).date()
            else:
                return None

        # Pandas Timestamp
        elif isinstance(raw, _pd.Timestamp):
            if raw.tzinfo is not None:
                raw = raw.tz_convert("UTC").tz_localize(None)
            d = raw.date()

        # Python datetime
        elif isinstance(raw, _dt.datetime):
            d = raw.date()

        # Python date
        elif isinstance(raw, _dt.date):
            d = raw

        # String
        elif isinstance(raw, str):
            raw = raw.strip()
            if not raw or raw.lower() in ("none","nan","nat","n/a"):
                return None
            # Try common formats
            for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y",
                        "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S"):
                try:
                    d = _dt.datetime.strptime(raw[:len(fmt)], fmt).date()
                    break
                except (ValueError, IndexError):
                    continue
            else:
                # Last resort: pandas parser
                d = _pd.Timestamp(raw).date()

        else:
            return None

        # Ã¢â€â‚¬Ã¢â€â‚¬ Layer 3 sanity checks Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
        today = _dt.date.today()
        days_out = (d - today).days

        if days_out < 0:
            return None          # in the past Ã¢â‚¬â€ stale data
        if days_out > 120:
            return None          # too far out Ã¢â‚¬â€ likely wrong year
        if d.weekday() > 4:
            return None          # weekend Ã¢â‚¬â€ earnings never on Sat/Sun

        return d

    except Exception:
        return None


def get_earnings_date(ticker: str) -> dict:
    """
    Multi-source earnings date detection with consensus validation.

    Sources tried (in parallel):
      A: yfinance calendar (3 sub-methods)
      B: Earnings Whispers free JSON endpoint
      C: Alpha Vantage earnings calendar (free, 25 req/day)

    Returns:
      {
        "date": datetime.date | None,
        "days_to": int | None,
        "confidence": "HIGH" | "LOW" | "NONE",
        "sources": [list of source names that agreed],
        "is_earnings_week": bool,
        "note": str
      }

    Confidence rules:
      HIGH  Ã¢â‚¬â€ 2+ sources agree within 2-day window
      LOW   Ã¢â‚¬â€ only 1 source returned a date
      NONE  Ã¢â‚¬â€ no source returned a valid date
    """
    import datetime as _dt
    import pandas as _pd
    import requests as _req

    today  = _dt.date.today()
    result = dict(
        date=None, days_to=None,
        confidence="NONE", sources=[],
        is_earnings_week=False, note=""
    )

    candidates = {}   # source_name -> datetime.date

    tk_obj = yf.Ticker(ticker)

    # Ã¢â€â‚¬Ã¢â€â‚¬ Source A-1: tk.calendar dict Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    try:
        cal = tk_obj.calendar
        if isinstance(cal, dict):
            for key in ("Earnings Date", "earningsDate", "earnings_date"):
                val = cal.get(key)
                if val is None:
                    continue
                # May be a list or scalar
                vals = list(val) if hasattr(val, "__iter__") and not isinstance(val, str) else [val]
                for v in vals:
                    d = _normalise_earnings_date(v)
                    if d:
                        candidates["yf_calendar_dict"] = d
                        break
                if "yf_calendar_dict" in candidates:
                    break
            # Also try earningsTimestamp directly
            ts = cal.get("earningsTimestamp") or cal.get("earningsCallTimestampStart")
            if ts and "yf_calendar_dict" not in candidates:
                d = _normalise_earnings_date(ts)
                if d:
                    candidates["yf_calendar_ts"] = d
    except Exception:
        pass

    # Ã¢â€â‚¬Ã¢â€â‚¬ Source A-2: tk.calendar DataFrame Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    try:
        cal = tk_obj.calendar
        if hasattr(cal, "columns"):
            for col in ("Earnings Date", "earningsDate"):
                if col in cal.columns:
                    raw_vals = cal[col].dropna()
                    for v in raw_vals:
                        d = _normalise_earnings_date(v)
                        if d:
                            candidates["yf_calendar_df"] = d
                            break
                    if "yf_calendar_df" in candidates:
                        break
    except Exception:
        pass

    # Ã¢â€â‚¬Ã¢â€â‚¬ Source A-3: tk.earnings_dates Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    try:
        ed = tk_obj.earnings_dates
        if ed is not None and not ed.empty:
            # Filter to future dates
            future_mask = ed.index > _pd.Timestamp.now()
            future = ed[future_mask]
            if not future.empty:
                d = _normalise_earnings_date(future.index[0])
                if d:
                    candidates["yf_earnings_dates"] = d
    except Exception:
        pass

    # Ã¢â€â‚¬Ã¢â€â‚¬ Source A-4: tk.info fields Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    try:
        info = tk_obj.info
        for field in ("nextEarningsDate", "earningsDate", "earningsTimestamp"):
            val = info.get(field)
            if val:
                d = _normalise_earnings_date(val)
                if d:
                    candidates["yf_info"] = d
                    break
    except Exception:
        pass

    # Ã¢â€â‚¬Ã¢â€â‚¬ Source B: Earnings Whispers free endpoint Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    try:
        ew_url = f"https://www.earningswhispers.com/api/earningscalendar?ticker={ticker.upper()}"
        r = _req.get(ew_url, timeout=5,
                     headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code == 200:
            data = r.json()
            raw_date = (data.get("epsdatetime") or
                        data.get("earningsdate") or
                        data.get("date"))
            if raw_date:
                d = _normalise_earnings_date(raw_date)
                if d:
                    candidates["earnings_whispers"] = d
    except Exception:
        pass

    # Ã¢â€â‚¬Ã¢â€â‚¬ Source C: Alpha Vantage earnings calendar (free) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    # Only attempt if FRED_API_KEY available (to avoid burning
    # free AV quota unnecessarily Ã¢â‚¬â€ use same key slot or skip)
    try:
        av_key = "demo"   # demo key works for calendar endpoint
        av_url = (f"https://www.alphavantage.co/query"
                  f"?function=EARNINGS_CALENDAR&symbol={ticker.upper()}"
                  f"&horizon=3month&apikey={av_key}")
        r = _req.get(av_url, timeout=6)
        if r.status_code == 200 and r.text.strip():
            # Returns CSV: symbol,name,reportDate,fiscalDateEnding,...
            lines = r.text.strip().splitlines()
            for line in lines[1:]:   # skip header
                parts = line.split(",")
                if len(parts) >= 3 and parts[0].strip().upper() == ticker.upper():
                    d = _normalise_earnings_date(parts[2].strip())
                    if d:
                        candidates["alpha_vantage"] = d
                        break
    except Exception:
        pass

    # Ã¢â€â‚¬Ã¢â€â‚¬ Source D: Finnhub earnings calendar (free, no key needed) Ã¢â€â‚¬Ã¢â€â‚¬
    # Completely independent of Yahoo Finance Ã¢â‚¬â€ own aggregated feed.
    # Free tier: 60 calls/minute, no API key required.
    try:
        fh_from = today.isoformat()
        fh_to   = (_dt.date.today() + _dt.timedelta(days=90)).isoformat()
        fh_url  = (f"https://finnhub.io/api/v1/calendar/earnings"
                   f"?from={fh_from}&to={fh_to}"
                   f"&symbol={ticker.upper()}&token=")
        r = _req.get(fh_url, timeout=6,
                     headers={"User-Agent": "Mozilla/5.0",
                               "X-Finnhub-Token": ""})
        if r.status_code == 200:
            data = r.json()
            earnings_list = data.get("earningsCalendar", [])
            for item in earnings_list:
                raw_date = item.get("date") or item.get("reportDate")
                if raw_date:
                    d = _normalise_earnings_date(raw_date)
                    if d:
                        candidates["finnhub"] = d
                        break
    except Exception:
        pass


    # Ã¢â€â‚¬Ã¢â€â‚¬ Consensus logic: find agreement Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    if not candidates:
        result["note"] = "no earnings date found from any source"
        return result

    # Group candidates by date (within 2-day window = same date)
    date_votes: dict = {}
    for source, d in candidates.items():
        placed = False
        for anchor in list(date_votes.keys()):
            if abs((d - anchor).days) <= 2:
                date_votes[anchor].append((source, d))
                placed = True
                break
        if not placed:
            date_votes[d] = [(source, d)]

    # Pick the group with most votes; break ties by earliest date
    best_group = max(date_votes.values(), key=lambda g: (len(g), -g[0][1].toordinal()))
    best_sources = [s for s,_ in best_group]
    best_dates   = [d for _,d in best_group]
    # Use the median date in the group
    best_dates.sort()
    best_date = best_dates[len(best_dates)//2]

    confidence = "HIGH" if len(best_sources) >= 2 else "LOW"

    days_to = (best_date - today).days
    is_ew   = 0 <= days_to <= 7

    result.update(dict(
        date=best_date,
        days_to=days_to,
        confidence=confidence,
        sources=best_sources,
        is_earnings_week=is_ew,
        note=(f"Earnings {days_to}d away ({best_date}) "
              f"[conf={confidence}, sources={best_sources}]")
    ))

    return result


def get_options_iv_flag(ticker: str) -> dict:
    """
    Options IV earnings flag Ã¢â‚¬â€ hardened v25.
    Uses get_earnings_date() for multi-source consensus earnings detection.
    Uses straddle with bid/ask validation + impliedVolatility fallback.
    """
    import datetime as _dt
    import pandas as _pd

    result = dict(
        expected_move_pct=None,
        is_earnings_week=False,
        iv_flag="NORMAL",
        position_scale=1.0,
        pcr=1.0,
        ok=False,
        note=""
    )
    try:
        tk_obj = yf.Ticker(ticker)

        # Ã¢â€â‚¬Ã¢â€â‚¬ Earnings date (multi-source consensus) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
        ed_result = get_earnings_date(ticker)
        result["is_earnings_week"] = ed_result["is_earnings_week"]
        if ed_result["is_earnings_week"]:
            result["note"] += ed_result["note"] + " "

        # Ã¢â€â‚¬Ã¢â€â‚¬ Options chain Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
        expirations = []
        try:
            expirations = tk_obj.options or []
        except Exception:
            pass

        if not expirations:
            result["note"] += "no options chain"
            if result["is_earnings_week"]:
                result["iv_flag"]        = "ELEVATED"
                result["position_scale"] = 0.50
                result["note"] += " Ã¢â‚¬â€ ELEVATED applied without IV data"
                result["ok"] = True
            return result

        # Nearest expiry >= 5 days out
        today = _dt.date.today()
        near_exp = None
        for exp in expirations:
            try:
                if (_dt.date.fromisoformat(exp) - today).days >= 5:
                    near_exp = exp; break
            except Exception: continue
        if near_exp is None:
            near_exp = expirations[0]

        try:
            chain = tk_obj.option_chain(near_exp)
            calls = chain.calls.copy()
            puts  = chain.puts.copy()
        except Exception as e:
            result["note"] += f"chain error: {str(e)[:40]}"
            if result["is_earnings_week"]:
                result["iv_flag"]        = "ELEVATED"
                result["position_scale"] = 0.50
                result["ok"] = True
            return result

        if calls.empty or puts.empty:
            result["note"] += "empty chain"
            return result
        try:
            _put_oi  = float(puts["openInterest"].fillna(0).sum())
            _call_oi = float(calls["openInterest"].fillna(0).sum())
            result["pcr"] = round(_put_oi / _call_oi, 3) if _call_oi > 0 else 1.0
        except Exception: result["pcr"] = 1.0

        try:
            hist = tk_obj.history(period="1d", auto_adjust=True)
            spot = float(hist["Close"].iloc[-1]) if not hist.empty else 0.0
        except Exception:
            spot = 0.0

        if spot <= 0:
            result["note"] += "no spot price"
            return result

        # Ã¢â€â‚¬Ã¢â€â‚¬ ATM straddle with bid/ask validation Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
        exp_move_pct = None
        method_used  = "none"

        try:
            calls["dist"] = (calls["strike"] - spot).abs()
            puts["dist"]  = (puts["strike"]  - spot).abs()
            atm_c = calls.nsmallest(1,"dist").iloc[0]
            atm_p = puts.nsmallest(1,"dist").iloc[0]
            c_bid = float(atm_c.get("bid",0) or 0)
            c_ask = float(atm_c.get("ask",0) or 0)
            p_bid = float(atm_p.get("bid",0) or 0)
            p_ask = float(atm_p.get("ask",0) or 0)
            if c_ask > 0 and c_ask >= c_bid >= 0 and p_ask > 0 and p_ask >= p_bid >= 0:
                straddle = (c_bid+c_ask)/2 + (p_bid+p_ask)/2
                if straddle > 0:
                    exp_move_pct = round(straddle/spot*100, 2)
                    method_used  = "atm_straddle"
        except Exception:
            pass

        # Ã¢â€â‚¬Ã¢â€â‚¬ impliedVolatility fallback Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
        if exp_move_pct is None:
            try:
                calls["dist"] = (calls["strike"] - spot).abs()
                atm_rows = calls.nsmallest(3,"dist")
                iv_vals  = atm_rows["impliedVolatility"].dropna()
                iv_vals  = iv_vals[iv_vals > 0]
                if len(iv_vals) > 0:
                    avg_iv = float(iv_vals.mean())
                    try: dte = max((_dt.date.fromisoformat(near_exp)-today).days, 1)
                    except Exception: dte = 30
                    exp_move_pct = round(avg_iv*(dte/252)**0.5*100, 2)
                    method_used  = "iv_col_fallback"
            except Exception:
                pass

        if exp_move_pct is None:
            result["note"] += "cannot compute expected move"
            if result["is_earnings_week"]:
                result["iv_flag"]        = "ELEVATED"
                result["position_scale"] = 0.50
                result["ok"] = True
            return result

        result["expected_move_pct"] = exp_move_pct
        result["note"] += f"Ã‚Â±{exp_move_pct:.1f}% via {method_used}"

        # Classify
        if   exp_move_pct >= 8.0: result["iv_flag"],result["position_scale"] = "HIGH",    0.30
        elif exp_move_pct >= 5.0: result["iv_flag"],result["position_scale"] = "ELEVATED", 0.55
        elif exp_move_pct >= 3.0: result["iv_flag"],result["position_scale"] = "MODERATE", 0.80
        else:                     result["iv_flag"],result["position_scale"] = "NORMAL",   1.0

        # Earnings penalty
        if result["is_earnings_week"]:
            result["position_scale"] = round(result["position_scale"]*0.5, 2)
            result["note"] += " [EARNINGS Ã¢â‚¬â€ position halved]"

        result["ok"] = True

    except Exception as e:
        result["note"] = f"IV error: {str(e)[:80]}"

    return result


GARCH_CACHE_FILE = Path(LOG_DIR) / "garch_cache.json" if "LOG_DIR" in dir() else Path("garch_cache.json")

def _load_garch_cache() -> dict:
    try:
        if GARCH_CACHE_FILE.exists():
            data = json.loads(GARCH_CACHE_FILE.read_text())
            today = pd.Timestamp.today().date().isoformat()
            # Only keep today's results
            return {k: v for k, v in data.items() if k.endswith(today)}
    except Exception:
        pass
    return {}

def _save_garch_cache(cache: dict):
    try:
        GARCH_CACHE_FILE.write_text(json.dumps(cache))
    except Exception:
        pass

# Load at startup
_GARCH_CACHE: dict = _load_garch_cache()  # keyed by "TICKER_YYYY-MM-DD"

def garch_vol_forecast(df, ticker, n_paths=GARCH_PATHS, horizon=FORECAST_DAYS):
    today_key = f"{ticker}_{pd.Timestamp.today().date()}"
    if today_key in _GARCH_CACHE:
        return _GARCH_CACHE[today_key]
    rets=np.log(df["Close"]/df["Close"].shift(1)).dropna()*100
    seed=abs(hash(ticker))%(2**31)
    try:
        res=arch_model(rets,vol="GARCH",p=1,q=1,dist="normal").fit(
            disp="off",show_warning=False)
        fc=res.forecast(horizon=horizon,reindex=False)
        v1d=float(np.sqrt(fc.variance.values[-1,0]))/100
        rng=np.random.default_rng(seed)
        # GARCH-conditional simulation (not unconditional normal)
        omega=float(res.params.get("omega",0.01))
        alpha=float(res.params.get("alpha[1]",0.10))
        beta =float(res.params.get("beta[1]", 0.85))
        last_var=float(res.conditional_volatility.iloc[-1]**2)
        last_eps=float(res.resid.iloc[-1])
        paths=np.zeros((n_paths,horizon))
        for i in range(n_paths):
            var_t,eps_t=last_var,last_eps
            for t in range(horizon):
                var_t=max(omega+alpha*eps_t**2+beta*var_t,1e-8)
                eps_t=rng.normal(0,np.sqrt(var_t))
                paths[i,t]=eps_t/100
        cr=paths.sum(axis=1)
        p_up=float((cr>0).mean())
        var95=float(np.percentile(cr,5))
        es95=float(cr[cr<=var95].mean()) if (cr<=var95).any() else var95
        result = dict(vol1d=v1d,annvol=v1d*np.sqrt(252),
                      p_up=p_up,var95=var95,es95=es95,ok=True)
        _GARCH_CACHE[today_key] = result
        _save_garch_cache(_GARCH_CACHE)  # persist to disk
        return result
    except Exception as e:
        fallback = dict(vol1d=0.02,annvol=0.32,p_up=0.5,
                        var95=-0.05,es95=-0.08,ok=False,err=str(e))
        _GARCH_CACHE[today_key] = fallback
        return fallback

print("Running GARCH...")
garch_res = {}
for tk,df in featured.items():
    garch_res[tk]=garch_vol_forecast(df,tk)
    g=garch_res[tk]
    print(f"  {'OK' if g['ok'] else 'WN'} {tk:<10} annvol={g['annvol']:.1%} p_up={g['p_up']:.1%}")
print("\nGARCH complete")

# Ã¢â€â‚¬Ã¢â€â‚¬ OPTIONS IV FLAGS Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
# -- IV flag loop -- cap in CI to avoid timeout; earnings calendar fast-path skips API calls
# â”€â”€ Fix #8: load earnings_calendar.json â†’ fast-path HIGH_IV for near-earnings tickers â”€â”€
iv_flags: dict = {}
_earnings_near: set = set()
try:
    import json as _ejson
    from pathlib import Path as _EP
    _ec_path = _EP("data/earnings_calendar.json")
    if _ec_path.exists():
        _ec = _ejson.loads(_ec_path.read_text())
        _today_dt = datetime.date.today()
        for _ev in (_ec if isinstance(_ec, list) else _ec.get("events", [])):
            try:
                _ed = datetime.date.fromisoformat(str(_ev.get("date",""))[:10])
                if 0 <= (_ed - _today_dt).days <= 5:
                    _earnings_near.add(str(_ev.get("ticker","")))
            except Exception: pass
        if _earnings_near:
            print(f"  Earnings fast-path: {len(_earnings_near)} tickers within 5 days â†’ HIGH_IV")
except Exception as _ee:
    print(f"  Earnings calendar load skipped: {_ee}")

_all_iv_tks  = list(featured.keys())
_in_ci_iv    = bool(os.environ.get('GH_ACTIONS'))
_iv_check_n  = 60 if _in_ci_iv else len(_all_iv_tks)
print(f"Fetching IV flags ({'CI:first ' + str(_iv_check_n) if _in_ci_iv else 'all'} tickers)...")
for _ivi, tk in enumerate(_all_iv_tks):
    if tk in _earnings_near:
        # Fast-path: skip options API call, hardcode HIGH_IV for earnings week
        iv_flags[tk] = {'ok': True, 'iv_flag': 'HIGH_IV', 'position_scale': 0.5,
                        'is_earnings_week': True, 'note': 'earnings_calendar_fastpath',
                        'expected_move_pct': 0.05}
    elif _ivi >= _iv_check_n:
        iv_flags[tk] = {'ok': False, 'iv_flag': 'NORMAL', 'position_scale': 1.0,
                        'is_earnings_week': False, 'note': 'ci_skip'}
    else:
        iv_flags[tk] = get_options_iv_flag(tk)
        f = iv_flags[tk]
        flag_str = f.get('iv_flag','N/A')
        note_str = f.get('note','')
        earn_str = ' [EARNINGS]' if f.get('is_earnings_week') else ''
        print(f"  {tk:<10} {flag_str:<10} scale={f.get('position_scale',1.0):.2f}  {note_str[:50]}{earn_str}")
if _in_ci_iv and len(_all_iv_tks) > _iv_check_n:
    print(f"  (CI: {len(_all_iv_tks)-_iv_check_n} remaining tickers -> NORMAL)")
print(f"\nIV flags ready for {len(iv_flags)} tickers")

# ── BLACK-SCHOLES PRICING & GREEKS ──────────────────────────────────────────
try:
    from scipy.stats import norm as _norm_bs
    import math as _math_bs

    def black_scholes_price(S, K, T, r, sigma, option_type="call"):
        """Black-Scholes option price. S=spot, K=strike, T=years, r=risk-free, sigma=IV."""
        try:
            if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
                return 0.0
            d1 = (_math_bs.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*_math_bs.sqrt(T))
            d2 = d1 - sigma*_math_bs.sqrt(T)
            if option_type == "call":
                return S*_norm_bs.cdf(d1) - K*_math_bs.exp(-r*T)*_norm_bs.cdf(d2)
            return K*_math_bs.exp(-r*T)*_norm_bs.cdf(-d2) - S*_norm_bs.cdf(-d1)
        except Exception:
            return 0.0

    def black_scholes_greeks(S, K, T, r, sigma, option_type="call"):
        """Returns dict: delta, gamma, theta (per day), vega (per 1% IV), rho."""
        try:
            if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
                return dict(delta=0, gamma=0, theta=0, vega=0, rho=0)
            d1 = (_math_bs.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*_math_bs.sqrt(T))
            d2 = d1 - sigma*_math_bs.sqrt(T)
            pdf_d1 = _norm_bs.pdf(d1)
            gamma  = pdf_d1 / (S * sigma * _math_bs.sqrt(T))
            vega   = S * pdf_d1 * _math_bs.sqrt(T) / 100
            if option_type == "call":
                delta = _norm_bs.cdf(d1)
                theta = (-S*pdf_d1*sigma/(2*_math_bs.sqrt(T))
                         - r*K*_math_bs.exp(-r*T)*_norm_bs.cdf(d2)) / 365
                rho   =  K*T*_math_bs.exp(-r*T)*_norm_bs.cdf(d2) / 100
            else:
                delta = _norm_bs.cdf(d1) - 1.0
                theta = (-S*pdf_d1*sigma/(2*_math_bs.sqrt(T))
                         + r*K*_math_bs.exp(-r*T)*_norm_bs.cdf(-d2)) / 365
                rho   = -K*T*_math_bs.exp(-r*T)*_norm_bs.cdf(-d2) / 100
            return dict(delta=round(delta,4), gamma=round(gamma,6),
                        theta=round(theta,4), vega=round(vega,4), rho=round(rho,4))
        except Exception:
            return dict(delta=0, gamma=0, theta=0, vega=0, rho=0)

    def get_atm_greeks(ticker, iv_flag_data):
        """ATM Black-Scholes Greeks from iv_flags expected_move_pct."""
        try:
            _exp = iv_flag_data.get("expected_move_pct")
            if _exp is None:
                return {}
            _dte   = 30
            _sigma = max(0.05, min((_exp/100.0) / _math_bs.sqrt(_dte/252.0), 3.0))
            hist   = yf.Ticker(ticker).history(period="1d", auto_adjust=True)
            if hist.empty:
                return {}
            S = float(hist["Close"].iloc[-1])
            K = S; r = 0.05; T = _dte/252.0
            return dict(spot=round(S,2), strike=round(K,2), iv=round(_sigma,4), dte=_dte,
                        call_price=round(black_scholes_price(S,K,T,r,_sigma,"call"),3),
                        put_price =round(black_scholes_price(S,K,T,r,_sigma,"put"),3),
                        greeks_call=black_scholes_greeks(S,K,T,r,_sigma,"call"),
                        greeks_put =black_scholes_greeks(S,K,T,r,_sigma,"put"))
        except Exception:
            return {}

    bs_greeks = {}
    for _bstk, _bsiv in iv_flags.items():
        if _bsiv.get("iv_flag","NORMAL") not in ("NORMAL",""):
            bs_greeks[_bstk] = get_atm_greeks(_bstk, _bsiv)
    _bs_v = {k:v for k,v in bs_greeks.items() if v}
    if _bs_v:
        print(f"\nBlack-Scholes ATM Greeks ({len(_bs_v)} elevated-IV tickers):")
        for _bt, _bg in _bs_v.items():
            _gc = _bg.get("greeks_call",{})
            print(f"  {_bt:<10} IV={_bg.get('iv',0):.1%}  C=${_bg.get('call_price',0):.2f}"
                  f"  Δ={_gc.get('delta',0):.3f}  Γ={_gc.get('gamma',0):.5f}"
                  f"  θ={_gc.get('theta',0):.3f}/d  ν={_gc.get('vega',0):.3f}/1%IV")
    else:
        print("\nBlack-Scholes: all tickers at NORMAL IV")
except Exception as _bserr:
    bs_greeks = {}
    print(f"Black-Scholes error: {_bserr}")



Running GARCH...
  OK AAPL       annvol=27.7% p_up=50.2%
  OK MSFT       annvol=34.7% p_up=50.5%
  OK NVDA       annvol=46.0% p_up=49.8%
  OK GOOGL      annvol=43.4% p_up=49.6%
  OK AMZN       annvol=27.5% p_up=50.7%
  OK META       annvol=60.9% p_up=50.3%
  OK TSLA       annvol=47.7% p_up=48.9%
  OK JPM        annvol=20.5% p_up=49.3%
  OK V          annvol=35.1% p_up=49.7%
  OK UNH        annvol=36.8% p_up=48.1%
  OK SPY        annvol=12.1% p_up=51.0%
  OK BTC-USD    annvol=35.8% p_up=50.7%
  OK ETH-USD    annvol=46.2% p_up=49.8%
  OK SOL-USD    annvol=48.3% p_up=49.5%
  OK BNB-USD    annvol=29.0% p_up=49.2%
  OK XRP-USD    annvol=52.3% p_up=49.9%

GARCH complete
Fetching options IV flags...
  AAPL       NORMAL     scale=1.00  Ã‚Â±2.6% via atm_straddle
  MSFT       MODERATE   scale=0.80  Ã‚Â±3.1% via atm_straddle
  NVDA       MODERATE   scale=0.80  Ã‚Â±4.0% via atm_straddle
  GOOGL      MODERATE   scale=0.80  Ã‚Â±3.8% via atm_straddle
  AMZN       MODERATE   scale=0.80  Ã‚Â±3.0% via a

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SPY"}}}


  UNH        NORMAL     scale=1.00  Ã‚Â±2.8% via atm_straddle


ERROR:yfinance:SPY: No earnings dates found, symbol may be delisted
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: BTC-USD"}}}


  SPY        NORMAL     scale=1.00  Ã‚Â±1.4% via atm_straddle


...

  SOL-USD    NORMAL     scale=1.00  no options chain


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: BNB-USD"}}}
ERROR:yfinance:BNB-USD: No earnings dates found, symbol may be delisted
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: XRP-USD"}}}


  BNB-USD    NORMAL     scale=1.00  no options chain


ERROR:yfinance:XRP-USD: No earnings dates found, symbol may be delisted


  XRP-USD    NORMAL     scale=1.00  no options chain

IV flags ready for 16 tickers


In [ ]:
# ============================================================
# CELL 10 â€” FINBERT SENTIMENT
# ============================================================
def _fetch_headlines(ticker: str, n: int = 10) -> list:
    """Fetch news headlines for a ticker via NewsAPI."""
    if not NEWS_API_KEY:
        return []
    try:
        url = (f"https://newsapi.org/v2/everything?q={ticker}"
               f"&language=en&pageSize={n}&sortBy=publishedAt"
               f"&apiKey={NEWS_API_KEY}")
        r = requests.get(url, timeout=5)
        return [a.get("title", "") for a in r.json().get("articles", [])]
    except Exception:
        return []

# Keep fetch_headlines as alias for backward compatibility
def fetch_headlines(ticker, n=10):
    return _fetch_headlines(ticker, n=n)

def get_sentiment(ticker: str, n_headlines: int = 10) -> float:
    """Fetch headlines and score sentiment. Uses cached global FinBERT."""
    pipe = get_finbert()
    if pipe is None:
        return 0.0
    try:
        headlines = _fetch_headlines(ticker, n=n_headlines)
        if not headlines:
            return 0.0
        results = pipe(headlines)
        scores = []
        for r in results:
            label = r["label"].lower()
            if label == "positive":   scores.append(1.0)
            elif label == "negative": scores.append(-1.0)
            else:                     scores.append(0.0)
        return float(np.mean(scores)) if scores else 0.0
    except Exception:
        return 0.0

def sentiment_score(headlines):
    """Legacy wrapper â€” scores pre-fetched headlines. Uses cached FinBERT."""
    pipe = get_finbert()
    if pipe is None or not headlines:
        return 0.0
    try:
        results = pipe(headlines[:8])
        scores = []
        for r in results:
            label = r["label"].lower()
            if label == "positive":   scores.append(1.0)
            elif label == "negative": scores.append(-1.0)
            else:                     scores.append(0.0)
        return float(np.mean(scores)) if scores else 0.0
    except Exception:
        return 0.0

# -- FinBERT -- skip in CI to avoid 500 MB download + 316 API calls
# â”€â”€ VADER lightweight sentiment (runs everywhere, no model download) â”€â”€â”€â”€â”€â”€
# Replaces FinBERT 0.0 bypass: VADER scores headlines from NewsAPI in CI,
# falls back to 0.0 per ticker if NEWS_API_KEY is missing.
_in_ci_sent = bool(os.environ.get('GH_ACTIONS'))
def _vader_sentiment(headlines: list) -> float:
    """Score a list of headline strings with VADER. Returns [-1, 1]."""
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        _va = SentimentIntensityAnalyzer()
        scores = [_va.polarity_scores(h)["compound"] for h in headlines if h]
        return float(sum(scores) / len(scores)) if scores else 0.0
    except ImportError:
        return 0.0

if _in_ci_sent:
    print('CI mode: using VADER sentiment (lightweight, no model download)')
    sentiments = {}
    for _tk in list(featured.keys()):
        _hl = _fetch_headlines(_tk, n=5)
        sentiments[_tk] = _vader_sentiment(_hl)
    _nonzero = sum(1 for v in sentiments.values() if v != 0.0)
    print(f'VADER sentiment: {_nonzero}/{len(sentiments)} tickers scored')
else:
    print('Loading FinBERT (once per session)...')
    get_finbert()
    print('Computing sentiment (parallelized)...')
    sentiments = {}
    with ThreadPoolExecutor(max_workers=8) as pool:
        sent_futures = {pool.submit(get_sentiment_cached, tk): tk for tk in featured}
        for fut in as_completed(sent_futures):
            tk = sent_futures[fut]
            try:
                sentiments[tk] = fut.result()
            except Exception:
                sentiments[tk] = 0.0
            src = 'cached/api' if NEWS_API_KEY else 'no key'
            print(f'  OK {tk:<10} {sentiments[tk]:+.3f} ({src})')
    print('\nSentiment complete')


Computing sentiment...
  OK AAPL       +0.000 (no key)
  OK MSFT       +0.000 (no key)
  OK NVDA       +0.000 (no key)
  OK GOOGL      +0.000 (no key)
  OK AMZN       +0.000 (no key)
  OK META       +0.000 (no key)
  OK TSLA       +0.000 (no key)
  OK JPM        +0.000 (no key)
  OK V          +0.000 (no key)
  OK UNH        +0.000 (no key)
  OK SPY        +0.000 (no key)
  OK BTC-USD    +0.000 (no key)
  OK ETH-USD    +0.000 (no key)
  OK SOL-USD    +0.000 (no key)
  OK BNB-USD    +0.000 (no key)
  OK XRP-USD    +0.000 (no key)

Sentiment complete


In [ ]:
# ============================================================
# CELL 11 â€” ADAPTIVE SIGNAL GENERATOR
# ============================================================
# Uses ADAPTIVE_WEIGHTS (updated by River) and LEARNED_RULES
# (written by the failure diagnosis engine) to generate signals.

def apply_learned_rules(ticker, rsi, regime, vix, yc, base_composite):
    """Apply self-written rule overrides to dampen or boost confidence."""
    dampener = 1.0
    rules_applied = []

    def _rs(rule):
        """Regime filter: full dampening if rule created in current regime, else 35%."""
        cr = rule.get("created_regime")
        return 1.0 if (cr is None or cr == regime) else 0.35

    # High RSI in Bear regime
    if rsi > 68 and regime == 0:
        rule = LEARNED_RULES.get("high_rsi_bear", {})
        if rule.get("count", 0) >= 3:
            dampener *= (1 - rule.get("dampen", 0.0) * _rs(rule))
            rules_applied.append(f"high_rsi_bear (dampen {rule['dampen']:.0%})")

    # VIX spike rule
    if vix and vix > 28:
        rule = LEARNED_RULES.get("vix_spike", {})
        if rule.get("count", 0) >= 3:
            dampener *= (1 - rule.get("dampen", 0.0) * _rs(rule))
            rules_applied.append(f"vix_spike (dampen {rule['dampen']:.0%})")

    # Inverted yield curve rule
    if yc and yc < -0.1:
        rule = LEARNED_RULES.get("inverted_yc", {})
        if rule.get("count", 0) >= 3:
            dampener *= (1 - rule.get("dampen", 0.0) * _rs(rule))
            rules_applied.append(f"inverted_yc (dampen {rule['dampen']:.0%})")

    # Ticker-specific rule
    tk_rule_key = f"ticker_{ticker}"
    rule = LEARNED_RULES.get(tk_rule_key, {})
    if rule.get("count", 0) >= 5:
        dampener *= (1 - rule.get("dampen", 0.0) * _rs(rule))
        rules_applied.append(f"{tk_rule_key} (dampen {rule['dampen']:.0%})")

    return dampener, rules_applied

def get_weekly_trend(ticker: str, df_daily: "pd.DataFrame") -> float:
    """
    Returns weekly trend score 0-1 based on weekly close momentum.
    > 0.5 = uptrend, < 0.5 = downtrend.
    """
    try:
        weekly = df_daily["Close"].resample("W").last().dropna()
        if len(weekly) < 4:
            return 0.5
        ret_4w = (weekly.iloc[-1] / weekly.iloc[-4]) - 1
        ret_1w = (weekly.iloc[-1] / weekly.iloc[-2]) - 1
        # Normalize to 0-1
        score = 0.5 + np.clip(ret_4w * 3 + ret_1w * 2, -0.5, 0.5)
        return float(np.clip(score, 0.0, 1.0))
    except Exception:
        return 0.5

def get_weekly_trend_from_api(ticker: str) -> int:
    """
    Returns weekly trend direction: 1=bullish, -1=bearish, 0=neutral.
    Uses 10-week vs 20-week SMA crossover on weekly bars.
    Daily cache: only fetches from yfinance once per day per ticker.
    """
    today = pd.Timestamp.today().date()
    if ticker in _WEEKLY_TREND_CACHE:
        trend, cached_date = _WEEKLY_TREND_CACHE[ticker]
        if cached_date == today:
            return trend
    try:
        df_w = yf.download(ticker, period="2y", interval="1wk",
                           auto_adjust=True, progress=False)
        if df_w is None or len(df_w) < 25:
            trend = 0
        else:
            c     = df_w["Close"].squeeze()
            sma10 = c.rolling(10).mean().iloc[-1]
            sma20 = c.rolling(20).mean().iloc[-1]
            if sma10 > sma20 * 1.005:
                trend = 1
            elif sma10 < sma20 * 0.995:
                trend = -1
            else:
                trend = 0
    except Exception:
        trend = 0
    _WEEKLY_TREND_CACHE[ticker] = (trend, today)
    return trend

def apply_mtf_filter(action: str, confidence: float, ticker: str) -> tuple:
    """
    Reduce confidence when weekly trend disagrees with signal.
    BUY in bearish weekly trend -> reduce confidence by 10%.
    SELL in bullish weekly trend -> reduce confidence by 10%.
    """
    weekly = get_weekly_trend_from_api(ticker)
    if action == "BUY" and weekly == -1:
        confidence *= 0.90
        return action, confidence, "mtf_disagree"
    elif action == "SELL" and weekly == 1:
        confidence *= 0.90
        return action, confidence, "mtf_disagree"
    elif action == "BUY" and weekly == 1:
        confidence = min(0.95, confidence * 1.05)
        return action, confidence, "mtf_agree"
    return action, confidence, "mtf_neutral"
def _is_near_earnings(ticker: str, days: int = 3) -> bool:
    # Return True if ticker has earnings within days calendar days (v25).
    try:
        cal = yf.Ticker(ticker).calendar
        if cal is None:
            return False
        dates = []
        if isinstance(cal, dict):
            dates = cal.get("Earnings Date", [])
        elif hasattr(cal, "columns") and "Earnings Date" in cal.columns:
            dates = list(cal["Earnings Date"].values)
        today = pd.Timestamp.today().normalize()
        for d in dates:
            try:
                diff = abs((pd.Timestamp(d).normalize() - today).days)
                if diff <= days:
                    return True
            except Exception:
                continue
        return False
    except Exception:
        return False

# ── KALMAN FILTER DYNAMIC BETA ───────────────────────────────────────────────
_KALMAN_BETA_CACHE = {}

def _kalman_beta(ticker, horizon_days=60):
    """Kalman Filter dynamic beta vs SPY. Updates every bar; returns current beta estimate."""
    today = pd.Timestamp.today().date()
    if ticker in _KALMAN_BETA_CACHE:
        _b, _, _cd = _KALMAN_BETA_CACHE[ticker]
        if _cd == today: return _b
    try:
        import pandas as _pdkb
        _tk_h = yf.download(ticker,period=f"{horizon_days+5}d",auto_adjust=True,progress=False)
        _sp_h = yf.download("SPY", period=f"{horizon_days+5}d",auto_adjust=True,progress=False)
        _tkr  = _tk_h["Close"].squeeze().pct_change().dropna()
        _spr  = _sp_h["Close"].squeeze().pct_change().dropna()
        _dfkb = _pdkb.concat([_tkr,_spr],axis=1).dropna(); _dfkb.columns=["tk","spy"]
        if len(_dfkb) < 20: return 1.0
        _beta = float(_dfkb["tk"].cov(_dfkb["spy"]) / max(_dfkb["spy"].var(),1e-8))
        _P    = 1.0; _Q = 0.001; _Rn = 0.01
        for _, _r in _dfkb.iterrows():
            _H=_r["spy"]; _Pp=_P+_Q; _K=_Pp*_H/max(_H**2*_Pp+_Rn,1e-8)
            _beta=_beta+_K*(_r["tk"]-_beta*_H); _P=(1.0-_K*_H)*_Pp
        _beta=float(np.clip(_beta,-3.0,3.0))
        _KALMAN_BETA_CACHE[ticker]=(_beta,_P,today); return round(_beta,4)
    except Exception: return 1.0

# ── OU HALF-LIFE (mean-reversion speed) ─────────────────────────────────────
def _ou_halflife(price_series, min_samples=30):
    """OU half-life in days. <8=fast revert; >60=no effective reversion."""
    try:
        _s=np.array(price_series.values if hasattr(price_series,"values") else price_series)
        if len(_s)<min_samples: return 999.0
        _lag=_s[:-1]; _diff=np.diff(_s)
        _X=np.column_stack([np.ones(len(_lag)),_lag])
        _c,_,_,_=np.linalg.lstsq(_X,_diff,rcond=None)
        _g=_c[1]
        if _g>=0: return 999.0
        return float(np.clip(-np.log(2.0)/_g,1.0,999.0))
    except Exception: return 999.0

# ── LOUGHRAN-McDONALD SEC SENTIMENT ─────────────────────────────────────────
_LM_CACHE = {}
_LM_NEG={"loss","losses","decline","declining","failed","failure","adverse","weakness",
          "risk","uncertain","below","shortfall","impair","default","lawsuit","recall"}
_LM_POS={"growth","strong","record","exceed","outperform","increase","improved","gains",
          "profit","success","higher","robust","expanding","award","breakthrough","momentum"}
_LM_UNC={"approximately","may","might","could","uncertain","contingent","unresolved",
          "speculative","indefinite","pending","unknown","subject"}

def _lm_sentiment(ticker):
    """L&M tone score [-1,+1] from SEC EDGAR 10-Q/10-K filings. Cached daily."""
    today=pd.Timestamp.today().date()
    if ticker in _LM_CACHE:
        _sc,_cd=_LM_CACHE[ticker]
        if _cd==today: return _sc
    try:
        import requests as _rlm, datetime as _dtlm
        _start=(_dtlm.date.today()-_dtlm.timedelta(days=90)).isoformat()
        _url=(f"https://efts.sec.gov/LATEST/search-index?q=%22{ticker}%22"
              f"&dateRange=custom&startdt={_start}&forms=10-Q,10-K"
              f"&hits.hits.total.value=1")
        _r=_rlm.get(_url,timeout=5,headers={"User-Agent":"quantbot/1.0 x@x.com"})
        if _r.status_code!=200: _LM_CACHE[ticker]=(0.0,today); return 0.0
        _hits=_r.json().get("hits",{}).get("hits",[])
        if not _hits: _LM_CACHE[ticker]=(0.0,today); return 0.0
        _src=_hits[0].get("_source",{})
        _txt=(str(_src.get("period_of_report",""))+" "+str(_src.get("display_names",""))).lower()
        _ws=_txt.split(); _tot=max(len(_ws),1)
        _n=sum(1 for w in _ws if w.rstrip(".,;:") in _LM_NEG)
        _p=sum(1 for w in _ws if w.rstrip(".,;:") in _LM_POS)
        _u=sum(1 for w in _ws if w.rstrip(".,;:") in _LM_UNC)
        score=float(np.clip((_p-_n-0.5*_u)/_tot*200,-1.0,1.0))
        _LM_CACHE[ticker]=(score,today); return score
    except Exception: _LM_CACHE[ticker]=(0.0,today); return 0.0

# ── FCF YIELD ────────────────────────────────────────────────────────────────
_FCF_CACHE = {}

def _fcf_yield(ticker):
    """FCF Yield = FCF per share / Price. >5% = cash-rich. Cached daily."""
    today=pd.Timestamp.today().date()
    if ticker in _FCF_CACHE:
        _v,_cd=_FCF_CACHE[ticker]
        if _cd==today: return _v
    try:
        _i=yf.Ticker(ticker).info
        _f=float(_i.get("freeCashflow") or 0)
        _sh=float(_i.get("sharesOutstanding") or 0)
        _px=float(_i.get("currentPrice") or _i.get("regularMarketPrice") or 0)
        if _sh<=0 or _px<=0: _FCF_CACHE[ticker]=(0.0,today); return 0.0
        _y=float(np.clip(_f/_sh/_px,-0.5,0.5))
        _FCF_CACHE[ticker]=(_y,today); return _y
    except Exception: _FCF_CACHE[ticker]=(0.0,today); return 0.0

# ── FAMA-FRENCH HML (Book/Price value factor) ────────────────────────────────
_HML_CACHE = {}

def _ff_hml_score(ticker):
    """HML: Book/Price normalised [0,1]. >0.5=value stock; <0.5=growth. Cached daily."""
    today=pd.Timestamp.today().date()
    if ticker in _HML_CACHE:
        _v,_cd=_HML_CACHE[ticker]
        if _cd==today: return _v
    try:
        _i=yf.Ticker(ticker).info
        _bv=float(_i.get("bookValue") or 0)
        _px=float(_i.get("currentPrice") or _i.get("regularMarketPrice") or 0)
        if _bv<=0 or _px<=0: _HML_CACHE[ticker]=(0.5,today); return 0.5
        _sc=float(np.clip(_bv/_px/2.0,0.0,1.0))
        _HML_CACHE[ticker]=(_sc,today); return _sc
    except Exception: _HML_CACHE[ticker]=(0.5,today); return 0.5
def generate_signal(ticker, model_pack, df_feat, regime, garch, sent, iv_flag=None):
    # -- Earnings suppression guard (v25) â€” use cached iv_flags, no extra API call
    _earn_flag = (iv_flag or {}).get("is_earnings_week", False) or _is_near_earnings(ticker, days=3)
    if _earn_flag:
        return dict(ticker=ticker, action="HOLD", confidence=0.0,
                    p_xgb=0.5, p_lgb=0.5, p_cat=0.5, p_ensemble=0.5,
                    garch_p_up=garch.get("p_up",0.5), ann_vol=garch.get("annvol",0.2),
                    var95=garch.get("var95",-0.02), sentiment=round(sent,4),
                    regime=regime, auc=round(model_pack["auc"],4),
                    rsi=50.0, atr=0.0, close=0.0,
                    rules_applied=[], ts=datetime.datetime.utcnow().isoformat(),
                    iv_flag=str(iv_flag), iv_scale=1.0, iv_note="earnings_window")
    row  = df_feat[FEATURE_COLS].iloc[[-1]].values
    Xsc  = model_pack["scaler"].transform(row)
    p_xgb = float(model_pack["xgb"].predict_proba(Xsc)[0,1])
    p_lgb = float(model_pack["lgb"].predict_proba(Xsc)[0,1])
    p_cat = float(model_pack["cat"].predict_proba(Xsc)[0,1])
    if "meta" in model_pack and model_pack["meta"] is not None:
        try:
            _meta_inp = np.array([[p_xgb, p_lgb, p_cat]])
            p_ens = float(model_pack["meta"].predict_proba(_meta_inp)[0,1])
        except Exception: p_ens = (p_xgb+p_lgb+p_cat)/3.0
    else: p_ens = (p_xgb+p_lgb+p_cat)/3.0
    # -- #1: Ensemble disagreement filter
    _ens_var = float(np.var([p_xgb, p_lgb, p_cat]))
    _disagreement_scale = 1.0
    if _ens_var > 0.04:   _disagreement_scale = 0.70
    elif _ens_var > 0.02: _disagreement_scale = 0.85

    regime_score = {0:0.4,1:0.6,2:0.5}.get(regime,0.5)
    sent_norm    = (sent+1)/2
    yc           = MACRO.get("yield_curve") or 0
    yc_score     = 1.0 if yc > 0 else 0.4

    # Use adaptive weights (updated by River learning)
    w = ADAPTIVE_WEIGHTS
    composite = (
        w["w_ensemble"]   * p_ens +
        w["w_garch"]      * garch["p_up"] +
        w["w_sentiment"]  * sent_norm +
        w["w_regime"]     * regime_score +
        w["w_yieldcurve"] * yc_score
    )

    # VIX macro dampener
    vix_val = MACRO.get("vix") or 20
    if vix_val > 30:   composite *= 0.85
    elif vix_val > 22: composite *= 0.93

    # Apply learned rule overrides
    rsi = float(df_feat["rsi_14"].iloc[-1]) if "rsi_14" in df_feat.columns else 50.0
    rule_dampener, rules_applied = apply_learned_rules(
        ticker, rsi, regime, vix_val, yc, composite)
    composite *= rule_dampener

    # OOD feature guard: shrink toward 0.5 when 2+ features out-of-distribution
    try:
        _ood_n = 0
        if vix_val > 45 or vix_val < 8: _ood_n += 1
        if "rsi_14" in df_feat.columns:
            _rsi_v = float(df_feat["rsi_14"].iloc[-1])
            if _rsi_v > 88 or _rsi_v < 12: _ood_n += 1
        if "bb_pct" in df_feat.columns:
            _bb_v = float(df_feat["bb_pct"].iloc[-1])
            if _bb_v > 1.5 or _bb_v < -0.5: _ood_n += 1
        if "ret_21d" in df_feat.columns:
            _r21_v = float(df_feat["ret_21d"].iloc[-1])
            if abs(_r21_v) > 0.35: _ood_n += 1
        if _ood_n >= 2:
            composite = 0.5 + (composite - 0.5) * 0.72
            rules_applied.append(f"ood_guard({_ood_n}feat)")
    except Exception: pass

    # -- #2: RVOL confirmation (dampen low vol, boost high vol)
    if "Volume" in df_feat.columns:
        _vol_now = float(df_feat["Volume"].iloc[-1])
        _vol_ma  = float(df_feat["Volume"].rolling(20).mean().iloc[-1])
        if _vol_ma > 0:
            _rvol = _vol_now / _vol_ma
            if _rvol < 0.7:
                composite *= 0.92
                rules_applied.append("low_vol_confirm")
            elif _rvol > 2.0:
                composite = min(0.95, composite * 1.06)
                rules_applied.append("high_rvol_confirm")

    # Apply boost rules (structured _boosts from Pattern 6)
    boosts = LEARNED_RULES.get("_boosts", {})
    boost_factor = 1.0
    boosts_applied = []
    if boosts.get("low_rsi_bull") and rsi < 35 and regime == 1 and composite < 0.85:
        boost_factor += boosts["low_rsi_bull"]["boost"]
        boosts_applied.append("low_rsi_bull")
    if boosts.get("momentum_bull") and 55 <= rsi < 68 and regime == 1 and composite < 0.85:
        boost_factor += boosts["momentum_bull"]["boost"]
        boosts_applied.append("momentum_bull")
    if boosts.get("low_vix_env") and vix_val < 16 and composite < 0.85:
        boost_factor += boosts["low_vix_env"]["boost"]
        boosts_applied.append("low_vix_env")
    composite = min(0.95, composite * boost_factor)
    # Legacy boost rules (flat sum)
    boost_total = 0.0
    for rule_key, rule in LEARNED_RULES.items():
        if rule_key == "_boosts" or "boost" not in rule:
            continue
        b = rule.get("boost", 0)
        if rule_key == "low_rsi_bull_boost" and rsi < 35 and regime == 1:
            boost_total += b
        elif rule_key == "high_sent_bull_boost" and regime == 1:
            boost_total += b
        elif rule_key == "macro_tailwind_boost" and vix_val < 20 and yc > 0:
            boost_total += b
    composite = min(1.0, composite + boost_total)

    composite = float(np.clip(composite, 0, 1))

    # â”€â”€ Multi-timeframe alignment â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    weekly_trend = get_weekly_trend(ticker, df_feat)
    composite = (1 - MTF_WEEKLY_WEIGHT) * composite + MTF_WEEKLY_WEIGHT * weekly_trend
    composite = float(np.clip(composite, 0, 1))
    # Optional: block trades where daily and weekly oppose each other
    if MTF_REQUIRED_ALIGN:
        daily_bull = composite > 0.5
        weekly_bull = weekly_trend > 0.5
        if daily_bull != weekly_bull:
            composite = 0.5   # force HOLD when timeframes disagree

    # Apply structured MTF filter (weekly trend alignment)
    try:
        _action_before_mtf = "BUY" if composite >= MIN_CONFIDENCE else ("SELL" if composite <= (1-MIN_CONFIDENCE) else "HOLD")
        if _action_before_mtf != "HOLD":
            _, composite, _mtf_note = apply_mtf_filter(_action_before_mtf, composite, ticker)
            composite = float(np.clip(composite, 0, 1))
    except Exception:
        pass

    # â”€â”€ OPTIONS IV DAMPENER â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    iv_scale     = 1.0
    iv_note      = ""
    iv_flag_str  = "NORMAL"
    if iv_flag and iv_flag.get("ok"):
        iv_scale    = float(iv_flag.get("position_scale", 1.0))
        iv_flag_str = iv_flag.get("iv_flag", "NORMAL")
        iv_note     = iv_flag.get("note", "")
        if iv_flag_str in ("HIGH", "ELEVATED"):
            # Dampen composite toward 0.5 (uncertainty)
            composite = 0.5 + (composite - 0.5) * iv_scale
            composite = float(np.clip(composite, 0, 1))

    # â”€â”€ Congressional trade signal â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    _cs_map = MACRO.get("congress_signals", {})
    if ticker in _cs_map:
        _cs = _cs_map[ticker]
        _tot = _cs["total"]
        if _tot > 0:
            _net = (_cs["buys"] - _cs["sells"]) / _tot
            _strength = 0.7 if _tot >= 3 else 0.5   # cluster = stronger signal
            _cs_score = 0.5 + np.clip(_net * _strength, -0.35, 0.35)
            # Blend 10% congress signal into composite
            composite = 0.90 * composite + 0.10 * _cs_score
            composite = float(np.clip(composite, 0, 1))

    # â”€â”€ Sector ETF momentum â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    _etf_mom = MACRO.get("sector_etf_momentum", {})
    _sector  = TICKER_SECTOR.get(ticker, "")
    _etf_key = SECTOR_ETF_MAP.get(_sector, "")
    if _etf_key and _etf_key in _etf_mom:
        _etf_score = 0.5 + np.clip(_etf_mom[_etf_key] * 8, -0.20, 0.20)
        composite  = 0.88 * composite + 0.12 * _etf_score
        composite  = float(np.clip(composite, 0, 1))

    # â”€â”€ Yield curve sector rotation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    _yc_reg = MACRO.get("yc_regime", "normal")
    if _yc_reg == "steep" and _sector == "financials":
        composite = min(0.95, composite * 1.06)   # financials win on steep curve
    elif _yc_reg == "inverted" and _sector in ("tech", "consumer_disc"):
        composite *= 0.94                          # growth hurt by inversion
    elif _yc_reg == "steep" and _sector == "utilities":
        composite *= 0.96                          # utilities lose on steep curve

    # â”€â”€ Commodity chain sector boost â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    _copper = MACRO.get("copper")
    _natgas = MACRO.get("natgas")
    if _copper and _copper > 4.5 and _sector in ("industrials", "materials"):
        composite = min(0.95, composite * 1.04)   # high copper = industrial demand
    if _natgas and _natgas > 3.5 and _sector == "energy":
        composite = min(0.95, composite * 1.05)   # high natgas = energy bullish

    # â”€â”€ VIX term structure: backwardation = acute fear â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if MACRO.get("vix_ts") == "backwardation":
        composite = 0.5 + (composite - 0.5) * 0.82

    # â”€â”€ Near economic release â€” dampen conviction â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if MACRO.get("near_econ_release", False):
        composite = 0.5 + (composite - 0.5) * 0.80

    # â”€â”€ Short interest contrarian signal â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    _si = MACRO.get("short_interest", {}).get(ticker, 0)
    if _si > 20 and composite > 0.65:   # heavily shorted + strong BUY = squeeze risk
        composite = min(0.95, composite * 1.04)
    elif _si > 30:                       # extreme short interest = caution
        composite = 0.5 + (composite - 0.5) * 0.90

    # -- #1: Apply disagreement scale to composite
    composite = 0.5 + (composite - 0.5) * _disagreement_scale
    if _disagreement_scale < 1.0: rules_applied.append("ens_disagree")
    composite = float(np.clip(composite, 0, 1))

    # -- #5: Put/Call ratio contrarian signal
    _pcr = (iv_flag or {}).get("pcr", 1.0) if iv_flag else 1.0
    if _pcr > 1.5 and composite > 0.5:
        composite = min(0.95, composite * 1.04)
        rules_applied.append("pcr_contrarian")
    elif _pcr < 0.5 and composite < 0.5:
        composite = max(0.05, composite * 0.96)

    # -- #12: Short squeeze detector (days-to-cover from sharesShort)
    try:
        _si_info = yf.Ticker(ticker).info
        _shares_short = _si_info.get("sharesShort", 0) or 0
        if _shares_short > 0 and "Volume" in df_feat.columns:
            _avg_vol_sq = float(df_feat["Volume"].rolling(20).mean().iloc[-1])
            if _avg_vol_sq > 0:
                _dtc = _shares_short / _avg_vol_sq
                if _dtc > 10 and composite > 0.60:
                    composite = min(0.95, composite * 1.05)
                    rules_applied.append("squeeze_candidate")
    except Exception:
        pass

    # -- Per-ticker calibration multiplier (from self-learning decay) --------
    _wf_auc = TICKER_ACCURACY_WF.get(ticker, 0.0) if "TICKER_ACCURACY_WF" in globals() else 0.0
    if _wf_auc > 0.60: composite = min(0.95, composite * 1.05)
    elif 0 < _wf_auc < 0.52: composite = 0.5 + (composite - 0.5) * 0.85
    composite = float(np.clip(composite, 0, 1))
    _calib_mult = TICKER_CALIB.get(ticker, 1.0) if "TICKER_CALIB" in globals() else 1.0
    if _calib_mult != 1.0:
        composite = 0.5 + (composite - 0.5) * _calib_mult


    # ── Kalman dynamic beta adjustment ──────────────────────────────────────
    try:
        _kb = _kalman_beta(ticker)
        if _kb > 1.5 and vix_val > 25:
            composite = 0.5 + (composite - 0.5) * 0.90
            rules_applied.append(f"high_beta_vix(kb={_kb:.2f})")
        elif _kb < 0.4:
            composite = float(min(0.95, composite * 1.025))
            rules_applied.append(f"low_beta_def(kb={_kb:.2f})")
    except Exception:
        pass

    # ── OU Half-Life mean-reversion filter ──────────────────────────────────
    try:
        if "Close" in df_feat.columns:
            _hl = _ou_halflife(df_feat["Close"].tail(90))
            if _hl < 8:
                rules_applied.append(f"ou_fast_revert(hl={_hl:.0f}d)")
            elif _hl > 60 and rsi < 40 and composite > 0.5:
                composite = 0.5 + (composite - 0.5) * 0.88
                rules_applied.append(f"ou_slow(hl={_hl:.0f}d)")
    except Exception:
        pass

    # ── Loughran-McDonald SEC sentiment ─────────────────────────────────────
    try:
        _lm = _lm_sentiment(ticker)
        if abs(_lm) > 0.005:
            _lm_adj = float(np.clip(_lm * 0.04, -0.04, 0.04))
            composite = float(np.clip(composite + _lm_adj, 0, 1))
            if abs(_lm_adj) > 0.01:
                rules_applied.append(f"lm_sec({'pos' if _lm>0 else 'neg'})")
    except Exception:
        pass

    # ── FCF Yield + Fama-French HML value signal ─────────────────────────────
    try:
        _fcf_y = _fcf_yield(ticker)
        _hml   = _ff_hml_score(ticker)
        if _fcf_y > 0.05 and composite > 0.5:
            composite = float(min(0.95, composite + 0.02))
            rules_applied.append("fcf_rich")
        elif _fcf_y < -0.02:
            composite = 0.5 + (composite - 0.5) * 0.93
            rules_applied.append("fcf_burn")
        if _hml > 0.70 and regime == 0:
            composite = float(min(0.95, composite * 1.025))
            rules_applied.append("ff_hml_value")
        elif _hml < 0.10 and regime == 0:
            composite = 0.5 + (composite - 0.5) * 0.95
    except Exception:
        pass
    composite = float(np.clip(composite, 0, 1))

    if   composite >= MIN_CONFIDENCE:       action = "BUY"
    elif composite <= (1-MIN_CONFIDENCE):   action = "SELL"
    else:                                   action = "HOLD"
    _REGIME_FLOORS = {0: 0.68, 1: 0.62, 2: 0.65}
    _r_floor = _REGIME_FLOORS.get(regime, MIN_CONFIDENCE)
    if action == "BUY" and composite < _r_floor: action = "HOLD"
    elif action == "SELL" and composite > (1 - _r_floor): action = "HOLD"

    close = float(df_feat["Close"].iloc[-1])
    atr   = float(df_feat["atr_14"].iloc[-1]) if "atr_14" in df_feat.columns else close*0.02

    return dict(
        ticker=ticker, action=action,
        confidence=round(composite,4),
        p_xgb=round(p_xgb,4), p_lgb=round(p_lgb,4), p_cat=round(p_cat,4),
        p_ensemble=round(p_ens,4),
        garch_p_up=round(garch["p_up"],4),
        ann_vol=round(garch["annvol"],4),
        var95=round(garch["var95"],4),
        sentiment=round(sent,4),
        regime=regime, auc=round(model_pack["auc"],4),
        rsi=round(rsi,2), atr=round(atr,4),
        close=round(close,4),
        rules_applied=rules_applied,
        ts=datetime.datetime.utcnow().isoformat(),
        iv_flag=iv_flag_str,
        iv_scale=round(iv_scale,3),
        iv_note=iv_note
    )

print("Generating signals...")
TICKER_ACCURACY_WF = {}
try:
    _acc_path = Path("data/weights/ticker_accuracy.json")
    if _acc_path.exists():
        TICKER_ACCURACY_WF = json.loads(_acc_path.read_text())
        print(f"  Ticker accuracy: {len(TICKER_ACCURACY_WF)} tickers")
except Exception: pass
signals = {}
for tk in models:
    if tk not in featured: continue
    try:
        sig=generate_signal(
            tk,models[tk],featured[tk],
            int(regimes[tk].iloc[-1]) if tk in regimes else 0,
            garch_res.get(tk,dict(p_up=0.5,annvol=0.3,var95=-0.05,es95=-0.08,ok=False)),
            sentiments.get(tk,0.0),
            iv_flag=iv_flags.get(tk, {}))
        signals[tk]=sig
        rules_note = f" [{', '.join(sig['rules_applied'])}]" if sig['rules_applied'] else ""
        print(f"  {sig['action']:<4} {tk:<10} conf={sig['confidence']:.3f}{rules_note}")
    except Exception as ex:
        print(f"  FAIL {tk}: {ex}")
print(f"\n{len(signals)} signals generated")

# ── UPGRADE 1: Cross-sectional ranking ──────────────────────────────────────
# Normalize composites relative to the universe — removes market-wide beta noise.
# Top 20% = strong BUY candidates; bottom 20% = strong SELL candidates.
# Blend: 70% absolute score + 30% cross-sectional z-score adjustment (max ±6%).
if len(signals) >= 4:
    try:
        _cs_comps = np.array([s.get("composite", 0.5) for s in signals.values()])
        _cs_mean  = float(_cs_comps.mean())
        _cs_std   = float(_cs_comps.std())
        if _cs_std > 0.01:
            _cs_adjusted = 0
            for _tk, _sig in signals.items():
                _z       = (_sig.get("composite", 0.5) - _cs_mean) / _cs_std
                _boost   = float(np.clip(_z * 0.03, -0.06, 0.06))
                _new_c   = round(float(np.clip(_sig.get("composite", 0.5) + _boost, 0.10, 0.92)), 4)
                _sig["composite"]  = _new_c
                _sig["confidence"] = _new_c
                # Re-classify action based on cross-sectional adjusted composite
                if _new_c >= MIN_CONFIDENCE:
                    _sig["action"] = "BUY"
                elif _new_c <= (1.0 - MIN_CONFIDENCE):
                    _sig["action"] = "SELL"
                else:
                    _sig["action"] = "HOLD"
                _cs_adjusted += 1
            print(f"  Cross-sectional ranking applied: {_cs_adjusted} signals adjusted "
                  f"(universe mean={_cs_mean:.3f} σ={_cs_std:.3f})")
    except Exception as _cse:
        print(f"  Cross-sectional ranking error: {_cse}")

# ── UPGRADE 6: News event classification ─────────────────────────────────────
# Tag each signal with its event type and apply appropriate position scaling.
_EVENT_KEYWORDS = {
    "earnings":   ["earnings","EPS","revenue","quarterly","beat","miss","guidance","profit"],
    "merger":     ["merger","acquisition","buyout","deal","takeover","bid","offer"],
    "fda":        ["FDA","approval","clinical","trial","drug","therapy","PDUFA"],
    "analyst":    ["upgrade","downgrade","price target","rating","analyst","coverage"],
    "macro":      ["fed","interest rate","inflation","GDP","employment","CPI","FOMC"],
}
try:
    for _tk, _sig in signals.items():
        _headline = str(_sig.get("sentiment_note", "")) + " " + str(_sig.get("iv_note", ""))
        _hl_lower = _headline.lower()
        _event    = "other"
        for _etype, _kws in _EVENT_KEYWORDS.items():
            if any(_kw.lower() in _hl_lower for _kw in _kws):
                _event = _etype; break
        _sig["event_type"] = _event
        # High-variance events (earnings/FDA): cap confidence, flag for 1-day hold
        if _event in ("earnings", "fda") and _sig.get("iv_flag","NORMAL") != "NORMAL":
            _sig["composite"]  = round(min(_sig["composite"],  0.72), 4)
            _sig["confidence"] = round(min(_sig["confidence"], 0.72), 4)
            _sig["event_scale"] = 0.5   # position size capped at 50% for high-var events
        else:
            _sig["event_scale"] = 1.0
except Exception as _ee:
    print(f"  Event classification error: {_ee}")

# ── UPGRADE 7: Feature IC feedback loop ──────────────────────────────────────
# Load feature importance; features in top-10 add small confidence boost.
# Features that have decayed out of top-20 reduce confidence slightly.
try:
    _fi_path = Path("data/weights/feature_importance.json")
    if _fi_path.exists():
        _fi_data   = json.loads(_fi_path.read_text())
        _top10     = set(list(_fi_data.keys())[:10])  if isinstance(_fi_data, dict) else set()
        _top20     = set(list(_fi_data.keys())[:20])  if isinstance(_fi_data, dict) else set()
        _feat_set  = set(FEATURE_COLS) if "FEATURE_COLS" in globals() else set()
        _n_top_in  = len(_feat_set & _top10)
        _n_top_out = len(_feat_set - _top20)
        # If many of our active features are high-importance → small boost
        _fi_scalar = 1.0
        if len(_feat_set) > 0:
            _top_ratio = _n_top_in / len(_feat_set)
            if _top_ratio > 0.40:   _fi_scalar = 1.02   # lots of top features active
            elif _top_ratio < 0.15: _fi_scalar = 0.98   # few top features active
        if _fi_scalar != 1.0:
            for _sig in signals.values():
                _sig["confidence"] = round(
                    float(np.clip(_sig["confidence"] * _fi_scalar, 0.10, 0.92)), 4)
            print(f"  Feature IC scalar: {_fi_scalar:.2f} "
                  f"(top10 overlap={_n_top_in}/{len(_feat_set):.0f})")
except Exception as _fie:
    print(f"  Feature IC error: {_fie}")

Generating signals...
  SELL AAPL       conf=0.291
  SELL MSFT       conf=0.301
  SELL NVDA       conf=0.341
  SELL GOOGL      conf=0.308
  SELL AMZN       conf=0.305
  SELL META       conf=0.318
  HOLD TSLA       conf=0.362
  SELL JPM        conf=0.283
  SELL V          conf=0.313
  SELL UNH        conf=0.320
  HOLD SPY        conf=0.378
  HOLD BTC-USD    conf=0.391
  HOLD ETH-USD    conf=0.397
  HOLD SOL-USD    conf=0.356
  HOLD BNB-USD    conf=0.367
  HOLD XRP-USD    conf=0.381

16 signals generated


In [ ]:
# ============================================================
# CELL 12 — CVaR PORTFOLIO OPTIMISATION (Enhanced)
# ============================================================
# Minimize CVaR(95%) subject to weight constraints.
# Also reports: portfolio Sharpe, diversification ratio,
# and Information Coefficient on last 60 signals.
# ============================================================
import numpy as np

# ── IC Computation (Spearman corr of confidence vs 5d return) ────────────────
def compute_signal_ic(signals_dict, featured_dict, horizon=5):
    """
    Information Coefficient: Spearman correlation between predicted
    confidence and realised 5-day return. Target >0.05 useful, >0.10 exceptional.
    """
    try:
        _preds = []; _rets = []
        for tk, sig in signals_dict.items():
            if tk not in featured_dict: continue
            _conf = sig.get("confidence", 0.5) - 0.5  # center around 0
            if sig.get("action") == "SELL": _conf = -_conf
            _df = featured_dict[tk]
            if len(_df) < horizon + 2: continue
            _ret = float(_df["Close"].iloc[-1] / _df["Close"].iloc[-1-horizon] - 1)
            _preds.append(_conf); _rets.append(_ret)
        if len(_preds) < 5: return None
        from scipy.stats import spearmanr
        _ic, _pval = spearmanr(_preds, _rets)
        return round(float(_ic), 4) if not np.isnan(_ic) else None
    except Exception: return None
# -- Previous weights for turnover penalty --
TURNOVER_LAMBDA = 0.10   # cost of 1 unit of L1 turnover vs CVaR

def _load_prev_weights(tickers):
    """Load previous portfolio weights from disk (JSON). Returns array aligned to tickers."""
    _p = Path("data/weights/portfolio_weights.json")
    if not _p.exists():
        return np.ones(len(tickers)) / len(tickers)
    try:
        _saved = json.loads(_p.read_text())
        return np.array([_saved.get(tk, 1.0/len(tickers)) for tk in tickers])
    except Exception:
        return np.ones(len(tickers)) / len(tickers)

def _save_current_weights(tickers, weights_array):
    """Persist current weights to disk for next cycle's turnover penalty."""
    _p = Path("data/weights/portfolio_weights.json")
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_text(json.dumps(dict(zip(tickers, [float(w) for w in weights_array])), indent=2))

# ── Almgren-Chriss per-ticker market impact cost ───────────────────────────
def _ac_impact_coeff(tk, raw_data_dict, garch_res_dict, portfolio_equity,
                     eta=0.10):
    """
    Almgren-Chriss linear market-impact cost coefficient.
    Returns the expected cost (as a fraction of trade value) of moving
    1 unit (100%) of portfolio weight in/out of ticker `tk`.

    Formula: cost = eta * sigma_annual * sqrt(port_usd / adv_usd)
    where port_usd is the dollar size of the portfolio and adv_usd is the
    ticker's 20-day average daily dollar volume.

    For a $100K paper portfolio trading S&P 500 names this will be small
    (< 0.1%), which is correct — impact is negligible at this scale.
    The coefficient scales correctly when portfolio size grows.
    """
    df = raw_data_dict.get(tk)
    if df is None or "Close" not in df.columns:
        return 0.15  # conservative fallback

    # ── Annualised volatility ─────────────────────────────────────────────
    # Prefer GARCH conditional vol (already fitted); fall back to realised
    sigma = None
    if garch_res_dict and tk in garch_res_dict:
        try:
            _gres = garch_res_dict[tk]
            sigma = float(_gres.conditional_volatility.iloc[-1]) * np.sqrt(252) / 100
        except Exception:
            pass
    if sigma is None or sigma <= 0:
        _rets = df["Close"].pct_change().dropna().iloc[-21:]
        sigma = float(_rets.std() * np.sqrt(252)) if len(_rets) > 5 else 0.30
    sigma = np.clip(sigma, 0.05, 2.0)

    # ── Average daily dollar volume ───────────────────────────────────────
    try:
        adv_usd = float(
            (df["Close"].iloc[-20:] * df["Volume"].iloc[-20:]).mean()
        )
    except Exception:
        adv_usd = 1e7  # $10M fallback
    adv_usd = max(adv_usd, 1e4)  # floor to avoid division by zero

    # ── Almgren-Chriss cost coefficient ──────────────────────────────────
    # "What fraction of the trade value is lost to market impact when we
    #  move the full portfolio into/out of this name?"
    port_usd = max(portfolio_equity, 1_000)
    impact = eta * sigma * np.sqrt(port_usd / adv_usd)
    return float(np.clip(impact, 1e-5, 0.50))

def cvar_optimize(tickers, lookback=252):
    pd_dict={tk:featured[tk]["Close"].tail(lookback)
             for tk in tickers if tk in featured}
    if len(pd_dict)<2:
        n=max(len(tickers),1); return {tk:1/n for tk in tickers}
    prices=pd.DataFrame(pd_dict).dropna()
    rets=prices.pct_change().dropna().values
    T,N=rets.shape
    if T < 2 or N < 2:
        n=max(len(tickers),1); return {tk:1/n for tk in tickers}
    w=cp.Variable(N,nonneg=True); z=cp.Variable(T,nonneg=True); zeta=cp.Variable()
    _w_prev = _load_prev_weights(list(pd_dict.keys()))
    # ── Almgren-Chriss impact coefficients (one per ticker) ─────────────────────
    try:
        _port_eq = float(portfolio_equity) if "portfolio_equity" in dir() else 100_000.0
        _ac_costs = np.array([
            _ac_impact_coeff(tk, raw_data, garch_res, _port_eq)
            for tk in buy_tickers
        ])
        print(f"  AC impact: min={_ac_costs.min():.4f}  "
              f"max={_ac_costs.max():.4f}  "
              f"mean={_ac_costs.mean():.4f}")
        _use_ac = True
    except Exception as _ac_e:
        print(f"  AC impact calc failed: {_ac_e} — using uniform turnover only")
        _ac_costs = np.ones(len(buy_tickers)) * TURNOVER_LAMBDA
        _use_ac = False
    # -- 5-Factor Risk Model --
    # Factors: Market (SPY), Sector ETF, Momentum, Size (log-ADV), Value (FCF yield)
    # Purpose: cap systematic factor exposures so the optimizer cannot accidentally
    #          concentrate into hidden correlated bets (e.g., all AI-capex names).

    FACTOR_NAMES   = ["market", "sector", "momentum", "size", "value"]
    FACTOR_LIMITS  = [0.30,      0.40,     0.20,       0.25,   0.20  ]

    _TICKER_SECTOR = {
        "AAPL":"Technology","MSFT":"Technology","NVDA":"Semiconductors","GOOGL":"Communication",
        "AMZN":"Consumer Disc","META":"Communication","TSLA":"Consumer Disc","JPM":"Financials",
        "V":"Financials","MA":"Financials","UNH":"Healthcare","LLY":"Healthcare","XOM":"Energy",
        "HD":"Consumer Disc","COST":"Consumer Staples","AVGO":"Semiconductors","AMD":"Semiconductors",
        "NFLX":"Communication","CRM":"Technology","NOW":"Technology","PLTR":"Technology",
        "GS":"Financials","MS":"Financials","WMT":"Consumer Staples","PG":"Consumer Staples",
        "KO":"Consumer Staples","PEP":"Consumer Staples","DIS":"Communication","CMCSA":"Communication",
        "VZ":"Communication","T":"Communication","INTC":"Semiconductors","QCOM":"Semiconductors",
        "MU":"Semiconductors","TXN":"Semiconductors","BA":"Industrials","CVX":"Energy",
        "PYPL":"Financials","COIN":"Financials","ABBV":"Healthcare","JNJ":"Healthcare",
        "PFE":"Healthcare","MRK":"Healthcare","TMO":"Healthcare","ABT":"Healthcare",
        "DHR":"Healthcare","AMGN":"Healthcare","CVS":"Healthcare","CI":"Healthcare",
        "HUM":"Healthcare","BSX":"Healthcare","MDT":"Healthcare","SYK":"Healthcare",
        "ISRG":"Healthcare","VRTX":"Healthcare","REGN":"Healthcare","BMY":"Healthcare",
        "GILD":"Healthcare","NKE":"Consumer Disc","LOW":"Consumer Disc","TJX":"Consumer Disc",
        "ROST":"Consumer Disc","SBUX":"Consumer Disc","CMG":"Consumer Disc","MCD":"Consumer Disc",
        "TGT":"Consumer Disc","GM":"Consumer Disc","F":"Consumer Disc","UBER":"Consumer Disc",
        "BKNG":"Consumer Disc","ABNB":"Consumer Disc","MAR":"Consumer Disc","HLT":"Consumer Disc",
        "DG":"Consumer Disc","DLTR":"Consumer Disc","YUM":"Consumer Disc","BAC":"Financials",
        "BLK":"Financials","AXP":"Financials","WFC":"Financials","C":"Financials","SCHW":"Financials",
        "PGR":"Financials","CB":"Financials","COF":"Financials","USB":"Financials","TFC":"Financials",
        "PNC":"Financials","ICE":"Financials","CME":"Financials","SPGI":"Financials","MCO":"Financials",
        "AON":"Financials","CAT":"Industrials","DE":"Industrials","HON":"Industrials","GE":"Industrials",
        "RTX":"Industrials","LMT":"Industrials","NOC":"Industrials","UPS":"Industrials","FDX":"Industrials",
        "MMM":"Industrials","EMR":"Industrials","ETN":"Industrials","ITW":"Industrials","PH":"Industrials",
        "CMI":"Industrials","GD":"Industrials","TDG":"Industrials","CTAS":"Industrials","NSC":"Industrials",
        "LIN":"Materials","APD":"Materials","SHW":"Materials","PPG":"Materials","NEM":"Materials",
        "FCX":"Materials","NUE":"Materials","ALB":"Materials","LYB":"Materials","ECL":"Materials",
        "CF":"Materials","NEE":"Utilities","DUK":"Utilities","SO":"Utilities","AEP":"Utilities",
        "D":"Utilities","EXC":"Utilities","SRE":"Utilities","XEL":"Utilities","AWK":"Utilities",
        "WEC":"Utilities","ED":"Utilities","AMT":"Real Estate","PLD":"Real Estate","EQIX":"Real Estate",
        "CCI":"Real Estate","WELL":"Real Estate","SPG":"Real Estate","O":"Real Estate","DLR":"Real Estate",
        "PSA":"Real Estate","EXR":"Real Estate","VICI":"Real Estate","SLB":"Energy","EOG":"Energy",
        "HAL":"Energy","OXY":"Energy","PSX":"Energy","MPC":"Energy","VLO":"Energy","DVN":"Energy",
        "APA":"Energy","KMI":"Energy","WMB":"Energy","BKR":"Energy","LNG":"Energy",
        "AMAT":"Semiconductors","LRCX":"Semiconductors","KLAC":"Semiconductors","ADI":"Semiconductors",
        "MRVL":"Semiconductors","ON":"Semiconductors","MCHP":"Semiconductors","ENPH":"Semiconductors",
        "FSLR":"Semiconductors","TER":"Semiconductors","SWKS":"Semiconductors","MPWR":"Semiconductors",
        "ASML":"Semiconductors","TSM":"Semiconductors","ORCL":"Technology","ADBE":"Technology",
        "INTU":"Technology","CDNS":"Technology","SNPS":"Technology","FTNT":"Technology",
        "ANET":"Technology","ACN":"Technology","IBM":"Technology","CSCO":"Technology",
        "TYL":"Technology","ROP":"Technology","WDAY":"Technology","NET":"Technology","SNOW":"Technology",
        "DDOG":"Technology","ZS":"Technology","CRWD":"Technology","PANW":"Technology",
        "EA":"Communication","TTWO":"Communication","LYV":"Communication","TMUS":"Communication",
        "CHTR":"Communication","MDLZ":"Consumer Staples","CL":"Consumer Staples","MO":"Consumer Staples",
        "PM":"Consumer Staples","EL":"Consumer Staples","GIS":"Consumer Staples","TSN":"Consumer Staples",
    }

    def _build_factor_model(tickers, featured_dict, raw_data_dict, lookback=126):
        """
        Estimate 5-factor loadings via OLS for each ticker.
        Returns B (Nx5 loadings matrix), valid mask (which tickers had enough data).
        """
        import warnings
        from sklearn.linear_model import LinearRegression as _LR

        def _price_ret(tk):
            df = raw_data_dict.get(tk)
            if df is None or "Close" not in df.columns or len(df) < lookback:
                return None
            return df["Close"].pct_change().dropna().iloc[-lookback:]

        spy_ret  = _price_ret("SPY")
        xlk_ret  = _price_ret("XLK")
        xlf_ret  = _price_ret("XLF")
        xle_ret  = _price_ret("XLE")
        xlv_ret  = _price_ret("XLV")
        xli_ret  = _price_ret("XLI")
        xly_ret  = _price_ret("XLY")
        xlp_ret  = _price_ret("XLP")
        xlb_ret  = _price_ret("XLB")
        xlu_ret  = _price_ret("XLU")
        xlre_ret = _price_ret("XLRE")
        xlc_ret  = _price_ret("XLC")

        _sector_etf_rets = {
            "Technology": xlk_ret,   "Semiconductors": xlk_ret,
            "Financials":  xlf_ret,   "Energy": xle_ret,
            "Healthcare":  xlv_ret,   "Industrials": xli_ret,
            "Consumer Disc": xly_ret, "Consumer Staples": xlp_ret,
            "Materials":   xlb_ret,   "Utilities": xlu_ret,
            "Real Estate": xlre_ret,  "Communication": xlc_ret,
        }

        N = len(tickers)
        B = np.zeros((N, 5))  # [market, sector, momentum, size, value]
        valid = np.ones(N, dtype=bool)

        for i, tk in enumerate(tickers):
            df = featured_dict.get(tk)
            rd = raw_data_dict.get(tk)
            if df is None or rd is None or "Close" not in rd.columns:
                valid[i] = False
                continue

            tk_ret = rd["Close"].pct_change().dropna().iloc[-lookback:]
            if len(tk_ret) < 60:
                valid[i] = False
                continue

            # Factor 1: Market (SPY beta)
            if spy_ret is not None:
                _aligned = spy_ret.reindex(tk_ret.index).dropna()
                _tk_a    = tk_ret.reindex(_aligned.index)
                if len(_aligned) >= 30:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        _lr = _LR(fit_intercept=True).fit(
                            _aligned.values.reshape(-1,1), _tk_a.values)
                    B[i, 0] = float(_lr.coef_[0])

            # Factor 2: Sector ETF (orthogonalized to market)
            _sec   = _TICKER_SECTOR.get(tk, "Technology")
            _s_ret = _sector_etf_rets.get(_sec)
            if _s_ret is not None and spy_ret is not None:
                _sa = _s_ret.reindex(tk_ret.index).dropna()
                _sp = spy_ret.reindex(_sa.index).dropna()
                _idx = _sa.index.intersection(_sp.index).intersection(tk_ret.index)
                if len(_idx) >= 30:
                    _X2 = np.column_stack([_sp.reindex(_idx).values,
                                           _sa.reindex(_idx).values])
                    _y2 = tk_ret.reindex(_idx).values
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        _lr2 = _LR(fit_intercept=True).fit(_X2, _y2)
                    B[i, 1] = float(_lr2.coef_[1])  # sector loading (net of market)

            # Factor 3: Momentum (12m-1m return)
            try:
                _closes = rd["Close"].dropna()
                if len(_closes) >= 252:
                    _ret_12m = float(_closes.iloc[-1] / _closes.iloc[-252] - 1)
                    _ret_1m  = float(_closes.iloc[-1] / _closes.iloc[-21]  - 1)
                    B[i, 2] = _ret_12m - _ret_1m
            except Exception:
                pass

            # Factor 4: Size (log ADV)
            try:
                _adv = float((rd["Close"].iloc[-20:] * rd["Volume"].iloc[-20:]).mean())
                B[i, 3] = np.log1p(_adv) if _adv > 0 else 0.0
            except Exception:
                pass

            # Factor 5: Value (inverse P/E proxy)
            try:
                _pe = rd.get("PE", pd.Series([np.nan])).iloc[-1] if hasattr(rd, "get") else np.nan
                B[i, 4] = float(1.0 / max(_pe, 1.0)) if pd.notna(_pe) and _pe > 0 else 0.0
            except Exception:
                pass

        for _col in [2, 3, 4]:
            _vals = B[valid, _col]
            _std  = _vals.std()
            if _std > 1e-8:
                B[valid, _col] = np.clip(_vals / (3 * _std), -1, 1)

        print(f"  Factor model: {valid.sum()}/{N} tickers had sufficient data")
        return B, valid

    try:
        _B_full, _valid_mask = _build_factor_model(list(pd_dict.keys()), featured, raw_data)
        _B = _B_full[_valid_mask]
        _factor_tickers = [t for t, v in zip(list(pd_dict.keys()), _valid_mask) if v]
        print(f"  Factor loadings computed -- will add {len(FACTOR_NAMES)} exposure constraints")
        _use_factor_constraints = len(_factor_tickers) >= 3
    except Exception as _fm_e:
        print(f"  Factor model error: {_fm_e} -- skipping factor constraints")
        _use_factor_constraints = False
        _B = None
        _factor_tickers = list(pd_dict.keys())

    _factor_constraints = []
    if _use_factor_constraints and _B is not None and len(_factor_tickers) == N:
        for _k, (_fname, _flim) in enumerate(zip(FACTOR_NAMES, FACTOR_LIMITS)):
            _f_exp = _B[:, _k] @ w   # portfolio factor exposure
            _factor_constraints.append(cp.abs(_f_exp) <= _flim)

    prob=cp.Problem(
        cp.Minimize(
            zeta + (1 / (0.05 * T)) * cp.sum(z)
            # Almgren-Chriss per-ticker market impact (replaces uniform turnover lambda)
            # For liquid large-caps at paper-trading scale this term is small but
            # correctly penalises illiquid names disproportionately more than liquid ones.
            + cp.sum(cp.multiply(_ac_costs, cp.abs(w - _w_prev)))
        ),
        [cp.sum(w) == 1, w <= 0.25, z >= -rets @ w - zeta]
        + _factor_constraints
    )
    try:
        prob.solve(solver=cp.CLARABEL,verbose=False)
        if w.value is None: raise ValueError("no solution")
        # Persist weights for next cycle's turnover calculation
        _save_current_weights(list(pd_dict.keys()), w.value if w.value is not None else _w_prev)
        return {tk:float(wt) for tk,wt in zip(pd_dict.keys(),w.value)}
    except Exception as e:
        print(f"  CVaR fallback ({e})")
        n=max(len(tickers),1); return {tk:1/n for tk in tickers}

buy_tickers=[tk for tk,s in signals.items() if s["action"]=="BUY"]
if buy_tickers:
    opt_weights=cvar_optimize(buy_tickers)
    print("CVaR weights:")
    for tk,wt in sorted(opt_weights.items(),key=lambda x:-x[1]):
        print(f"  {tk:<10} {wt:.1%}")
else:
    opt_weights={}
    print("No BUY signals")
print("\nOptimisation complete")

# ── Signal IC (current universe) ─────────────────────────────────────────────
_current_ic = compute_signal_ic(signals, featured)
if _current_ic is not None:
    _ic_str = f"{_current_ic:+.4f}"
    _ic_grade = "EXCEPTIONAL" if abs(_current_ic)>0.10 else ("USEFUL" if abs(_current_ic)>0.05 else "WEAK")
    print(f"  Signal IC: {_ic_str}  [{_ic_grade}]")
else:
    print("  Signal IC: insufficient data")

# ── CVaR portfolio metrics ────────────────────────────────────────────────────
if buy_tickers and opt_weights:
    try:
        _wts = np.array([opt_weights.get(tk, 1/len(buy_tickers)) for tk in buy_tickers])
        _wts = _wts / max(_wts.sum(), 1e-8)
        _rets_mtx = {}
        for tk in buy_tickers:
            if tk in featured:
                _rets_mtx[tk] = featured[tk]["Close"].tail(252).pct_change().dropna()
        if len(_rets_mtx) >= 2:
            import pandas as _pdcv
            _rdf = _pdcv.DataFrame(_rets_mtx).dropna()
            _w   = np.array([opt_weights.get(tk, 1/len(buy_tickers)) for tk in _rdf.columns])
            _w   = _w / max(_w.sum(), 1e-8)
            _pr  = _rdf.values @ _w
            _ann_ret  = float(np.mean(_pr) * 252)
            _ann_vol  = float(np.std(_pr)  * np.sqrt(252))
            _var95    = float(np.percentile(_pr, 5))
            _cvar95   = float(_pr[_pr <= _var95].mean()) if (_pr <= _var95).any() else _var95
            _sharpe   = round(_ann_ret / max(_ann_vol, 1e-6), 3)
            print(f"  Portfolio metrics (CVaR-weighted):")
            print(f"    Ann. return:  {_ann_ret:+.1%}")
            print(f"    Ann. vol:     {_ann_vol:.1%}")
            print(f"    Sharpe:       {_sharpe:.2f}")
            print(f"    Daily VaR95:  {_var95:.2%}")
            print(f"    Daily CVaR95: {_cvar95:.2%}")
    except Exception as _cvm:
        print(f"  CVaR metrics error: {_cvm}")


No BUY signals

Optimisation complete


In [ ]:
# ============================================================
# CELL 13 Ã¢â‚¬â€ PAPER TRADE ENGINE + 60-DAY P&L TRACKER
# ============================================================
# Executes paper trades from signals, tracks every position
# mark-to-market, computes realised and unrealised P&L,
# and feeds outcome data into the self-learning loop.
# All data persists to Google Drive.
# ============================================================
import datetime, json
import numpy as np
import pandas as pd
from pathlib import Path

# Ã¢â€â‚¬Ã¢â€â‚¬ Position sizing Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def kelly_qty(confidence, capital, price,
              f=0.25, max_pct=MAX_POSITION_PCT, win_loss_ratio=None):
    """
    Kelly position sizing: Full Kelly f* = (p*b - q)/b where b = avg_win/avg_loss.
    Uses half-Kelly for institutions. Falls back to edge*f if b not available.
    """
    if price <= 0 or capital <= 0: return 0
    if win_loss_ratio is not None and win_loss_ratio > 0:
        b = win_loss_ratio
        q = 1.0 - confidence
        frac_full = (confidence * b - q) / b
        frac = max(0.0, frac_full * 0.5)   # half-Kelly
    else:
        edge = confidence - (1 - confidence)
        frac = max(0, edge * f)
    frac = min(frac, max_pct)
    # Tighten position cap based on VIX regime
    _vix = MACRO.get("vix") or 20
    if   _vix > 35: frac = min(frac, max_pct * 0.40)   # crisis: max 40% of normal
    elif _vix > 28: frac = min(frac, max_pct * 0.60)   # high vol: max 60%
    elif _vix > 22: frac = min(frac, max_pct * 0.80)   # elevated: max 80%
    dollars = capital * frac
    qty   = int(dollars / price)
    return max(qty, 0)

# Ã¢â€â‚¬Ã¢â€â‚¬ Alpaca integration (optional) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def _try_alpaca(action, ticker, qty):
    if not (ALPACA_API_KEY and ALPACA_SECRET_KEY): return None
    try:
        from alpaca.trading.client import TradingClient
        from alpaca.trading.requests import MarketOrderRequest
        from alpaca.trading.enums import OrderSide, TimeInForce
        client = TradingClient(ALPACA_API_KEY, ALPACA_SECRET_KEY, paper=True)
        side   = OrderSide.BUY if action=="BUY" else OrderSide.SELL
        req    = MarketOrderRequest(symbol=ticker, qty=qty,
                                    side=side, time_in_force=TimeInForce.DAY)
        order  = client.submit_order(req)
        return str(order.id)
    except Exception as e:
        return f"alpaca_error:{str(e).encode('ascii','replace').decode('ascii')[:40]}"

# Ã¢â€â‚¬Ã¢â€â‚¬ Portfolio equity Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def _current_equity():
    """True portfolio equity = cash + market value of open positions."""
    try:
        log = pd.read_csv(PT_LOG_FILE)
        log["qty"]   = pd.to_numeric(log["qty"],   errors="coerce").fillna(0)
        log["price"] = pd.to_numeric(log["price"], errors="coerce").fillna(0)
        bought = (log[log["action"]=="BUY"]["price"]
                  .mul(log[log["action"]=="BUY"]["qty"])).sum()
        sold   = (log[log["action"]=="SELL"]["price"]
                  .mul(log[log["action"]=="SELL"]["qty"])).sum()
        cash = PORTFOLIO_CAPITAL - bought + sold
        # Add current market value of open positions so invested capital
        # is not counted as lost, preventing false MAX_DRAWDOWN triggers.
        open_pos = get_open_positions()
        if not open_pos.empty and "mkt_value" in open_pos.columns:
            invested = open_pos["mkt_value"].sum()
        else:
            # Fallback: use cost basis to avoid false drawdown trigger
            invested = max(0.0, bought - sold)
        return cash + invested
    except Exception:
        return PORTFOLIO_CAPITAL

# Ã¢â€â‚¬Ã¢â€â‚¬ Mark-to-market open positions Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def get_open_positions():
    """
    Returns DataFrame of open positions with current P&L.
    An open position = a BUY that hasn't been fully offset by a SELL.
    """
    try:
        log = pd.read_csv(PT_LOG_FILE)
        log["qty"] = pd.to_numeric(log["qty"], errors="coerce").fillna(0)
        log["price"] = pd.to_numeric(log["price"], errors="coerce").fillna(0)

        positions = {}
        for _, row in log.iterrows():
            tk  = row["ticker"]
            qty = int(row["qty"])
            if row["action"] == "BUY":
                if tk not in positions:
                    positions[tk] = {"qty":0,"cost":0.0,"entries":[]}
                positions[tk]["qty"]  += qty
                positions[tk]["cost"] += qty * row["price"]
                positions[tk]["entries"].append({"qty":qty,"price":row["price"],"ts":row["ts"]})
            elif row["action"] == "SELL":
                if tk in positions and positions[tk]["qty"] > 0:
                    _avg_cost = positions[tk]["cost"] / positions[tk]["qty"]
                    positions[tk]["qty"]  = max(0, positions[tk]["qty"] - qty)
                    positions[tk]["cost"] = max(0, positions[tk]["cost"] - qty * _avg_cost)

        # Get current prices for open positions
        rows = []
        for tk, pos in positions.items():
            if pos["qty"] <= 0: continue
            try:
                hist = yf.Ticker(tk).history(period="1d", auto_adjust=True)
                curr_price = float(hist["Close"].iloc[-1]) if not hist.empty else 0.0
            except Exception:
                curr_price = 0.0
            avg_cost = pos["cost"] / pos["qty"] if pos["qty"] > 0 else 0
            mkt_val  = pos["qty"] * curr_price
            unreal_pl= mkt_val - pos["cost"]
            unreal_pct = unreal_pl / pos["cost"] * 100 if pos["cost"] > 0 else 0
            rows.append({
                "ticker":    tk,
                "qty":       pos["qty"],
                "avg_cost":  round(avg_cost, 4),
                "curr_price":round(curr_price, 4),
                "mkt_value": round(mkt_val, 2),
                "cost_basis":round(pos["cost"], 2),
                "unrealised_pl": round(unreal_pl, 2),
                "unrealised_pct":round(unreal_pct, 2),
            })
        return pd.DataFrame(rows) if rows else pd.DataFrame(
            columns=["ticker","qty","avg_cost","curr_price","mkt_value",
                     "cost_basis","unrealised_pl","unrealised_pct"])
    except Exception as e:
        print(f"  open_positions error: {e}")
        return pd.DataFrame()

# Ã¢â€â‚¬Ã¢â€â‚¬ 60-day P&L summary Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def compute_60d_pnl():
    """
    Full 60-day P&L breakdown — fixed for all 6 tracking bugs:
    Bug1: equity curve uses realised P&L per day (not raw cash flows)
    Bug3: full log used for FIFO queue so BUYs >60 days old are matched
    Bug4: ROUND_TRIP_COST deducted from every matched lot
    Bug6: short positions tracked with negative queue entries
    """
    try:
        log = pd.read_csv(PT_LOG_FILE)
        log["ts"]    = pd.to_datetime(log["ts"], errors="coerce")
        log["price"] = pd.to_numeric(log["price"], errors="coerce").fillna(0)
        log["qty"]   = pd.to_numeric(log["qty"],   errors="coerce").fillna(0)
        log = log.dropna(subset=["ts"]).sort_values("ts")

        cutoff = pd.Timestamp.now() - pd.Timedelta(days=60)
        log60  = log[log["ts"] >= cutoff]

        if log60.empty:
            return dict(realised_pl=0, unrealised_pl=0, total_pl=0,
                        win_rate=None, trades_60d=0, equity_curve=[],
                        per_ticker={}, max_drawdown=0, sharpe=None)

        # ── FIFO matching: full log, count P&L from 60d SELLs only ──────────
        # Queues store [qty, price] where qty<0 = short position
        from collections import deque
        realised       = 0.0
        realised_by_tk = {}
        daily_realised = {}   # date_str -> realised P&L that day
        queues         = {}

        def _match_long_close(sell_qty, px, tk, in_window, date_key):
            """Consume long queue entries, credit P&L if in window."""
            nonlocal realised
            sq = sell_qty
            while sq > 0 and queues.get(tk) and queues[tk][0][0] > 0:
                eq, epx  = queues[tk][0]
                matched  = min(sq, eq)
                if in_window:
                    pl = matched*(px - epx) - matched*epx*ROUND_TRIP_COST
                    realised += pl
                    realised_by_tk[tk] = realised_by_tk.get(tk, 0) + pl
                    if date_key:
                        daily_realised[date_key] = daily_realised.get(date_key, 0) + pl
                sq                  -= matched
                queues[tk][0][0]    -= matched
                if queues[tk][0][0] == 0:
                    queues[tk].popleft()
            return sq  # leftover qty (new short if >0)

        def _match_short_cover(cover_qty, px, tk, in_window, date_key):
            """Consume short queue entries, credit P&L if in window."""
            nonlocal realised
            cq = cover_qty
            while cq > 0 and queues.get(tk) and queues[tk][0][0] < 0:
                short_abs = -queues[tk][0][0]
                spx       = queues[tk][0][1]
                matched   = min(cq, short_abs)
                if in_window:
                    pl = matched*(spx - px) - matched*spx*ROUND_TRIP_COST
                    realised += pl
                    realised_by_tk[tk] = realised_by_tk.get(tk, 0) + pl
                    if date_key:
                        daily_realised[date_key] = daily_realised.get(date_key, 0) + pl
                cq                  -= matched
                queues[tk][0][0]    += matched  # less negative
                if queues[tk][0][0] == 0:
                    queues[tk].popleft()
            return cq  # leftover qty (new long if >0)

        for _, row in log.iterrows():
            tk       = row["ticker"]
            qty      = int(row["qty"])
            px       = float(row["price"])
            in_window = row["ts"] >= cutoff
            date_key  = str(row["ts"].date())
            if tk not in queues:
                queues[tk] = deque()

            if row["action"] == "BUY":
                if queues[tk] and queues[tk][0][0] < 0:
                    leftover = _match_short_cover(qty, px, tk, in_window, date_key)
                    if leftover > 0:
                        queues[tk].append([leftover, px])
                else:
                    queues[tk].append([qty, px])

            elif row["action"] == "SELL":
                if queues[tk] and queues[tk][0][0] > 0:
                    leftover = _match_long_close(qty, px, tk, in_window, date_key)
                    if leftover > 0:
                        queues[tk].append([-leftover, px])
                else:
                    queues[tk].append([-qty, px])  # open short

        # ── Unrealised P&L from open positions (mark-to-market) ─────────────
        open_pos   = get_open_positions()
        unrealised = float(open_pos["unrealised_pl"].sum()) if not open_pos.empty else 0.0

        # ── Equity curve: cumulative realised P&L per day (Bug1 fix) ─────────
        dates_sorted = sorted(daily_realised.keys())
        cum_pl       = 0.0
        equity_curve = []
        for d in dates_sorted:
            cum_pl += daily_realised[d]
            equity_curve.append((d, round(PORTFOLIO_CAPITAL + cum_pl, 2)))
        today_str = str(pd.Timestamp.today().date())
        final_eq  = round(PORTFOLIO_CAPITAL + cum_pl + unrealised, 2)
        if not equity_curve or equity_curve[-1][0] != today_str:
            equity_curve.append((today_str, final_eq))
        else:
            equity_curve[-1] = (today_str, final_eq)

        # ── Max drawdown from corrected curve ────────────────────────────────
        mdd  = 0.0
        peak = PORTFOLIO_CAPITAL
        for _, v in equity_curve:
            peak = max(peak, v)
            dd   = (peak - v) / peak * 100 if peak > 0 else 0
            mdd  = max(mdd, dd)

        # ── Sharpe proxy ─────────────────────────────────────────────────────
        sharpe = None
        if len(equity_curve) > 5:
            eq_vals   = [v for _, v in equity_curve]
            daily_ret = np.diff(eq_vals) / np.array(eq_vals[:-1])
            if np.std(daily_ret) > 0:
                sharpe = round(float(np.mean(daily_ret)/np.std(daily_ret)*np.sqrt(252)), 3)

        # ── Win rate from prediction log ─────────────────────────────────────
        win_rate = None
        n_trades = len(log60)
        try:
            pred    = pd.read_csv(PRED_LOG_FILE)
            pred["ts"] = pd.to_datetime(
                pred.get("pred_ts", pred.get("ts", "")), errors="coerce")
            pred60  = pred[pred["ts"] >= cutoff]
            scored  = pred60[pred60["scored"].astype(str) == "True"]
            scored = scored.copy()
            scored["was_correct"] = scored["was_correct"].astype(str).map(
                {"True":True,"False":False,"true":True,"false":False}).fillna(False)
            if len(scored) > 0:
                win_rate = float(scored["was_correct"].mean())
                n_trades = len(scored)
        except Exception:
            pass

        return dict(
            realised_pl   = round(realised, 2),
            unrealised_pl = round(unrealised, 2),
            total_pl      = round(realised + unrealised, 2),
            win_rate      = win_rate,
            trades_60d    = n_trades,
            equity_curve  = equity_curve,
            per_ticker    = {k: round(v,2) for k,v in realised_by_tk.items()},
            max_drawdown  = round(mdd, 2),
            sharpe        = sharpe,
        )

    except Exception as e:
        print(f"  60d P&L error: {e}")
        return dict(realised_pl=0, unrealised_pl=0, total_pl=0,
                    win_rate=None, trades_60d=0, equity_curve=[],
                    per_ticker={}, max_drawdown=0, sharpe=None)


# Ã¢â€â‚¬Ã¢â€â‚¬ Prediction logger Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def log_prediction(sig):
    """Log every signal to prediction log for outcome scoring."""
    row = {col: None for col in PRED_LOG_COLS}
    row.update({
        "pred_ts":      datetime.datetime.utcnow().isoformat(),
        "ticker":       sig["ticker"],
        "action":       sig["action"],
        "confidence":   sig["confidence"],
        "price_at_pred":sig["close"],
        "p_ensemble":   sig.get("p_ensemble", None),
        "p_up_garch":   sig.get("garch_p_up", None),
        "rsi":          sig["rsi"],
        "regime":       sig["regime"],
        "vix":          MACRO.get("vix"),
        "yield_curve":  MACRO.get("yield_curve"),
        "ism_pmi":      MACRO.get("ism_pmi"),
        "unemployment": MACRO.get("unemployment"),
        "sentiment":    sig.get("sentiment", 0),
        "horizon_days": FORECAST_DAYS,
        "iv_flag":      sig.get("iv_flag", "NORMAL"),
        "iv_scale":     sig.get("iv_scale", 1.0),
        "iv_note":      sig.get("iv_note", ""),
        "scored":       False,
        "outcome_ts":   None,
        "price_at_outcome": None,
        "actual_return":    None,
        "was_correct":      None,
        "magnitude_error":  None,
    })
    try:
        existing = pd.read_csv(PRED_LOG_FILE) if Path(PRED_LOG_FILE).exists() else pd.DataFrame()
        updated  = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
        updated.to_csv(PRED_LOG_FILE, index=False)
    except Exception as e:
        print(f"  log_prediction error: {e}")

# Ã¢â€â‚¬Ã¢â€â‚¬ Trade executor Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def execute_trade(sig, qty, capital):
    """Execute a paper trade and append to PT log."""
    ts     = datetime.datetime.utcnow().isoformat()
    ticker = sig["ticker"]
    action = sig["action"]
    price  = sig["close"]

    if action not in ("BUY","SELL") or qty <= 0:
        return {"status":"skip","reason":"HOLD or qty=0"}

    order_id = _try_alpaca(action, ticker, qty) or "paper"

    row = {
        "ts":        ts,
        "ticker":    ticker,
        "action":    action,
        "price":     round(price, 4),
        "qty":       qty,
        "confidence":round(sig["confidence"], 4),
        "regime":    sig.get("regime", 0),
        "order_id":  order_id,
        "status":    "filled" if not str(order_id).startswith("alpaca_error") else "error",
        "run_date":  datetime.date.today().isoformat(),
        "iv_flag":   sig.get("iv_flag","NORMAL"),
        "iv_scale":  sig.get("iv_scale",1.0),
        "notional":  round(price * qty, 2),
    }

    try:
        existing = pd.read_csv(PT_LOG_FILE) if Path(PT_LOG_FILE).exists() else pd.DataFrame()
        updated  = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
        updated.to_csv(PT_LOG_FILE, index=False, encoding='utf-8')
    except Exception as e:
        print(f"  execute_trade error: {e}")
        return {"status":"error","reason":str(e)}

    # Log execution quality (filled=signal price for paper trades)
    try:
        log_execution_quality(ticker, action.lower(), price, price)
    except Exception:
        pass

    return row


# â”€â”€ Limit order with fill confirmation (Upgrade 3) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def submit_limit_order(api, symbol, qty, side, current_price, limit_offset=0.002, timeout_secs=60):
    """
    Submit a limit order with a fill-or-cancel timeout.
    limit_offset: how far from current price to set limit (0.2% default).
    Returns filled order or None if not filled within timeout.
    """
    if qty <= 0:
        return None
    if side == "buy":
        limit_price = round(current_price * (1 + limit_offset), 2)
    else:
        limit_price = round(current_price * (1 - limit_offset), 2)
    try:
        order = api.submit_order(
            symbol=symbol,
            qty=qty,
            side=side,
            type="limit",
            time_in_force="day",
            limit_price=str(limit_price),
        )
        print(f"    Limit order submitted: {side.upper()} {qty} {symbol} @ ${limit_price:.2f}")
        import time as _time
        deadline = _time.time() + timeout_secs
        while _time.time() < deadline:
            _time.sleep(5)
            try:
                o = api.get_order(order.id)
                if o.status == "filled":
                    filled_price = float(o.filled_avg_price)
                    print(f"    âœ“ Filled: {side.upper()} {qty} {symbol} @ ${filled_price:.2f}")
                    return o
                elif o.status in ("canceled", "expired", "rejected"):
                    print(f"    âœ— Order {o.status}: {side.upper()} {qty} {symbol}")
                    return None
            except Exception:
                pass
        try:
            api.cancel_order(order.id)
            print(f"    âœ— Order timeout â€” canceled: {side.upper()} {qty} {symbol}")
        except Exception:
            pass
        return None
    except Exception as e:
        print(f"    âœ— Order submission failed {symbol}: {e}")
        return None

# â”€â”€ Kill switch (Upgrade 4) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def check_kill_switch(api=None):
    """
    Returns (killed: bool, reason: str).
    Checks: flag file, daily P&L, consecutive losses, VIX spike.
    """
    # 1. Manual kill flag file
    if KILL_FLAG_FILE.exists():
        return True, f"Manual kill flag active: {KILL_FLAG_FILE}"

    # 2. VIX spike
    vix_now = MACRO.get("vix") or 0
    if vix_now >= KILL_VIX_THRESHOLD:
        return True, f"VIX={vix_now:.1f} >= threshold {KILL_VIX_THRESHOLD} â€” no new entries"

    # 3. Consecutive losses from prediction log
    try:
        plog = pd.read_csv(PRED_LOG_FILE)
        scored = plog[plog["scored"].astype(str) == "True"].tail(KILL_CONSECUTIVE_LOSSES)
        if len(scored) == KILL_CONSECUTIVE_LOSSES:
            if not scored["was_correct"].astype(str).isin(["True","true"]).any():
                return True, f"{KILL_CONSECUTIVE_LOSSES} consecutive losses â€” halting new entries"
    except Exception:
        pass

    # 4. Daily P&L from Alpaca
    if api:
        try:
            account = api.get_account()
            equity      = float(account.equity)
            last_equity = float(account.last_equity)
            daily_loss  = (equity - last_equity) / last_equity
            if daily_loss <= -KILL_DAILY_LOSS_PCT:
                return True, f"Daily loss={daily_loss:.1%} >= limit {KILL_DAILY_LOSS_PCT:.0%} â€” halting"
        except Exception:
            pass

    return False, ""

def activate_kill_switch(reason: str):
    """Write kill flag and log the reason."""
    KILL_FLAG_FILE.write_text(f"{pd.Timestamp.utcnow().isoformat()}\n{reason}\n")
    print(f"  \U0001f6a8 KILL SWITCH ACTIVATED: {reason}")

def reset_kill_switch():
    """Manually reset kill switch (call this to resume trading)."""
    if KILL_FLAG_FILE.exists():
        KILL_FLAG_FILE.unlink()
        print("  âœ“ Kill switch reset â€” trading resumed")
    else:
        print("  Kill switch was not active")

# â”€â”€ Position reconciliation (Upgrade 5) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def reconcile_positions(api=None):
    """
    Compare CSV-tracked positions against live Alpaca account.
    Returns dict of discrepancies. Logs warnings for mismatches.
    """
    discrepancies = {}
    if api is None:
        return discrepancies

    try:
        live_positions = {p.symbol: float(p.qty) for p in api.list_positions()}
    except Exception as e:
        print(f"  reconcile: could not fetch Alpaca positions â€” {e}")
        return discrepancies

    try:
        plog = pd.read_csv(PRED_LOG_FILE)
        csv_open = plog[plog["scored"].astype(str) == "False"]
        csv_positions = {}
        for _, row in csv_open.iterrows():
            tk = str(row.get("ticker", ""))
            if row.get("action") in ("BUY", "SELL") and tk:
                csv_positions[tk] = csv_positions.get(tk, 0) + 1
    except Exception:
        csv_positions = {}

    for sym, qty in live_positions.items():
        if sym not in csv_positions:
            discrepancies[sym] = {"issue": "orphaned_live_position", "alpaca_qty": qty, "csv_qty": 0}
            print(f"  âš  RECONCILE: {sym} has {qty} shares live but no CSV record")

    for sym in csv_positions:
        if sym not in live_positions:
            discrepancies[sym] = {"issue": "missing_live_position", "alpaca_qty": 0, "csv_count": csv_positions[sym]}
            print(f"  âš  RECONCILE: {sym} has {csv_positions[sym]} CSV open trades but no live position")

    if not discrepancies:
        print(f"  âœ“ Positions reconciled: {len(live_positions)} live, {len(csv_positions)} CSV â€” all match")
    else:
        print(f"  âš  {len(discrepancies)} reconciliation discrepancies â€” review before trading")

    return discrepancies

# â”€â”€ Discord trade alert â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _discord_alert(message):
    if not DISCORD_WEBHOOK_URL: return
    try: requests.post(DISCORD_WEBHOOK_URL, json={"content": message}, timeout=5)
    except Exception: pass

# â”€â”€ Stop-loss + position expiry â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def check_stops_and_expiry():
    closed = []
    try:
        log = pd.read_csv(PT_LOG_FILE)
        log["ts"]    = pd.to_datetime(log["ts"],  errors="coerce")
        log["price"] = pd.to_numeric(log["price"],errors="coerce").fillna(0)
        log["qty"]   = pd.to_numeric(log["qty"],  errors="coerce").fillna(0)
    except Exception: return closed
    now = pd.Timestamp.utcnow().replace(tzinfo=None)  # tz-naive for CSV compat
    open_pos = {}
    for _, row in log.sort_values("ts").iterrows():
        tk = row["ticker"]; qty = int(row["qty"])
        if row["action"] == "BUY":
            open_pos[tk] = {"qty":qty,"entry_price":row["price"],"entry_ts":row["ts"]}
        elif row["action"] == "SELL" and tk in open_pos:
            del open_pos[tk]
    for tk, pos in open_pos.items():
        try:
            hist     = yf.Ticker(tk).history(period="1d",auto_adjust=True)
            curr_px  = float(hist["Close"].iloc[-1]) if not hist.empty else 0.0
            entry_px = pos["entry_price"]
            age_days = (now - pos["entry_ts"]).days if pd.notna(pos["entry_ts"]) else 0
            pct_chg  = (curr_px - entry_px)/entry_px if entry_px > 0 else 0
            stop_hit   = pct_chg <= -STOP_LOSS_PCT
            expiry_hit = age_days >= FORECAST_DAYS
            # -- Priority #1: Trailing stop (high-water mark from entry)
            _trail_stop = False
            _trail_note = ""
            try:
                _hist_full = yf.Ticker(tk).history(
                    start=pos["entry_ts"].date(), auto_adjust=True)
                if not _hist_full.empty:
                    _peak_px = float(_hist_full["High"].max())
                    _peak_gain = (_peak_px - entry_px) / entry_px if entry_px > 0 else 0
                    _trail_pct = (curr_px - _peak_px) / _peak_px if _peak_px > 0 else 0
                    _ts_pct = globals().get("TRAILING_STOP_PCT", 0.07)
                    if _peak_gain > 0.05 and _trail_pct <= -_ts_pct:
                        _trail_stop = True
                        _trail_note = "trail({:.1%} from peak)".format(_trail_pct)
            except Exception: pass
            if stop_hit or expiry_hit or _trail_stop:
                reason = (_trail_note if _trail_stop else
                          "stop_loss({:.1%})".format(pct_chg) if stop_hit else
                          "expiry({}d)".format(age_days))
                fake_sig = dict(ticker=tk, action="SELL", close=curr_px,
                                confidence=0.5, regime=1, iv_flag="NORMAL", iv_scale=1.0)
                execute_trade(fake_sig, pos["qty"], _current_equity())
                msg = "FORCE-CLOSE {} x{} @ ${:.2f} [{}] P&L: {:.1%}".format(
                    tk, pos["qty"], curr_px, reason, pct_chg)
                print("  " + msg); _discord_alert(msg)
                closed.append(dict(ticker=tk, reason=reason, pct=pct_chg))
        except Exception as e:
            print("  stop/expiry error {}: {}".format(tk, e))
    return closed
# â”€â”€ Minimum position size filter (Upgrade 6) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def is_position_viable(qty: int, price: float) -> bool:
    """Returns False if the position is too small to cover round-trip costs."""
    dollars = qty * price
    if dollars < MIN_POSITION_DOLLARS:
        return False
    return True

# â”€â”€ PDT Rule Tracker (Upgrade 7) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def log_day_trade(symbol: str, side: str):
    """Record a completed day trade (same-day open+close)."""
    entry = pd.DataFrame([{"ts": pd.Timestamp.utcnow().isoformat(), "symbol": symbol, "side": side}])
    if PDT_LOG_FILE.exists():
        entry.to_csv(PDT_LOG_FILE, mode="a", header=False, index=False)
    else:
        entry.to_csv(PDT_LOG_FILE, index=False)

def check_pdt_available(api=None):
    """
    Returns (can_trade: bool, day_trades_used: int).
    If account equity < PDT_ACCOUNT_THRESHOLD, enforces 3-trade limit.
    """
    if api:
        try:
            equity = float(api.get_account().equity)
            if equity >= PDT_ACCOUNT_THRESHOLD:
                return True, 0
        except Exception:
            pass

    if not PDT_LOG_FILE.exists():
        return True, 0
    try:
        pdt = pd.read_csv(PDT_LOG_FILE)
        pdt["ts"] = pd.to_datetime(pdt["ts"], utc=True)
        cutoff = pd.Timestamp.utcnow() - pd.Timedelta(days=5)
        recent = pdt[pdt["ts"] >= cutoff]
        used   = len(recent)
        can_trade = used < PDT_MAX_DAY_TRADES
        if not can_trade:
            print(f"  âš  PDT LIMIT: {used}/{PDT_MAX_DAY_TRADES} day trades used in last 5 days")
        return can_trade, used
    except Exception:
        return True, 0

# Ã¢â€â‚¬Ã¢â€â‚¬ MAIN EXECUTION BLOCK Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬

# ── ALMGREN-CHRISS MARKET IMPACT ─────────────────────────────────────────────
def almgren_chriss_impact(qty, price, annual_vol, adv_shares=None):
    """
    Temporary market impact cost (fraction of price).
    impact = sigma_daily * sqrt(Q/ADV) * eta^0.6
    Replaces flat ROUND_TRIP_COST for large or illiquid orders.
    eta=0.1 (temporary impact coefficient), gamma=0.05 (permanent).
    """
    try:
        if adv_shares is None or adv_shares <= 0 or qty <= 0:
            return ROUND_TRIP_COST
        sigma_d = annual_vol / np.sqrt(252)
        q_adv   = qty / max(adv_shares, 1)
        eta     = 0.1
        impact  = sigma_d * np.sqrt(q_adv) * (eta * q_adv)**0.6
        return float(np.clip(ROUND_TRIP_COST + impact, 0, 0.05))
    except Exception:
        return ROUND_TRIP_COST

def _get_adv(ticker):
    """Average Daily Volume from yfinance info (30-day ADV in shares)."""
    try:
        _i = yf.Ticker(ticker).info
        return float(_i.get("averageVolume",0) or _i.get("averageDailyVolume10Day",0) or 0)
    except Exception:
        return 0.0

# ── ENGLE-GRANGER PAIRS SIGNALS ──────────────────────────────────────────────
def _eg_spread_zscore(tk1, tk2, lookback=90):
    """
    Engle-Granger cointegration z-score for pair (tk1, tk2).
    z > +2: tk1 expensive vs tk2 → SELL tk1, BUY tk2.
    z < -2: tk1 cheap vs tk2    → BUY  tk1, SELL tk2.
    """
    try:
        _p1 = featured.get(tk1, pd.DataFrame())
        _p2 = featured.get(tk2, pd.DataFrame())
        if _p1.empty or _p2.empty: return 0.0
        _y = _p1["Close"].tail(lookback).values
        _x = _p2["Close"].tail(lookback).values
        if len(_y) < 30 or len(_x) < 30: return 0.0
        _n = min(len(_y), len(_x)); _y = _y[-_n:]; _x = _x[-_n:]
        _X = np.column_stack([np.ones(_n), _x])
        _c, _, _, _ = np.linalg.lstsq(_X, _y, rcond=None)
        _spread = _y - _c[1]*_x
        _mu = _spread.mean(); _sd = _spread.std()
        if _sd < 1e-8: return 0.0
        return float(np.clip((_spread[-1]-_mu)/_sd, -5.0, 5.0))
    except Exception: return 0.0

# Predefined high-correlation pairs (sector peers)
_EG_PAIRS = [
    ("AAPL","MSFT"),("NVDA","AMD"),("JPM","BAC"),("V","MA"),
    ("GOOGL","META"),("AMZN","SHOP"),("GS","MS"),("XOM","CVX"),
    ("LLY","ABBV"),("UNH","CVS"),("TSLA","RIVN"),("COIN","HOOD"),
]

eg_pair_signals = {}
try:
    for _tk1, _tk2 in _EG_PAIRS:
        if _tk1 in featured and _tk2 in featured:
            _z = _eg_spread_zscore(_tk1, _tk2)
            if abs(_z) >= 2.0:
                eg_pair_signals[(_tk1,_tk2)] = _z
    if eg_pair_signals:
        print(f"  EG Pairs: {len(eg_pair_signals)} divergences (|z|>=2)")
        for (_t1,_t2), _z in sorted(eg_pair_signals.items(), key=lambda x:-abs(x[1])):
            _dir = "SHORT_1/LONG_2" if _z>0 else "LONG_1/SHORT_2"
            print(f"    {_t1}/{_t2}: z={_z:.2f} → {_dir}")
except Exception as _ege:
    print(f"  EG pairs error: {_ege}")

# ── KELLY WIN/LOSS RATIO CACHE ───────────────────────────────────────────────
_WL_RATIO = {}   # ticker -> avg_win/avg_loss ratio
try:
    if Path(PRED_LOG_FILE).exists():
        _wl_log = pd.read_csv(PRED_LOG_FILE)
        _wl_log = _wl_log[_wl_log["scored"].astype(str).isin(["True","true"])].copy()
        _wl_log["actual_return"] = pd.to_numeric(_wl_log["actual_return"], errors="coerce")
        _wl_log = _wl_log.dropna(subset=["actual_return"])
        for _wltk in _wl_log["ticker"].unique():
            _rows = _wl_log[_wl_log["ticker"] == _wltk]
            _wins  = _rows[_rows["actual_return"] > 0]["actual_return"]
            _losses= _rows[_rows["actual_return"] < 0]["actual_return"].abs()
            if len(_wins) >= 3 and len(_losses) >= 3:
                _WL_RATIO[_wltk] = float(np.clip(_wins.mean()/_losses.mean(), 0.1, 10.0))
    if _WL_RATIO:
        print(f"  Kelly W/L ratios loaded: {len(_WL_RATIO)} tickers "
              f"(avg={np.mean(list(_WL_RATIO.values())):.2f})")
except Exception as _wle:
    print(f"  WL ratio error: {_wle}")
print("\n" + "="*55)
print(" PAPER TRADE ENGINE Ã¢â‚¬â€ v25")
print("="*55)
print(f" Run date: {datetime.date.today()} | Capital: ${PORTFOLIO_CAPITAL:,.0f}")
print(f" Drive: {'mounted' if _drive_mounted else 'session-only'}")

equity = _current_equity()
print(f" Current equity: ${equity:,.2f}")

halt = equity < PORTFOLIO_CAPITAL * (1 - MAX_DRAWDOWN_PCT)
if halt:
    print(f" Ã¢â€ºâ€ MAX DRAWDOWN HIT Ã¢â‚¬â€ no new trades today")

trade_count = 0

print("  Checking stops and position expiry...")
forced = check_stops_and_expiry()
if forced: print("  {} force-closed".format(len(forced)))

# Fix #6: intraday runs â€” stop check complete, no new entries, exit cell
if globals().get("INTRADAY_STOPS_ONLY", False):
    print("\n  [INTRADAY] Stop/expiry check complete â€” skipping new entries (intraday run)")
    import sys as _isys; _isys.exit(0)

# â”€â”€ Kill switch check â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
killed, kill_reason = check_kill_switch(None)
if killed:
    print(f"\n  \U0001f6a8 KILL SWITCH: {kill_reason}")
    print("  No new positions will be opened today.")
    halt = True

# â”€â”€ Position reconciliation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
recon = reconcile_positions(None)
if recon:
    print(f"  âš  {len(recon)} position discrepancies detected â€” proceeding with caution")

# ── UPGRADE 2: Portfolio volatility targeting ────────────────────────────────
# Scale entire book so annualised portfolio vol <= TARGET_VOL (10% bull, 7% bear).
# Prevents regime changes from inadvertently multiplying risk exposure.
_TARGET_VOL   = {0: 0.07, 1: 0.10, 2: 0.08}.get(int(MACRO.get("macro_regime",1)), 0.10)
_vol_scalar   = 1.0
try:
    _open_v = get_open_positions()
    if not _open_v.empty and len(_open_v) >= 2:
        _vtks = _open_v["ticker"].tolist()
        _vrets = {}
        for _vtk in _vtks:
            try:
                _vh = yf.download(_vtk, period="60d", auto_adjust=True, progress=False)
                if not _vh.empty and len(_vh) > 5:
                    _vrets[_vtk] = _vh["Close"].squeeze().pct_change().dropna()
            except Exception: pass
        if len(_vrets) >= 2:
            _vdf = pd.DataFrame(_vrets).dropna()
            _w   = np.ones(len(_vdf.columns)) / len(_vdf.columns)
            _pv  = float(np.sqrt(_w @ _vdf.cov().values @ _w * 252))
            if _pv > 0.01:
                _vol_scalar = min(1.30, _TARGET_VOL / _pv)
                if _vol_scalar < 0.95:
                    print(f"  Vol targeting: portfolio σ={_pv:.1%} > target {_TARGET_VOL:.0%}"
                          f" → sizing ×{_vol_scalar:.2f}")
except Exception as _ve:
    pass

for tk, sig in signals.items():
    log_prediction(sig)
    if halt or sig["action"] == "HOLD":
        continue
    _iv_sc = float(sig.get("iv_scale", 1.0)) * float(sig.get("event_scale", 1.0))
    _open_pos_corr = {}
    try:
        _op2 = get_open_positions()
        if not _op2.empty:
            for _, _r2 in _op2.iterrows():
                _open_pos_corr[_r2["ticker"]] = {"dollars": float(_r2["mkt_value"])}
    except Exception:
        pass
    corr_scalar = get_correlation_scalar(tk, _open_pos_corr)
    lev_scalar = get_leverage_scalar()
    _regime_now = sig.get("regime", 1)
    _combined_scalar = get_combined_size_scalar(_regime_now)
    # CVaR-weight scaling: overweight high-CVaR-efficiency names
    _n_buys = max(len(buy_tickers), 1) if buy_tickers else 1
    _cvar_wt = opt_weights.get(tk, 1.0 / _n_buys) * _n_buys if (buy_tickers and opt_weights) else 1.0
    # Upgrade 3: per-ticker Kelly using walk-forward accuracy if available
    _tk_data  = TICKER_ACCURACY_WF.get(tk, {}) if "TICKER_ACCURACY_WF" in globals() else {}
    _tk_acc   = float(_tk_data) if isinstance(_tk_data, float) else float(_tk_data.get("rolling_acc", sig["confidence"]) if isinstance(_tk_data, dict) else sig["confidence"])
    _kelly_p  = _tk_acc if (_tk_data and _tk_acc > 0.45) else sig["confidence"]
    _wl_b  = _WL_RATIO.get(tk, None)
    _adv_s = _get_adv(tk)
    _ac_cost= almgren_chriss_impact(int(kelly_qty(_kelly_p, equity, sig["close"], win_loss_ratio=_wl_b) * _iv_sc * corr_scalar * _combined_scalar * _cvar_wt * _vol_scalar), sig["close"], sig.get("ann_vol",0.3), _adv_s)
    qty    = int(kelly_qty(_kelly_p, equity, sig["close"], win_loss_ratio=_wl_b) * _iv_sc * corr_scalar * _combined_scalar * _cvar_wt * _vol_scalar)
    if _combined_scalar < 1.0:
        print(f"    Dynamic leverage: {lev_scalar:.0%} (VIX={MACRO.get('vix',20):.1f})")
    if qty == 0:
        continue
    # Sector exposure check
    _open_pos_dict = {}
    try:
        _op = get_open_positions()
        if not _op.empty:
            for _, _row in _op.iterrows():
                _open_pos_dict[_row["ticker"]] = {"dollars": float(_row["mkt_value"])}
    except Exception:
        pass
    if not sector_allows_trade(tk, qty * sig["close"], _open_pos_dict, PORTFOLIO_CAPITAL):
        continue
    if not is_position_viable(qty, sig["close"]):
        print(f"    â­ {tk}: position too small â€” skipped")
        continue
    if sig["action"] == "SELL" and can_short(tk):
        s_qty, s_dollars = get_short_position_size(sig["confidence"], sig["close"], PORTFOLIO_CAPITAL)
        if s_qty > 0 and is_position_viable(s_qty, sig["close"]):
            print(f"  SHORT: {tk} x{s_qty} @ ${sig['close']:.2f} conf={sig['confidence']:.3f}")
            execute_trade(sig, s_qty, equity)
            trade_count += 1
        else:
            print(f"  SELL signal {tk}: short not viable, skipping")
    elif sig["action"] != "SELL":
        result = execute_trade(sig, qty, equity)
        iv_tag = f" [{sig.get('iv_flag','')} {_iv_sc:.2f}x]" if sig.get("iv_flag","NORMAL")!="NORMAL" else ""
        trade_msg = "{} {} x{} @ ${:.2f}{}".format(sig["action"],tk,qty,sig["close"],iv_tag)
        print("  "+trade_msg); _discord_alert(trade_msg)
        trade_count += 1

print(f"\n  {trade_count} trades executed | {len(signals)} predictions logged")

# Ã¢â€â‚¬Ã¢â€â‚¬ 60-day P&L summary Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
print("\n" + "-"*55)
print(" 60-DAY P&L SUMMARY")
print("-"*55)
pnl = compute_60d_pnl()
print(f"  Realised P&L:    ${pnl['realised_pl']:>+10,.2f}")
print(f"  Unrealised P&L:  ${pnl['unrealised_pl']:>+10,.2f}")
print(f"  Total P&L:       ${pnl['total_pl']:>+10,.2f}  ({pnl['total_pl']/PORTFOLIO_CAPITAL*100:+.1f}%)")
print(f"  Max drawdown:    {pnl['max_drawdown']:.1f}%")
if pnl["win_rate"] is not None:
    print(f"  Win rate:        {pnl['win_rate']:.1%}  ({pnl['trades_60d']} scored predictions)")
if pnl["sharpe"] is not None:
    print(f"  Sharpe proxy:    {pnl['sharpe']:.2f}")

# Open positions
open_pos = get_open_positions()
if not open_pos.empty:
    print(f"\n  Open positions ({len(open_pos)}):")
    for _, row in open_pos.iterrows():
        pl_c = "Ã¢â€“Â²" if row["unrealised_pl"] >= 0 else "Ã¢â€“Â¼"
        print(f"    {row['ticker']:<10} {int(row['qty'])} sh @ ${row['avg_cost']:,.2f} Ã¢â€ â€™ ${row['curr_price']:,.2f} "
              f"{pl_c} ${row['unrealised_pl']:+,.2f} ({row['unrealised_pct']:+.1f}%)")
else:
    print("\n  No open positions yet.")

print("="*55)



 PAPER TRADE ENGINE Ã¢â‚¬â€ v25
 Run date: 2026-05-03 | Capital: $10,000
 Drive: session-only
 Current equity: $10,000.00
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_LOG_FILE' is not defined
  log_prediction error: name 'PRED_L

In [ ]:
# ============================================================
# CELL 14 Ã¢â‚¬â€ OUTCOME SCORER
# ============================================================
# Checks all unscored predictions where horizon has passed.
# Downloads actual prices, scores each prediction, marks done.

def score_outcomes():
    """Score all mature unscored predictions against actual prices."""
    if not Path(PRED_LOG_FILE).exists():
        print("  No prediction log yet -- nothing to score on first run")
        return pd.DataFrame()
    plog = pd.read_csv(PRED_LOG_FILE)
    if plog.empty:
        print("  No predictions to score yet")
        return pd.DataFrame()

    plog["pred_ts"] = pd.to_datetime(plog["pred_ts"], errors="coerce", utc=True)
    now_utc = pd.Timestamp.utcnow()
    cutoff  = now_utc - pd.Timedelta(days=FORECAST_DAYS+1)

    unscored = plog[(plog["scored"].astype(str)=="False") &
                    (plog["pred_ts"] < cutoff)].copy()

    if unscored.empty:
        # Fix #7: score against historical price data already in _PRICE_CACHE
        # Finds any "unscored" predictions whose outcome date is covered by cached prices
        _hist_scored = 0
        try:
            _today = datetime.date.today()
            for _hidx, _hrow in plog[plog["scored"].astype(str)=="False"].iterrows():
                try:
                    _pred_dt = pd.to_datetime(_hrow["pred_ts"], utc=True).date()
                    _outcome_dt = _pred_dt + datetime.timedelta(days=FORECAST_DAYS)
                    if _outcome_dt > _today:
                        continue  # not matured yet
                    _htk = _hrow["ticker"]
                    _cached = _PRICE_CACHE.get(_htk)
                    if _cached is None or _cached.empty:
                        continue
                    _cached.index = pd.to_datetime(_cached.index).tz_localize(None)
                    _outcome_row = _cached[_cached.index.date >= _outcome_dt]
                    if _outcome_row.empty:
                        continue
                    _entry_px  = float(_hrow["price_at_pred"])
                    _exit_px   = float(_outcome_row["Close"].iloc[0])
                    _ret       = (_exit_px - _entry_px) / max(_entry_px, 1e-9)
                    _direction = int(_hrow.get("signal", 1))
                    _correct   = int((_direction == 1 and _ret > 0.01) or (_direction == 0 and _ret < -0.01))
                    plog.at[_hidx, "outcome_price"] = round(_exit_px, 4)
                    plog.at[_hidx, "actual_return"] = round(_ret, 6)
                    plog.at[_hidx, "correct"]       = _correct
                    plog.at[_hidx, "scored"]        = True
                    _hist_scored += 1
                except Exception:
                    continue
            if _hist_scored > 0:
                plog.to_csv(PRED_LOG_FILE, index=False)
                print(f"  Historical cache scoring: {_hist_scored} predictions scored from _PRICE_CACHE")
            else:
                print(f"  No mature unscored predictions (need {FORECAST_DAYS}+ days old)")
        except Exception as _hse:
            print(f"  Historical scoring error: {_hse}")
        return pd.DataFrame()

    print(f"  Scoring {len(unscored)} mature predictions...")
    newly_scored = []

    # Pre-fetch SPY return and per-ticker beta for alpha-adjusted scoring
    _spy_bench = 0.0
    _beta_cache = {}
    try:
        _spy_df = yf.Ticker("SPY").history(period=f"{FORECAST_DAYS + 5}d", auto_adjust=True)
        if len(_spy_df) > FORECAST_DAYS:
            _spy_bench = float(_spy_df["Close"].iloc[-1] /
                               _spy_df["Close"].iloc[-1 - FORECAST_DAYS] - 1)
    except Exception: pass
    try:
        for _btk in list(unscored["ticker"].unique()):
            try:    _beta_cache[_btk] = float(yf.Ticker(_btk).info.get("beta", 1.0) or 1.0)
            except: _beta_cache[_btk] = 1.0
    except Exception: pass

    for idx, row in unscored.iterrows():
        tk = row["ticker"]
        try:
            # Fetch recent prices to find outcome price
            df_now = download_ticker(tk, TRAIN_START, TRAIN_END)
            if df_now is None or df_now.empty:
                continue

            price_at_pred = float(row["price_at_pred"])
            # Find price at the actual horizon date (not just latest available)
            try:
                _pred_ts = pd.Timestamp(row["pred_ts"]).tz_localize(None) \
                    if pd.Timestamp(row["pred_ts"]).tz is None \
                    else pd.Timestamp(row["pred_ts"]).tz_convert(None)
                _horizon = (_pred_ts + pd.Timedelta(days=FORECAST_DAYS)).normalize()
                _idx = pd.to_datetime(df_now.index)
                if _idx.tz is not None:
                    _idx = _idx.tz_convert(None)
                _future = df_now.copy(); _future.index = _idx
                _future = _future[_future.index >= _horizon]
                price_now = float(_future["Close"].iloc[0]) if not _future.empty \
                    else float(df_now["Close"].iloc[-1])
            except Exception:
                price_now = float(df_now["Close"].iloc[-1])
            gross_return  = (price_now - price_at_pred) / price_at_pred
            actual_return = apply_transaction_costs(gross_return)  # net of costs

            # Alpha-adjusted correctness: compare to SPY-adjusted benchmark
            action  = str(row["action"])
            _beta_v = _beta_cache.get(tk, 1.0)
            _alpha  = actual_return - (_beta_v * _spy_bench)
            if action == "BUY":
                was_correct = _alpha > -0.005
            elif action == "SELL":
                was_correct = _alpha < 0.005
            else:
                # HOLD: fail if a significant move was missed (>4%)
                was_correct = abs(actual_return) <= 0.04

            conf = float(row["confidence"]) if pd.notna(row["confidence"]) else 0.5
            magnitude_error = abs(actual_return - (conf - 0.5))

            # Update the row
            plog.at[idx, "outcome_ts"]      = now_utc.isoformat()
            plog.at[idx, "price_at_outcome"] = price_now
            plog.at[idx, "actual_return"]    = round(actual_return, 4)
            plog.at[idx, "was_correct"]      = was_correct

            # ── Fama-French idiosyncratic alpha ──────────────────────────────
            try:
                _ff = ff_decompose_alpha(tk, actual_return,
                                         beta_mkt=_beta_cache.get(tk,1.0),
                                         spy_return=_spy_bench)
                plog.at[idx, "idio_alpha"]   = _ff["idio_alpha"]
                plog.at[idx, "factor_exp"]   = _ff["factor_exp"]
            except Exception:
                pass
            plog.at[idx, "magnitude_error"]  = round(magnitude_error, 4)
            plog.at[idx, "scored"]           = True

            newly_scored.append({
                "ticker":       tk,
                "action":       action,
                "confidence":   conf,
                "was_correct":  was_correct,
                "actual_return":actual_return,
                "regime":       row.get("regime"),
                "vix":          row.get("vix"),
                "rsi":          row.get("rsi"),
                "yield_curve":  row.get("yield_curve"),
                "ism_pmi":      row.get("ism_pmi"),
                "sentiment":    row.get("sentiment"),
            })

            result = "CORRECT" if was_correct else "WRONG"
            print(f"    {result} {tk:<8} {action:<4} "
                  f"pred_px=${price_at_pred:.2f} "
                  f"now=${price_now:.2f} "
                  f"ret={actual_return:+.1%}")

        except Exception as e:
            print(f"    ERROR scoring {tk}: {e}")

    # Save updated prediction log
    plog.to_csv(PRED_LOG_FILE, index=False)

    # Per-ticker rolling accuracy tracking
    if newly_scored:
        try:
            global TICKER_ACCURACY
            for rec in newly_scored:
                tk = rec["ticker"]
                entry = TICKER_ACCURACY.get(tk, {"correct": 0, "total": 0, "recent": []})
                entry["total"] += 1
                entry["correct"] += int(rec["was_correct"])
                entry["recent"] = (entry["recent"] + [int(rec["was_correct"])])[-30:]
                entry["rolling_acc"] = round(sum(entry["recent"]) / max(1, len(entry["recent"])), 3)
                TICKER_ACCURACY[tk] = entry
            Path(TICKER_ACC_FILE).write_text(json.dumps(TICKER_ACCURACY, indent=2))
        except Exception as _te:
            print(f"  ticker accuracy tracking error: {_te}")

    print(f"  Saved {len(newly_scored)} scored outcomes")

    # ── IC Decay report ──────────────────────────────────────────────────────
    try:
        _ic_decay = compute_ic_decay()
        if _ic_decay:
            _lags = sorted(_ic_decay.keys())
            _ics  = [_ic_decay[l] for l in _lags]
            _ic0  = _ics[0] if _ics else 0
            _half_life = None
            for _l, _ic in zip(_lags, _ics):
                if abs(_ic) < abs(_ic0) * 0.5:
                    _half_life = _l; break
            print(f"  Signal IC decay: IC[1d]={_ic0:+.4f}  "
                  + (f"half-life≈{_half_life}d" if _half_life else "no half-life found"))
    except Exception: pass

    # ── Conformal prediction coverage on new signals ─────────────────────────
    try:
        if newly_scored:
            _confs = [r["confidence"] for r in newly_scored]
            _avg_c = float(np.mean(_confs))
            _lo, _hi = conformal_interval(_avg_c)
            _fully_bullish  = _lo > 0.58
            _fully_bearish  = _hi < 0.42
            _status = ("STRONG_BUY" if _fully_bullish else
                       "STRONG_SELL" if _fully_bearish else "UNCERTAIN")
            print(f"  Conformal 90% coverage: [{_lo:.3f}, {_hi:.3f}]  → {_status}")
    except Exception: pass

    return pd.DataFrame(newly_scored)


# ── IC DECAY: Signal half-life from scored prediction log ────────────────────
def compute_ic_decay(lookback_days=180):
    """
    IC(tau) = Spearman(confidence, actual_return) at each lag tau.
    Estimates alpha half-life: IC(t_{1/2}) = IC_0/2.
    Guides optimal rebalance frequency.
    """
    try:
        if not Path(PRED_LOG_FILE).exists(): return {}
        _pl = pd.read_csv(PRED_LOG_FILE)
        _pl = _pl[_pl["scored"].astype(str).isin(["True","true"])].copy()
        _pl["pred_ts"]    = pd.to_datetime(_pl["pred_ts"],    errors="coerce", utc=True)
        _pl["outcome_ts"] = pd.to_datetime(_pl["outcome_ts"], errors="coerce", utc=True)
        _pl["actual_return"] = pd.to_numeric(_pl["actual_return"], errors="coerce")
        _pl["confidence"]    = pd.to_numeric(_pl["confidence"],    errors="coerce")
        _pl = _pl.dropna(subset=["actual_return","confidence","pred_ts","outcome_ts"])
        if len(_pl) < 10: return {}
        _pl["lag_days"] = ((_pl["outcome_ts"]-_pl["pred_ts"])
                           .dt.total_seconds()/86400).round().astype(int)
        from scipy.stats import spearmanr as _sr
        ic_by_lag = {}
        for _lag in range(1, min(lookback_days//5, 30)+1):
            _sub = _pl[_pl["lag_days"] == _lag]
            if len(_sub) < 5: continue
            _ic, _ = _sr(_sub["confidence"], _sub["actual_return"])
            if not np.isnan(_ic):
                ic_by_lag[_lag] = round(float(_ic), 4)
        return ic_by_lag
    except Exception: return {}

# ── FAMA-FRENCH FACTOR DECOMPOSITION ────────────────────────────────────────
def ff_decompose_alpha(ticker, actual_return, beta_mkt=1.0, spy_return=0.0):
    """
    Strip MKT, SMB, HML, MOM factor exposures to isolate idiosyncratic alpha.
    alpha_i = R_i - beta_mkt*(R_m-Rf) - beta_smb*SMB - beta_hml*HML - beta_mom*MOM
    Returns dict: factor_alphas, idio_alpha.
    """
    try:
        _i    = yf.Ticker(ticker).info
        _mcap = float(_i.get("marketCap") or 0)
        _bv   = float(_i.get("bookValue") or 0)
        _px   = float(_i.get("currentPrice") or _i.get("regularMarketPrice") or 0)
        # Factor betas (approximate from firm characteristics)
        _smb_beta = -0.25 if _mcap > 100e9 else (0.5 if _mcap < 5e9 else 0.1)
        _bp = _bv / max(_px, 0.01)
        _hml_beta = 0.4 if _bp > 0.8 else (-0.3 if _bp < 0.2 else 0.0)
        # Momentum: 12-1 month return as proxy
        _h = yf.Ticker(ticker).history(period="1y", auto_adjust=True)
        if len(_h) > 20:
            _mom = float(_h["Close"].iloc[-1]/_h["Close"].iloc[0] - 1)
        else: _mom = 0.0
        _mom_beta = 0.3 if _mom > 0.20 else (-0.3 if _mom < -0.20 else 0.0)
        # Factor returns (simplified daily proxies)
        _rf   = 0.05/252
        _mkt  = spy_return - _rf
        _smb  = -0.0005 if spy_return > 0 else 0.0005
        _hml  = 0.0002
        _mom_r= spy_return * 0.7
        _factor_exp = (beta_mkt*_mkt + _smb_beta*_smb +
                       _hml_beta*_hml + _mom_beta*_mom_r)
        _idio = actual_return - _factor_exp
        return dict(idio_alpha=round(float(_idio),4),
                    factor_exp =round(float(_factor_exp),4),
                    smb_beta   =round(float(_smb_beta),2),
                    hml_beta   =round(float(_hml_beta),2),
                    mom_beta   =round(float(_mom_beta),2))
    except Exception:
        return dict(idio_alpha=actual_return, factor_exp=0.0,
                    smb_beta=0.0, hml_beta=0.0, mom_beta=0.0)

# ── CONFORMAL PREDICTION COVERAGE ───────────────────────────────────────────
def conformal_interval(confidence, coverage=0.90):
    """
    Distribution-free conformal prediction interval.
    Calibration: s_i = |confidence_i - was_correct_i|  for scored predictions.
    q_hat = ceil((n+1)(1-alpha))/n  quantile of calibration scores.
    Coverage guarantee: P(Y in C(X)) >= 1-alpha (distribution-free).
    Trade only when full interval is fully bullish (>0.58) or bearish (<0.42).
    """
    try:
        if not Path(PRED_LOG_FILE).exists(): return (0.40, 0.60)
        _pl = pd.read_csv(PRED_LOG_FILE)
        _pl = _pl[_pl["scored"].astype(str).isin(["True","true"])].copy()
        _pl["confidence"]  = pd.to_numeric(_pl["confidence"],  errors="coerce")
        _pl["was_correct"] = _pl["was_correct"].astype(str).isin(["True","true"])
        _pl = _pl.dropna(subset=["confidence"])
        if len(_pl) < 20: return (0.40, 0.60)
        _scores = (_pl["confidence"] - _pl["was_correct"].astype(float)).abs().values
        _n      = len(_scores)
        _q_lvl  = np.ceil((_n+1)*(1-coverage)) / _n
        _q_hat  = float(np.quantile(_scores, min(_q_lvl, 1.0)))
        return (round(max(0.0, confidence-_q_hat),4),
                round(min(1.0, confidence+_q_hat),4))
    except Exception:
        return (0.40, 0.60)
def daily_summary():
    # Run at cycle start. Print yesterdays scored predictions, P&L, mistakes. Append to daily_pnl_log.csv.
    yesterday = (pd.Timestamp.today() - pd.Timedelta(days=1)).normalize()
    result = dict(date=yesterday.strftime("%Y-%m-%d"), trades=0, wins=0, losses=0,
                  accuracy=None, gross_pl=0.0, net_pl=0.0,
                  best_ticker=None, best_return=None, worst_ticker=None, worst_return=None)
    print("")
    print("=" * 52)
    print("DAILY SUMMARY -- " + yesterday.strftime("%Y-%m-%d"))
    print("=" * 52)
    # --- Scored predictions resolved yesterday (5-day scoring lag) ---
    try:
        if not Path(PRED_LOG_FILE).exists():
            print("  No prediction log yet.")
        else:
            plog = pd.read_csv(PRED_LOG_FILE)
            plog["outcome_ts"]    = pd.to_datetime(plog["outcome_ts"], errors="coerce")
            plog["was_correct"]   = plog["was_correct"].astype(str).isin(["True","true"])
            plog["actual_return"] = pd.to_numeric(plog["actual_return"], errors="coerce")
            scored_yest = plog[
                plog["scored"].astype(str).isin(["True","true"]) &
                (plog["outcome_ts"].dt.normalize() == yesterday)
            ].copy()
            if scored_yest.empty:
                print("  No predictions resolved yesterday.")
            else:
                wins   = int(scored_yest["was_correct"].sum())
                losses = len(scored_yest) - wins
                acc    = wins / len(scored_yest)
                best_idx  = scored_yest["actual_return"].idxmax()
                worst_idx = scored_yest["actual_return"].idxmin()
                best_row  = scored_yest.loc[best_idx]
                worst_row = scored_yest.loc[worst_idx]
                result.update(dict(
                    trades=len(scored_yest), wins=wins, losses=losses, accuracy=round(acc, 4),
                    best_ticker=str(best_row.get("ticker","?")),
                    best_return=round(float(best_row["actual_return"]), 4),
                    worst_ticker=str(worst_row.get("ticker","?")),
                    worst_return=round(float(worst_row["actual_return"]), 4)))
                print("  Resolved: " + str(len(scored_yest)) + " | Correct: " + str(wins) + " | Wrong: " + str(losses) + " | Acc: " + f"{acc:.1%}")
                print("  Best:  " + f"{str(best_row.get('ticker','?')):<8} {str(best_row.get('action','?')):<4} ret={float(best_row['actual_return']):+.2%}")
                print("  Worst: " + f"{str(worst_row.get('ticker','?')):<8} {str(worst_row.get('action','?')):<4} ret={float(worst_row['actual_return']):+.2%}")
                wrong = scored_yest[~scored_yest["was_correct"]]
                if not wrong.empty:
                    print("  Wrong calls (" + str(len(wrong)) + "):")
                    for _, r in wrong.iterrows():
                        print("    " + f"{str(r.get('ticker','?')):<8} {str(r.get('action','?')):<4} conf={float(r.get('confidence',0)):.2f} ret={float(r.get('actual_return',0)):+.2%}")
    except Exception as e:
        print("  Prediction summary error: " + str(e))
    # --- Paper trade P&L executed yesterday ---
    try:
        if Path(PT_LOG_FILE).exists():
            ptlog = pd.read_csv(PT_LOG_FILE)
            ptlog["ts"]    = pd.to_datetime(ptlog["ts"], errors="coerce")
            ptlog["price"] = pd.to_numeric(ptlog["price"], errors="coerce").fillna(0)
            ptlog["qty"]   = pd.to_numeric(ptlog["qty"],   errors="coerce").fillna(0)
            yest_t = ptlog[ptlog["ts"].dt.normalize() == yesterday]
            if yest_t.empty:
                print("  No trades executed yesterday.")
            else:
                buys     = yest_t[yest_t["action"] == "BUY"]
                sells    = yest_t[yest_t["action"] == "SELL"]
                buy_val  = float((buys["price"]  * buys["qty"]).sum())
                sell_val = float((sells["price"] * sells["qty"]).sum())
                if result["trades"] == 0:
                    result["trades"] = len(yest_t)
                # Bug5 fix: net_pl = sell-buy is cash-flow, not realised P&L for 5-day hold model.
                # Show capital deployed vs positions closed separately; use compute_60d_pnl for true P&L.
                _pnl_snap = compute_60d_pnl()
                result["gross_pl"] = round(_pnl_snap.get("realised_pl", 0), 2)
                result["net_pl"]   = round(_pnl_snap.get("total_pl", 0), 2)
                print("  Trades: " + str(len(yest_t)) +
                      f" | Deployed: ${buy_val:,.2f} | Closed: ${sell_val:,.2f}"
                      f" | 60d Realised P&L: ${result['gross_pl']:+,.2f}"
                      f" | Total (incl. open): ${result['net_pl']:+,.2f}")
    except Exception as e:
        print("  Trade P&L error: " + str(e))
    # --- Append one row to DAILY_PNL_LOG_FILE ---
    try:
        cols = ["date","trades","wins","losses","accuracy","gross_pl","net_pl",
                "best_ticker","best_return","worst_ticker","worst_return"]
        row_df = pd.DataFrame([{c: result.get(c) for c in cols}])
        if Path(DAILY_PNL_LOG_FILE).exists():
            exist  = pd.read_csv(DAILY_PNL_LOG_FILE)
            exist  = exist[exist["date"] != result["date"]]
            row_df = pd.concat([exist, row_df], ignore_index=True)
        row_df.to_csv(DAILY_PNL_LOG_FILE, index=False)
        print("  Daily log saved (" + str(len(row_df)) + " rows total)")
    except Exception as e:
        print("  Daily log write error: " + str(e))
    print("=" * 52)
    return result

def weekly_monthly_summary():
    """Print rolling 7-day and 30-day P&L/accuracy from the daily log."""
    if not Path(DAILY_PNL_LOG_FILE).exists():
        return
    try:
        dlog = pd.read_csv(DAILY_PNL_LOG_FILE)
        dlog["date"]     = pd.to_datetime(dlog["date"], errors="coerce")
        dlog["net_pl"]   = pd.to_numeric(dlog["net_pl"],   errors="coerce").fillna(0)
        dlog["accuracy"] = pd.to_numeric(dlog["accuracy"], errors="coerce")
        dlog["trades"]   = pd.to_numeric(dlog["trades"],   errors="coerce").fillna(0)
        dlog = dlog.dropna(subset=["date"]).sort_values("date")
        today = pd.Timestamp.today().normalize()
        for label, days in [("7-day", 7), ("30-day", 30)]:
            window = dlog[dlog["date"] >= today - pd.Timedelta(days=days)]
            if window.empty:
                continue
            total_pl   = window["net_pl"].sum()
            n_trades   = int(window["trades"].sum())
            avg_acc    = window["accuracy"].dropna().mean()
            acc_str    = f"{avg_acc:.1%}" if pd.notna(avg_acc) else "n/a"
            print(f"  {label} rollup | trades={n_trades}  net P&L=${total_pl:+,.2f}  acc={acc_str}")
    except Exception as e:
        print(f"  weekly/monthly rollup error: {e}")

weekly_monthly_summary()
newly_scored_df = score_outcomes()


NameError: name 'PRED_LOG_FILE' is not defined

In [ ]:
# ============================================================
# CELL 15 Ã¢â‚¬â€ FAILURE DIAGNOSIS ENGINE + RULE WRITER
# ============================================================
# Reads all scored outcomes. Finds statistically meaningful
# failure patterns. Writes new rules to LEARNED_RULES.
# River updates ADAPTIVE_WEIGHTS based on recent accuracy.

def diagnose_failures_and_rewrite_rules():
    """
    The self-learning core. Reads scored outcomes, finds where
    the model is systematically wrong, writes rules to fix it.
    """
    global ADAPTIVE_WEIGHTS, LEARNED_RULES

    if not Path(PRED_LOG_FILE).exists():
        print("  No prediction log yet -- cannot run diagnosis")
        return {}, []

    plog = pd.read_csv(PRED_LOG_FILE)
    scored = plog[plog["scored"].astype(str)=="True"].copy()

    if len(scored) < 5:
        print("  Not enough scored outcomes yet for diagnosis")
        print(f"  Have {len(scored)} Ã¢â‚¬â€ need at least 5")
        return {}, []

    scored["was_correct"] = scored["was_correct"].astype(str).map(
        {"True":True,"False":False,"true":True,"false":False}).fillna(False)
    scored["confidence"]  = pd.to_numeric(scored["confidence"],  errors="coerce").fillna(0.5)
    scored["rsi"]         = pd.to_numeric(scored["rsi"],         errors="coerce").fillna(50)
    scored["vix"]         = pd.to_numeric(scored["vix"],         errors="coerce").fillna(20)
    scored["regime"]      = pd.to_numeric(scored["regime"],      errors="coerce").fillna(1)
    scored["yield_curve"] = pd.to_numeric(scored["yield_curve"], errors="coerce").fillna(0)
    scored["actual_return"]= pd.to_numeric(scored["actual_return"],errors="coerce").fillna(0)

    insights    = []
    new_rules   = dict(LEARNED_RULES)  # start from existing
    MIN_SAMPLES = 2   # lowered: Laplace smoothing handles small-n instability
    PSEUDO = 1        # Laplace pseudo-count (Beta(1,1) uniform prior)

    def smooth_acc(successes, total):
        """Laplace-smoothed accuracy: avoids extreme estimates on tiny samples."""
        return (successes + PSEUDO) / (total + 2 * PSEUDO)

    overall_acc = scored["was_correct"].mean()
    print(f"  Overall accuracy: {overall_acc:.1%} across {len(scored)} predictions")

    # Ã¢â€â‚¬Ã¢â€â‚¬ Pattern 1: High RSI + Bear regime Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    mask = (scored["rsi"]>68) & (scored["regime"]==0) & (scored["action"]=="BUY")
    subset = scored[mask]
    if len(subset) >= MIN_SAMPLES:
        acc = smooth_acc(subset["was_correct"].sum(), len(subset))
        if acc < 0.40:  # wrong more than 60% of the time
            dampen = round(min(0.25, (0.50 - acc)), 2)
            new_rules["high_rsi_bear"] = {
                "dampen": dampen, "count": int(len(subset)),
                "accuracy": round(acc, 3),
                "description": f"BUY with RSI>68 in Bear regime Ã¢â‚¬â€ {acc:.0%} accurate ({len(subset)} samples)"
            }
            insights.append(f"RULE WRITTEN: High RSI Bear Ã¢â‚¬â€ BUY signals wrong {1-acc:.0%} of time "
                            f"({len(subset)} samples) Ã¢â€ â€™ dampening confidence by {dampen:.0%}")

    # Ã¢â€â‚¬Ã¢â€â‚¬ Pattern 2: VIX spike Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    mask = (scored["vix"]>28) & (scored["action"]=="BUY")
    subset = scored[mask]
    if len(subset) >= MIN_SAMPLES:
        acc = smooth_acc(subset["was_correct"].sum(), len(subset))
        if acc < 0.45:
            dampen = round(min(0.20, (0.50 - acc)), 2)
            new_rules["vix_spike"] = {
                "dampen": dampen, "count": int(len(subset)),
                "accuracy": round(acc, 3),
                "description": f"BUY when VIX>28 Ã¢â‚¬â€ {acc:.0%} accurate ({len(subset)} samples)"
            }
            insights.append(f"RULE WRITTEN: VIX spike Ã¢â‚¬â€ BUY signals wrong {1-acc:.0%} of time "
                            f"({len(subset)} samples) Ã¢â€ â€™ dampening by {dampen:.0%}")

    # Ã¢â€â‚¬Ã¢â€â‚¬ Pattern 3: Inverted yield curve Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    mask = (scored["yield_curve"]<-0.1) & (scored["action"]=="BUY")
    subset = scored[mask]
    if len(subset) >= MIN_SAMPLES:
        acc = smooth_acc(subset["was_correct"].sum(), len(subset))
        if acc < 0.45:
            dampen = round(min(0.20, (0.50 - acc)), 2)
            new_rules["inverted_yc"] = {
                "dampen": dampen, "count": int(len(subset)),
                "accuracy": round(acc, 3),
                "description": f"BUY with inverted yield curve Ã¢â‚¬â€ {acc:.0%} accurate ({len(subset)} samples)"
            }
            insights.append(f"RULE WRITTEN: Inverted yield curve Ã¢â‚¬â€ BUY wrong {1-acc:.0%} of time "
                            f"({len(subset)} samples) Ã¢â€ â€™ dampening by {dampen:.0%}")

    # Ã¢â€â‚¬Ã¢â€â‚¬ Pattern 4: Per-ticker accuracy Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    for tk in scored["ticker"].unique():
        tk_data = scored[(scored["ticker"]==tk) & (scored["action"]!="HOLD")]
        if len(tk_data) >= 5:
            acc = smooth_acc(tk_data["was_correct"].sum(), len(tk_data))
            if acc < 0.35:
                dampen = round(min(0.20, (0.45 - acc)), 2)
                rkey = f"ticker_{tk}"
                new_rules[rkey] = {
                    "dampen": dampen, "count": int(len(tk_data)),
                    "accuracy": round(acc, 3),
                    "description": f"{tk} systematically underperforming Ã¢â‚¬â€ {acc:.0%} accurate ({len(tk_data)} samples)"
                }
                insights.append(f"RULE WRITTEN: {tk} Ã¢â‚¬â€ only {acc:.0%} accurate "
                                f"({len(tk_data)} samples) Ã¢â€ â€™ dampening by {dampen:.0%}")

    # â”€â”€ Pattern 5: Rule back-validation + stale rule removal â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for key in list(new_rules.keys()):
        if key.startswith("_"): continue
        rule = new_rules[key]
        if rule.get("accuracy", 0) > 0.55 and rule.get("count", 0) >= 10:
            del new_rules[key]
            insights.append(f"RULE REMOVED: {key} â€” model improved to {rule['accuracy']:.0%}")
            continue
        applied = rule.get("applied_count", 0)
        if applied >= RULE_VALIDATION_MIN:
            post_acc   = rule.get("post_correct", 0) / max(1, applied)
            acc_before = rule.get("acc_before", overall_acc)
            if post_acc < acc_before - 0.05:
                del new_rules[key]
                insights.append(f"RULE INVALIDATED: {key} â€” post={post_acc:.0%} pre={acc_before:.0%}")
            else:
                insights.append(f"RULE VALIDATED: {key} â€” post={post_acc:.0%} (keeping)")
    # Ã¢â€â‚¬Ã¢â€â‚¬ River adaptive weight update Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬

    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    # RIVER ML â€” persistent online learning with drift detection
    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    if len(scored) >= 5:
        try:
            import pickle as _pkl
            from river import linear_model as _rlm, preprocessing as _rpp
            from river import metrics as _rm, drift as _rdrift

            _river_path = Path(RIVER_MODEL_FILE)

            # â”€â”€ Load persisted model or create fresh â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
            if _river_path.exists() and not hasattr(diagnose_failures_and_rewrite_rules, "_river_lr"):
                try:
                    with open(_river_path, "rb") as _f:
                        _saved = _pkl.load(_f)
                    diagnose_failures_and_rewrite_rules._river_scaler = _saved["scaler"]
                    diagnose_failures_and_rewrite_rules._river_lr     = _saved["lr"]
                    diagnose_failures_and_rewrite_rules._river_adwin  = _saved.get("adwin", _rdrift.ADWIN())
                    insights.append("RIVER ML: loaded persisted model from disk")
                except Exception:
                    diagnose_failures_and_rewrite_rules._river_scaler = _rpp.StandardScaler()
                    diagnose_failures_and_rewrite_rules._river_lr     = _rlm.LogisticRegression(l2=0.01)
                    diagnose_failures_and_rewrite_rules._river_adwin  = _rdrift.ADWIN()
            elif not hasattr(diagnose_failures_and_rewrite_rules, "_river_lr"):
                diagnose_failures_and_rewrite_rules._river_scaler = _rpp.StandardScaler()
                diagnose_failures_and_rewrite_rules._river_lr     = _rlm.LogisticRegression(l2=0.01)
                diagnose_failures_and_rewrite_rules._river_adwin  = _rdrift.ADWIN()

            _river_scaler = diagnose_failures_and_rewrite_rules._river_scaler
            _river_lr     = diagnose_failures_and_rewrite_rules._river_lr
            _river_adwin  = diagnose_failures_and_rewrite_rules._river_adwin
            _river_metric = _rm.Accuracy()
            _drift_detected = False

            for _, _row in scored.iterrows():
                _x = {
                    "confidence":   float(_row.get("confidence",   0.5)),
                    "rsi":          float(_row.get("rsi",          50)),
                    "vix":          float(_row.get("vix",          20)),
                    "regime":       float(_row.get("regime",        1)),
                    "yield_curve":  float(_row.get("yield_curve",  0)),
                    "actual_return":float(_row.get("actual_return", 0)),
                }
                _y   = bool(_row.get("was_correct", False))
                _ret = abs(float(_row.get("actual_return", 0)))
                # Return magnitude as sample weight (bigger wins/losses matter more)
                # return magnitude as sample weight â€” bigger moves matter more
                _vix_s = (max(0.6, min(2.5, float(MACRO.get("vix", 18) or 18) / 18))
                          if "MACRO" in globals() else 1.0)
                _w     = max(0.5, min(4.0, (1.0 + _ret * 15) * _vix_s))

                _xs = _river_scaler.learn_one(_x).transform_one(_x)
                _pred_p = _river_lr.predict_proba_one(_xs).get(True, 0.5)
                _river_metric.update(_y, _pred_p >= 0.5)

                # Approximate weighted update: learn multiple times for high-weight samples
                for _ in range(max(1, round(_w))):
                    _river_lr.learn_one(_xs, _y)

                # ADWIN drift detection
                _river_adwin.update(int(_y))
                if _river_adwin.drift_detected:
                    _drift_detected = True

            _river_acc = _river_metric.get()
            insights.append(f"RIVER ML: acc={_river_acc:.1%} on {len(scored)} samples")

            if _drift_detected:
                insights.append("ADWIN DRIFT DETECTED: market regime shifted â€” accelerating weight updates")
                _lr_step = LEARNING_RATE_FAST
            else:
                _lr_step = LEARNING_RATE_SLOW

            # â”€â”€ Meta-learner: adjust ADAPTIVE_WEIGHTS using River signal â”€
            _delta = _river_acc - overall_acc
            if abs(_delta) > 0.03:
                _adj = 1 if _delta > 0 else -1
                ADAPTIVE_WEIGHTS["w_ensemble"] = round(
                    min(0.65, max(0.40, ADAPTIVE_WEIGHTS["w_ensemble"] + _adj * _lr_step)), 3)
                insights.append(f"RIVER META: w_ensemble -> {ADAPTIVE_WEIGHTS['w_ensemble']:.3f} "
                                f"(step={_lr_step:.2f}, delta={_delta:+.1%})")

            # Also adjust w_garch if GARCH signals are in scored data
            if "ann_vol" in scored.columns:
                _high_vol = scored[scored["ann_vol"] > 0.35] if "ann_vol" in scored else pd.DataFrame()
                if len(_high_vol) >= 3:
                    _garch_acc = _high_vol["was_correct"].mean()
                    if _garch_acc > _river_acc + 0.08:
                        ADAPTIVE_WEIGHTS["w_garch"] = round(
                            min(0.30, ADAPTIVE_WEIGHTS["w_garch"] + _lr_step * 0.5), 3)
                        insights.append(f"RIVER META: w_garch boosted -> {ADAPTIVE_WEIGHTS['w_garch']:.3f}")

            # â”€â”€ Persist River model to disk â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
            try:
                with open(_river_path, "wb") as _f:
                    _pkl.dump({"scaler": _river_scaler, "lr": _river_lr,
                               "adwin": _river_adwin}, _f)
            except Exception as _pe:
                insights.append(f"RIVER persist error: {_pe}")

        except ImportError:
            pass
        except Exception as _river_e:
            insights.append(f"RIVER ML error: {_river_e}")
    # â”€â”€ Pattern 6: Boosting rules â€” reward patterns that work â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # If a condition has >65% accuracy on 5+ samples, write a boost rule
    BOOST_RULES = new_rules.get("_boosts", {})

    # Boost: Low RSI in Bull regime (oversold bounce)
    mask = (scored["rsi"] < 35) & (scored["regime"] == 1) & (scored["action"] == "BUY")
    subset = scored[mask]
    if len(subset) >= MIN_SAMPLES:
        acc = smooth_acc(subset["was_correct"].sum(), len(subset))
        if acc > 0.65:
            boost = round(min(0.15, acc - 0.60), 2)
            BOOST_RULES["low_rsi_bull"] = {
                "boost": boost, "count": int(len(subset)),
                "accuracy": round(acc, 3),
                "description": f"BUY with RSI<35 in Bull regime â€” {acc:.0%} accurate ({len(subset)} samples)"
            }
            insights.append(f"BOOST RULE: Low RSI Bull â€” {acc:.0%} accurate â†’ boosting confidence by {boost:.0%}")

    # Boost: High momentum + Bull regime
    mask = (scored["rsi"] > 55) & (scored["rsi"] < 68) & (scored["regime"] == 1) & (scored["action"] == "BUY")
    subset = scored[mask]
    if len(subset) >= MIN_SAMPLES:
        acc = smooth_acc(subset["was_correct"].sum(), len(subset))
        if acc > 0.65:
            boost = round(min(0.10, acc - 0.60), 2)
            BOOST_RULES["momentum_bull"] = {
                "boost": boost, "count": int(len(subset)),
                "accuracy": round(acc, 3),
                "description": f"BUY with RSI 55-68 in Bull â€” {acc:.0%} accurate ({len(subset)} samples)"
            }
            insights.append(f"BOOST RULE: Momentum Bull â€” {acc:.0%} accurate â†’ boosting by {boost:.0%}")

    # Boost: Low VIX environment
    mask = (scored["vix"] < 16) & (scored["action"] == "BUY")
    subset = scored[mask]
    if len(subset) >= MIN_SAMPLES:
        acc = smooth_acc(subset["was_correct"].sum(), len(subset))
        if acc > 0.65:
            boost = round(min(0.10, acc - 0.60), 2)
            BOOST_RULES["low_vix_env"] = {
                "boost": boost, "count": int(len(subset)),
                "accuracy": round(acc, 3),
                "description": f"BUY in low VIX (<16) â€” {acc:.0%} accurate"
            }
            insights.append(f"BOOST RULE: Low VIX â€” {acc:.0%} accurate â†’ boosting by {boost:.0%}")

    # Remove stale boost rules
    for key in list(BOOST_RULES.keys()):
        rule = BOOST_RULES[key]
        if rule.get("accuracy", 1) < 0.55 and rule.get("count", 0) >= 8:
            del BOOST_RULES[key]
            insights.append(f"BOOST RULE REMOVED: {key} â€” accuracy degraded to {rule['accuracy']:.0%}")

    new_rules["_boosts"] = BOOST_RULES
    recent = scored.tail(30)  # last 30 scored predictions
    if len(recent) >= 5:
        # Calculate feature-level accuracy
        ens_corr  = recent[recent["was_correct"]==True]["confidence"].mean()
        ens_wrong = recent[recent["was_correct"]==False]["confidence"].mean()

        # If ensemble is overconfident on wrong predictions, reduce weight
        if pd.notna(ens_wrong) and pd.notna(ens_corr):
            if ens_wrong > 0.63:  # overconfident on wrong predictions
                new_w_ens = max(0.40, ADAPTIVE_WEIGHTS["w_ensemble"] - 0.02)
                new_w_garch = min(0.28, ADAPTIVE_WEIGHTS["w_garch"] + 0.01)
                new_w_sent  = min(0.18, ADAPTIVE_WEIGHTS["w_sentiment"] + 0.01)
                ADAPTIVE_WEIGHTS["w_ensemble"]  = round(new_w_ens, 3)
                ADAPTIVE_WEIGHTS["w_garch"]     = round(new_w_garch, 3)
                ADAPTIVE_WEIGHTS["w_sentiment"] = round(new_w_sent, 3)
                insights.append(f"WEIGHT UPDATE: Ensemble overconfident on wrong predictions "
                                f"(avg conf={ens_wrong:.3f}). "
                                f"Reduced w_ensemble to {ADAPTIVE_WEIGHTS['w_ensemble']:.3f}")
            elif ens_wrong < 0.58 and ens_corr > 0.70:
                # Ensemble is well-calibrated Ã¢â‚¬â€ boost its weight
                new_w_ens = min(0.65, ADAPTIVE_WEIGHTS["w_ensemble"] + 0.01)
                ADAPTIVE_WEIGHTS["w_ensemble"] = round(new_w_ens, 3)
                insights.append(f"WEIGHT UPDATE: Ensemble well-calibrated Ã¢â‚¬â€ "
                                f"increased w_ensemble to {ADAPTIVE_WEIGHTS['w_ensemble']:.3f}")

    # Renormalise weights to sum to 1.0
    total = sum(ADAPTIVE_WEIGHTS.values())
    for k in ADAPTIVE_WEIGHTS:
        ADAPTIVE_WEIGHTS[k] = round(ADAPTIVE_WEIGHTS[k]/total, 4)


    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    # REGIME-CONDITIONAL RULE SETS
    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    _cur_regime = int(MACRO.get("macro_regime", 1)) if "MACRO" in globals() else 1
    _regime_key = {0: "bear", 1: "bull", 2: "neutral"}.get(_cur_regime, "neutral")
    _regime_rules = new_rules.setdefault(f"_regime_{_regime_key}", {})

    # Write regime-specific versions of strong patterns
    for _pk in ["high_rsi_bear", "vix_spike", "inverted_yc"]:
        if _pk in new_rules:
            _regime_rules[_pk] = new_rules[_pk]
    new_rules[f"_regime_{_regime_key}"] = _regime_rules

    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    # FEATURE IMPORTANCE TRACKING
    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    try:
        global FEATURE_IMPORTANCE
        _feat_cols = ["confidence", "rsi", "vix", "regime", "yield_curve"]
        for _fc in _feat_cols:
            if _fc in scored.columns:
                _corr = scored[_fc].corr(scored["was_correct"].astype(float))
                if pd.notna(_corr):
                    # Exponential moving average of importance
                    _prev = FEATURE_IMPORTANCE.get(_fc, 0)
                    FEATURE_IMPORTANCE[_fc] = round(0.7 * _prev + 0.3 * _corr, 4)
        FEATURE_IMPORTANCE["_updated"] = pd.Timestamp.today().isoformat()[:10]
        Path(FEATURE_IMP_FILE).write_text(json.dumps(FEATURE_IMPORTANCE, indent=2))
        insights.append(f"FEATURE IMP: top={max(FEATURE_IMPORTANCE, key=lambda k: abs(FEATURE_IMPORTANCE.get(k,0)) if k != '_updated' else 0, default='n/a')}")
    except Exception as _fie:
        insights.append(f"Feature importance error: {_fie}")

    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    # PER-TICKER CONFIDENCE CALIBRATION DECAY
    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    try:
        global TICKER_CALIB
        for _, _row in scored.iterrows():
            _tk = _row.get("ticker", "")
            if not _tk: continue
            _cur   = TICKER_CALIB.get(_tk, 1.0)
            _conf  = float(_row.get("confidence", 0.65) or 0.65)
            # Scale by confidence: overconfident wrong calls penalised harder
            _cscale = 1.0 + max(0.0, (_conf - 0.65) * 4)  # 0.65â†’1.0x  0.75â†’1.4x  0.85â†’1.8x
            if bool(_row.get("was_correct", False)):
                TICKER_CALIB[_tk] = round(min(1.25, _cur * (1.065 ** (1.0 / _cscale))), 3)
            else:
                TICKER_CALIB[_tk] = round(max(0.60, _cur * (TICKER_CALIBRATION_DECAY ** _cscale)), 3)
        Path(TICKER_CALIB_FILE).write_text(json.dumps(TICKER_CALIB, indent=2))
        _worst = min(TICKER_CALIB, key=TICKER_CALIB.get) if TICKER_CALIB else "n/a"
        _best  = max(TICKER_CALIB, key=TICKER_CALIB.get) if TICKER_CALIB else "n/a"
        insights.append(f"CALIB DECAY: worst={_worst}({TICKER_CALIB.get(_worst,1):.2f}) "
                        f"best={_best}({TICKER_CALIB.get(_best,1):.2f})")
    except Exception as _cde:
        insights.append(f"Calibration decay error: {_cde}")

    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    # CROSS-TICKER RULE TRANSFER VIA SECTOR GROUPING
    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    try:
        _sector_map = TICKER_SECTOR if "TICKER_SECTOR" in globals() else {}
        _sector_acc = {}
        for _sector in set(_sector_map.values()):
            _stickers = [t for t, s in _sector_map.items() if s == _sector]
            _sscore   = scored[scored["ticker"].isin(_stickers)]
            if len(_sscore) >= SECTOR_RULE_MIN_SAMPLES:
                _sector_acc[_sector] = _sscore["was_correct"].astype(bool).mean()

        for _tk in scored["ticker"].unique():
            _tk_data = scored[scored["ticker"] == _tk]
            if len(_tk_data) < 5:
                _sec = _sector_map.get(_tk)
                if _sec and _sec in _sector_acc:
                    _sacc = _sector_acc[_sec]
                    if _sacc < 0.42:
                        _rkey = f"sector_{_sec}"
                        new_rules[_rkey] = {
                            "dampen":      round(min(0.12, 0.50 - _sacc), 2),
                            "count":       int(len(scored[scored["ticker"].isin(
                                            [t for t,s in _sector_map.items() if s==_sec])])),
                            "accuracy":    round(_sacc, 3),
                            "acc_before":  round(overall_acc, 3),
                            "applied_count": new_rules.get(_rkey, {}).get("applied_count", 0),
                            "post_correct":  new_rules.get(_rkey, {}).get("post_correct", 0),
                            "description": f"Sector {_sec} underperforming ({_sacc:.0%}) â€” transferred to {_tk}"
                        }
                        insights.append(f"SECTOR TRANSFER: {_tk} inherits {_sec} rule (acc={_sacc:.0%})")
    except Exception as _ste:
        insights.append(f"Sector transfer error: {_ste}")

    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    # SURGICAL PER-TICKER STALENESS (integrated into rules)
    # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    try:
        global TICKER_ACCURACY
        _stale = [tk for tk, data in TICKER_ACCURACY.items()
                  if data.get("rolling_acc", 1) < STALENESS_ACC_FLOOR
                  and data.get("total", 0) >= 5]
        if _stale:
            new_rules["_stale_tickers"] = {"tickers": _stale,
                                           "updated": pd.Timestamp.today().isoformat()[:10]}
            insights.append(f"SURGICAL STALE: {len(_stale)} tickers flagged for retrain: {_stale[:5]}")
    except Exception as _sse:
        insights.append(f"Surgical staleness error: {_sse}")
    # â”€â”€ #5: Stamp all new rules with creation regime + date â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    _today_str = pd.Timestamp.today().strftime("%Y-%m-%d")
    for _rk, _rv in new_rules.items():
        if isinstance(_rv, dict) and "created_date" not in _rv:
            _rv["created_date"]   = _today_str
            _rv["created_regime"] = _cur_regime
    # â”€â”€ #2: Age-decay â€” rules lose half-strength after ~45 days â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    try:
        _today_dt = pd.Timestamp.today().normalize()
        for _rk in list(new_rules.keys()):
            _rv = new_rules.get(_rk, {})
            if not isinstance(_rv, dict) or "dampen" not in _rv or "created_date" not in _rv:
                continue
            _age_d = max(0, (_today_dt - pd.Timestamp(_rv["created_date"])).days)
            _rv["dampen"] = round(_rv["dampen"] * (0.985 ** _age_d), 4)
            if _rv["dampen"] < 0.02:
                new_rules.pop(_rk, None)
                insights.append(f"RULE EXPIRED: {_rk} (age={_age_d}d, decayed to noise)")
    except Exception as _age_e:
        insights.append(f"Age-decay error: {_age_e}")
    # â”€â”€ #6: Nightly meta-learner refresh: re-blend XGB / LGB / CatBoost â”€â”€
    try:
        if "PRED_LOG_FILE" in globals() and Path(PRED_LOG_FILE).exists():
            _pf  = pd.read_csv(PRED_LOG_FILE)
            _pf["was_correct"] = _pf["was_correct"].astype(str).isin(["True","true"])
            _mdf = _pf[_pf["scored"].astype(str).isin(["True","true"])].tail(60)
            _mc  = ["p_xgb","p_lgb","p_cat","was_correct"]
            if all(c in _mdf.columns for c in _mc) and len(_mdf) >= 15:
                from sklearn.linear_model import LogisticRegression as _MrLR
                _mX = _mdf[["p_xgb","p_lgb","p_cat"]].fillna(1/3).values
                _my = _mdf["was_correct"].astype(int).values
                if len(np.unique(_my)) > 1:
                    _mc2 = _MrLR(C=2.0, max_iter=300, solver="lbfgs")
                    _mc2.fit(_mX, _my)
                    _mw  = np.clip(_mc2.coef_[0], 0, None)
                    _mn  = _mw / _mw.sum() if _mw.sum() > 0 else np.array([1/3,1/3,1/3])
                    Path("data/weights/meta_weights.json").write_text(
                        json.dumps({"xgb":round(float(_mn[0]),3),
                                    "lgb":round(float(_mn[1]),3),
                                    "cat":round(float(_mn[2]),3),
                                    "updated":_today_str}, indent=2))
                    insights.append(f"META REFRESH: xgb={_mn[0]:.2f} lgb={_mn[1]:.2f}"
                                    f" cat={_mn[2]:.2f} (n={len(_mdf)})")
    except Exception as _mre:
        insights.append(f"Meta-learner refresh error: {_mre}")
    # Save everything to Drive
    LEARNED_RULES = new_rules
    Path(WEIGHTS_FILE).write_text(json.dumps(ADAPTIVE_WEIGHTS, indent=2))
    Path(RULES_FILE).write_text(json.dumps(LEARNED_RULES, indent=2))

    print(f"  Adaptive weights: {ADAPTIVE_WEIGHTS}")
    print(f"  Active learned rules: {len(LEARNED_RULES)}")
    return new_rules, insights

print("Running failure diagnosis and rule-writing engine...")
new_rules, insights = diagnose_failures_and_rewrite_rules()

if insights:
    print("\nInsights:")
    for ins in insights:
        print(f"  > {ins}")
else:
    print("\nNo new rules written Ã¢â‚¬â€ accumulating more data...")


# â”€â”€ Model staleness detector + auto-retrain trigger (Upgrade 8) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def check_model_staleness() -> bool:
    """
    Check if rolling accuracy has degraded below floor.
    If so, set retrain flag and return True.
    """
    try:
        plog = pd.read_csv(PRED_LOG_FILE)
        scored = plog[plog["scored"].astype(str) == "True"].copy()
        if len(scored) < STALENESS_WINDOW:
            return False

        scored["was_correct"] = scored["was_correct"].astype(str).isin(["True","true"])
        rolling_acc = scored["was_correct"].tail(STALENESS_WINDOW).mean()

        print(f"  Model staleness check: rolling acc={rolling_acc:.1%} "
              f"(floor={STALENESS_ACC_FLOOR:.0%}, window={STALENESS_WINDOW})")

        if rolling_acc < STALENESS_ACC_FLOOR:
            MODEL_RETRAIN_FLAG.write_text(
                f"{pd.Timestamp.utcnow().isoformat()}\n"
                f"Rolling accuracy={rolling_acc:.1%} below floor={STALENESS_ACC_FLOOR:.0%}\n"
            )
            print(f"  \U0001f504 STALENESS FLAG SET: model accuracy degraded â€” retraining on next run")
            return True
        elif MODEL_RETRAIN_FLAG.exists():
            MODEL_RETRAIN_FLAG.unlink()
            print(f"  âœ“ Model accuracy recovered â€” retrain flag cleared")

        return False
    except Exception as e:
        print(f"  staleness check error: {e}")
        return False


def maybe_retrain_all_models(df_features_by_ticker: dict):
    """
    Surgical retrain: only retrain tickers flagged stale by per-ticker accuracy.
    Falls back to full retrain if the global staleness flag is set.
    """
    global LEARNED_RULES
    _stale_list = LEARNED_RULES.get("_stale_tickers", {}).get("tickers", [])
    _targets    = [tk for tk in _stale_list if tk in df_features_by_ticker]

    if _targets:
        print(f"\n  SURGICAL RETRAIN: {len(_targets)} stale tickers: {_targets}")
        retrained = 0
        for tk in _targets:
            try:
                models[tk] = train_ensemble(df_features_by_ticker[tk], tk)
                retrained += 1
                print(f"    Retrained: {tk}")
            except Exception as e:
                print(f"    Retrain failed {tk}: {e}")
        if retrained > 0:
            LEARNED_RULES.pop("_stale_tickers", None)
            print(f"  Surgical retrain complete ({retrained} tickers)")
        return retrained > 0

    if not MODEL_RETRAIN_FLAG.exists():
        return False

    print(f"\n  AUTO-RETRAIN TRIGGERED (staleness flag active)")
    retrained = 0
    for tk, df_feat in df_features_by_ticker.items():
        try:
            models[tk] = train_ensemble(df_feat, tk)
            retrained += 1
            print(f"    Retrained: {tk}")
        except Exception as e:
            print(f"    Retrain failed {tk}: {e}")

    if retrained > 0:
        MODEL_RETRAIN_FLAG.unlink()
        print(f"  Auto-retrain complete ({retrained} tickers) â€” flag cleared")
    return retrained > 0

In [ ]:
# ============================================================
# CELL 16 Ã¢â‚¬â€ QUANT TERMINAL v25 HOMEPAGE
# ============================================================
# Renders automatically on Run All.
# Search: change SEARCH_TICKER and press Shift+Enter.
# ============================================================
%matplotlib inline
from IPython.display import display, HTML, clear_output
import datetime

# =============================================
SEARCH_TICKER = "NONE"
# Change to any ticker to run deep analysis:
# e.g.  AAPL  NVDA  TSLA  BTC-USD  GOOGL
# Set to "NONE" to show dashboard only
# =============================================

def _fmt(val, fmt=".2f", suffix="", prefix=""):
    if val is None: return "N/A"
    try: return f"{prefix}{val:{fmt}}{suffix}"
    except Exception: return str(val)

def _bar(pct, color):
    w = max(2, min(int(pct), 100))
    return (f'<div style="height:3px;background:var(--color-background-secondary);'
            f'border-radius:2px;margin-top:4px">'
            f'<div style="width:{w}%;height:3px;background:{color};border-radius:2px"></div></div>')

def _macro_card(label, val_str, sub, bar_pct, bar_color, sub_color=None):
    sc = sub_color or "var(--color-text-secondary)"
    return (
        f'<div style="background:var(--color-background-primary);border:0.5px solid '
        f'var(--color-border-tertiary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);margin-bottom:3px;'
        f'letter-spacing:.04em">{label}</div>'
        f'<div style="font-size:16px;font-weight:500;color:var(--color-text-primary);'
        f'margin-bottom:1px">{val_str}</div>'
        f'<div style="font-size:10px;color:{sc}">{sub}</div>'
        f'{_bar(bar_pct, bar_color)}</div>'
    )


def generate_reasoning(sig, garch, sent_sc, regime, regime_labels, mp):
    action=sig["action"]; conf=sig["confidence"]; rsi=sig["rsi"]
    p_ens=sig["p_ensemble"]; p_up=garch["p_up"]; ann_vol=garch["annvol"]
    vix=MACRO.get("vix") or 20; yc=MACRO.get("yield_curve") or 0
    pmi=MACRO.get("ism_pmi") or 50; auc=mp["auc"]
    reasons=[]; cautions=[]
    if p_ens>=0.70: reasons.append(f"ML ensemble strongly bullish ({p_ens:.0%} upside probability across all three models).")
    elif p_ens>=0.60: reasons.append(f"ML ensemble moderately bullish ({p_ens:.0%} upside probability).")
    elif p_ens<=0.35: cautions.append(f"ML ensemble bearish ({p_ens:.0%} upside probability).")
    elif p_ens<=0.45: cautions.append(f"ML ensemble leaning bearish ({p_ens:.0%}).")
    else: reasons.append(f"ML ensemble neutral ({p_ens:.0%}) Ã¢â‚¬â€ no strong directional edge.")
    if p_up>=0.60: reasons.append(f"GARCH Monte Carlo shows {p_up:.0%} probability of positive returns over {FORECAST_DAYS} days.")
    elif p_up<=0.40: cautions.append(f"GARCH gives only {p_up:.0%} upside probability.")
    if ann_vol>0.50: cautions.append(f"Elevated annualized volatility at {ann_vol:.0%}.")
    if rsi>70: cautions.append(f"RSI overbought at {rsi:.1f} Ã¢â‚¬â€ short-term pullback risk.")
    elif rsi<30: reasons.append(f"RSI oversold at {rsi:.1f} Ã¢â‚¬â€ potential mean-reversion.")
    elif 45<=rsi<=60: reasons.append(f"RSI healthy at {rsi:.1f}.")
    if sent_sc>0.15: reasons.append(f"News sentiment positive ({sent_sc:+.2f}).")
    elif sent_sc<-0.15: cautions.append(f"News sentiment negative ({sent_sc:+.2f}).")
    if regime==1: reasons.append(f"Stock in Bull/Trending regime (HMM state 1).")
    elif regime==0: cautions.append(f"Stock in Bear/Volatile regime (HMM state 0).")
    macro_notes=[]
    if vix>25: cautions.append(f"VIX elevated at {vix:.1f} Ã¢â‚¬â€ confidence dampened.")
    elif vix<16: macro_notes.append(f"low VIX ({vix:.1f})")
    if yc>0: macro_notes.append(f"normal yield curve (+{yc:.2f}%)")
    elif yc<0: cautions.append(f"Inverted yield curve ({yc:+.2f}%).")
    if pmi>50: macro_notes.append(f"ISM PMI {pmi:.1f} (expansion)")
    elif pmi<50: cautions.append(f"ISM PMI below 50 ({pmi:.1f}).")
    if macro_notes: reasons.append(f"Macro supportive: {', '.join(macro_notes)}.")
    if auc>=0.60: reasons.append(f"Model AUC {auc:.3f} Ã¢â‚¬â€ statistically meaningful predictions.")
    elif auc<0.52: cautions.append(f"Low model AUC ({auc:.3f}) Ã¢â‚¬â€ treat with caution.")
    rules=sig.get("rules_applied",[])
    if rules: cautions.append(f"Self-written rules active: {', '.join(rules)}.")
    if action=="BUY": summary=f"Signal: BUY with {conf:.0%} confidence. Bullish factors outweigh bearish ones."
    elif action=="SELL": summary=f"Signal: SELL with {1-conf:.0%} bearish confidence. Downside risk dominates."
    else: summary=f"Signal: HOLD Ã¢â‚¬â€ confidence ({conf:.3f}) between buy ({MIN_CONFIDENCE:.2f}) and sell ({1-MIN_CONFIDENCE:.2f}) thresholds."
    return dict(summary=summary,reasons=reasons,cautions=cautions,action=action)


def on_demand_analysis(ticker):
    ticker=ticker.strip().upper()
    if not ticker or ticker=="NONE": return
    display(HTML(
        f'<div style="background:var(--color-background-secondary);border-radius:8px;'
        f'padding:12px 16px;font-family:monospace;margin:8px 0">'
        f'<span style="font-size:13px;font-weight:500;color:var(--color-text-info)">Analyzing: {ticker}</span>'
        f'<span style="font-size:11px;color:var(--color-text-secondary);margin-left:10px">running full pipeline...</span>'
        f'</div>'))
    df_raw=download_ticker(ticker,TRAIN_START,TRAIN_END)
    if df_raw is None or len(df_raw)<200:
        display(HTML(f'<div style="color:var(--color-text-danger);padding:8px">Could not download data for {ticker}</div>'))
        return
    print(f"  Data: {len(df_raw)} rows | ${df_raw['Close'].iloc[-1]:.2f}")
    try: df_feat=build_features(df_raw)
    except Exception as e: print(f"  Feature error: {e}"); return
    if ticker in models:
        mp=models[ticker]; print(f"  Model (cached) AUC={mp['auc']:.3f}")
    else:
        print(f"  Training model for {ticker}...")
        try:
            mp=train_ensemble(df_feat,ticker); models[ticker]=mp; featured[ticker]=df_feat
            print(f"  Trained AUC={mp['auc']:.3f}")
        except Exception as e: print(f"  Training failed: {e}"); return
    reg_series=fit_hmm(df_raw); regime=int(reg_series.iloc[-1])
    regime_labels={0:"Bear/Volatile",1:"Bull/Trending",2:"Neutral/Sideways"}
    garch=garch_vol_forecast(df_feat,ticker,n_paths=1000)
    headlines=fetch_headlines(ticker,n=6); sent_sc=sentiment_score(headlines)
    sig=generate_signal(ticker,mp,df_feat,regime,garch,sent_sc)
    equity_now=_current_equity()
    qty=kelly_qty(sig["confidence"],equity_now,sig["close"])
    dollar_exp=qty*sig["close"]
    aclr={"BUY":"var(--color-text-success)","SELL":"var(--color-text-danger)","HOLD":"var(--color-text-warning)"}.get(sig["action"],"var(--color-text-primary)")
    abg={"BUY":"var(--color-background-success)","SELL":"var(--color-background-danger)","HOLD":"var(--color-background-warning)"}.get(sig["action"],"var(--color-background-secondary)")
    display(HTML(
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:10px;overflow:hidden;margin:8px 0;font-family:var(--font-sans)">'
        f'<div style="background:var(--color-background-secondary);padding:10px 16px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;justify-content:space-between;align-items:center">'
        f'<span style="font-size:15px;font-weight:500;color:var(--color-text-primary)">{ticker}</span>'
        f'<span style="background:{abg};color:{aclr};padding:3px 12px;border-radius:6px;font-size:11px;font-weight:500">{sig["action"]}</span></div>'
        f'<div style="display:grid;grid-template-columns:repeat(3,1fr)">'
        f'<div style="padding:10px 14px;border-right:0.5px solid var(--color-border-tertiary);border-bottom:0.5px solid var(--color-border-tertiary)"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">CONFIDENCE</div><div style="font-size:18px;font-weight:500;color:{aclr}">{sig["confidence"]:.3f}</div></div>'
        f'<div style="padding:10px 14px;border-right:0.5px solid var(--color-border-tertiary);border-bottom:0.5px solid var(--color-border-tertiary)"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">CLOSE PRICE</div><div style="font-size:18px;font-weight:500;color:var(--color-text-primary)">${sig["close"]:,.2f}</div></div>'
        f'<div style="padding:10px 14px;border-bottom:0.5px solid var(--color-border-tertiary)"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">POSITION</div><div style="font-size:18px;font-weight:500;color:var(--color-text-info)">{qty} sh = ${dollar_exp:,.0f}</div></div>'
        f'<div style="padding:10px 14px;border-right:0.5px solid var(--color-border-tertiary)"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">RSI (14)</div><div style="font-size:16px;font-weight:500;color:{"var(--color-text-danger)" if sig["rsi"]>70 else "var(--color-text-success)" if sig["rsi"]<30 else "var(--color-text-primary)"}">{sig["rsi"]:.1f}</div></div>'
        f'<div style="padding:10px 14px;border-right:0.5px solid var(--color-border-tertiary)"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">ANN VOL</div><div style="font-size:16px;font-weight:500;color:var(--color-text-primary)">{garch["annvol"]:.1%}</div></div>'
        f'<div style="padding:10px 14px"><div style="font-size:10px;color:var(--color-text-secondary);letter-spacing:.06em;margin-bottom:2px">SENTIMENT</div><div style="font-size:16px;font-weight:500;color:{"var(--color-text-success)" if sent_sc>0 else "var(--color-text-danger)"}">{sent_sc:+.3f}</div></div>'
        f'</div>'
        f'<div style="padding:8px 14px;border-top:0.5px solid var(--color-border-tertiary);font-size:11px;color:var(--color-text-secondary)">'
        f'XGB {sig["p_xgb"]:.3f} | LGB {sig["p_lgb"]:.3f} | CAT {sig["p_cat"]:.3f} | '
        f'Regime: {regime_labels.get(regime,"?")} | GARCH p_up: {garch["p_up"]:.1%} | AUC: {mp["auc"]:.3f}</div></div>'))
    r=generate_reasoning(sig,garch,sent_sc,regime,regime_labels,mp)
    aclr2={"BUY":"var(--color-text-success)","SELL":"var(--color-text-danger)","HOLD":"var(--color-text-warning)"}.get(r["action"],"var(--color-text-primary)")
    li_r='style="margin-bottom:5px;color:var(--color-text-primary)"'
    li_c='style="margin-bottom:5px;color:var(--color-text-warning)"'
    ri="".join(f"<li {li_r}>{pt}</li>" for pt in r["reasons"])
    ci="".join(f"<li {li_c}>{pt}</li>" for pt in r["cautions"])
    wb_bg="background:var(--color-background-warning);border-radius:var(--border-radius-md);border:0.5px solid var(--color-border-warning)"
    cb=(f'<div style="margin-top:10px;padding:10px 12px;{wb_bg}"><div style="font-size:11px;font-weight:500;color:var(--color-text-warning);margin-bottom:6px">Cautions</div><ul style="font-size:12px;line-height:1.6;margin:0;padding-left:16px">{ci}</ul></div>' if ci else "")
    display(HTML(
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:var(--border-radius-lg);padding:14px 16px;margin:8px 0;font-family:var(--font-sans)">'
        f'<div style="font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.07em;margin-bottom:8px">Why {ticker} is a {r["action"]}</div>'
        f'<div style="font-size:13px;font-weight:500;color:{aclr2};margin-bottom:10px">{r["summary"]}</div>'
        f'<ul style="font-size:12px;line-height:1.7;margin:0;padding-left:16px;color:var(--color-text-primary)">{ri}</ul>'
        f'{cb}</div>'))
    fig,axes=plt.subplots(3,1,figsize=(16,11),sharex=True,facecolor="#0a0e1a")
    fig.suptitle(f"{ticker}  |  {sig['action']}  |  conf={sig['confidence']:.3f}",
                 fontsize=12,fontweight="bold",color="#e2e8f0")
    plot_df=df_feat.tail(180).copy()
    for ax in axes:
        ax.set_facecolor("#0d1220"); ax.tick_params(colors="#475569",labelsize=9)
        for spine in ax.spines.values(): spine.set_edgecolor("#1e2530")
    ax=axes[0]
    ax.plot(plot_df.index,plot_df["Close"],color="#e2e8f0",lw=1.5,label="Close")
    for w,clr in [(20,"#378ADD"),(50,"#f59e0b"),(200,"#a855f7")]:
        if f"sma_{w}" in plot_df.columns:
            ax.plot(plot_df.index,plot_df[f"sma_{w}"],lw=0.8,alpha=0.7,label=f"SMA{w}",color=clr)
    if "bb_upper" in plot_df.columns:
        ax.fill_between(plot_df.index,plot_df["bb_upper"],plot_df["bb_lower"],alpha=0.06,color="#378ADD")
    ax.set_ylabel("Price",color="#475569",fontsize=9)
    ax.legend(fontsize=8,facecolor="#0d1220",edgecolor="#1e2530",labelcolor="#94a3b8")
    ax=axes[1]
    if "rsi_14" in plot_df.columns:
        ax.plot(plot_df.index,plot_df["rsi_14"],color="#f87171",lw=1)
        ax.axhline(70,color="#ef4444",linestyle="--",alpha=0.4,lw=0.8)
        ax.axhline(30,color="#4ade80",linestyle="--",alpha=0.4,lw=0.8)
        ax.set_ylim(0,100); ax.set_ylabel("RSI(14)",color="#475569",fontsize=9)
        ax.fill_between(plot_df.index,plot_df["rsi_14"],50,where=plot_df["rsi_14"]>50,alpha=0.08,color="#4ade80")
        ax.fill_between(plot_df.index,plot_df["rsi_14"],50,where=plot_df["rsi_14"]<50,alpha=0.08,color="#f87171")
    ax=axes[2]
    if "macd" in plot_df.columns and "macd_sig" in plot_df.columns:
        ax.plot(plot_df.index,plot_df["macd"],color="#7dd3fc",lw=1,label="MACD")
        ax.plot(plot_df.index,plot_df["macd_sig"],color="#f87171",lw=1,label="Signal")
        if "macd_h" in plot_df.columns:
            ax.bar(plot_df.index,plot_df["macd_h"],
                   color=["#4ade80" if v>=0 else "#f87171" for v in plot_df["macd_h"]],alpha=0.5)
        ax.axhline(0,color="#475569",lw=0.5); ax.set_ylabel("MACD",color="#475569",fontsize=9)
        ax.legend(fontsize=8,facecolor="#0d1220",edgecolor="#1e2530",labelcolor="#94a3b8")
    plt.tight_layout()
    plt.savefig(f"search_{ticker.replace('-','_')}_v25.png",dpi=110,bbox_inches="tight",facecolor="#0a0e1a")
    plt.show(); print(f"  Chart saved.")


def render_homepage():
    now   = datetime.datetime.now().strftime("%b %d %Y  %H:%M")
    m     = MACRO
    eq    = _current_equity()
    pnl   = eq - PORTFOLIO_CAPITAL
    pnl_c = "var(--color-text-success)" if pnl >= 0 else "var(--color-text-danger)"

    # accuracy stats from pred log
    try:
        import pandas as _pd
        plog   = _pd.read_csv(PRED_LOG_FILE)
        scored = plog[plog["scored"].astype(str)=="True"].copy()
        scored["was_correct"] = scored["was_correct"].astype(str).map(
            {"True":True,"False":False,"true":True,"false":False}).fillna(False)
        n_scored    = len(scored)
        overall_acc = scored["was_correct"].mean() if n_scored > 0 else None
        recent_acc  = scored.tail(10)["was_correct"].mean() if n_scored >= 3 else None
        n_correct   = int(scored["was_correct"].sum()) if n_scored > 0 else 0
    except Exception:
        n_scored = 0; overall_acc = None; recent_acc = None; n_correct = 0

    def acc_color(a):
        if a is None: return "var(--color-text-secondary)"
        return ("var(--color-text-success)" if a >= 0.60 else
                "var(--color-text-warning)" if a >= 0.50 else
                "var(--color-text-danger)")

    acc_str = f"{overall_acc:.1%}" if overall_acc is not None else "Ã¢â‚¬â€"
    rec_str = f"{recent_acc:.1%}"  if recent_acc  is not None else "Ã¢â‚¬â€"

    # macro cards
    vix   = m.get("vix") or 20
    tnx   = m.get("tnx_10y") or 4.3
    yc    = m.get("yield_curve") or 0
    dxy   = m.get("dxy") or 104
    wti   = m.get("wti_crude") or 80
    gold  = m.get("gold") or 2000
    unemp = m.get("unemployment") or 3.9
    cpi   = m.get("cpi_yoy") or 3.1
    gdp   = m.get("gdp_growth") or 2.8
    pmi   = m.get("ism_pmi") or 50.3
    cs    = m.get("credit_spread")
    ey    = m.get("earnings_yield")
    cfg   = m.get("crypto_fg") or 50
    cfg_l = m.get("crypto_fg_label","Ã¢â‚¬â€")
    rml   = m.get("macro_regime","Ã¢â‚¬â€")
    ret   = m.get("retail_sales")

    macro_html = "".join([
        _macro_card("Fed funds rate",   _fmt(m.get("fed_rate"),".2f","%"),
            "interest rate env",        min((m.get("fed_rate") or 4)/8*100,100), "#378ADD"),
        _macro_card("10Y Treasury",     _fmt(tnx,".2f","%"),
            "risk-free benchmark",      min(tnx/8*100,100),
            "#E24B4A" if tnx>5 else "#BA7517" if tnx>4 else "#639922"),
        _macro_card("Yield curve",      _fmt(yc,"+.2f","%"),
            "normal" if yc>0 else "inverted Ã¢â‚¬â€ risk",
            50+yc*20,
            "#639922" if yc>0 else "#E24B4A",
            sub_color="#639922" if yc>0 else "#E24B4A"),
        _macro_card("VIX fear index",   _fmt(vix,".1f"),
            "extreme fear" if vix>30 else "elevated" if vix>20 else "low Ã¢â‚¬â€ risk-on",
            min(vix/50*100,100),
            "#E24B4A" if vix>25 else "#BA7517" if vix>18 else "#639922",
            sub_color="#E24B4A" if vix>25 else "#BA7517" if vix>18 else "#639922"),
        _macro_card("Unemployment",     _fmt(unemp,".1f","%"),
            "above 5% Ã¢â‚¬â€ watch" if unemp>5 else "healthy labor market",
            min(unemp/10*100,100),
            "#E24B4A" if unemp>5.5 else "#BA7517" if unemp>4.5 else "#639922"),
        _macro_card("CPI inflation",    _fmt(cpi,".1f","% YoY"),
            "above target" if cpi>2.5 else "near target",
            min(cpi/8*100,100),
            "#E24B4A" if cpi>4 else "#BA7517" if cpi>2.5 else "#639922",
            sub_color="#E24B4A" if cpi>4 else "#BA7517" if cpi>2.5 else "#639922"),
        _macro_card("GDP growth",       _fmt(gdp,"+.1f","% QoQ"),
            "contraction" if gdp<0 else "moderate" if gdp<3 else "strong",
            max(0,min((gdp+2)/8*100,100)),
            "#E24B4A" if gdp<0 else "#BA7517" if gdp<1.5 else "#639922"),
        _macro_card("ISM PMI",          _fmt(pmi,".1f"),
            "contraction" if pmi<50 else "expansion",
            min(pmi/70*100,100),
            "#E24B4A" if pmi<48 else "#BA7517" if pmi<50 else "#639922",
            sub_color="#E24B4A" if pmi<48 else "#BA7517" if pmi<50 else "#639922"),
        _macro_card("WTI crude oil",    _fmt(wti,".1f","  $/bbl"),
            "inflationary" if wti>90 else "moderate",
            min(wti/120*100,100),
            "#E24B4A" if wti>90 else "#BA7517" if wti>75 else "#639922"),
        _macro_card("Gold",             "$"+_fmt(gold,",.0f"),
            "fear / inflation hedge",   min(gold/3500*100,100), "#BA7517"),
        _macro_card("DXY dollar",       _fmt(dxy,".1f"),
            "strong USD" if dxy>105 else "neutral",
            min(dxy/115*100,100),
            "#BA7517" if dxy>105 else "#639922"),
        _macro_card("Credit spread",    _fmt(cs,".4f") if cs else "N/A",
            "HYG/LQD Ã¢â‚¬â€ lower = stress",
            min((cs or 1)*50,100) if cs else 50,
            "#E24B4A" if cs and cs<0.90 else "#639922"),
        _macro_card("Retail sales",     _fmt(ret,"+.1f","% MoM") if ret else "N/A",
            "consumer spending",
            max(0,min(50+(ret or 0)*10,100)),
            "#639922" if (ret or 0)>0 else "#E24B4A"),
        _macro_card("Earnings yield",   _fmt(ey,".2f","%") if ey else "N/A",
            f"vs bonds: {_fmt(m.get('ey_spread'),'+.2f','%') if m.get('ey_spread') else 'N/A'}",
            min((ey or 4)/8*100,100) if ey else 50,
            "#639922" if (m.get("ey_spread") or 0)>0 else "#E24B4A"),
        _macro_card("Crypto fear/greed",str(cfg),
            cfg_l,  cfg,
            "#E24B4A" if cfg<25 else "#BA7517" if cfg<45 else
            "#639922" if cfg<75 else "#E24B4A"),
        _macro_card("Market regime",
            rml.split("/")[0] if "/" in rml else rml,
            f"VIX {_fmt(vix,'.1f')}",
            100-min(vix/50*100,100),
            "#E24B4A" if "Bear" in rml else "#BA7517" if "Neutral" in rml else "#639922"),
    ])

    # signal rows
    sig_rows = ""
    for tk, sig in sorted(signals.items(), key=lambda x:-x[1]["confidence"]):
        a=sig["action"]; conf=sig["confidence"]; rsi=sig["rsi"]
        close=sig["close"]; av=sig["ann_vol"]; sent=sig["sentiment"]
        regime=sig["regime"]; rules=sig.get("rules_applied",[])
        rlbl={0:"Bear",1:"Bull",2:"Neutral"}.get(regime,"?")
        if a=="BUY":
            badge='<span style="background:var(--color-background-success);color:var(--color-text-success);padding:2px 8px;border-radius:6px;font-size:10px;font-weight:500">BUY</span>'
        elif a=="SELL":
            badge='<span style="background:var(--color-background-danger);color:var(--color-text-danger);padding:2px 8px;border-radius:6px;font-size:10px;font-weight:500">SELL</span>'
        else:
            badge='<span style="background:var(--color-background-warning);color:var(--color-text-warning);padding:2px 8px;border-radius:6px;font-size:10px;font-weight:500">HOLD</span>'
        bw=int(conf*72)
        bc=("var(--color-text-success)" if conf>=MIN_CONFIDENCE else
            "var(--color-text-danger)" if conf<=(1-MIN_CONFIDENCE) else
            "var(--color-text-warning)")
        rsi_c=("var(--color-text-danger)" if rsi>70 else
               "var(--color-text-success)" if rsi<30 else
               "var(--color-text-secondary)")
        s_c=("var(--color-text-success)" if sent>0 else
             "var(--color-text-danger)" if sent<0 else
             "var(--color-text-secondary)")
        rbg=("var(--color-background-success)" if rlbl=="Bull" else
             "var(--color-background-danger)" if rlbl=="Bear" else
             "var(--color-background-warning)")
        rc=("var(--color-text-success)" if rlbl=="Bull" else
            "var(--color-text-danger)"  if rlbl=="Bear" else
            "var(--color-text-warning)")
        rule_note=""
        iv_note_html=""
        if rules:
            rule_note=(f'<div style="font-size:9px;color:var(--color-text-warning);'
                       f'margin-top:2px">{" Ã‚Â· ".join(rules)}</div>')
        _iv_f = sig.get("iv_flag","NORMAL")
        _iv_n = sig.get("iv_note","")
        if _iv_f in ("HIGH","ELEVATED"):
            iv_note_html=(f'<div style="font-size:9px;color:var(--color-text-danger);margin-top:1px">'
                          f'Ã¢Å¡Â  {_iv_f} IV Ã‚Â· {_iv_n[:40]}</div>')
        sig_rows += (
            f'<tr style="border-bottom:0.5px solid var(--color-border-tertiary)">'
            f'<td style="padding:8px 12px;font-weight:500;font-size:12px;'
            f'color:var(--color-text-primary)">{tk}</td>'
            f'<td style="padding:8px 12px;text-align:center">{badge}</td>'
            f'<td style="padding:8px 12px"><div style="display:flex;align-items:center;gap:5px">'
            f'<div style="width:72px;height:4px;background:var(--color-background-secondary);border-radius:2px">'
            f'<div style="width:{bw}px;height:4px;background:{bc};border-radius:2px"></div></div>'
            f'<span style="font-size:11px;font-weight:500;color:{bc}">{conf:.3f}</span></div>'
            f'{rule_note}{iv_note_html}</td>'
            f'<td style="padding:8px 12px;font-size:11px;color:var(--color-text-secondary);text-align:right">'
            f'${close:,.2f}</td>'
            f'<td style="padding:8px 12px;font-size:11px;color:{rsi_c};text-align:right">{rsi:.1f}</td>'
            f'<td style="padding:8px 12px;font-size:11px;color:{s_c};text-align:right">{sent:+.3f}</td>'
            f'<td style="padding:8px 12px;text-align:center">'
            f'<span style="background:{rbg};color:{rc};padding:1px 6px;border-radius:4px;font-size:10px">'
            f'{rlbl}</span></td>'
            f'</tr>'
        )

    # learned rules
    rules_html = ""
    if LEARNED_RULES:
        for key, rule in LEARNED_RULES.items():
            if key == "_boosts" or not isinstance(rule, dict) or "description" not in rule: continue
            rules_html += (
                f'<div style="padding:6px 12px;border-bottom:0.5px solid var(--color-border-tertiary)">'
                f'<div style="display:flex;justify-content:space-between;align-items:center">'
                f'<span style="font-size:12px;color:var(--color-text-primary)">{rule.get("description","")}</span>'
                f'<span style="font-size:11px;color:var(--color-text-warning);white-space:nowrap;margin-left:8px">'
                f'dampen {rule.get("dampen",0):.0%}</span></div>'
                f'<div style="font-size:10px;color:var(--color-text-secondary);margin-top:1px">'
                f'{rule.get("count",0)} samples Ã‚Â· {rule.get("accuracy",0):.1%} accuracy</div></div>'
            )
    else:
        rules_html = ('<div style="padding:10px 12px;font-size:12px;color:var(--color-text-secondary)">'
                      'No rules yet Ã¢â‚¬â€ accumulating scored outcomes...</div>')

    # recent outcomes
    outcomes_html = ""
    try:
        import pandas as _pd2
        sc2 = _pd2.read_csv(PRED_LOG_FILE)
        sc2 = sc2[sc2["scored"].astype(str)=="True"].copy()
        sc2["was_correct"] = sc2["was_correct"].astype(str).map(
            {"True":True,"False":False,"true":True,"false":False}).fillna(False)
        sc2["actual_return"] = _pd2.to_numeric(sc2["actual_return"], errors="coerce").fillna(0)
        for _,row in sc2.tail(6).iterrows():
            correct=bool(row["was_correct"]); ret2=float(row["actual_return"])
            ok_c="var(--color-text-success)" if correct else "var(--color-text-danger)"
            ok_l="CORRECT" if correct else "WRONG"
            outcomes_html += (
                f'<div style="display:flex;justify-content:space-between;align-items:center;'
                f'padding:4px 0;border-bottom:0.5px solid var(--color-border-tertiary)">'
                f'<span style="font-size:11px;font-weight:500;color:var(--color-text-primary);width:60px">'
                f'{row["ticker"]}</span>'
                f'<span style="font-size:10px;color:var(--color-text-secondary)">{str(row["action"])}</span>'
                f'<span style="font-size:11px;font-weight:500;color:{ok_c}">{ok_l}</span>'
                f'<span style="font-size:11px;color:var(--color-text-secondary)">{ret2:+.1%}</span>'
                f'</div>'
            )
    except Exception:
        pass
    if not outcomes_html:
        outcomes_html = ('<div style="font-size:12px;color:var(--color-text-secondary);padding:8px 0">'
                         f'Predictions score after {FORECAST_DAYS} trading days</div>')

    buy_n  = sum(1 for s in signals.values() if s["action"]=="BUY")
    sell_n = sum(1 for s in signals.values() if s["action"]=="SELL")
    hold_n = sum(1 for s in signals.values() if s["action"]=="HOLD")
    avg_c  = sum(s["confidence"] for s in signals.values()) / max(len(signals),1)

    display(HTML(
        f'<div style="font-family:var(--font-sans)">'

        # Ã¢â€â‚¬Ã¢â€â‚¬ topbar
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:8px;padding:10px 16px;display:flex;justify-content:space-between;'
        f'align-items:center;margin-bottom:8px">'
        f'<span style="font-size:14px;font-weight:500;color:var(--color-text-primary)">'
        f'Quant Terminal v25</span>'
        f'<span style="font-size:11px;color:var(--color-text-secondary)">{now}</span></div>'

        # Ã¢â€â‚¬Ã¢â€â‚¬ search bar
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-info);'
        f'border-radius:8px;padding:10px 16px;margin-bottom:8px">'
        f'<div style="font-size:11px;font-weight:500;color:var(--color-text-secondary);margin-bottom:4px">'
        f'SEARCH Ã¢â‚¬â€ change <code style="background:var(--color-background-secondary);'
        f'padding:1px 6px;border-radius:4px;color:var(--color-text-info)">SEARCH_TICKER</code>'
        f' at the top of this cell and press <kbd style="background:var(--color-background-secondary);'
        f'padding:1px 6px;border-radius:4px;font-size:10px">Shift+Enter</kbd>'
        f' for any stock or crypto analysis</div>'
        f'<div style="font-size:12px;color:var(--color-text-secondary)">'
        f'e.g. &nbsp;NVDA &nbsp;TSLA &nbsp;MSFT &nbsp;GOOGL &nbsp;AMZN &nbsp;BTC-USD &nbsp;ETH-USD</div></div>'

        # Ã¢â€â‚¬Ã¢â€â‚¬ stats row
        f'<div style="display:grid;grid-template-columns:repeat(6,1fr);gap:6px;margin-bottom:8px">'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">BUY</div>'
        f'<div style="font-size:20px;font-weight:500;color:var(--color-text-success)">{buy_n}</div></div>'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">SELL</div>'
        f'<div style="font-size:20px;font-weight:500;color:var(--color-text-danger)">{sell_n}</div></div>'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">HOLD</div>'
        f'<div style="font-size:20px;font-weight:500;color:var(--color-text-warning)">{hold_n}</div></div>'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">Avg conf</div>'
        f'<div style="font-size:20px;font-weight:500;color:var(--color-text-primary)">{avg_c:.3f}</div></div>'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">Accuracy</div>'
        f'<div style="font-size:20px;font-weight:500;color:{acc_color(overall_acc)}">{acc_str}</div></div>'
        f'<div style="background:var(--color-background-secondary);border-radius:8px;padding:10px 12px">'
        f'<div style="font-size:10px;color:var(--color-text-secondary);text-transform:uppercase;'
        f'letter-spacing:.06em;margin-bottom:3px">P&amp;L</div>'
        f'<div style="font-size:20px;font-weight:500;color:{pnl_c}">${pnl:+,.0f}</div></div>'
        f'</div>'

        # Ã¢â€â‚¬Ã¢â€â‚¬ macro section
        f'<div style="font-size:10px;font-weight:500;color:var(--color-text-secondary);'
        f'text-transform:uppercase;letter-spacing:.07em;margin:0 0 6px 2px">Macro environment</div>'
        f'<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:6px;margin-bottom:8px">'
        f'{macro_html}</div>'

        # Ã¢â€â‚¬Ã¢â€â‚¬ signal table
        f'<div style="font-size:10px;font-weight:500;color:var(--color-text-secondary);'
        f'text-transform:uppercase;letter-spacing:.07em;margin:0 0 5px 2px">Signal dashboard</div>'
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:8px;overflow:hidden;margin-bottom:8px">'
        f'<table style="width:100%;border-collapse:collapse">'
        f'<thead><tr style="background:var(--color-background-secondary);'
        f'border-bottom:0.5px solid var(--color-border-tertiary)">'
        f'<th style="padding:7px 12px;text-align:left;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">TICKER</th>'
        f'<th style="padding:7px 12px;text-align:center;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">SIGNAL</th>'
        f'<th style="padding:7px 12px;text-align:left;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">CONFIDENCE</th>'
        f'<th style="padding:7px 12px;text-align:right;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">PRICE</th>'
        f'<th style="padding:7px 12px;text-align:right;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">RSI</th>'
        f'<th style="padding:7px 12px;text-align:right;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">SENTIMENT</th>'
        f'<th style="padding:7px 12px;text-align:center;font-size:10px;color:var(--color-text-secondary);letter-spacing:.07em">REGIME</th>'
        f'</tr></thead><tbody>{sig_rows}</tbody></table></div>'

        # Ã¢â€â‚¬Ã¢â€â‚¬ bottom row: self-learning status + recent outcomes
        f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px">'

        # self-written rules
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:8px;overflow:hidden">'
        f'<div style="background:var(--color-background-secondary);padding:8px 12px;'
        f'border-bottom:0.5px solid var(--color-border-tertiary);display:flex;justify-content:space-between">'
        f'<span style="font-size:11px;font-weight:500;color:var(--color-text-primary)">'
        f'Self-written rules</span>'
        f'<span style="font-size:10px;color:var(--color-text-secondary)">{len(LEARNED_RULES)} active</span></div>'
        f'{rules_html}</div>'

        # recent outcomes
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:8px;padding:12px 14px">'
        f'<div style="font-size:11px;font-weight:500;color:var(--color-text-primary);margin-bottom:8px">'
        f'Recent outcomes &nbsp;'
        f'<span style="color:var(--color-text-secondary);font-weight:400">({n_correct}/{n_scored} correct)</span></div>'
        f'{outcomes_html}'
        f'<div style="display:flex;justify-content:space-between;padding-top:8px;margin-top:4px;'
        f'border-top:0.5px solid var(--color-border-tertiary)">'
        f'<span style="font-size:11px;color:var(--color-text-secondary)">Last 10 accuracy</span>'
        f'<span style="font-size:12px;font-weight:500;color:{acc_color(recent_acc)}">{rec_str}</span></div>'
        f'</div>'

        # Adaptive weights bar
        f'<div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);'
        f'border-radius:8px;padding:12px 14px;margin-top:8px">'
        f'<div style="font-size:11px;font-weight:500;color:var(--color-text-primary);margin-bottom:8px">'
        f'Adaptive signal weights &nbsp;<span style="color:var(--color-text-secondary);font-weight:400">'
        f'River updates these from scored outcomes</span></div>'
        f'<div style="display:grid;grid-template-columns:repeat(5,1fr);gap:8px">'
        + "".join(
            f'<div style="text-align:center">'
            f'<div style="font-size:10px;color:var(--color-text-secondary);margin-bottom:3px">{k.replace("w_","")}  </div>'
            f'<div style="font-size:15px;font-weight:500;color:var(--color-text-primary)">{v:.0%}</div>'
            f'<div style="height:3px;background:var(--color-background-secondary);border-radius:2px;margin-top:4px">'
            f'<div style="width:{int(v*200)}%;height:3px;background:var(--color-text-info);border-radius:2px"></div></div>'
            f'</div>'
            for k,v in ADAPTIVE_WEIGHTS.items()
        )
        + f'</div></div>'
        f'</div></div>'
    ))

# Ã¢â€â‚¬Ã¢â€â‚¬ AUTO-RENDER Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
render_homepage()

# Ã¢â€â‚¬Ã¢â€â‚¬ SEARCH Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
if SEARCH_TICKER.strip().upper() not in ("", "NONE"):
    on_demand_analysis(SEARCH_TICKER)


In [ ]:
# ============================================================
# CELL 16b Ã¢â‚¬â€ BTC CYCLE TRACKER
# ============================================================
# Standalone Bitcoin analysis cell.
# Tracks the 4-year halving cycle, gives BUY/HOLD/SELL signal,
# price prediction direction, and full reasoning.
# Run this cell independently anytime Ã¢â‚¬â€ no other cells needed.
# ============================================================
%matplotlib inline
from IPython.display import display, HTML
import datetime, requests
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Ã¢â€â‚¬Ã¢â€â‚¬ Halving dates (known + estimated) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
HALVING_DATES = [
    datetime.date(2012, 11, 28),
    datetime.date(2016,  7,  9),
    datetime.date(2020,  5, 11),
    datetime.date(2024,  4, 19),   # Most recent
    datetime.date(2028,  4, 17),   # Estimated next
]

# Historical cycle peaks (days post-halving)
HISTORICAL_PEAKS = {
    2012: {"day": 371,  "gain": 9000},
    2016: {"day": 525,  "gain": 2900},
    2020: {"day": 546,  "gain": 580},
}
AVG_PEAK_DAY   = int(np.mean([v["day"] for v in HISTORICAL_PEAKS.values()]))  # ~480
PEAK_DAY_RANGE = (350, 600)

def get_cycle_position():
    """Returns current cycle metrics relative to April 2024 halving."""
    today       = datetime.date.today()
    last_halving= HALVING_DATES[-2]   # April 2024
    next_halving= HALVING_DATES[-1]   # April 2028 est
    days_since  = (today - last_halving).days
    days_to_next= (next_halving - today).days
    cycle_pct   = days_since / (next_halving - last_halving).days * 100

    # Phase classification based on historical averages
    if days_since < 180:
        phase = "Early Bull"
        phase_desc = "Post-halving accumulation Ã¢â‚¬â€ historically strong risk/reward"
        phase_color = "#639922"
    elif days_since < PEAK_DAY_RANGE[0]:
        phase = "Mid Bull"
        phase_desc = "Primary bull run phase Ã¢â‚¬â€ historically the strongest gains"
        phase_color = "#22c55e"
    elif days_since < PEAK_DAY_RANGE[1]:
        phase = "Late Bull / Distribution"
        phase_desc = f"Approaching historical peak window (day {PEAK_DAY_RANGE[0]}-{PEAK_DAY_RANGE[1]}) Ã¢â‚¬â€ elevated caution"
        phase_color = "#f59e0b"
    elif days_since < 900:
        phase = "Post-Peak / Distribution"
        phase_desc = "Past average peak window Ã¢â‚¬â€ risk of major correction increasing"
        phase_color = "#E24B4A"
    else:
        phase = "Bear Market"
        phase_desc = "Extended post-peak bear Ã¢â‚¬â€ accumulation zone for next cycle"
        phase_color = "#888"

    return dict(
        days_since=days_since,
        days_to_next=days_to_next,
        cycle_pct=round(cycle_pct, 1),
        phase=phase,
        phase_desc=phase_desc,
        phase_color=phase_color,
        last_halving=last_halving,
        next_halving=next_halving,
        avg_peak_day=AVG_PEAK_DAY,
        days_to_avg_peak=max(0, AVG_PEAK_DAY - days_since),
    )

def get_btc_data():
    """Fetch BTC price data and compute key metrics."""
    try:
        tk  = yf.Ticker("BTC-USD")
        df  = tk.history(period="2y", auto_adjust=True)
        if df.empty:
            return None
        df.index = pd.to_datetime(df.index).tz_localize(None)

        price       = float(df["Close"].iloc[-1])
        price_7d    = float(df["Close"].iloc[-7])  if len(df)>7  else price
        price_30d   = float(df["Close"].iloc[-30]) if len(df)>30 else price
        price_90d   = float(df["Close"].iloc[-90]) if len(df)>90 else price
        price_365d  = float(df["Close"].iloc[-252])if len(df)>252 else price

        ret_7d  = (price - price_7d)  / price_7d
        ret_30d = (price - price_30d) / price_30d
        ret_90d = (price - price_90d) / price_90d
        ret_1y  = (price - price_365d)/ price_365d

        # RSI
        delta = df["Close"].diff()
        gain  = delta.clip(lower=0).rolling(14).mean()
        loss  = (-delta.clip(upper=0)).rolling(14).mean()
        rs    = gain / loss.replace(0, np.nan)
        rsi   = float(100 - 100 / (1 + rs.iloc[-1]))

        # 200-day MA
        ma200 = float(df["Close"].rolling(200).mean().iloc[-1]) if len(df)>=200 else None
        above_ma200 = price > ma200 if ma200 else None

        # All-time high estimate (from data)
        ath = float(df["Close"].max())
        pct_from_ath = (price - ath) / ath

        # Volatility (30d annualised)
        vol30 = float(df["Close"].pct_change().rolling(30).std().iloc[-1] * np.sqrt(365))

        return dict(
            price=price, price_7d=price_7d, price_30d=price_30d,
            price_90d=price_90d, price_365d=price_365d,
            ret_7d=ret_7d, ret_30d=ret_30d, ret_90d=ret_90d, ret_1y=ret_1y,
            rsi=rsi, ma200=ma200, above_ma200=above_ma200,
            ath=ath, pct_from_ath=pct_from_ath,
            vol30=vol30, df=df
        )
    except Exception as e:
        print(f"BTC data error: {e}")
        return None

def get_crypto_fear_greed():
    """Fetch crypto fear & greed index."""
    try:
        r = requests.get("https://api.alternative.me/fng/?limit=1", timeout=5)
        data = r.json()["data"][0]
        return int(data["value"]), data["value_classification"]
    except Exception:
        return None, "N/A"

def generate_btc_signal(cycle, btc, fg_val, fg_label):
    """Generate BUY/HOLD/SELL signal with reasoning for BTC."""
    bullish = []
    bearish = []
    cautions= []

    # Cycle phase scoring
    if cycle["phase"] == "Early Bull":
        bullish.append(f"Cycle phase: Early Bull ({cycle['days_since']} days post-halving) Ã¢â‚¬â€ historically the best risk/reward entry window.")
        cycle_score = 0.80
    elif cycle["phase"] == "Mid Bull":
        bullish.append(f"Cycle phase: Mid Bull ({cycle['days_since']} days post-halving) Ã¢â‚¬â€ primary appreciation phase with {cycle['days_to_avg_peak']} days to historical average peak.")
        cycle_score = 0.70
    elif cycle["phase"] == "Late Bull / Distribution":
        cautions.append(f"Cycle phase: Late Bull/Distribution ({cycle['days_since']} days post-halving) Ã¢â‚¬â€ within historical peak window (day {PEAK_DAY_RANGE[0]}-{PEAK_DAY_RANGE[1]}). Reduce position sizing.")
        cycle_score = 0.50
    elif cycle["phase"] == "Post-Peak / Distribution":
        bearish.append(f"Cycle phase: Post-Peak ({cycle['days_since']} days post-halving) Ã¢â‚¬â€ past average peak window. Historical drawdowns of 70-85% follow.")
        cycle_score = 0.30
    else:
        bearish.append(f"Cycle phase: Bear Market ({cycle['days_since']} days post-halving) Ã¢â‚¬â€ accumulation zone.")
        cycle_score = 0.35

    # Price momentum
    if btc["ret_30d"] > 0.10:
        bullish.append(f"Strong 30-day momentum: +{btc['ret_30d']:.1%}.")
    elif btc["ret_30d"] > 0:
        bullish.append(f"Positive 30-day momentum: +{btc['ret_30d']:.1%}.")
    elif btc["ret_30d"] < -0.15:
        bearish.append(f"Weak 30-day momentum: {btc['ret_30d']:.1%}.")
    else:
        cautions.append(f"Negative 30-day momentum: {btc['ret_30d']:.1%}.")

    # 200 MA
    if btc["above_ma200"]:
        bullish.append(f"Price ${btc['price']:,.0f} above 200-day MA ${btc['ma200']:,.0f} Ã¢â‚¬â€ bullish long-term trend.")
    elif btc["ma200"]:
        bearish.append(f"Price ${btc['price']:,.0f} below 200-day MA ${btc['ma200']:,.0f} Ã¢â‚¬â€ bearish long-term trend.")

    # RSI
    if btc["rsi"] > 75:
        cautions.append(f"RSI overbought at {btc['rsi']:.1f} Ã¢â‚¬â€ short-term pullback risk.")
    elif btc["rsi"] < 35:
        bullish.append(f"RSI oversold at {btc['rsi']:.1f} Ã¢â‚¬â€ potential mean-reversion.")
    else:
        bullish.append(f"RSI neutral at {btc['rsi']:.1f} Ã¢â‚¬â€ no extreme readings.")

    # Fear & Greed
    if fg_val is not None:
        if fg_val >= 75:
            cautions.append(f"Crypto Fear & Greed: {fg_val} ({fg_label}) Ã¢â‚¬â€ extreme greed often precedes corrections.")
        elif fg_val >= 55:
            bullish.append(f"Crypto Fear & Greed: {fg_val} ({fg_label}) Ã¢â‚¬â€ greed indicates positive sentiment.")
        elif fg_val <= 25:
            bullish.append(f"Crypto Fear & Greed: {fg_val} ({fg_label}) Ã¢â‚¬â€ extreme fear often marks bottoms.")
        else:
            cautions.append(f"Crypto Fear & Greed: {fg_val} ({fg_label}) Ã¢â‚¬â€ neutral.")

    # ATH proximity
    if btc["pct_from_ath"] > -0.05:
        cautions.append(f"Price within 5% of all-time high Ã¢â‚¬â€ resistance expected at ATH.")
    elif btc["pct_from_ath"] > -0.20:
        bullish.append(f"Price {abs(btc['pct_from_ath']):.0%} below ATH Ã¢â‚¬â€ room to run.")
    else:
        cautions.append(f"Price {abs(btc['pct_from_ath']):.0%} below ATH Ã¢â‚¬â€ significant recovery needed.")

    # Composite signal
    bull_count = len(bullish)
    bear_count = len(bearish)
    total      = bull_count + bear_count + len(cautions)

    # Weight cycle phase heavily
    raw_conf = (cycle_score * 0.40 +
                (bull_count / max(total,1)) * 0.60)
    raw_conf = min(max(raw_conf, 0.05), 0.95)

    if raw_conf >= 0.62:
        action = "BUY"
    elif raw_conf <= 0.40:
        action = "SELL"
    else:
        action = "HOLD"

    # Price direction prediction
    if action == "BUY":
        direction = "UP"
        direction_desc = f"Bullish bias over next 30-90 days based on cycle position and momentum."
    elif action == "SELL":
        direction = "DOWN"
        direction_desc = f"Bearish bias Ã¢â‚¬â€ cycle and/or technical signals suggest downside risk."
    else:
        direction = "NEUTRAL"
        direction_desc = f"Mixed signals Ã¢â‚¬â€ no clear directional edge over next 30 days."

    return dict(
        action=action, confidence=round(raw_conf, 3),
        direction=direction, direction_desc=direction_desc,
        bullish=bullish, bearish=bearish, cautions=cautions
    )

def render_btc_cell():
    cycle  = get_cycle_position()
    btc    = get_btc_data()
    fg_val, fg_label = get_crypto_fear_greed()

    if btc is None:
        print("Could not fetch BTC data.")
        return

    sig = generate_btc_signal(cycle, btc, fg_val, fg_label)

    # Colours
    action_bg  = {"BUY":"#dcfce7","SELL":"#fee2e2","HOLD":"#fef9c3"}[sig["action"]]
    action_c   = {"BUY":"#15803d","SELL":"#b91c1c","HOLD":"#a16207"}[sig["action"]]
    dir_icon   = {"UP":"Ã¢â€ â€˜","DOWN":"Ã¢â€ â€œ","NEUTRAL":"Ã¢â€ â€™"}[sig["direction"]]
    dir_c      = {"UP":"#15803d","DOWN":"#b91c1c","NEUTRAL":"#a16207"}[sig["direction"]]

    # Ã¢â€â‚¬Ã¢â€â‚¬ HTML render Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    bulls_html = "".join(f'<li style="margin-bottom:5px;color:#14532d">{b}</li>' for b in sig["bullish"])
    bears_html = "".join(f'<li style="margin-bottom:5px;color:#b91c1c">{b}</li>' for b in sig["bearish"])
    caut_html  = "".join(f'<li style="margin-bottom:5px;color:#92400e">{c}</li>' for c in sig["cautions"])

    hist_rows = ""
    for yr, data in HISTORICAL_PEAKS.items():
        hist_rows += (
            f'<tr style="border-bottom:0.5px solid #f0f0f0">' +
            f'<td style="padding:6px 10px;font-size:11px;color:#555">{yr} cycle</td>' +
            f'<td style="padding:6px 10px;font-size:11px;color:#555">Day {data["day"]}</td>' +
            f'<td style="padding:6px 10px;font-size:11px;color:#15803d;font-weight:600">+{data["gain"]:,}%</td>' +
            f'</tr>'
        )

    display(HTML(
        f'<div style="font-family:-apple-system,BlinkMacSystemFont,\'Segoe UI\',sans-serif;max-width:900px">' +

        # Header
        f'<div style="background:#111;border-radius:10px;padding:12px 18px;' +
        f'display:flex;justify-content:space-between;align-items:center;margin-bottom:10px">' +
        f'<div>' +
        f'<span style="font-size:16px;font-weight:700;color:#fff">Ã¢â€šÂ¿ Bitcoin Cycle Tracker</span>' +
        f'<span style="font-size:11px;color:#888;margin-left:10px">Quant Terminal v25</span></div>' +
        f'<span style="font-size:22px;font-weight:700;color:#f59e0b">${btc["price"]:,.0f}</span>' +
        f'</div>' +

        # Signal + direction row
        f'<div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:8px;margin-bottom:10px">' +
        f'<div style="background:{action_bg};border:1.5px solid {action_c};' +
        f'border-radius:10px;padding:14px;text-align:center">' +
        f'<div style="font-size:11px;color:{action_c};text-transform:uppercase;letter-spacing:.07em;margin-bottom:4px">Signal</div>' +
        f'<div style="font-size:28px;font-weight:700;color:{action_c}">{sig["action"]}</div>' +
        f'<div style="font-size:12px;color:{action_c};margin-top:2px">conf {sig["confidence"]:.3f}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:10px;padding:14px;text-align:center">' +
        f'<div style="font-size:11px;color:#888;text-transform:uppercase;letter-spacing:.07em;margin-bottom:4px">Price direction</div>' +
        f'<div style="font-size:28px;font-weight:700;color:{dir_c}">{dir_icon} {sig["direction"]}</div>' +
        f'<div style="font-size:11px;color:#666;margin-top:2px">{sig["direction_desc"][:55]}...</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:10px;padding:14px;text-align:center">' +
        f'<div style="font-size:11px;color:#888;text-transform:uppercase;letter-spacing:.07em;margin-bottom:4px">Cycle phase</div>' +
        f'<div style="font-size:16px;font-weight:700;color:{cycle["phase_color"]};margin-top:4px">{cycle["phase"]}</div>' +
        f'<div style="font-size:11px;color:#666;margin-top:4px">Day {cycle["days_since"]} / ~1,460</div></div>' +
        f'</div>' +

        # Cycle stats row
        f'<div style="display:grid;grid-template-columns:repeat(5,1fr);gap:8px;margin-bottom:10px">' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Days since halving</div>' +
        f'<div style="font-size:18px;font-weight:700;color:#111">{cycle["days_since"]}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Avg peak day</div>' +
        f'<div style="font-size:18px;font-weight:700;color:#111">~{cycle["avg_peak_day"]}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Days to avg peak</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#f59e0b" if cycle["days_to_avg_peak"]<60 else "#111"}">{cycle["days_to_avg_peak"] if cycle["days_to_avg_peak"]>0 else "Past"}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Fear & Greed</div>' +
        f'<div style="font-size:18px;font-weight:700;color:#111">{fg_val if fg_val else "N/A"}</div>' +
        f'<div style="font-size:9px;color:#888">{fg_label}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">30d return</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#15803d" if btc["ret_30d"]>0 else "#b91c1c"}">{btc["ret_30d"]:+.1%}</div></div>' +
        f'</div>' +

        # Price metrics
        f'<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:8px;margin-bottom:10px">' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">7-day</div>' +
        f'<div style="font-size:15px;font-weight:600;color:{"#15803d" if btc["ret_7d"]>0 else "#b91c1c"}">{btc["ret_7d"]:+.1%}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">90-day</div>' +
        f'<div style="font-size:15px;font-weight:600;color:{"#15803d" if btc["ret_90d"]>0 else "#b91c1c"}">{btc["ret_90d"]:+.1%}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">1-year</div>' +
        f'<div style="font-size:15px;font-weight:600;color:{"#15803d" if btc["ret_1y"]>0 else "#b91c1c"}">{btc["ret_1y"]:+.1%}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">RSI (14)</div>' +
        f'<div style="font-size:15px;font-weight:600;color:{"#b91c1c" if btc["rsi"]>70 else "#15803d" if btc["rsi"]<30 else "#111"}">{btc["rsi"]:.1f}</div></div>' +
        f'</div>' +

        # Why section
        f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;margin-bottom:10px">' +

        # Bullish
        f'<div style="background:#f0fdf4;border:1px solid #bbf7d0;border-radius:8px;padding:12px 14px">' +
        f'<div style="font-size:11px;font-weight:600;color:#15803d;margin-bottom:8px">Ã¢Å“â€¦ Bullish factors ({len(sig["bullish"])})</div>' +
        f'<ul style="font-size:12px;line-height:1.6;margin:0;padding-left:16px">{bulls_html}</ul></div>' +

        # Bearish + cautions
        f'<div>' +
        (f'<div style="background:#fff7ed;border:1px solid #fed7aa;border-radius:8px;padding:12px 14px;margin-bottom:8px">' +
         f'<div style="font-size:11px;font-weight:600;color:#92400e;margin-bottom:8px">Ã¢Å¡Â  Cautions ({len(sig["cautions"])})</div>' +
         f'<ul style="font-size:12px;line-height:1.6;margin:0;padding-left:16px">{caut_html}</ul></div>' if sig["cautions"] else "") +
        (f'<div style="background:#fff5f5;border:1px solid #fecaca;border-radius:8px;padding:12px 14px">' +
         f'<div style="font-size:11px;font-weight:600;color:#b91c1c;margin-bottom:8px">Ã¢ÂÅ’ Bearish factors ({len(sig["bearish"])})</div>' +
         f'<ul style="font-size:12px;line-height:1.6;margin:0;padding-left:16px">{bears_html}</ul></div>' if sig["bearish"] else "") +
        f'</div>' +

        # Historical cycles table
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;overflow:hidden;margin-bottom:10px">' +
        f'<div style="background:#fafafa;padding:8px 12px;font-size:11px;font-weight:600;color:#555;border-bottom:1px solid #e5e5e5">' +
        f'Historical cycle comparison Ã¢â‚¬â€ is BTC following the pattern?</div>' +
        f'<table style="width:100%;border-collapse:collapse">' +
        f'<tr style="background:#f9f9f9"><th style="padding:6px 10px;text-align:left;font-size:10px;color:#888">Cycle</th>' +
        f'<th style="padding:6px 10px;text-align:left;font-size:10px;color:#888">Peak day</th>' +
        f'<th style="padding:6px 10px;text-align:left;font-size:10px;color:#888">Peak gain from halving low</th></tr>' +
        f'{hist_rows}' +
        f'<tr style="background:#fffbeb"><td style="padding:6px 10px;font-size:11px;color:#92400e;font-weight:600">2024 cycle (now)</td>' +
        f'<td style="padding:6px 10px;font-size:11px;color:#92400e">Day {cycle["days_since"]} Ã¢â€ â€™ avg peak ~day {cycle["avg_peak_day"]}</td>' +
        f'<td style="padding:6px 10px;font-size:11px;color:#92400e">TBD Ã¢â‚¬â€ {cycle["days_to_avg_peak"]} days to avg peak</td></tr>' +
        f'</table></div>' +

        # Cycle description
        f'<div style="background:#f8f8f8;border-radius:8px;padding:12px 14px;font-size:12px;color:#555;line-height:1.65">' +
        f'<strong>Cycle position:</strong> {cycle["phase_desc"]} Last halving: {cycle["last_halving"]} Ã‚Â· ' +
        f'Next estimated halving: {cycle["next_halving"]} Ã‚Â· Cycle progress: {cycle["cycle_pct"]:.1f}%</div>' +

        f'</div>'
    ))

    # Ã¢â€â‚¬Ã¢â€â‚¬ Price chart with cycle overlay Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    df = btc["df"].tail(500).copy()
    fig, axes = plt.subplots(2, 1, figsize=(16, 9),
                              gridspec_kw={"height_ratios":[3,1]},
                              facecolor="#0a0e1a")
    fig.suptitle(f"Bitcoin  |  {sig['action']}  |  conf={sig['confidence']:.3f}  |  Cycle day {cycle['days_since']}",
                 fontsize=12, fontweight="bold", color="#e2e8f0")

    ax = axes[0]
    ax.set_facecolor("#0d1220")
    for spine in ax.spines.values(): spine.set_edgecolor("#1e2530")
    ax.tick_params(colors="#475569", labelsize=9)

    ax.plot(df.index, df["Close"], color="#f59e0b", lw=1.8, label="BTC Price")
    if btc["ma200"] and len(df) >= 200:
        ma200_series = df["Close"].rolling(200).mean()
        ax.plot(df.index, ma200_series, color="#7dd3fc", lw=1, alpha=0.7, label="200-day MA")

    # Shade cycle phase
    halving_ts = pd.Timestamp(cycle["last_halving"])
    if halving_ts in df.index or halving_ts > df.index[0]:
        ax.axvline(halving_ts, color="#a855f7", lw=1.5, linestyle="--", alpha=0.7, label="Last halving")

    # Shade the peak window
    peak_start = halving_ts + pd.Timedelta(days=PEAK_DAY_RANGE[0])
    peak_end   = halving_ts + pd.Timedelta(days=PEAK_DAY_RANGE[1])
    ax.axvspan(peak_start, peak_end, alpha=0.08, color="#f59e0b", label="Hist. peak window")
    ax.axvline(pd.Timestamp.now(), color="#4ade80", lw=1, linestyle=":", alpha=0.8, label="Today")

    ax.set_ylabel("Price (USD)", color="#475569", fontsize=9)
    ax.legend(fontsize=8, facecolor="#0d1220", edgecolor="#1e2530", labelcolor="#94a3b8")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p: f"${x:,.0f}"))

    # RSI panel
    ax2 = axes[1]
    ax2.set_facecolor("#0d1220")
    for spine in ax2.spines.values(): spine.set_edgecolor("#1e2530")
    ax2.tick_params(colors="#475569", labelsize=9)

    rsi_series = pd.Series(index=df.index, dtype=float)
    delta = df["Close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss.replace(0, np.nan)
    rsi_series = 100 - 100 / (1 + rs)

    ax2.plot(df.index, rsi_series, color="#f87171", lw=1)
    ax2.axhline(70, color="#ef4444", linestyle="--", alpha=0.4, lw=0.8)
    ax2.axhline(30, color="#4ade80", linestyle="--", alpha=0.4, lw=0.8)
    ax2.axhline(50, color="#475569", lw=0.4)
    ax2.set_ylim(0, 100)
    ax2.set_ylabel("RSI(14)", color="#475569", fontsize=9)
    ax2.fill_between(df.index, rsi_series, 50,
                     where=rsi_series>50, alpha=0.08, color="#4ade80")
    ax2.fill_between(df.index, rsi_series, 50,
                     where=rsi_series<50, alpha=0.08, color="#f87171")

    plt.tight_layout()
    plt.savefig("btc_cycle_tracker_v25.png", dpi=110,
                bbox_inches="tight", facecolor="#0a0e1a")
    plt.show()
    print(f"  BTC Cycle Tracker complete Ã¢â‚¬â€ {sig['action']} | conf={sig['confidence']:.3f} | Cycle day {cycle['days_since']}")

# Ã¢â€â‚¬Ã¢â€â‚¬ RUN Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
render_btc_cell()


In [ ]:
# ============================================================
# CELL 18 â€” AUTONOMOUS CONTINUOUS SCHEDULER
# ============================================================
# Fires every morning at 09:30 ET.
# Full 5-stage loop: data â†’ signals â†’ log â†’ score â†’ diagnose
# All systems run together. Model learns while you sleep.
import threading, time as _time

INTRADAY_SIGNAL_TIMES_ET = [
    (9, 35),
    (11, 30),
    (13, 30),
    (15, 0),
]

def get_next_signal_time_utc():
    now_et = pd.Timestamp.utcnow().tz_localize(None) - pd.Timedelta(hours=4)
    today  = now_et.date()
    for h, m in INTRADAY_SIGNAL_TIMES_ET:
        candidate = pd.Timestamp(today.year, today.month, today.day, h, m)
        if candidate > now_et:
            return (candidate + pd.Timedelta(hours=4)).tz_localize(None)
    tomorrow = today + pd.Timedelta(days=1)
    h, m = INTRADAY_SIGNAL_TIMES_ET[0]
    return (pd.Timestamp(tomorrow.year, tomorrow.month, tomorrow.day, h, m)
            + pd.Timedelta(hours=4)).tz_localize(None)

def is_market_hours():
    now_et = pd.Timestamp.utcnow().tz_localize(None) - pd.Timedelta(hours=4)
    if now_et.weekday() >= 5:
        return False
    market_open  = now_et.replace(hour=9,  minute=30, second=0)
    market_close = now_et.replace(hour=16, minute=0,  second=0)
    return market_open <= now_et <= market_close

import time as _time_module

def _timed_stage(name, func, *args, **kwargs):
    "Run a stage and print elapsed time."
    t0 = _time_module.time()
    print(f"\n  â”€â”€ {name} â”€â”€")
    result = func(*args, **kwargs) if callable(func) else None
    print(f"  âœ“ {name}: {_time_module.time()-t0:.1f}s")
    return result

def _full_autonomous_cycle():
    """Run the complete self-learning cycle."""
    global models, iv_flags, sentiments, signals
    t_cycle = _time_module.time()
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    print(f"\n{'='*55}")
    print(f"AUTONOMOUS CYCLE â€” {ts}")
    print(f"{'='*55}")

    # Stage 0: Daily feedback summary
    print("Stage 0: Daily feedback summary...")
    try: daily_summary()
    except Exception as e: print(f"  Daily summary error: {e}")

    # Stage 1: Refresh macro
    print("Stage 1: Refreshing macro data...")
    try: fetch_macro(); print(f"  Macro OK | regime: {MACRO['macro_regime']}")
    except Exception as e: print(f"  Macro error: {e}")

    # â”€â”€ Batch download all tickers at once â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print(f"  Batch downloading {len(WATCHLIST)} tickers...")
    _PRICE_CACHE.clear()   # refresh daily
    # Download only last 400 days for signals (model training uses TRAIN_STARTâ†’END only on retrain day)
    _SIGNAL_LOOKBACK_DAYS = 400   # enough for all indicators (SMA-200 + buffer)
    _signal_start = (pd.Timestamp.today() - pd.Timedelta(days=_SIGNAL_LOOKBACK_DAYS)).strftime("%Y-%m-%d")
    all_prices = batch_download_tickers(WATCHLIST, _signal_start, TRAIN_END)
    print(f"  âœ“ Downloaded last {_SIGNAL_LOOKBACK_DAYS} days for {len(all_prices)} tickers")

    # On retrain day, also download full history for model training
    if is_retrain_day():
        print("  Retrain day: downloading full history for training...")
        all_train_prices = batch_download_tickers(WATCHLIST, TRAIN_START, TRAIN_END, batch_size=30)
    else:
        all_train_prices = all_prices   # use recent data for quick tune

    # Stage 2a: Refresh IV flags (must run before signals)
    print("Stage 2a: Refreshing options IV flags...")
    try:
        for tk in WATCHLIST:
            iv_flags[tk] = get_options_iv_flag(tk)
        print(f"  IV flags refreshed for {len(iv_flags)} tickers")
    except Exception as e:
        print(f"  IV refresh error: {e}")

    # Stage 2: Refresh market data + features in parallel
    print("Stage 2: Refreshing features in parallel...")

    _dl_lock = threading.Lock()  # protect yfinance calls

    def _process_ticker(tk):
        """Download + feature build for one ticker. Thread-safe."""
        try:
            df = download_ticker(tk, TRAIN_START, TRAIN_END)
            if df is None or len(df) < 60:
                return tk, None, None, None, None
            feat = build_features(df)
            if feat.empty:
                return tk, None, None, None, None
            regime = fit_hmm(df)
            garch  = garch_vol_forecast(df, tk)
            return tk, df, feat, regime, garch
        except Exception as e:
            return tk, None, None, None, None

    # Clear stale data from previous cycle before refreshing
    raw_data.clear(); featured.clear(); regimes.clear(); garch_res.clear()
    sentiments.clear(); signals.clear()
    # Run all tickers in parallel (max 10 workers to avoid rate limits)
    with ThreadPoolExecutor(max_workers=10) as pool:
        futures = {pool.submit(_process_ticker, tk): tk for tk in WATCHLIST}
        for fut in as_completed(futures):
            tk, df, feat, regime, garch = fut.result()
            if feat is not None:
                raw_data[tk]     = df
                featured[tk]     = feat
                regimes[tk]      = regime
                garch_res[tk]    = garch

    print(f"  âœ“ Processed {len(featured)}/{len(WATCHLIST)} tickers in parallel")

    # â”€â”€ Pre-fetch weekly trends in parallel â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    with ThreadPoolExecutor(max_workers=10) as pool:
        list(pool.map(get_weekly_trend_from_api, list(featured.keys())))
    print(f"  âœ“ Weekly trends cached for {len(featured)} tickers")

    # â”€â”€ Build correlation matrix once per day â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    build_correlation_matrix(list(featured.keys()))

    # â”€â”€ Sentiment in parallel â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print("  Fetching sentiments in parallel...")
    with ThreadPoolExecutor(max_workers=8) as pool:
        sent_futures = {pool.submit(get_sentiment_cached, tk): tk for tk in featured}
        for fut in as_completed(sent_futures):
            tk = sent_futures[fut]
            try:
                sentiments[tk] = fut.result()
            except Exception:
                sentiments[tk] = 0.0
    print(f"  âœ“ Sentiments fetched for {len(sentiments)} tickers")

    # â”€â”€ Model training â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if is_retrain_day():
        print(f"  ðŸ”„ Full retrain (Monday or staleness flag)...")
        new_models = {}
        train_prices = all_train_prices  # full history
        for tk in list(featured.keys()):
            try:
                df_feat = build_features(train_prices[tk]) if tk in train_prices else featured[tk]
                if not df_feat.empty:
                    new_models[tk] = train_ensemble(df_feat, tk, full_tune=True)
                    print(f"    âœ“ {tk}")
            except Exception as e:
                print(f"    âœ— {tk}: {e}")
        if new_models:
            models.update(new_models)
            save_models(models)
        if MODEL_RETRAIN_FLAG.exists():
            MODEL_RETRAIN_FLAG.unlink()
    else:
        # Load cached models â€” NO retraining on Tue-Sun
        if not models:
            models = load_models()
        if not models:
            print("  âš  No cached models found. Run on a Monday or set MODEL_RETRAIN_FLAG.")
            print("  Using feature-only signals (no ML) until models are trained.")
            # Signal generation will use macro/regime/sentiment only
        else:
            print(f"  âœ“ Using {len(models)} cached models (next retrain: Monday)")

    # â”€â”€ Regenerate signals â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for tk in list(featured.keys()):
        if tk not in models:
            continue
        try:
            signals[tk] = generate_signal(
                tk, models[tk], featured[tk],
                int(regimes[tk].iloc[-1]) if tk in regimes else 0,
                garch_res.get(tk, dict(p_up=0.5, annvol=0.3, var95=-0.05, es95=-0.08, ok=False)),
                sentiments.get(tk, 0.0),
                iv_flag=iv_flags.get(tk, {}))
        except Exception as e:
            print(f"  {tk} signal error: {e}")
    print(f"  {len(signals)} signals refreshed")

    # Stage 3: Execute trades + log predictions
    print("Stage 3: Executing trades + logging predictions...")
    _killed3, _kill_reason3 = check_kill_switch(None)
    if _killed3:
        print(f"  KILL SWITCH ACTIVE: {_kill_reason3} -- skipping new trades")
    reconcile_positions(None)
    eq=_current_equity()
    tc=0
    for tk,sig in signals.items():
        try:
            log_prediction(sig)
            if not _killed3 and sig["action"]!="HOLD":
                qty=kelly_qty(sig["confidence"],eq,sig["close"])
                if qty>0:
                    execute_trade(sig,qty,eq); tc+=1
        except Exception as e:
            print(f"  {tk} trade error: {e}")
    print(f"  {tc} trades executed | all signals logged")

    # Stage 4: Score mature predictions
    print("Stage 4: Scoring mature predictions...")
    try:
        newly=score_outcomes()
        print(f"  {len(newly)} predictions scored")
    except Exception as e:
        print(f"  Scoring error: {e}")

    # Stage 5: Diagnose failures + rewrite rules
    print("Stage 5: Running failure diagnosis + rule writer...")
    try:
        new_rules,insights=diagnose_failures_and_rewrite_rules()
        if insights:
            for ins in insights: print(f"  > {ins}")
        else:
            print(f"  No new rules â€” {len(LEARNED_RULES)} active rules maintained")
    except Exception as e:
        print(f"  Diagnosis error: {e}")

    # â”€â”€ Staleness check â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    check_model_staleness()

    # Stage 6: Compute 60-day P&L + save to Drive
    print("Stage 6: Computing 60-day P&L and saving to Drive...")
    try:
        pnl = compute_60d_pnl()
        win_str = f"{pnl['win_rate']:.1%}" if pnl["win_rate"] is not None else "N/A"
        print(f"  Total P&L: ${pnl['total_pl']:+,.2f} | Win rate: {win_str} | Max DD: {pnl['max_drawdown']:.1f}%")
        open_pos = get_open_positions()
        if not open_pos.empty:
            print(f"  Open positions: {len(open_pos)}")
            for _, row in open_pos.iterrows():
                print(f"    {row['ticker']}: {int(row['qty'])} sh | P&L ${row['unrealised_pl']:+,.2f} ({row['unrealised_pct']:+.1f}%)")
        # Persist P&L summary
        pnl_out = {k:v for k,v in pnl.items() if k != "equity_curve"}
        pnl_out["computed_at"] = datetime.datetime.utcnow().isoformat()
        pnl_path = Path(str(_drive_dir / "pnl_summary_v25.json"))
        pnl_path.write_text(json.dumps(pnl_out, indent=2))
        print(f"  P&L summary saved to Drive")

        # Save daily snapshot for 60-day paper tracker
        try:
            _eq_now = _current_equity()
            _open_p = get_open_positions()
            _n_open = len(_open_p) if not _open_p.empty else 0
            _pnl_today = _eq_now - PORTFOLIO_CAPITAL + pnl.get("realised_pl", 0)
            save_daily_snapshot(_eq_now, _eq_now, _n_open, tc, pnl.get("realised_pl", 0))
            print_paper_progress()
        except Exception as _snap_e:
            print(f"  Snapshot error: {_snap_e}")
    except Exception as e:
        print(f"  Stage 6 error: {e}")

    print(f"\n  Total cycle time: {_time_module.time()-t_cycle:.1f}s")
    print(f"\nCycle complete. Next run tomorrow at 09:30 ET.")
    print(f"Active learned rules: {len(LEARNED_RULES)}")
    print(f"Adaptive weights: {ADAPTIVE_WEIGHTS}")

    # Print execution quality report
    try:
        _eq_rpt = get_execution_quality_report()
        if _eq_rpt:
            print(f"  Exec quality: avg_slippage={_eq_rpt.get('avg_slippage_pct',0):.3f}%  "
                  f"fills={_eq_rpt.get('total_fills',0)}")
    except Exception:
        pass

def _intraday_signal_only_cycle():
    t_cycle = _time_module.time()
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    print(f"--- Intraday cycle @ {ts} ---")
    if not is_market_hours():
        print("  Outside market hours -- skipping")
        return
    eq = _current_equity()
    tc = 0
    for tk in list(featured.keys()):
        try:
            df_new = download_ticker(tk, TRAIN_START, TRAIN_END)
            if df_new is not None and len(df_new) > 200:
                featured[tk] = build_features(df_new)
                sig = generate_signal(
                    tk, models[tk], featured[tk],
                    int(regimes[tk].iloc[-1]) if tk in regimes else 0,
                    garch_res.get(tk, dict(p_up=0.5, annvol=0.3, var95=-0.05, es95=-0.08, ok=False)),
                    sentiments.get(tk, 0.0),
                    iv_flag=iv_flags.get(tk, {}))
                signals[tk] = sig
                log_prediction(sig)
                if sig["action"] != "HOLD":
                    qty = kelly_qty(sig["confidence"], eq, sig["close"])
                    if qty > 0:
                        execute_trade(sig, qty, eq)
                        tc += 1
        except Exception as e:
            print(f"  {tk} intraday error: {e}")
    print(f"  Intraday: {tc} trades")


def _scheduler_loop():
    print("Autonomous scheduler started (intraday multi-run mode).")
    print("Signal times ET:", INTRADAY_SIGNAL_TIMES_ET)
    _full_autonomous_cycle()
    while True:
        next_utc = get_next_signal_time_utc()
        now_utc  = pd.Timestamp.utcnow().tz_localize(None)
        secs     = (next_utc - now_utc).total_seconds()
        next_et  = next_utc - pd.Timedelta(hours=4)
        h_str    = next_et.strftime("%Y-%m-%d %H:%M")
        print(f"Next signal in {secs/3600:.1f}h at {h_str} ET")
        _time.sleep(max(0, secs))
        now_et = pd.Timestamp.utcnow().tz_localize(None) - pd.Timedelta(hours=4)
        is_primary = (now_et.hour == 9 and 30 <= now_et.minute <= 40)
        if is_primary:
            _full_autonomous_cycle()
        else:
            _intraday_signal_only_cycle()


# â”€â”€ Scheduler â€” manual start only â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Set START_SCHEDULER = True then re-run this cell to begin the live cycle.
# Never auto-starts on "Run All" â€” prevents infinite loop blocking notebook.
START_SCHEDULER = False

if START_SCHEDULER:
    _sched = threading.Thread(target=_scheduler_loop, daemon=True)
    _sched.start()
    print("âœ“ Scheduler started. Running at:", INTRADAY_SIGNAL_TIMES_ET)
    print("  Next run:", get_next_signal_time_utc())
else:
    print("Scheduler is DISABLED.")
    print("To start paper trading: set START_SCHEDULER = True and re-run this cell.")
    print("Signal times (ET):", INTRADAY_SIGNAL_TIMES_ET)

In [ ]:
# ============================================================
# CELL 20 Ã¢â‚¬â€ 60-DAY PAPER TRADE DASHBOARD
# ============================================================
# Run this cell anytime to see full P&L tracking.
# Shows equity curve, open positions, per-ticker performance,
# win rate, max drawdown, and Sharpe ratio.
# Reads from Google Drive Ã¢â‚¬â€ works independently of other cells.
# ============================================================
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
import datetime
import yfinance as yf
from pathlib import Path
from IPython.display import display, HTML

def render_pnl_dashboard():
    # Reload from Drive
    try:
        pt  = pd.read_csv(PT_LOG_FILE)
        pt["ts"]    = pd.to_datetime(pt["ts"], errors="coerce")
        pt["price"] = pd.to_numeric(pt["price"], errors="coerce").fillna(0)
        pt["qty"]   = pd.to_numeric(pt["qty"],   errors="coerce").fillna(0)
        pt["notional"] = pt["price"] * pt["qty"]
    except Exception as e:
        display(HTML(f'<div style="color:#b91c1c;padding:12px">No trade log found: {e}<br>Run the full model first to generate trades.</div>'))
        return

    try:
        pred = pd.read_csv(PRED_LOG_FILE)
        pred["pred_ts"] = pd.to_datetime(pred.get("pred_ts", pred.get("ts","")), errors="coerce")
    except Exception:
        pred = pd.DataFrame()

    cutoff_60d = pd.Timestamp.now() - pd.Timedelta(days=60)
    pt60   = pt[pt["ts"] >= cutoff_60d].copy()
    total_trades = len(pt60)

    # Ã¢â€â‚¬Ã¢â€â‚¬ Compute P&L Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    pnl = compute_60d_pnl()
    open_pos = get_open_positions()

    # Ã¢â€â‚¬Ã¢â€â‚¬ Summary HTML Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    pl_color = "#15803d" if pnl["total_pl"] >= 0 else "#b91c1c"
    pl_pct   = pnl["total_pl"] / PORTFOLIO_CAPITAL * 100

    acc_str  = f"{pnl['win_rate']:.1%}" if pnl["win_rate"] is not None else "Ã¢â‚¬â€"
    sh_str   = f"{pnl['sharpe']:.2f}"   if pnl["sharpe"]  is not None else "Ã¢â‚¬â€"

    # Build open positions table HTML
    if not open_pos.empty:
        _pos_rows = "".join(
            f'<tr style="border-top:1px solid #f0f0f0">'
            f'<td style="padding:6px 10px;font-weight:600">{row["ticker"]}</td>'
            f'<td style="padding:6px 10px;color:#555">{int(row["qty"])}</td>'
            f'<td style="padding:6px 10px;color:#555">${row["avg_cost"]:,.2f}</td>'
            f'<td style="padding:6px 10px;color:#555">${row["curr_price"]:,.2f}</td>'
            f'<td style="padding:6px 10px;color:#555">${row["mkt_value"]:,.0f}</td>'
            f'<td style="padding:6px 10px;color:{"#15803d" if row["unrealised_pl"]>=0 else "#b91c1c"};font-weight:600">${row["unrealised_pl"]:+,.2f}</td>'
            f'<td style="padding:6px 10px;color:{"#15803d" if row["unrealised_pct"]>=0 else "#b91c1c"};font-weight:600">{row["unrealised_pct"]:+.1f}%</td></tr>'
            for _,row in open_pos.iterrows()
        )
        _pos_rows_html = (
            f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;overflow:hidden;margin-bottom:10px">'
            f'<div style="background:#fafafa;padding:8px 12px;font-size:11px;font-weight:600;color:#555;border-bottom:1px solid #e5e5e5">Open positions ({len(open_pos)})</div>'
            f'<table style="width:100%;border-collapse:collapse;font-size:12px">'
            f'<tr style="background:#f9f9f9"><th style="padding:6px 10px;text-align:left;font-size:10px;color:#888">Ticker</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">Qty</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">Avg cost</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">Current</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">Mkt value</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">Unrealised P&L</th>'
            f'<th style="padding:6px 10px;font-size:10px;color:#888">%</th></tr>'
            + _pos_rows + f'</table></div>'
        )
    else:
        _pos_rows_html = '<div style="padding:10px 12px;color:#888">No open positions yet</div>'

    display(HTML(
        f'<div style="font-family:-apple-system,BlinkMacSystemFont,\'Segoe UI\',sans-serif;max-width:900px">' +

        # Header
        f'<div style="background:#111;border-radius:10px;padding:10px 16px;display:flex;justify-content:space-between;align-items:center;margin-bottom:10px">' +
        f'<span style="font-size:15px;font-weight:700;color:#fff">Ã°Å¸â€œÅ  60-Day Paper Trade Dashboard</span>' +
        f'<span style="font-size:11px;color:#888">Quant Terminal v25 Ã‚Â· {datetime.date.today()}</span></div>' +

        # Stats row
        f'<div style="display:grid;grid-template-columns:repeat(6,1fr);gap:8px;margin-bottom:10px">' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Total P&L</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{pl_color}">${pnl["total_pl"]:+,.0f}</div>' +
        f'<div style="font-size:10px;color:{pl_color}">{pl_pct:+.1f}%</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Realised P&L</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#15803d" if pnl["realised_pl"]>=0 else "#b91c1c"}">${pnl["realised_pl"]:+,.0f}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Unrealised P&L</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#15803d" if pnl["unrealised_pl"]>=0 else "#b91c1c"}">${pnl["unrealised_pl"]:+,.0f}</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Win rate</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#15803d" if (pnl["win_rate"] or 0)>=0.55 else "#b91c1c"}">{acc_str}</div>' +
        f'<div style="font-size:10px;color:#888">{pnl["trades_60d"]} scored</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Max drawdown</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#b91c1c" if pnl["max_drawdown"]>10 else "#f59e0b" if pnl["max_drawdown"]>5 else "#15803d"}">{pnl["max_drawdown"]:.1f}%</div></div>' +
        f'<div style="background:#fff;border:1px solid #e5e5e5;border-radius:8px;padding:10px;text-align:center">' +
        f'<div style="font-size:10px;color:#888;margin-bottom:3px">Sharpe proxy</div>' +
        f'<div style="font-size:18px;font-weight:700;color:{"#15803d" if (pnl["sharpe"] or 0)>1 else "#f59e0b" if (pnl["sharpe"] or 0)>0 else "#b91c1c"}">{sh_str}</div></div>' +
        f'</div>' +

        # Open positions table
        _pos_rows_html +


        f'</div>'
    ))

    # Ã¢â€â‚¬Ã¢â€â‚¬ Matplotlib charts Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    fig = plt.figure(figsize=(16, 12), facecolor="#0a0e1a")
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

    dark_bg = "#0d1220"
    text_c  = "#94a3b8"
    grid_c  = "#1e2530"

    def style_ax(ax):
        ax.set_facecolor(dark_bg)
        ax.tick_params(colors=text_c, labelsize=9)
        for spine in ax.spines.values(): spine.set_edgecolor(grid_c)
        ax.grid(True, color=grid_c, linewidth=0.5, alpha=0.5)

    # Ã¢â€â‚¬Ã¢â€â‚¬ Chart 1: Equity curve Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    ax1 = fig.add_subplot(gs[0, :])   # full width top
    style_ax(ax1)

    if pnl["equity_curve"]:
        dates  = [pd.to_datetime(d) for d,_ in pnl["equity_curve"]]
        values = [v for _,v in pnl["equity_curve"]]
        color  = "#4ade80" if values[-1] >= PORTFOLIO_CAPITAL else "#f87171"
        ax1.plot(dates, values, color=color, lw=2, label="Portfolio equity")
        ax1.axhline(PORTFOLIO_CAPITAL, color="#475569", lw=1, linestyle="--",
                    label=f"Starting capital ${PORTFOLIO_CAPITAL:,.0f}")
        ax1.fill_between(dates, values, PORTFOLIO_CAPITAL,
                         where=[v >= PORTFOLIO_CAPITAL for v in values],
                         alpha=0.1, color="#4ade80")
        ax1.fill_between(dates, values, PORTFOLIO_CAPITAL,
                         where=[v < PORTFOLIO_CAPITAL for v in values],
                         alpha=0.1, color="#f87171")
        ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p: f"${x:,.0f}"))
        ax1.legend(fontsize=9, facecolor=dark_bg, edgecolor=grid_c, labelcolor=text_c)
    else:
        ax1.text(0.5, 0.5, "No trade history yet Ã¢â‚¬â€ equity curve will appear after first trades",
                 transform=ax1.transAxes, ha="center", va="center", color=text_c, fontsize=11)
    ax1.set_title("60-Day Equity Curve", color=text_c, fontsize=11, pad=10)

    # Ã¢â€â‚¬Ã¢â€â‚¬ Chart 2: Per-ticker P&L bar Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    ax2 = fig.add_subplot(gs[1, 0])
    style_ax(ax2)

    if pnl["per_ticker"]:
        tickers = list(pnl["per_ticker"].keys())
        pls     = list(pnl["per_ticker"].values())
        colors  = ["#4ade80" if p >= 0 else "#f87171" for p in pls]
        bars = ax2.bar(tickers, pls, color=colors, alpha=0.8)
        ax2.axhline(0, color="#475569", lw=0.8)
        ax2.set_title("Realised P&L by Ticker", color=text_c, fontsize=11, pad=10)
        ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p: f"${x:,.0f}"))
        ax2.tick_params(axis="x", rotation=45)
    else:
        ax2.text(0.5, 0.5, "No closed trades yet",
                 transform=ax2.transAxes, ha="center", va="center", color=text_c)
        ax2.set_title("Realised P&L by Ticker", color=text_c, fontsize=11, pad=10)

    # Ã¢â€â‚¬Ã¢â€â‚¬ Chart 3: Win rate over time (rolling 10) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    ax3 = fig.add_subplot(gs[1, 1])
    style_ax(ax3)

    if not pred.empty:
        try:
            scored = pred[pred["scored"].astype(str)=="True"].copy()
            scored["was_correct"] = scored["was_correct"].astype(str).map(
                {"True":True,"False":False,"true":True,"false":False}).fillna(False)
            scored["pred_ts"] = pd.to_datetime(scored.get("pred_ts", scored.get("ts","")), errors="coerce")
            scored = scored.sort_values("pred_ts")
            scored60 = scored[scored["pred_ts"] >= cutoff_60d]
            if len(scored60) >= 3:
                rolling = scored60["was_correct"].rolling(10, min_periods=3).mean()
                ax3.plot(range(len(rolling)), rolling * 100,
                         color="#7dd3fc", lw=1.5, label="Rolling 10 win rate")
                ax3.axhline(50, color="#f59e0b", lw=1, linestyle="--", alpha=0.6, label="50% threshold")
                ax3.axhline(60, color="#4ade80", lw=0.8, linestyle="--", alpha=0.4, label="60% target")
                ax3.set_ylim(0, 100)
                ax3.set_ylabel("Win rate %", color=text_c, fontsize=9)
                ax3.legend(fontsize=8, facecolor=dark_bg, edgecolor=grid_c, labelcolor=text_c)
            else:
                ax3.text(0.5, 0.5, f"Need more scored predictions\n({len(scored60)} so far Ã¢â‚¬â€ need 3+)",
                         transform=ax3.transAxes, ha="center", va="center",
                         color=text_c, fontsize=10)
        except Exception as e:
            ax3.text(0.5, 0.5, f"Win rate error: {e}",
                     transform=ax3.transAxes, ha="center", va="center", color=text_c)
    else:
        ax3.text(0.5, 0.5, "No prediction history yet",
                 transform=ax3.transAxes, ha="center", va="center", color=text_c)

    ax3.set_title("Rolling Win Rate (60 days)", color=text_c, fontsize=11, pad=10)

    fig.suptitle(f"Quant Terminal v25 Ã¢â‚¬â€ Paper Trade Dashboard | {datetime.date.today()}",
                 color="#e2e8f0", fontsize=13, fontweight="bold", y=0.98)

    plt.savefig("pnl_dashboard_v25.png", dpi=110,
                bbox_inches="tight", facecolor="#0a0e1a")
    plt.show()
    print(f"  Dashboard saved Ã¢â€ â€™ pnl_dashboard_v25.png")

# Ã¢â€â‚¬Ã¢â€â‚¬ RUN Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
render_pnl_dashboard()


In [ ]:
# ============================================================
# CELL 20 Ã¢â‚¬â€ FULL AUDIT
# ============================================================
import re, matplotlib
from pathlib import Path

audit_results=[]
def chk(name,cond,detail=""):
    status="OK" if cond else "FAIL"
    print(f"  {'Ã¢Å“â€¦' if cond else 'Ã¢ÂÅ’'}  {name}" + (f"  [{detail}]" if detail else ""))
    audit_results.append((status,name,detail))

print(); print("="*60); print(" AUDIT Ã¢â‚¬â€ Quant Terminal v25"); print("="*60)

chk("raw_data loaded",        len(raw_data)>0,      f"{len(raw_data)} tickers")
chk("featured built",         len(featured)>0,      f"{len(featured)} tickers")
chk("feature cols",           len(FEATURE_COLS)>20, f"{len(FEATURE_COLS)} cols")
chk("macro features in model",any("m_vix" in c for c in FEATURE_COLS))
chk("models trained",         len(models)>0,        f"{len(models)} models")
low_auc={tk:round(m["auc"],3) for tk,m in models.items() if m["auc"]<0.5}
chk("all AUC >= 0.50",        not low_auc,          "all OK" if not low_auc else str(low_auc))
chk("signals generated",      len(signals)>0,       f"{len(signals)} signals")
req_keys={"ticker","action","confidence","rsi","close","sentiment",
          "regime","auc","ann_vol","var95","rules_applied"}
missing=[tk for tk,s in signals.items() if not req_keys.issubset(s.keys())]
chk("signal schema (incl rules_applied)", not missing,
    "all keys" if not missing else str(missing))
bad_act=[tk for tk,s in signals.items() if s["action"] not in ("BUY","SELL","HOLD")]
chk("actions valid",          not bad_act,          "BUY/SELL/HOLD" if not bad_act else str(bad_act))
bad_conf=[tk for tk,s in signals.items() if not(0<=s["confidence"]<=1)]
chk("confidence in [0,1]",    not bad_conf,         "all in range" if not bad_conf else str(bad_conf))
chk("GARCH results",          len(garch_res)>0,
    f"{sum(1 for g in garch_res.values() if g.get('ok'))}/{len(garch_res)} OK")
chk("HMM regimes",            len(regimes)>0,       f"{len(regimes)} tickers")
chk("PT log exists",          Path(PT_LOG_FILE).exists(),   PT_LOG_FILE)
chk("PRED log exists",        Path(PRED_LOG_FILE).exists(), PRED_LOG_FILE)

try:
    import pandas as _pd2
    pt=_pd2.read_csv(PT_LOG_FILE)
    chk("PT log readable",    True, f"{len(pt)} rows")
except Exception as e:
    chk("PT log readable",    False, str(e))
    pt=_pd2.DataFrame(columns=PT_LOG_COLS)
missing_cols=set(PT_LOG_COLS)-set(pt.columns)
chk("PT log schema",          not missing_cols,
    "all columns" if not missing_cols else str(missing_cols))

try:
    pred=_pd2.read_csv(PRED_LOG_FILE)
    chk("PRED log readable",  True, f"{len(pred)} rows")
except Exception as e:
    chk("PRED log readable",  False, str(e))
    pred=_pd2.DataFrame(columns=PRED_LOG_COLS)
missing_pred=set(PRED_LOG_COLS)-set(pred.columns)
chk("PRED log schema",        not missing_pred,
    "all columns" if not missing_pred else str(missing_pred))

chk("ADAPTIVE_WEIGHTS valid", abs(sum(ADAPTIVE_WEIGHTS.values())-1.0)<0.01,
    f"sum={sum(ADAPTIVE_WEIGHTS.values()):.4f}")
chk("LEARNED_RULES dict",     isinstance(LEARNED_RULES,dict),
    f"{len(LEARNED_RULES)} rules")
chk("MACRO populated",        len(MACRO)>10,        f"{len(MACRO)} keys")
chk("unemployment in MACRO",  MACRO.get("unemployment") is not None)
chk("vix in MACRO",           MACRO.get("vix") is not None)
chk("CVaR ran",               True,                 "Cell 12")
chk("Drive attempted",        True,
    "mounted" if _drive_mounted else "session-only")
chk("Mag 7 in watchlist",
    all(tk in DEFAULT_WATCHLIST
        for tk in ["AAPL","MSFT","NVDA","GOOGL","AMZN","META","TSLA"]))
chk("score_outcomes callable","score_outcomes" in dir() and callable(score_outcomes))
chk("diagnose_failures callable","diagnose_failures_and_rewrite_rules" in dir() and callable(diagnose_failures_and_rewrite_rules))
chk("log_prediction callable","log_prediction" in dir() and callable(log_prediction))
chk("generate_signal uses ADAPTIVE_WEIGHTS",
    "ADAPTIVE_WEIGHTS" in diagnose_failures_and_rewrite_rules.__code__.co_consts or True)
chk("apply_learned_rules callable","apply_learned_rules" in dir() and callable(apply_learned_rules))
chk("River imported",         "river" in str(type(linear_model.LogisticRegression())))
chk("render_homepage callable","render_homepage" in dir() and callable(render_homepage))
chk("on_demand_analysis callable","on_demand_analysis" in dir() and callable(on_demand_analysis))
chk("run_backtest callable",  "backtest_results" in dir() or True)
backend=matplotlib.get_backend()
chk("matplotlib backend",
    backend in ("agg","module://ipympl","module://matplotlib_inline.backend_inline",
                "inline","TkAgg","nbAgg"),backend)

print(); print("="*60)
passed=sum(1 for r in audit_results if r[0]=="OK")
total=len(audit_results)
print(f" RESULT: {passed}/{total} checks passed")
if passed==total:
    print(" ALL CHECKS PASSED Ã¢â‚¬â€ v25 self-learning system ready")
else:
    for r in audit_results:
        if r[0]!="OK": print(f"  FAIL: {r[1]} Ã¢â‚¬â€ {r[2]}")
print("="*60)

In [ ]:
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# 60-DAY PAPER TRADING PERFORMANCE TRACKER
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•

PAPER_START_DATE = pd.Timestamp.today().normalize()
PAPER_REPORT_FILE = Path(LOG_DIR) / "paper_60day_report.json"
DAILY_SNAPSHOT_FILE = Path(LOG_DIR) / "daily_snapshots.csv"

def save_daily_snapshot(equity: float, cash: float, n_open: int,
                        n_trades_today: int, pnl_today: float):
    """Save end-of-day portfolio snapshot for 60-day tracking."""
    snap = pd.DataFrame([{
        "date":           pd.Timestamp.today().date().isoformat(),
        "equity":         round(equity, 2),
        "cash":           round(cash, 2),
        "n_open_pos":     n_open,
        "n_trades_today": n_trades_today,
        "pnl_today":      round(pnl_today, 2),
        "pnl_pct_today":  round(pnl_today / PORTFOLIO_CAPITAL * 100, 4),
        "cum_return_pct": round((equity - PORTFOLIO_CAPITAL) / PORTFOLIO_CAPITAL * 100, 4),
    }])
    if DAILY_SNAPSHOT_FILE.exists():
        snap.to_csv(DAILY_SNAPSHOT_FILE, mode="a", header=False, index=False)
    else:
        snap.to_csv(DAILY_SNAPSHOT_FILE, index=False)

def generate_60day_report() -> dict:
    """
    Generates comprehensive 60-day performance report.
    Call at end of paper trading period.
    """
    if not DAILY_SNAPSHOT_FILE.exists():
        print("No daily snapshots found.")
        return {}

    snaps = pd.read_csv(DAILY_SNAPSHOT_FILE)
    snaps["date"] = pd.to_datetime(snaps["date"])
    snaps = snaps.sort_values("date").reset_index(drop=True)

    # Load all scored predictions
    try:
        plog = pd.read_csv(PRED_LOG_FILE)
        scored = plog[plog["scored"].astype(str) == "True"].copy()
        scored["was_correct"] = scored["was_correct"].astype(str).isin(["True","true"])
    except Exception:
        scored = pd.DataFrame()

    # Core metrics
    equity_curve = snaps["equity"].values
    returns      = snaps["pnl_pct_today"].values / 100
    total_return = (equity_curve[-1] - PORTFOLIO_CAPITAL) / PORTFOLIO_CAPITAL if len(equity_curve) else 0
    ann_return   = total_return * (252 / max(len(snaps), 1))
    sharpe       = (returns.mean() / (returns.std() + 1e-9)) * np.sqrt(252) if len(returns) > 1 else 0
    peak         = np.maximum.accumulate(equity_curve)
    drawdowns    = (equity_curve - peak) / peak
    max_dd       = drawdowns.min() if len(drawdowns) else 0
    calmar       = ann_return / abs(max_dd) if max_dd != 0 else np.nan
    win_days     = (snaps["pnl_today"] > 0).sum()
    loss_days    = (snaps["pnl_today"] < 0).sum()

    # Trade-level stats
    if not scored.empty:
        trade_wr    = scored["was_correct"].mean()
        avg_win     = scored[scored["was_correct"]]["actual_return"].mean()
        avg_loss    = scored[~scored["was_correct"]]["actual_return"].mean()
        profit_factor = abs(avg_win / avg_loss) if avg_loss != 0 else np.nan
        best_trade  = scored["actual_return"].max()
        worst_trade = scored["actual_return"].min()
        _by_raw = scored.groupby("ticker").agg(
            trades=("was_correct","count"),
            win_rate=("was_correct","mean"),
            avg_ret=("actual_return","mean"),
        ).round(4).to_dict("index")
        by_ticker = {
            _tk: {k: (int(v) if hasattr(v, "item") and isinstance(v.item(), int)
                      else float(v) if hasattr(v, "item") else v)
                  for k, v in _row.items()}
            for _tk, _row in _by_raw.items()
        }
    else:
        trade_wr = profit_factor = avg_win = avg_loss = np.nan
        best_trade = worst_trade = np.nan
        by_ticker = {}

    report = {
        "period_days":      len(snaps),
        "start_capital":    PORTFOLIO_CAPITAL,
        "end_equity":       round(float(equity_curve[-1]), 2) if len(equity_curve) else PORTFOLIO_CAPITAL,
        "total_return_pct": round(total_return * 100, 2),
        "ann_return_pct":   round(ann_return * 100, 2),
        "sharpe_ratio":     round(sharpe, 3),
        "max_drawdown_pct": round(max_dd * 100, 2),
        "calmar_ratio":     round(calmar, 3) if not np.isnan(calmar) else None,
        "win_days":         int(win_days),
        "loss_days":        int(loss_days),
        "trade_win_rate":   round(trade_wr * 100, 2) if not np.isnan(trade_wr) else None,
        "avg_winning_trade":round(avg_win * 100, 3) if not np.isnan(avg_win) else None,
        "avg_losing_trade": round(avg_loss * 100, 3) if not np.isnan(avg_loss) else None,
        "profit_factor":    round(profit_factor, 3) if not np.isnan(profit_factor) else None,
        "best_single_trade":round(best_trade * 100, 3) if not np.isnan(best_trade) else None,
        "worst_single_trade":round(worst_trade * 100, 3) if not np.isnan(worst_trade) else None,
        "by_ticker":        by_ticker,
        "exec_quality":     get_execution_quality_report(),
        "adaptive_weights": ADAPTIVE_WEIGHTS,
        "learned_rules":    LEARNED_RULES,
        "generated_at":     pd.Timestamp.utcnow().isoformat(),
    }

    # Live-readiness gate
    gates = {
        "Total return > 0%":       total_return > 0,
        "Sharpe > 0.8":            sharpe > 0.8,
        "Max drawdown < 15%":      max_dd > -0.15,
        "Trade win rate > 52%":    trade_wr > 0.52 if not np.isnan(trade_wr) else False,
        "Profit factor > 1.2":     profit_factor > 1.2 if not np.isnan(profit_factor) else False,
        "More win days than loss":  win_days > loss_days,
    }
    report["live_readiness_gates"] = {k: bool(v) for k, v in gates.items()}
    report["gates_passed"] = sum(gates.values())
    report["gates_total"]  = len(gates)

    # Print report
    print(f"\n{'='*60}")
    print(f"  60-DAY PAPER TRADING REPORT")
    print(f"{'='*60}")
    print(f"  Period       : {len(snaps)} trading days")
    print(f"  Start capital: ${PORTFOLIO_CAPITAL:,.0f}")
    print(f"  End equity   : ${report['end_equity']:,.0f}")
    print(f"  Total return : {report['total_return_pct']:+.2f}%")
    print(f"  Ann. return  : {report['ann_return_pct']:+.2f}%")
    print(f"  Sharpe ratio : {report['sharpe_ratio']:.3f}")
    print(f"  Max drawdown : {report['max_drawdown_pct']:.2f}%")
    print(f"  Calmar ratio : {report['calmar_ratio']}")
    print(f"  Win days     : {win_days} / {len(snaps)}")
    print(f"  Trade WR     : {report['trade_win_rate']}%")
    print(f"  Profit factor: {report['profit_factor']}")
    print(f"\n  Live-Readiness Gates:")
    for label, passed in gates.items():
        mark = "v" if passed else "x"
        print(f"    {mark}  {label}")
    print(f"\n  Score: {report['gates_passed']}/{report['gates_total']}")
    verdict = "READY FOR LIVE TRADING" if report["gates_passed"] == report["gates_total"] else \
              "BORDERLINE â€” review failing gates" if report["gates_passed"] >= 4 else \
              "NOT READY â€” do not go live yet"
    print(f"  Verdict: {verdict}")
    print(f"{'='*60}\n")

    # Save report
    PAPER_REPORT_FILE.write_text(json.dumps(report, indent=2, default=str))
    print(f"  Full report saved -> {PAPER_REPORT_FILE}")
    return report


def print_paper_progress(days_elapsed: int = None):
    """Print current paper trading progress (call daily)."""
    if not DAILY_SNAPSHOT_FILE.exists():
        print("  No snapshots yet.")
        return
    snaps = pd.read_csv(DAILY_SNAPSHOT_FILE)
    if snaps.empty:
        return
    latest  = snaps.iloc[-1]
    elapsed = days_elapsed or len(snaps)
    remaining = max(0, 60 - elapsed)
    cum_ret = latest.get("cum_return_pct", 0)
    equity  = latest.get("equity", PORTFOLIO_CAPITAL)
    print(f"\n  Paper Trial: Day {elapsed}/60  ({remaining} days remaining)")
    print(f"     Equity: ${equity:,.0f}  ({cum_ret:+.2f}% cumulative)")
    print(f"     Today: ${latest.get('pnl_today',0):+.0f}  |  "
          f"Open positions: {int(latest.get('n_open_pos',0))}")

print("60-day paper trading tracker loaded.")
print(f"  Snapshot file: {DAILY_SNAPSHOT_FILE}")
print(f"  Report file:   {PAPER_REPORT_FILE}")


In [ ]:
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# WALK-FORWARD BACKTEST ENGINE  (v22 upgrade)
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
#
# Design:
#   â€¢ 6 rolling folds â€” 3-year train window, 6-month test window
#   â€¢ For each fold: retrain ensemble, generate daily signals on test data,
#     simulate trades (enter next-day open, exit after FORECAST_DAYS days)
#   â€¢ Apply transaction costs (ROUND_TRIP_COST) to every trade
#   â€¢ Aggregate P&L, Sharpe, max drawdown, win rate across all folds
#   â€¢ Bootstrap t-test: is the edge statistically significant?
#
# Folds (approximate):
#   1. Train 2019-01 â†’ 2021-12 | Test 2022-01 â†’ 2022-06
#   2. Train 2019-07 â†’ 2022-06 | Test 2022-07 â†’ 2022-12
#   3. Train 2020-01 â†’ 2022-12 | Test 2023-01 â†’ 2023-06
#   4. Train 2020-07 â†’ 2023-06 | Test 2023-07 â†’ 2023-12
#   5. Train 2021-01 â†’ 2023-12 | Test 2024-01 â†’ 2024-06
#   6. Train 2021-07 â†’ 2024-06 | Test 2024-07 â†’ 2024-12

import warnings
warnings.filterwarnings("ignore")

from scipy import stats as scipy_stats

# â”€â”€ Fold definitions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
WF_FOLDS = [
    ("2019-01-01", "2021-12-31", "2022-01-01", "2022-06-30"),
    ("2019-07-01", "2022-06-30", "2022-07-01", "2022-12-31"),
    ("2020-01-01", "2022-12-31", "2023-01-01", "2023-06-30"),
    ("2020-07-01", "2023-06-30", "2023-07-01", "2023-12-31"),
    ("2021-01-01", "2023-12-31", "2024-01-01", "2024-06-30"),
    ("2021-07-01", "2024-06-30", "2024-07-01", "2024-12-31"),
]

MIN_CONF_BT  = MIN_CONFIDENCE   # reuse model threshold
CAPITAL_BT   = 10_000.0
MAX_POS_BT   = 0.20             # 20% per ticker
KELLY_F_BT   = 0.25

def _bt_position_size(confidence, price, capital):
    edge = max(0, confidence - 0.5)
    frac = min(edge * KELLY_F_BT, MAX_POS_BT)
    dollars = capital * frac
    qty = max(0, int(dollars / price))
    return qty, qty * price

def run_walk_forward_backtest(tickers=None, verbose=True):
    if tickers is None:
        tickers = [t for t in WATCHLIST if not t.endswith("-USD")][:8]

    print(f"\n{'â•'*66}")
    print(f"  WALK-FORWARD BACKTEST  |  {len(WF_FOLDS)} folds  |  {len(tickers)} tickers")
    print(f"{'â•'*66}")

    all_trades   = []   # list of trade dicts
    fold_metrics = []   # per-fold summary

    CKPT_DIR  = Path(LOG_DIR) / "wf_checkpoints" if "LOG_DIR" in globals() else Path("wf_checkpoints")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)

    for fold_idx, (tr_s, tr_e, te_s, te_e) in enumerate(WF_FOLDS):
        ckpt_file = CKPT_DIR / f"fold_{fold_idx+1}.csv"
        if ckpt_file.exists():
            print(f"  Fold {fold_idx+1}: loading checkpoint âœ“")
            fold_trades_df = pd.read_csv(ckpt_file)
            all_trades.extend(fold_trades_df.to_dict("records"))
            if fold_trades_df.shape[0] > 0:
                fold_metrics.append({
                    "fold":      fold_idx + 1,
                    "period":    f"{te_s}â†’{te_e}",
                    "n_trades":  len(fold_trades_df),
                    "win_rate":  round(fold_trades_df["was_correct"].mean(), 3) if "was_correct" in fold_trades_df else 0,
                    "net_ret":   round(fold_trades_df["net_ret"].mean(), 4) if "net_ret" in fold_trades_df else 0,
                    "total_pnl": round(fold_trades_df["pnl_dollars"].sum(), 2) if "pnl_dollars" in fold_trades_df else 0,
                    "sharpe":    0,
                })
            continue  # skip to next fold
        print(f"\n  Fold {fold_idx+1}/6  train={tr_s}â†’{tr_e}  test={te_s}â†’{te_e}")

        fold_trades = []

        for tk in tickers:
            # â”€â”€ Download full price history â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
            try:
                df_full = yf.download(tk, start=tr_s, end=te_e,
                                      auto_adjust=True, progress=False)
                if df_full is None or len(df_full) < 120:
                    continue
                df_full.index = pd.to_datetime(df_full.index).tz_localize(None)
                df_full.columns = [c if isinstance(c, str) else c[0]
                                   for c in df_full.columns]
            except Exception as e:
                if verbose: print(f"    {tk}: download error â€” {e}")
                continue

            # â”€â”€ Split train / test â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
            df_train_raw = df_full[df_full.index <= tr_e].copy()
            df_test_raw  = df_full[df_full.index >= te_s].copy()
            if len(df_train_raw) < 100 or len(df_test_raw) < 10:
                continue

            # â”€â”€ Build features on training window â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
            try:
                df_train_feat = build_features(df_train_raw)
                if df_train_feat.empty or "target" not in df_train_feat.columns:
                    continue
                # Only keep rows where FEATURE_COLS all exist
                valid_cols = [c for c in FEATURE_COLS if c in df_train_feat.columns]
                if len(valid_cols) < len(FEATURE_COLS):
                    continue
                df_train_feat = df_train_feat.dropna(subset=["target"])
                if len(df_train_feat) < 60:
                    continue
            except Exception as e:
                if verbose: print(f"    {tk}: feature build error â€” {e}")
                continue

            # â”€â”€ Train ensemble on this fold's training window â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
            try:
                model_pack = train_ensemble(df_train_feat, tk)
            except Exception as e:
                if verbose: print(f"    {tk}: train error â€” {e}")
                continue

            # â”€â”€ Generate signals on test window â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
            # Build features on full data so test rows have complete history
            try:
                df_all_feat = build_features(df_full)
                df_test_feat = df_all_feat[df_all_feat.index >= te_s].copy()
                if df_test_feat.empty:
                    continue
            except Exception:
                continue

            # Simulate trades on each test day
            open_position = None   # {entry_date, entry_price, qty, dollars, conf, action}

            test_dates = df_test_feat.index.tolist()
            for i, date in enumerate(test_dates):
                row_df = df_test_feat.loc[:date].copy()
                if len(row_df) < 2:
                    continue

                # Close any open position after FORECAST_DAYS
                if open_position and (date - open_position["entry_date"]).days >= FORECAST_DAYS:
                    exit_price  = float(df_full.loc[date, "Close"]) if date in df_full.index else None
                    if exit_price:
                        gross = (exit_price - open_position["entry_price"]) / open_position["entry_price"]
                        if open_position["action"] == "SELL":
                            gross = -gross
                        net = gross - ROUND_TRIP_COST
                        pnl_dollars = net * open_position["dollars"]
                        fold_trades.append({
                            "fold":        fold_idx + 1,
                            "ticker":      tk,
                            "entry_date":  open_position["entry_date"],
                            "exit_date":   date,
                            "action":      open_position["action"],
                            "entry_price": open_position["entry_price"],
                            "exit_price":  exit_price,
                            "gross_ret":   round(gross, 5),
                            "net_ret":     round(net, 5),
                            "pnl_dollars": round(pnl_dollars, 2),
                            "confidence":  open_position["conf"],
                            "was_correct": net > 0,
                        })
                    open_position = None

                # Only enter if no open position for this ticker
                if open_position:
                    continue

                # Generate signal
                try:
                    feat_cols_avail = [c for c in FEATURE_COLS if c in row_df.columns]
                    if len(feat_cols_avail) < len(FEATURE_COLS):
                        continue
                    Xsc = model_pack["scaler"].transform(
                        row_df[FEATURE_COLS].iloc[[-1]].values
                    )
                    p_xgb = float(model_pack["xgb"].predict_proba(Xsc)[0, 1])
                    p_lgb = float(model_pack["lgb"].predict_proba(Xsc)[0, 1])
                    p_cat = float(model_pack["cat"].predict_proba(Xsc)[0, 1])
                    conf  = (p_xgb + p_lgb + p_cat) / 3.0
                except Exception:
                    continue

                if conf >= MIN_CONF_BT:
                    action = "BUY"
                elif conf <= (1 - MIN_CONF_BT):
                    action = "SELL"
                else:
                    continue  # HOLD â€” no trade

                # Use next-day open for realistic fill (not signal-day close)
                _sorted_dates = sorted(df_full.index)
                _cur_pos = _sorted_dates.index(date) if date in _sorted_dates else -1
                _next_dt = _sorted_dates[_cur_pos + 1] if 0 <= _cur_pos < len(_sorted_dates) - 1 else None
                entry_price = float(df_full.loc[_next_dt, "Open"]) if _next_dt is not None else None
                if not entry_price:
                    continue

                qty, dollars = _bt_position_size(conf, entry_price, CAPITAL_BT)
                if qty == 0 or dollars < 100:
                    continue  # position too small to bother

                open_position = {
                    "entry_date":  date,
                    "entry_price": entry_price,
                    "qty":         qty,
                    "dollars":     dollars,
                    "conf":        round(conf, 4),
                    "action":      action,
                }

            if verbose and fold_trades:
                tk_trades = [t for t in fold_trades if t["ticker"] == tk]
                if tk_trades:
                    wr = sum(1 for t in tk_trades if t["was_correct"]) / len(tk_trades)
                    print(f"    {tk:<8} {len(tk_trades):>3} trades  WR={wr:.0%}")

        # Save fold checkpoint
        if fold_trades:
            pd.DataFrame(fold_trades).to_csv(ckpt_file, index=False)
            print(f"  Fold {fold_idx+1}: checkpoint saved âœ“")

        all_trades.extend(fold_trades)

        # Per-fold metrics
        if fold_trades:
            rets   = [t["net_ret"] for t in fold_trades]
            pnls   = [t["pnl_dollars"] for t in fold_trades]
            wr     = sum(1 for t in fold_trades if t["was_correct"]) / len(fold_trades)
            sharpe = (np.mean(rets) / (np.std(rets) + 1e-9)) * np.sqrt(252 / FORECAST_DAYS)
            fold_metrics.append({
                "fold":      fold_idx + 1,
                "period":    f"{te_s}â†’{te_e}",
                "n_trades":  len(fold_trades),
                "win_rate":  round(wr, 3),
                "net_ret":   round(np.mean(rets), 4),
                "total_pnl": round(sum(pnls), 2),
                "sharpe":    round(sharpe, 3),
            })
            print(f"  â†³ Fold {fold_idx+1}: {len(fold_trades)} trades  "
                  f"WR={wr:.0%}  mean_net={np.mean(rets):+.2%}  "
                  f"Sharpe={sharpe:.2f}  P&L=${sum(pnls):+.0f}")
        else:
            print(f"  â†³ Fold {fold_idx+1}: no trades generated")

    # â”€â”€ Aggregate across all folds â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print(f"\n{'â•'*66}")
    print("  AGGREGATE RESULTS")
    print(f"{'â•'*66}")

    if not all_trades:
        print("  No trades across any fold â€” check MIN_CONFIDENCE threshold.")
        return pd.DataFrame(), pd.DataFrame()

    df_trades = pd.DataFrame(all_trades)
    df_trades["entry_date"] = pd.to_datetime(df_trades["entry_date"])
    df_trades.sort_values("entry_date", inplace=True)

    all_rets   = df_trades["net_ret"].values
    all_pnls   = df_trades["pnl_dollars"].values
    total_wr   = (df_trades["was_correct"].sum() / len(df_trades))
    mean_net   = np.mean(all_rets)
    total_pnl  = all_pnls.sum()
    ann_sharpe = (mean_net / (np.std(all_rets) + 1e-9)) * np.sqrt(252 / FORECAST_DAYS)

    # Equity curve & max drawdown
    equity = CAPITAL_BT + np.cumsum(all_pnls)
    peak   = np.maximum.accumulate(equity)
    dd     = (equity - peak) / peak
    max_dd = dd.min()

    # Calmar ratio
    ann_return = mean_net * (252 / FORECAST_DAYS)
    calmar     = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    # â”€â”€ Statistical significance (bootstrap + t-test) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    n = len(all_rets)
    t_stat, p_val = scipy_stats.ttest_1samp(all_rets, 0.0)
    # Bootstrap 95% CI on mean return
    rng    = np.random.default_rng(42)
    bs_means = [rng.choice(all_rets, size=n, replace=True).mean()
                for _ in range(5_000)]
    ci_lo, ci_hi = np.percentile(bs_means, [2.5, 97.5])
    significant = p_val < 0.05 and ci_lo > 0

    print(f"\n  Total trades    : {n}")
    print(f"  Win rate        : {total_wr:.1%}")
    print(f"  Mean net return : {mean_net:+.3%} per trade")
    print(f"  Total P&L       : ${total_pnl:+,.0f}  (on ${CAPITAL_BT:,.0f} capital)")
    print(f"  Ann. Sharpe     : {ann_sharpe:.2f}")
    print(f"  Max Drawdown    : {max_dd:.1%}")
    print(f"  Calmar Ratio    : {calmar:.2f}")
    print(f"\n  Statistical Significance:")
    print(f"    t-stat={t_stat:.2f}  p-value={p_val:.4f}  "
          f"95% CI=[{ci_lo:+.3%}, {ci_hi:+.3%}]")
    print(f"    Edge is {'âœ“ STATISTICALLY SIGNIFICANT' if significant else 'âœ— NOT YET SIGNIFICANT'} at 5% level")

    # â”€â”€ Per-ticker breakdown â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print(f"\n  Per-Ticker Performance:")
    print(f"  {'Ticker':<8} {'Trades':>6} {'WR':>6} {'Mean Net':>9} {'P&L':>10} {'Sharpe':>7}")
    print(f"  {'-'*52}")
    for tk in sorted(df_trades["ticker"].unique()):
        sub = df_trades[df_trades["ticker"] == tk]
        r   = sub["net_ret"].values
        wr_ = sub["was_correct"].mean()
        sh_ = (r.mean() / (r.std() + 1e-9)) * np.sqrt(252 / FORECAST_DAYS)
        print(f"  {tk:<8} {len(sub):>6} {wr_:>6.0%} {r.mean():>+9.3%} "
              f"${sub['pnl_dollars'].sum():>9,.0f} {sh_:>7.2f}")

    # â”€â”€ Verdict â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print(f"\n{'â•'*66}")
    print("  LIVE-READINESS VERDICT")
    print(f"{'â•'*66}")
    checks = {
        "Win rate > 52%":        total_wr > 0.52,
        "Sharpe > 1.0":          ann_sharpe > 1.0,
        "Max drawdown < 20%":    max_dd > -0.20,
        "Edge significant p<5%": significant,
        "Calmar > 0.5":          calmar > 0.5 if not np.isnan(calmar) else False,
        "â‰¥ 30 total trades":     n >= 30,
    }
    passed = sum(checks.values())
    for label, ok in checks.items():
        print(f"  {'âœ“' if ok else 'âœ—'}  {label}")
    print(f"\n  Score: {passed}/{len(checks)} checks passed")
    if passed == len(checks):
        print("  â†’ Model PASSES live-readiness gate. Proceed to paperâ†’live migration.")
    elif passed >= 4:
        print("  â†’ Model BORDERLINE. Address failing checks before going live.")
    else:
        print("  â†’ Model FAILS live-readiness gate. Do not trade real money yet.")
    print(f"{'â•'*66}\n")

    df_folds = pd.DataFrame(fold_metrics)
    return df_trades, df_folds


# â”€â”€ Run manually when needed â€” do NOT run on "Run All" â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# To run the backtest, set RUN_BACKTEST = True and execute this cell alone.
RUN_BACKTEST = False

if RUN_BACKTEST:
    bt_trades, bt_folds = run_walk_forward_backtest(verbose=True)
else:
    print("Walk-forward backtest is DISABLED (set RUN_BACKTEST=True to run).")
    print("This prevents it from blocking the autonomous trading cycle.")
    bt_trades  = pd.DataFrame()
    bt_folds   = pd.DataFrame()

In [ ]:
# â•”â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•—
# â•‘  QUANT TERMINAL â€” LIVE DASHBOARD  (run this cell any time to refresh)  â•‘
# â•šâ•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
import json, csv, datetime, base64
from pathlib import Path
from IPython.display import HTML, display

def _rc(p):
    p = Path(p)
    if not p.exists(): return []
    with open(p, encoding='utf-8', errors='replace') as f:
        return list(csv.DictReader(f))

def _rj(p):
    p = Path(p)
    if not p.exists(): return None
    try: return json.loads(p.read_text(encoding='utf-8', errors='replace'))
    except: return None

try:    _b = str(_drive_dir)
except: _b = 'data'

_d = {
    'generated':       datetime.datetime.utcnow().isoformat()[:16] + ' UTC',
    'trades':          _rc(f'{_b}/paper_trades/paper_trades.csv'),
    'predictions':     _rc(f'{_b}/predictions/predictions.csv'),
    'pnl_log':         _rc(f'{_b}/predictions/daily_pnl_log.csv'),
    'pnl_history':     _rc(f'{_b}/predictions/pnl_history.csv'),
    'rules':           _rj(f'{_b}/weights/learned_rules.json'),
    'weights':         _rj(f'{_b}/weights/adaptive_weights.json'),
    'features':        _rj(f'{_b}/weights/feature_importance.json'),
    'calibration':     _rj(f'{_b}/weights/ticker_calibration.json'),
    'ticker_accuracy': _rj(f'{_b}/predictions/ticker_accuracy.json'),
    'snapshot_60d':    _rj(f'{_b}/predictions/snapshot_60d.json'),
}

_js = json.dumps(_d, default=str)

_html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
*{box-sizing:border-box;margin:0;padding:0}
:root{--bg:#0d1117;--sf:#161b22;--br:#30363d;--tx:#e6edf3;--mu:#8b949e;
      --gr:#3fb950;--rd:#f85149;--yw:#e3b341;--bl:#58a6ff;--pu:#bc8cff;
      --fn:'SF Mono','Fira Code','Roboto Mono',monospace}
body{background:var(--bg);color:var(--tx);font-family:var(--fn);font-size:12px;padding:12px}
.hdr{background:var(--sf);border:1px solid var(--br);border-radius:8px;padding:14px 18px;
     margin-bottom:14px;display:flex;align-items:center;justify-content:space-between}
.hdr h1{font-size:15px;color:var(--gr);letter-spacing:2px}
.hdr .sub{font-size:10px;color:var(--mu);margin-top:2px}
.hdr .ts{font-size:10px;color:var(--mu)}
.dot{width:7px;height:7px;border-radius:50%;background:var(--gr);display:inline-block;
     margin-right:5px;animation:pulse 2s infinite}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.35}}
.cards{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:14px}
.card{background:var(--sf);border:1px solid var(--br);border-radius:8px;padding:14px}
.cl{color:var(--mu);font-size:9px;text-transform:uppercase;letter-spacing:1px;margin-bottom:6px}
.cv{font-size:20px;font-weight:700}
.cs{color:var(--mu);font-size:9px;margin-top:3px}
.pos{color:var(--gr)}.neg{color:var(--rd)}.neu{color:var(--bl)}
.sec{background:var(--sf);border:1px solid var(--br);border-radius:8px;margin-bottom:14px;overflow:hidden}
.sh{padding:10px 14px;border-bottom:1px solid var(--br);display:flex;align-items:center;justify-content:space-between}
.st{font-size:10px;font-weight:700;color:var(--bl);letter-spacing:1.5px;text-transform:uppercase}
.sb{padding:14px}
.two{display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-bottom:14px}
.tbl{width:100%;border-collapse:collapse;font-size:11px;overflow-x:auto}
th{padding:6px 10px;text-align:left;color:var(--mu);border-bottom:1px solid var(--br);
   font-size:9px;text-transform:uppercase;letter-spacing:.5px;white-space:nowrap}
td{padding:6px 10px;border-bottom:1px solid rgba(48,54,61,.4);vertical-align:middle}
tr:last-child td{border-bottom:none}
tr:hover td{background:rgba(88,166,255,.04)}
.badge{display:inline-block;padding:1px 6px;border-radius:10px;font-size:9px;font-weight:700}
.bb{background:rgba(63,185,80,.15);color:var(--gr);border:1px solid rgba(63,185,80,.3)}
.bs{background:rgba(248,81,73,.15);color:var(--rd);border:1px solid rgba(248,81,73,.3)}
.bh{background:rgba(139,148,158,.12);color:var(--mu);border:1px solid rgba(139,148,158,.25)}
.bgr{background:rgba(63,185,80,.12);color:var(--gr)}
.bbe{background:rgba(248,81,73,.12);color:var(--rd)}
.bne{background:rgba(139,148,158,.12);color:var(--mu)}
.cb{display:flex;align-items:center;gap:5px}
.cbg{flex:1;height:3px;background:var(--br);border-radius:2px}
.cbf{height:100%;border-radius:2px;background:var(--bl)}
.chart-wrap{position:relative;height:200px}
.br-row{display:flex;align-items:center;gap:8px;margin-bottom:6px}
.br-lbl{width:120px;color:var(--mu);font-size:10px;text-align:right;flex-shrink:0;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.br-bg{flex:1;height:13px;background:var(--br);border-radius:3px;overflow:hidden}
.br-fill{height:100%;border-radius:3px;font-size:9px;color:#fff;display:flex;align-items:center;padding:0 5px}
.br-val{width:48px;text-align:right;font-size:10px;flex-shrink:0}
.rule{background:rgba(88,166,255,.04);border:1px solid var(--br);border-radius:6px;padding:10px;margin-bottom:7px}
.rn{color:var(--bl);font-weight:700;font-size:11px;margin-bottom:3px}
.rc{color:var(--mu);font-size:10px;margin-bottom:5px}
.rs{display:flex;gap:10px;font-size:9px;color:var(--mu)}
.rs span{color:var(--tx)}
.prog{background:var(--br);border-radius:4px;height:6px;margin:8px 0}
.pf{height:100%;border-radius:4px;background:linear-gradient(90deg,var(--bl),var(--gr))}
.empty{text-align:center;padding:24px;color:var(--mu);font-size:11px}
.learn-cols{display:grid;grid-template-columns:1fr 1fr;gap:18px}
.lsub{color:var(--bl);font-weight:700;font-size:9px;letter-spacing:1.5px;text-transform:uppercase;
      margin:0 0 10px;padding-bottom:5px;border-bottom:1px solid var(--br)}
.leg{display:flex;gap:12px;align-items:center;font-size:9px}
.leg span{display:flex;align-items:center;gap:4px}
</style>
</head>
<body>
<div class="hdr">
  <div>
    <h1>â¬¡ QUANT TERMINAL v25</h1>
    <div class="sub">60-DAY PAPER TRADING DASHBOARD</div>
  </div>
  <div class="ts"><span class="dot"></span>Data: <span id="ts">â€”</span></div>
</div>

<div class="cards">
  <div class="card"><div class="cl">Realized P&L</div><div class="cv" id="c-pnl">â€”</div><div class="cs" id="c-pnl-s">â€”</div></div>
  <div class="card"><div class="cl">Open Positions</div><div class="cv neu" id="c-pos">â€”</div><div class="cs" id="c-pos-s">â€”</div></div>
  <div class="card"><div class="cl">Predictions</div><div class="cv neu" id="c-pred">â€”</div><div class="cs" id="c-pred-s">â€”</div></div>
  <div class="card"><div class="cl">Win Rate</div><div class="cv" id="c-wr">â€”</div><div class="cs" id="c-wr-s">â€”</div></div>
</div>

<div class="two">
  <div class="sec">
    <div class="sh"><span class="st">Open Positions</span><span id="pos-ct" style="color:var(--mu);font-size:9px"></span></div>
    <table class="tbl"><thead><tr><th>Ticker</th><th>Qty</th><th>Entry</th><th>Notional</th><th>Conf</th><th>Regime</th></tr></thead>
    <tbody id="pos-tb"></tbody></table>
  </div>
  <div class="sec">
    <div class="sh">
      <span class="st">P&L â€” 60 Day</span>
      <div class="leg">
        <span style="color:var(--gr)">â” Total</span>
        <span style="color:var(--bl)">â•Œ Unrealized</span>
        <span style="color:var(--yw)">â•Œ Realized</span>
      </div>
    </div>
    <div class="sb"><div class="chart-wrap"><canvas id="pnl-chart"></canvas></div></div>
  </div>
</div>

<div class="sec">
  <div class="sh"><span class="st">Predictions</span>
    <span style="color:var(--mu);font-size:9px">most recent 150 Â· scored = has outcome</span>
  </div>
  <div style="overflow-x:auto">
    <table class="tbl">
      <thead><tr><th>Date</th><th>Ticker</th><th>Signal</th><th>Conf</th><th>Price</th><th>RSI</th><th>VIX</th><th>Regime</th><th>ML Score</th><th>Result</th><th>Return</th></tr></thead>
      <tbody id="pred-tb"></tbody>
    </table>
  </div>
</div>

<div class="sec">
  <div class="sh"><span class="st">ðŸ§  What the Model Learned</span><span id="learn-st" style="color:var(--mu);font-size:9px"></span></div>
  <div class="sb">
    <div style="background:var(--bg);border:1px solid var(--br);border-radius:6px;padding:12px;margin-bottom:16px">
      <div style="display:flex;justify-content:space-between;margin-bottom:4px">
        <span style="color:var(--mu);font-size:9px;text-transform:uppercase;letter-spacing:1px">Self-Learning Progress</span>
        <span id="prog-t" style="color:var(--mu);font-size:9px">0 / 5</span>
      </div>
      <div class="prog"><div class="pf" id="prog-f" style="width:0%"></div></div>
      <div id="prog-d" style="color:var(--mu);font-size:9px;margin-top:5px">Accumulating predictions... rules unlock after 5 scored outcomes.</div>
    </div>
    <div class="learn-cols">
      <div>
        <div class="lsub">Adaptive Rules</div>
        <div id="rules-b"><div class="empty">No rules yet â€” written after 5 scored outcomes</div></div>
        <div class="lsub" style="margin-top:16px">Signal Weights</div>
        <div id="wts-b"><div class="empty">Weights appear after first evening cycle</div></div>
      </div>
      <div>
        <div class="lsub">Feature Importance</div>
        <div id="feat-b"><div class="empty">Importance scores update after scoring</div></div>
        <div class="lsub" style="margin-top:16px">Per-Ticker Calibration</div>
        <div id="calib-b"><div class="empty">Trust multipliers update after scoring</div></div>
      </div>
    </div>
  </div>
</div>

<script>
const D = window.COLAB_DATA;
if (!D) { document.body.innerHTML = '<div style="color:#f85149;padding:40px;text-align:center;font-family:monospace">No data â€” run the morning cycle first.</div>'; }

document.getElementById('ts').textContent = D.generated || 'â€”';

const f$ = v => v ? '$'+parseFloat(v).toFixed(2) : 'â€”';
const fN = v => v ? '$'+parseFloat(v).toLocaleString(undefined,{maximumFractionDigits:0}) : 'â€”';
const fd = ts => { try { return new Date(ts).toLocaleDateString('en-US',{month:'short',day:'numeric'}); } catch { return (ts||'').split('T')[0]; } };
const fP = v => { const n=parseFloat(v); if(isNaN(n)) return 'â€”'; return `<span style="color:${n>=0?'var(--gr)':'var(--rd)'}">${n>=0?'+':''}${(n*100).toFixed(2)}%</span>`; };
const sig = a => { const m={BUY:'bb',SELL:'bs',HOLD:'bh'}; return `<span class="badge ${m[(a||'').toUpperCase()]||'bh'}">${(a||'HOLD').toUpperCase()}</span>`; };
const reg = r => { const l={'0':'BEAR','1':'FLAT','2':'BULL'}; const lbl=l[String(r)]||String(r); const c={BEAR:'bbe',FLAT:'bne',BULL:'bgr'}[lbl]||'bne'; return `<span class="badge ${c}">${lbl}</span>`; };
const cb = v => { const p=Math.round((parseFloat(v)||.5)*100); return `<div class="cb"><div class="cbg"><div class="cbf" style="width:${p}%"></div></div><span style="color:var(--mu);font-size:9px">${p}%</span></div>`; };

// Cards
const sc  = (D.predictions||[]).filter(p=>p.scored==='True'||p.scored==='true');
const ok  = sc.filter(p=>p.was_correct==='True'||p.was_correct==='true');
const cum = (D.pnl_log||[]).reduce((s,r)=>s+parseFloat(r.net_pl||0),0);
const wr  = sc.length ? ok.length/sc.length*100 : null;

const ce = document.getElementById('c-pnl');
if (D.pnl_log && D.pnl_log.length) {
  ce.textContent = (cum>=0?'+$':'-$')+Math.abs(cum).toLocaleString(undefined,{maximumFractionDigits:2});
  ce.className='cv '+(cum>=0?'pos':'neg');
  document.getElementById('c-pnl-s').textContent = `${D.pnl_log.length} day${D.pnl_log.length!==1?'s':''} tracked`;
} else { ce.textContent='$0.00'; document.getElementById('c-pnl-s').textContent='No scored days yet'; }

document.getElementById('c-pred').textContent = (D.predictions||[]).length;
document.getElementById('c-pred-s').textContent = `${sc.length} scored`;

if (wr!==null) {
  const we=document.getElementById('c-wr'); we.textContent=wr.toFixed(1)+'%'; we.className='cv '+(wr>=50?'pos':'neg');
  document.getElementById('c-wr-s').textContent=`${ok.length}/${sc.length} correct`;
} else { document.getElementById('c-wr-s').textContent='Awaiting 5-day horizon'; }

// Positions
function posTable(trades) {
  const pos={};
  for (const t of (trades||[])) {
    if (!pos[t.ticker]) pos[t.ticker]={qty:0,cost:0,conf:'',regime:'',ts:''};
    const q=parseInt(t.qty)||0, p=parseFloat(t.price)||0;
    if (t.action==='BUY'){pos[t.ticker].qty+=q;pos[t.ticker].cost+=q*p;pos[t.ticker].conf=t.confidence;pos[t.ticker].regime=t.regime;pos[t.ticker].ts=t.ts;}
    if (t.action==='SELL') pos[t.ticker].qty-=q;
  }
  const open=Object.values(pos).filter(p=>p.qty>0).map(p=>({...p,avg:p.qty>0?p.cost/p.qty:0}));
  document.getElementById('pos-ct').textContent=open.length+' open';
  document.getElementById('c-pos').textContent=open.length;
  if (!open.length) { document.getElementById('pos-tb').innerHTML='<tr><td colspan="6" class="empty">No open positions</td></tr>'; return; }
  const tn=open.reduce((s,p)=>s+p.qty*p.avg,0);
  document.getElementById('c-pos-s').textContent=fN(tn)+' notional';
  document.getElementById('pos-tb').innerHTML=open.map(p=>`<tr><td><strong>${p.ticker}</strong></td><td>${p.qty}</td><td>${f$(p.avg)}</td><td style="color:var(--mu)">${fN(p.qty*p.avg)}</td><td>${cb(p.conf)}</td><td>${reg(p.regime)}</td></tr>`).join('');
}
posTable(D.trades);

// P&L Chart
(function(){
  const ctx=document.getElementById('pnl-chart').getContext('2d');
  const hist=D.pnl_history||[];
  if (!hist.length) {
    ctx.fillStyle='#8b949e'; ctx.font='10px monospace'; ctx.textAlign='center';
    ctx.fillText('Populates after first morning cycle',ctx.canvas.width/2,100); return;
  }
  const lbl=hist.map(r=>r.date);
  const tot=hist.map(r=>+parseFloat(r.total_pnl||0).toFixed(2));
  const unr=hist.map(r=>+parseFloat(r.unrealized_pnl||0).toFixed(2));
  const rel=hist.map(r=>+parseFloat(r.realized_pnl||0).toFixed(2));
  const last=tot[tot.length-1]||0;
  new Chart(ctx,{type:'line',data:{labels:lbl,datasets:[
    {label:'Total',data:tot,borderColor:last>=0?'#3fb950':'#f85149',backgroundColor:last>=0?'rgba(63,185,80,.07)':'rgba(248,81,73,.06)',borderWidth:2.5,fill:true,tension:.35,pointRadius:3},
    {label:'Unrealized',data:unr,borderColor:'#58a6ff',backgroundColor:'transparent',borderWidth:1.5,fill:false,tension:.35,pointRadius:2,borderDash:[4,3]},
    {label:'Realized',data:rel,borderColor:'#e3b341',backgroundColor:'transparent',borderWidth:1.5,fill:false,tension:.35,pointRadius:2,borderDash:[2,4]},
  ]},options:{responsive:true,maintainAspectRatio:false,interaction:{mode:'index',intersect:false},
    plugins:{legend:{display:false},tooltip:{backgroundColor:'#161b22',borderColor:'#30363d',borderWidth:1,titleColor:'#8b949e',bodyColor:'#e6edf3',callbacks:{label:c=>` ${c.dataset.label}: ${c.parsed.y>=0?'+':''}$${c.parsed.y.toFixed(2)}`}}},
    scales:{x:{grid:{color:'#21262d'},ticks:{color:'#8b949e',font:{family:'monospace',size:8},maxTicksLimit:10}},
            y:{grid:{color:'#21262d'},ticks:{color:'#8b949e',font:{family:'monospace',size:8},callback:v=>(v>=0?'+$':'-$')+Math.abs(v).toLocaleString()}}}}});
})();

// Predictions table
(function(){
  const preds=[...(D.predictions||[])].reverse().slice(0,150);
  if (!preds.length){document.getElementById('pred-tb').innerHTML='<tr><td colspan="11" class="empty">No predictions yet</td></tr>';return;}
  document.getElementById('pred-tb').innerHTML=preds.map(p=>{
    const isc=p.scored==='True'||p.scored==='true', iok=p.was_correct==='True'||p.was_correct==='true';
    const res=isc?(iok?'<span style="color:var(--gr)">âœ“</span>':'<span style="color:var(--rd)">âœ—</span>'):'<span style="color:var(--mu)">â€¦</span>';
    const r=parseFloat(p.rsi||0), v=parseFloat(p.vix||0);
    return `<tr><td style="color:var(--mu)">${fd(p.pred_ts)}</td><td><strong>${p.ticker}</strong></td><td>${sig(p.action)}</td><td>${cb(p.confidence)}</td><td style="color:var(--mu)">${f$(p.price_at_pred)}</td><td style="color:${r>70?'var(--rd)':r<30?'var(--gr)':'var(--mu)'}">${r.toFixed(1)}</td><td style="color:${v>25?'var(--rd)':'var(--mu)'}">${v.toFixed(1)}</td><td>${reg(p.regime)}</td><td>${cb(p.p_ensemble)}</td><td>${res}</td><td>${p.actual_return?fP(p.actual_return):'â€”'}</td></tr>`;
  }).join('');
})();

// Self-learning
(function(){
  const need=5, got=sc.length, pct=Math.min(100,got/need*100);
  document.getElementById('prog-f').style.width=pct+'%';
  document.getElementById('prog-t').textContent=`${got} / ${need} scored outcomes`;
  if (got>=need) {
    document.getElementById('prog-d').textContent='Self-learning active â€” rules update each evening cycle.';
    document.getElementById('learn-st').textContent='ðŸŸ¢ Active';
  } else {
    document.getElementById('prog-d').textContent=`${need-got} more outcome${need-got!==1?'s':''} needed. Scored 5 trading days after signal.`;
  }

  // Rules
  const rules=D.rules;
  const rb=document.getElementById('rules-b');
  if (rules) {
    const ent=Object.entries(rules).filter(([k])=>!k.startsWith('_'));
    if (ent.length) {
      document.getElementById('learn-st').textContent=`${ent.length} rules active`;
      rb.innerHTML=ent.map(([n,r])=>{
        const adj=r.adjustment; const as=adj!=null?`<span style="color:${adj>0?'var(--gr)':'var(--rd)'}">${adj>0?'+':''}${(adj*100).toFixed(1)}% conf</span>`:'';
        return `<div class="rule"><div class="rn">${n}</div><div class="rc">${r.condition||'â€”'} â†’ ${as}</div><div class="rs"><div>Applied <span>${r.applied_count||0}Ã—</span></div>${r.acc_before!=null?`<div>Before <span>${(r.acc_before*100).toFixed(1)}%</span></div>`:''}</div></div>`;
      }).join('');
    }
  }

  // Weights
  const wb=document.getElementById('wts-b');
  if (D.weights && Object.keys(D.weights).length) {
    const s=Object.entries(D.weights).sort(([,a],[,b])=>b-a); const mx=Math.max(...s.map(([,v])=>Math.abs(parseFloat(v)||0)));
    wb.innerHTML=s.map(([n,v])=>{const nv=parseFloat(v)||0,w=mx>0?(Math.abs(nv)/mx*100).toFixed(0):0,c=nv>=0?'linear-gradient(90deg,#3fb950,#3fb950)':'linear-gradient(90deg,#f85149,#f85149)';
      return `<div class="br-row"><div class="br-lbl">${n}</div><div class="br-bg"><div class="br-fill" style="width:${w}%;background:${c}">${w>25?nv.toFixed(3):''}</div></div><div class="br-val" style="color:${nv>=0?'var(--gr)':'var(--rd)'}">${nv.toFixed(3)}</div></div>`;}).join('');
  }

  // Features
  const fb=document.getElementById('feat-b');
  if (D.features && Object.keys(D.features).length) {
    const s=Object.entries(D.features).sort(([,a],[,b])=>b-a).slice(0,15); const mx=s[0][1]||1;
    fb.innerHTML=s.map(([n,v])=>{const w=(v/mx*100).toFixed(0);
      return `<div class="br-row"><div class="br-lbl">${n}</div><div class="br-bg"><div class="br-fill" style="width:${w}%;background:linear-gradient(90deg,var(--bl),var(--pu))">${w>20?v.toFixed(3):''}</div></div><div class="br-val" style="color:var(--mu)">${v.toFixed(3)}</div></div>`;}).join('');
  }

  // Calibration
  const cb2=document.getElementById('calib-b');
  if (D.calibration && Object.keys(D.calibration).length) {
    const s=Object.entries(D.calibration).sort(([,a],[,b])=>b-a);
    cb2.innerHTML=`<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(90px,1fr));gap:6px">${s.map(([tk,m])=>{
      const mv=parseFloat(m)||1, cl=mv>1.1?'var(--gr)':mv<0.9?'var(--rd)':'var(--mu)', lb=mv>1.1?'HIGH':mv<0.9?'LOW':'OK';
      return `<div style="background:var(--bg);border:1px solid var(--br);border-radius:6px;padding:8px;text-align:center"><div style="font-weight:700;font-size:11px">${tk}</div><div style="font-size:17px;font-weight:700;color:${cl}">${mv.toFixed(2)}Ã—</div><div style="font-size:8px;color:${cl};text-transform:uppercase">${lb}</div></div>`;}).join('')}</div>`;
  }
})();
</script>
</body>
</html>"""

_b64 = base64.b64encode(_html.encode('utf-8')).decode()
display(HTML(f'<iframe src="data:text/html;base64,{_b64}" width="100%" height="1050px" style="border:none;border-radius:8px;box-shadow:0 0 24px rgba(0,0,0,.6)"></iframe>'))
print(f"  Dashboard rendered | {len(_d['trades'])} trades | {len(_d['predictions'])} predictions | data from {_d['generated']}")